## CONFI

In [1]:
import os
import re
import glob
import pandas as pd
import numpy as np

from io import StringIO
from datetime import date
import datetime

# ══════════════════════════════════════════════════════════════════════════
# ROOT CONFIG
# ══════════════════════════════════════════════════════════════════════════

BASE = (
    r"C:\Users\Shambhavi Dixit"
    r"\OneDrive - Incircle Ecoproducts Pvt Ltd"
    r"\Thrivebrands - Client Database"
    r"\Emami"
)

YEAR = 2026

MONTHS = {
    "jan": ["jan", "january"],
    "feb": ["feb", "february"],
    "mar": ["mar", "march"],
    "apr": ["apr", "april"],
    "may": ["may"],
    "jun": ["jun", "june"],
    "jul": ["jul", "july"],
    "aug": ["aug", "august"],
    "sep": ["sep", "sept", "september"],
    "oct": ["oct", "october"],
    "nov": ["nov", "november"],
    "dec": ["dec", "december"],
}

# ══════════════════════════════════════════════════════════════════════════
# AMAZON PATHS
# ══════════════════════════════════════════════════════════════════════════

AMZ_INV_ROOT = os.path.join(BASE, "input", "amazon", "uae", "inventory", str(YEAR))
AMZ_REV_ROOT = os.path.join(BASE, "input", "amazon", "uae", "revenue",   str(YEAR))
AMZ_ADS_ROOT = os.path.join(BASE, "input", "amazon", "uae", "marketing", str(YEAR))

# ══════════════════════════════════════════════════════════════════════════
# NOON PATHS
# ══════════════════════════════════════════════════════════════════════════

NOON_REV_ROOT = os.path.join(BASE, "input", "noon", "uae", "revenue",   str(YEAR))
NOON_ADS_ROOT = os.path.join(BASE, "input", "noon", "uae", "marketing", str(YEAR))

# Inventory: .../noon/uae/inventory/2026/<month>/<date_folder>/<file.csv>
# Inventory column: Quantity
NOON_INV_ROOT = os.path.join(BASE, "input", "noon", "uae", "inventory", str(YEAR))

# ══════════════════════════════════════════════════════════════════════════
# NOON MINUTES PATHS
# ══════════════════════════════════════════════════════════════════════════

NMS_REV_ROOT = os.path.join(BASE, "input", "noon_minutes", "uae", "revenue",   str(YEAR))
NMS_ADS_ROOT = os.path.join(BASE, "input", "noon_minutes", "uae", "marketing", str(YEAR))

# Inventory: .../noon_minutes/uae/inventory/2026/<month>/<date_folder>/<file.csv>
# Inventory column: Stock
NMS_INV_ROOT = os.path.join(BASE, "input", "noon_minutes", "uae", "inventory", str(YEAR))

# ══════════════════════════════════════════════════════════════════════════
# HELPER FILE
# ══════════════════════════════════════════════════════════════════════════

HELPER_FILE = os.path.join(BASE, "input", "helper_file", "Emami (UAE)- Item Master Sheet.xlsx")

# ══════════════════════════════════════════════════════════════════════════
# OUTPUT FOLDER
# ══════════════════════════════════════════════════════════════════════════

# OUTPUT_FOLDER = os.path.join(BASE, "output")



OUTPUT_FOLDER = (
    r"C:\Users\Shambhavi Dixit"
    r"\OneDrive - Incircle Ecoproducts Pvt Ltd"
    r"\Thrivebrands - Client Database"
    r"\client_mamangement_team"
    r"\EMAMI"
)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

OUT_INV    = os.path.join(OUTPUT_FOLDER, f"inventory_output_{YEAR}.xlsx")
OUT_REV    = os.path.join(OUTPUT_FOLDER, f"revenue_output_{YEAR}.xlsx")
OUT_MARK   = os.path.join(OUTPUT_FOLDER, f"marketing_output_{YEAR}.xlsx")
OUT_BUYBOX = os.path.join(OUTPUT_FOLDER, f"buybox_output_{YEAR}.xlsx")

BUYBOX_DAILY_ROOT = os.path.join(BASE, "output", "emami_amazon_buybox", str(YEAR), "emami", "ae")

# ══════════════════════════════════════════════════════════════════════════
# AZURE CONFIG
# ══════════════════════════════════════════════════════════════════════════

AZ_CONN = (
    "DefaultEndpointsProtocol=https;"
    "AccountName=thriveclientdatabase;"
    "AccountKey=t3YQjkSXWXafdSIzp48QxNgmuW8BSh+6FzHaEy6VhGbgINYgo7Sica0BZw9sAKEPbwBcjqqhK5p6+AStuxDzSA==;"
    "EndpointSuffix=core.windows.net"
)
AZ_CONTAINER = "thrive-client-emami"

# ══════════════════════════════════════════════════════════════════════════
# GLOBAL HELPERS
# ══════════════════════════════════════════════════════════════════════════

def clean_asin(series):
    return (series.astype(str).str.upper().str.strip()
            .str.replace(".0", "", regex=False))

def clean_date(series):
    # dayfirst=True: treats DD/MM/YYYY correctly (prevents 01/06/2026 → year 2001 or 2008)
    return pd.to_datetime(series, dayfirst=True, errors="coerce").dt.normalize()

HELPER_COLS = [
    "ASIN", "Brand", "Product Name", "Product Category",
    "Product Sub Category", "Pack Type", "Pack Size",
    "SC Ideal Price", "VC Ideal Price"
]

print("\n✅ CONFIG READY")
print(f"📅 YEAR : {YEAR}")
print(f"📁 OUTPUT : {OUTPUT_FOLDER}")
print(f"📦 MONTHS : {MONTHS}")
print("\nInventory Roots:")
print(f"  AMZ : {AMZ_INV_ROOT}")
print(f"  NC  : {NOON_INV_ROOT}")
print(f"  NMs : {NMS_INV_ROOT}")



✅ CONFIG READY
📅 YEAR : 2026
📁 OUTPUT : C:\Users\Shambhavi Dixit\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\client_mamangement_team\EMAMI
📦 MONTHS : {'jan': ['jan', 'january'], 'feb': ['feb', 'february'], 'mar': ['mar', 'march'], 'apr': ['apr', 'april'], 'may': ['may'], 'jun': ['jun', 'june'], 'jul': ['jul', 'july'], 'aug': ['aug', 'august'], 'sep': ['sep', 'sept', 'september'], 'oct': ['oct', 'october'], 'nov': ['nov', 'november'], 'dec': ['dec', 'december']}

Inventory Roots:
  AMZ : C:\Users\Shambhavi Dixit\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\Emami\input\amazon\uae\inventory\2026
  NC  : C:\Users\Shambhavi Dixit\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\Emami\input\noon\uae\inventory\2026
  NMs : C:\Users\Shambhavi Dixit\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\Emami\input\noon_minutes\uae\inventory\2026


In [2]:
# Utility: safe diagnostics for undefined variables and column checks
import pandas as pd

def safe_col_info(df, col):
    if not isinstance(df, pd.DataFrame):
        print(f"Provided object is not a DataFrame: {type(df).__name__}")
        return
    if col in df.columns:
        print(f"Column '{col}' in DataFrame — sample values:")
        print(df[col].head(5).to_string(index=False))
    else:
        print(f"Column '{col}' NOT in DataFrame. Available columns (first 30):\n  {df.columns.tolist()[:30]}")


def check_globals(vars_list):
    missing = [v for v in vars_list if v not in globals()]
    if missing:
        print('Missing variables:', missing)
    else:
        print('All expected variables present')
    for v in vars_list:
        if v in globals():
            obj = globals()[v]
            t = type(obj).__name__
            if isinstance(obj, pd.DataFrame):
                print(f"- {v}: DataFrame shape={obj.shape}")
            else:
                print(f"- {v}: {t}")

# Quick check list (adjust as needed)
_expected = [
    'marketing_output','OUT_MARK',
    'nms_inv_df','nms_clean','nms_raw','nms_rev_df',
    'noon_inv_df','nc_clean','amz_inv_df','helper_df'
]
print('\n== Quick Globals Check ==')
check_globals(_expected)

# Check ASIN presence if DataFrames available
for df_name in ['nms_inv_df','nms_rev_df','nms_clean','noon_inv_df','amz_inv_df']:
    if df_name in globals() and isinstance(globals()[df_name], pd.DataFrame):
        df = globals()[df_name]
        if 'ASIN' in df.columns:
            print(f"{df_name}: ASIN present — sample:")
            try:
                print(df['ASIN'].head(3).to_string(index=False))
            except Exception:
                print('  (Could not print sample)')
        else:
            print(f"{df_name}: ASIN NOT present. Columns: {df.columns.tolist()[:20]}")



== Quick Globals Check ==
Missing variables: ['marketing_output', 'nms_inv_df', 'nms_clean', 'nms_raw', 'nms_rev_df', 'noon_inv_df', 'nc_clean', 'amz_inv_df', 'helper_df']
- OUT_MARK: str


## AMAZON INVENTORY INPUT

In [3]:
# ══════════════════════════════════════════════════════════════════════════
# AMAZON INVENTORY MASTER BUILD
# INVENTORY = PRIORITY BASE
# ══════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore", message="Workbook contains no default style")

amz_inventory_parts = []


# for process_month in MONTHS:


#     input_folder = os.path.join(
#         AMZ_INV_ROOT,
#         process_month
#     )


#     output_folder = os.path.join(
#         AMZ_INV_ROOT,
#         f"{process_month}_merged"
#     )


#     os.makedirs(
#         output_folder,
#         exist_ok=True
#     )


#     # Skip Missing Folder
#     if not os.path.exists(input_folder):


#         print(f"⚠️ Missing Inventory Folder: {process_month}")
#         continue

# ══════════════════════════════════════════════════════════════════════
# LOOP MONTHS (supports jan/january, jul/july, jun/june, etc.)
# ══════════════════════════════════════════════════════════════════════

for month_key, month_aliases in MONTHS.items():

    input_folder = None
    folder_name = None

    # Find whichever folder exists
    for alias in month_aliases:

        candidate = os.path.join(AMZ_INV_ROOT, alias)

        if os.path.exists(candidate):
            input_folder = candidate
            folder_name = alias
            break

    if input_folder is None:
        print(f"⚠️ Missing Inventory Folder: {month_key}")
        continue

    print(f"\n📂 Processing Inventory Folder: {folder_name}")

    output_folder = os.path.join(
        AMZ_INV_ROOT,
        f"{folder_name}_merged"
    )

    os.makedirs(output_folder, exist_ok=True)

    


    month_data = []


    # ══════════════════════════════════════════════════════════════════════
    # Read Daily Inventory Files
    # ══════════════════════════════════════════════════════════════════════


    for file in sorted(os.listdir(input_folder)):


        file_lower = file.lower()


        # Only Excel or CSV
        if not (file_lower.endswith(".xlsx") or file_lower.endswith(".csv")):
            continue


        # Only Inventory Files
        if "amazon_emami_inventory" not in file_lower:
            continue


        path = os.path.join(
            input_folder,
            file
        )


        print(f"📖 Reading: {file}")


        try:


            # ── Read Proper Data ───────────────────────────────────────
            if file_lower.endswith(".csv"):


                # Amazon VC inventory CSVs always have exactly 1 metadata line
                # before the real header (e.g. "Program=[Retail],Distributor View=...").
                # on_bad_lines="skip" drops the ACTUAL header row (15 cols) because
                # it looks jagged vs the 10-col metadata row → use skiprows=1 directly.
                #
                # Fallback: if "ASIN" is NOT in the first row after skiprows=1,
                # scan first 5 rows for the ASIN header (handles rare format variants).
                try:
                    df = pd.read_csv(
                        path,
                        skiprows=1,
                        dtype=str,
                        encoding="utf-8-sig",
                        encoding_errors="ignore",
                        on_bad_lines="skip"
                    )
                    df.columns = df.columns.astype(str).str.strip()
                    if "ASIN" not in df.columns:
                        raise ValueError("ASIN not in columns after skiprows=1")
                except Exception:
                    # Fallback: scan up to 10 rows for ASIN header
                    raw = pd.read_csv(
                        path, header=None, dtype=str,
                        encoding="utf-8-sig", encoding_errors="ignore",
                        engine="python", sep=",", on_bad_lines="skip"
                    )
                    found = False
                    for _ri in range(min(10, len(raw))):
                        if "ASIN" in raw.iloc[_ri].astype(str).values:
                            df = pd.read_csv(
                                path, skiprows=_ri, dtype=str,
                                encoding="utf-8-sig", encoding_errors="ignore",
                                on_bad_lines="skip"
                            )
                            df.columns = df.columns.astype(str).str.strip()
                            found = True
                            break
                    if not found:
                        raise ValueError("ASIN header row not found in CSV")


            else:


                # Find actual header row in Excel
                raw = pd.read_excel(
                    path,
                    header=None
                )


                header_matches = raw.eq("ASIN").any(axis=1)
                if not header_matches.any():
                    raise ValueError("ASIN header row not found")


                header_row = header_matches[header_matches].index[0]


                df = pd.read_excel(
                    path,
                    skiprows=header_row,
                    dtype=str
                )


            # ── Clean Columns ──────────────────────────────────────────
            df.columns = (
                df.columns
                .astype(str)
                .str.strip()
            )


            # ── Extract Date From Filename ────────────────────────────
            # match = re.search(
            #     r'(\d{1,2}_\w+_\d{4})',
            #     file
            # )


            # if match:


            #     file_date = pd.to_datetime(
            #         match.group(1),
            #         format="%d_%b_%Y",
            #         errors="coerce"
            #     )


            #     df["Date"] = file_date


            # else:


            #     df["Date"] = pd.NaT
            match = re.search(
                r'(\d{1,2})_(\w+)_(\d{4})',
                file.lower()
            )

            if match:

                day = match.group(1)
                month = match.group(2)
                year = match.group(3)

                # Try abbreviated month first
                file_date = pd.to_datetime(
                    f"{day}_{month}_{year}",
                    format="%d_%b_%Y",
                    errors="coerce"
                )

                # If failed, try full month name
                if pd.isna(file_date):
                    file_date = pd.to_datetime(
                        f"{day}_{month}_{year}",
                        format="%d_%B_%Y",
                        errors="coerce"
                    )

                df["Date"] = file_date

            else:

                df["Date"] = pd.NaT

            # ── Clean ASIN ────────────────────────────────────────────
            if "ASIN" in df.columns:


                df["ASIN"] = clean_asin(
                    df["ASIN"]
                )


            # ── Sales Channel ─────────────────────────────────────────
            df["Sales Channel"] = "Vendor Central"


            # ── Source Tracking ──────────────────────────────────────
            df["source_file"] = file


            month_data.append(df)


        except Exception as e:


            print(f"❌ Skipped: {file}")
            print("Reason:", e)


    # ══════════════════════════════════════════════════════════════════════
    # Merge Month Inventory
    # ══════════════════════════════════════════════════════════════════════


    if month_data:


        merged = pd.concat(
            month_data,
            ignore_index=True
        )


        # ── Clean Dates ───────────────────────────────────────────────
        merged["Date"] = clean_date(
            merged["Date"]
        )


        # ── Remove Blank ASIN Rows ───────────────────────────────────
        if "ASIN" in merged.columns:
            merged = merged[
                merged["ASIN"].notna()
            ]


        # ── Sort ─────────────────────────────────────────────────────
        sort_cols = ["Date"]
        if "ASIN" in merged.columns:
            sort_cols.append("ASIN")
        merged = merged.sort_values(sort_cols)


        # ── Remove Duplicate Rows ────────────────────────────────────
        merged = merged.drop_duplicates()


        # ── Save Monthly Master ──────────────────────────────────────
        out = os.path.join(
            output_folder,
            f"merged_output_inventory_{folder_name}_{YEAR}.xlsx"
        )


        merged.to_excel(
            out,
            index=False
        )


        print(f"\n✅ Saved Monthly Inventory:")
        print(out)


        print(f"📊 Shape: {merged.shape}")


        amz_inventory_parts.append(merged)


    else:


        print(f"⚠️ No Inventory Files Found: {folder_name}")


# ══════════════════════════════════════════════════════════════════════════
# FINAL AMAZON INVENTORY MASTER
# THIS BECOMES INVENTORY PRIORITY DATASET
# ══════════════════════════════════════════════════════════════════════════


if amz_inventory_parts:


    amz_inv_df = pd.concat(
        amz_inventory_parts,
        ignore_index=True
    )


    # Final Cleaning
    if "ASIN" in amz_inv_df.columns:
        amz_inv_df["ASIN"] = clean_asin(amz_inv_df["ASIN"])


    amz_inv_df["Date"] = clean_date(
        amz_inv_df["Date"]
    )


    # Final Sort
    sort_cols = ["Date"]
    if "ASIN" in amz_inv_df.columns:
        sort_cols.append("ASIN")
    amz_inv_df = amz_inv_df.sort_values(sort_cols)


    print("\n✅ FINAL AMAZON INVENTORY READY")


    print("Shape:", amz_inv_df.shape)


    print("\n📅 Available Months:")


    print(
        sorted(
            amz_inv_df["Date"]
            .dt.strftime("%Y-%m")
            .dropna()
            .unique()
        )
    )


    display(
        amz_inv_df.head(10)
    )


else:


    print("❌ No Amazon Inventory Data Found")

⚠️ Missing Inventory Folder: jan

📂 Processing Inventory Folder: feb
📖 Reading: amazon_emami_inventory_01_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_02_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_03_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_04_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_05_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_06_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_07_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_08_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_09_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_10_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_11_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_12_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_13_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_14_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_15_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_16_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_17_feb_2026.xlsx
📖 Reading: amazon_emami_inventory_18_feb_2026.xlsx
📖 Reading: am

,ASIN,Product Title,Brand,Vendor Confirmation %,Net Received,Net Received Units,Open Purchase Order Quantity,Receive Fill %,Overall Vendor Lead Time (days),Aged 90+ Days Sellable Inventory,Aged 90+ Days Sellable Units,Sellable On Hand Inventory,Sellable On Hand Units,Unsellable On Hand Inventory,Unsellable On Hand Units,Date,Sales Channel,source_file
0,B00F93HFME,Navratna Herbal Cool Oil | With 9 Active Herba...,Navratna,0,0,0,NaN,0,0,0,0,773.1,45,17.18,1,2026-02-01,Vendor Central,amazon_emami_inventory_01_feb_2026.xlsx
1,B071JYP7NQ,Emami 7 Oils in One Damage Control Hair Oil | ...,Emami,0,0,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-01,Vendor Central,amazon_emami_inventory_01_feb_2026.xlsx
2,B07MVWQJQ5,Emami Smart And Handsome Men Radiant Glow Crea...,UNKNOWN,0,0,0,NaN,0,0,0,0,73.32,6,0,0,2026-02-01,Vendor Central,amazon_emami_inventory_01_feb_2026.xlsx
3,B07MW28F51,Emami Smart And Handsome Men Face Wash-100g | ...,UNKNOWN,0,0,0,12,0,0,0,0,114.3,6,0,0,2026-02-01,Vendor Central,amazon_emami_inventory_01_feb_2026.xlsx
4,B07P82TMDJ,Creme 21 Body Lotion for Dry to Ultra Dry Skin...,Creme 21,0,0,0,12,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-01,Vendor Central,amazon_emami_inventory_01_feb_2026.xlsx
5,B07PBC4HJC,Creme 21 Body Lotion for Normal Skin - 250ml |...,Creme 21,0,0,0,12,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-01,Vendor Central,amazon_emami_inventory_01_feb_2026.xlsx
6,B087JL5FCR,Navratna Herbal Cool Oil | With 9 Active Herba...,Navratna,0,0,0,NaN,0,0,0,0,117,5,0,0,2026-02-01,Vendor Central,amazon_emami_inventory_01_feb_2026.xlsx
7,B08FC7TLJL,Emami 7 Oils in One Shea Butter Hair Oil | Hai...,UNKNOWN,0,0,0,12,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-01,Vendor Central,amazon_emami_inventory_01_feb_2026.xlsx
8,B08R8F3YZ2,Creme 21 Aqua Soft Cream (250 Ml),Crème 21,0,0,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-01,Vendor Central,amazon_emami_inventory_01_feb_2026.xlsx
9,B096F56YHT,Creme 21 Soft Care Moisturising Cream Soap – 1...,Crème 21,0,0,0,12,0,0,NaN,NaN,NaN,NaN,NaN,NaN,2026-02-01,Vendor Central,amazon_emami_inventory_01_feb_2026.xlsx


In [4]:
# amz_inv_df.to_csv("input.csv", index=False)

# NOON_MINUTES_INVENTORY

In [5]:
# ══════════════════════════════════════════════════════════════════════════
# NOON MINUTES INVENTORY
# MONTH-WISE CONSOLIDATION
# ══════════════════════════════════════════════════════════════════════════

import os
import re
import pandas as pd

YEAR = 2026

MONTHS = [
    "may",
    "jun",
    "july",
    "aug"
]

NMS_INV_ROOT = (
    r"C:\Users\Shambhavi Dixit"
    r"\OneDrive - Incircle Ecoproducts Pvt Ltd"
    r"\Thrivebrands - Client Database"
    r"\Emami\input\noon_minutes\uae\inventory\2026"
)

nms_inventory_parts = []

for process_month in MONTHS:

    month_folder = os.path.join(
        NMS_INV_ROOT,
        process_month
    )

    if not os.path.exists(month_folder):

        print(f"⚠️ Missing Folder : {month_folder}")
        continue

    all_data = []

    # ==========================================================
    # READ ALL DATE FOLDERS
    # ==========================================================

    for day_folder in sorted(os.listdir(month_folder)):

        day_path = os.path.join(
            month_folder,
            day_folder
        )

        if not os.path.isdir(day_path):
            continue

        print(f"📂 {process_month.upper()} -> {day_folder}")

        for file in sorted(os.listdir(day_path)):

            if not file.lower().endswith(
                (".csv", ".xlsx")
            ):
                continue

            file_path = os.path.join(
                day_path,
                file
            )

            print(f"📖 Reading : {file}")

            try:

                if file.lower().endswith(".csv"):

                    df = pd.read_csv(
                        file_path,
                        dtype=str,
                        encoding="utf-8-sig"
                    )

                else:

                    df = pd.read_excel(
                        file_path,
                        dtype=str
                    )

                df.columns = (
                    df.columns
                    .astype(str)
                    .str.strip()
                )

                # ==================================================
                # EXTRACT DATE FROM FILE NAME
                # Example:
                # noon_minuts_stock_12_june_2026.csv
                # ==================================================

                match = re.search(
                    r'(\d{1,2})_([A-Za-z]+)_(\d{4})',
                    file
                )

                if match:

                    file_date = pd.to_datetime(
                        f"{match.group(1)}-{match.group(2)}-{match.group(3)}",
                        errors="coerce"
                    )

                else:

                    file_date = pd.NaT

                df["Date"] = file_date
                df["source_file"] = file
                df["Sales Channel"] = "Noon Minutes"

                all_data.append(df)

            except Exception as e:

                print(f"❌ Failed : {file}")
                print(e)

    # ==========================================================
    # SAVE MONTH FILE
    # ==========================================================

    if all_data:

        merged_df = pd.concat(
            all_data,
            ignore_index=True
        )

        merged_df["Date"] = pd.to_datetime(
            merged_df["Date"],
            errors="coerce"
        )

        merged_df = merged_df.drop_duplicates()

        sort_cols = ["Date"]

        if "NIN" in merged_df.columns:
            sort_cols.append("NIN")

        merged_df = merged_df.sort_values(
            sort_cols
        )

        output_file = os.path.join(
            month_folder,
            f"merged_nms_{process_month}_{YEAR}.xlsx"
        )

        merged_df.to_excel(
            output_file,
            index=False
        )

        print(f"\n✅ Saved : {output_file}")
        print(f"Rows : {len(merged_df):,}")

        nms_inventory_parts.append(
            merged_df
        )

# ==============================================================
# FINAL MASTER
# ==============================================================

if nms_inventory_parts:

    nms_inv_df = pd.concat(
        nms_inventory_parts,
        ignore_index=True
    )

    nms_inv_df["Date"] = pd.to_datetime(
        nms_inv_df["Date"],
        errors="coerce"
    )

    nms_inv_df = nms_inv_df.sort_values(
        "Date"
    )

    print("\n✅ FINAL NMS INVENTORY READY")
    print("Shape :", nms_inv_df.shape)

    display(nms_inv_df.head())

else:

    print("❌ No Noon Minutes Inventory Found")

📂 MAY -> 18
📖 Reading : noon_minuts_stock_18_may_2026.csv
📂 MAY -> 19
📖 Reading : noon_minuts_stock_19_may_2026.csv
📂 MAY -> 20
📖 Reading : noon_minuts_stock_20_may_2026.csv
📂 MAY -> 21
📖 Reading : noon_minuts_stock_21_may_2026.csv
📂 MAY -> 22
📖 Reading : noon_minuts_stock_22_may_2026.csv
📂 MAY -> 25
📖 Reading : noon_minuts_stock_25_may_2026.csv
📂 MAY -> 26
📖 Reading : noon_minuts_stock_26_may_2026.csv
📂 MAY -> 27
📖 Reading : noon_minuts_stock_27_may_2026.csv
📂 MAY -> 28
📖 Reading : noon_minuts_stock_28_may_2026.csv
📂 MAY -> 29
📖 Reading : noon_minuts_stock_29_may_2026.csv
📂 MAY -> 30
📖 Reading : noon_minuts_stock_30_may_2026.csv
📂 MAY -> 31
📖 Reading : noon_minuts_stock_31_may_2026.csv

✅ Saved : C:\Users\Shambhavi Dixit\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\Emami\input\noon_minutes\uae\inventory\2026\may\merged_nms_may_2026.xlsx
Rows : 15,411
📂 JUN -> 1
📖 Reading : noon_minuts_stock_1_june_2026.csv
📂 JUN -> 10
📖 Reading : noon_minuts_stock_10_june_202

,Warehouse,Warehouse Code,Category,ZSKU,Barcode,Brand,Product Title,Product Type,Product Sub-Type,Refresh Time,Stock,Date,source_file,Sales Channel,Row Labels,Sum of Stock
0,Noon grocery technopark - Jebel Ali,W00068387A,Personal Care,Z3472F77ACFFA103E8ED6Z-1,6294007502627,creme21,Creme 21 Pure Glycerin Oil - 200ml | 99.5% Pur...,personal_care,bath_body_bath_set,"18 May 2026, 03:35:00",301,2026-05-18,noon_minuts_stock_18_may_2026.csv,Noon Minutes,NaN,NaN
1074,Dubai South WH - Dubai South Logistics Distric...,W00833473AE,Personal Care,Z5540F7088905D2C64FFBZ-1,5060068529528,emami,Emami Smart And Handsome Men Refreshing Face W...,skin_care,treatment_serums_acne_blemish_treatments,"18 May 2026, 03:35:00",11,2026-05-18,noon_minuts_stock_18_may_2026.csv,Noon Minutes,NaN,NaN
1073,Al Garhoud,W00809698AE,Personal Care,Z38685AD13D6A297BDF7DZ-1,6294007503358,creme_21,Creme 21 XTRA Bright Face Serum – 30ml | With ...,skin_care,treatment_serums_acne_blemish_treatments,"18 May 2026, 03:35:00",1,2026-05-18,noon_minuts_stock_18_may_2026.csv,Noon Minutes,NaN,NaN
1072,PP4W+Q6G - Al Ma`Mourah - KHIA 8 - (Khezad) Ab...,W00121548A,Personal Care,Z3472F77ACFFA103E8ED6Z-1,6294007502627,creme21,Creme 21 Pure Glycerin Oil - 200ml | 99.5% Pur...,personal_care,bath_body_bath_set,"18 May 2026, 03:35:00",140,2026-05-18,noon_minuts_stock_18_may_2026.csv,Noon Minutes,NaN,NaN
1071,Dubai South WH - Dubai South Logistics Distric...,W00833473AE,Personal Care,Z3472F77ACFFA103E8ED6Z-1,6294007502627,creme21,Creme 21 Pure Glycerin Oil - 200ml | 99.5% Pur...,personal_care,bath_body_bath_set,"18 May 2026, 03:35:00",124,2026-05-18,noon_minuts_stock_18_may_2026.csv,Noon Minutes,NaN,NaN


In [6]:
# nms_inv_df.to_csv("nms_inventory.csv", index=False)

# NOON_INVENTORY_PROCESS

In [7]:
import os
import re
import pandas as pd

YEAR = 2026
MONTHS = ["may", "jun","july","aug"]

NOON_INV_ROOT = (
    r"C:\Users\Shambhavi Dixit"
    r"\OneDrive - Incircle Ecoproducts Pvt Ltd"
    r"\Thrivebrands - Client Database"
    r"\Emami\input\noon\uae\inventory\2026"
)

for process_month in MONTHS:

    month_folder = os.path.join(
        NOON_INV_ROOT,
        process_month
    )

    if not os.path.exists(month_folder):
        continue

    all_data = []

    # Loop Date Folders
    for date_folder in sorted(os.listdir(month_folder)):

        date_path = os.path.join(
            month_folder,
            date_folder
        )

        if not os.path.isdir(date_path):
            continue

        print(f"Reading Folder : {date_folder}")

        # Loop Files
        for file in os.listdir(date_path):

            if not file.lower().endswith(
                (".csv", ".xlsx")
            ):
                continue

            file_path = os.path.join(
                date_path,
                file
            )

            try:

                if file.lower().endswith(".csv"):
                    df = pd.read_csv(
                        file_path,
                        dtype=str
                    )
                else:
                    df = pd.read_excel(
                        file_path,
                        dtype=str
                    )

                # Extract Date
                match = re.search(
                    r'(\d{1,2})_([A-Za-z]+)_(\d{4})',
                    file
                )

                if match:

                    file_date = pd.to_datetime(
                        f"{match.group(1)}-{match.group(2)}-{match.group(3)}",
                        errors="coerce"
                    )

                    df["Date"] = file_date

                df["source_file"] = file

                all_data.append(df)

            except Exception as e:

                print(f"Failed : {file}")
                print(e)

    # Save Monthly File
    if all_data:

        merged_df = pd.concat(
            all_data,
            ignore_index=True
        )

        output_file = os.path.join(
            month_folder,
            f"merged_noon_{process_month}_{YEAR}.xlsx"
        )

        merged_df.to_excel(
            output_file,
            index=False
        )

        print(
            f"✅ Saved : {output_file}"
        )
        print(
            f"Rows : {len(merged_df):,}"
        )

Reading Folder : 18
Reading Folder : 19
Reading Folder : 20
Reading Folder : 21
Reading Folder : 22
Reading Folder : 25
Reading Folder : 26
Reading Folder : 27
Reading Folder : 28
Reading Folder : 29
Reading Folder : 30
Reading Folder : 31
✅ Saved : C:\Users\Shambhavi Dixit\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\Emami\input\noon\uae\inventory\2026\may\merged_noon_may_2026.xlsx
Rows : 1,162
Reading Folder : 1
Reading Folder : 10
Reading Folder : 11
Reading Folder : 12
Reading Folder : 13
Reading Folder : 14
Reading Folder : 15
Reading Folder : 16
Reading Folder : 17
Reading Folder : 18
Reading Folder : 19
Reading Folder : 2
Reading Folder : 20
Reading Folder : 21
Reading Folder : 22
Reading Folder : 23
Reading Folder : 24
Reading Folder : 25
Reading Folder : 26
Reading Folder : 27
Reading Folder : 28
Reading Folder : 29
Reading Folder : 3
Reading Folder : 30
Reading Folder : 4
Reading Folder : 5
Reading Folder : 6
Reading Folder : 7
Reading Folder : 8
Rea

## REVENUE INPUT

In [8]:
# NOTE: standardise_noon_inv is defined once, definitively, in the
# "SECTION A — INVENTORY PROCESSING" cell below (search for "A-0  SHARED HELPERS").
# This cell used to contain a duplicate/conflicting definition with a dead
# "skip_date_col" parameter that was never called anywhere — removed to avoid
# two different functions silently overwriting each other depending on run order.
print("ℹ️  standardise_noon_inv is defined in the Section A cell (A-0 SHARED HELPERS).")


ℹ️  standardise_noon_inv is defined in the Section A cell (A-0 SHARED HELPERS).


## ## 4 · Amazon – Inventory (`amz_inv_df`)

In [9]:
# ══════════════════════════════════════════════════════════════════════════
# AMAZON INVENTORY MASTER
# INVENTORY = PRIORITY BASE
# ══════════════════════════════════════════════════════════════════════════

inv_parts = []

# ══════════════════════════════════════════════════════════════════════════
# Load Monthly Inventory Masters
# ══════════════════════════════════════════════════════════════════════════

for month in MONTHS:

    file_path = os.path.join(

        AMZ_INV_ROOT,

        f"{month}_merged",

        f"merged_output_inventory_{month}_{YEAR}.xlsx"
    )

    if os.path.exists(file_path):

        print(f"✅ Loading Inventory: {month}")

        try:

            df = pd.read_excel(file_path)

            df["processed_month"] = month

            inv_parts.append(df)

        except Exception as e:

            print(f"❌ Error Loading: {file_path}")
            print("Reason:", e)

    else:

        print(f"⚠️ Inventory File Not Found: {month}")

# ══════════════════════════════════════════════════════════════════════════
# Combine Inventory
# ══════════════════════════════════════════════════════════════════════════

if inv_parts:

    amz_inv_df = pd.concat(

        inv_parts,

        ignore_index=True
    )

else:

    amz_inv_df = pd.DataFrame()

    print("❌ No Inventory Data Loaded")

# ══════════════════════════════════════════════════════════════════════════
# Clean Columns
# ══════════════════════════════════════════════════════════════════════════

amz_inv_df.columns = (

    amz_inv_df.columns

    .astype(str)

    .str.strip()
)

# Remove Unwanted
amz_inv_df.drop(

    columns=["Product Title"],

    errors="ignore",

    inplace=True
)

# ══════════════════════════════════════════════════════════════════════════
# Rename Columns
# ══════════════════════════════════════════════════════════════════════════

amz_inv_df.rename(columns={

    "Sellable On Hand Units"       : "Available Inventory",

    "Sellable On Hand Inventory"   : "Sellable Inventory Amount",

    "Open Purchase Order Quantity" : "Open POs",

    "Net Received"                 : "PO_received_amount",

    "Net Received Units"           : "PO_received_Units",

    "Receive Fill %"               : "Receive_Fill_Rate",

    "Unsellable On Hand Inventory" : "Unsellable On Hand Inventory Amount",

    "Subcategory"                  : "Product Sub Category",

}, inplace=True)

# ══════════════════════════════════════════════════════════════════════════
# Clean ASIN + Date
# ══════════════════════════════════════════════════════════════════════════

amz_inv_df["ASIN"] = (

    amz_inv_df["ASIN"]

    .astype(str)

    .str.upper()

    .str.strip()

    # .str.replace("-1", "", regex=False)

    .str.replace(".0", "", regex=False)
)

amz_inv_df["Date"] = (

    pd.to_datetime(

        amz_inv_df["Date"],

        errors="coerce"
    )

    .dt.normalize()
)

# ══════════════════════════════════════════════════════════════════════════
# Sales Channel
# ══════════════════════════════════════════════════════════════════════════

amz_inv_df["Sales Channel"] = "Vendor Central"

# ══════════════════════════════════════════════════════════════════════════
# Inventory Numeric Cleaning
# ══════════════════════════════════════════════════════════════════════════

inventory_numeric_cols = [

    "Available Inventory",

    "Sellable Inventory Amount",

    "Open POs",

    "PO_received_amount",

    "PO_received_Units",

    "Receive_Fill_Rate",

    "Unsellable On Hand Inventory Amount"
]

for col in inventory_numeric_cols:

    if col in amz_inv_df.columns:

        amz_inv_df[col] = (

            amz_inv_df[col]

            .astype(str)

            .str.replace(",", "", regex=False)

            .str.replace("%", "", regex=False)

            .str.strip()

            .pipe(pd.to_numeric, errors="coerce")

            .fillna(0)
        )

if "Available Inventory" in amz_inv_df.columns:

    amz_inv_df["Available Inventory"] = (

        amz_inv_df["Available Inventory"]

        .astype(int)
    )

# ══════════════════════════════════════════════════════════════════════════
# Inventory Priority Logic
# KEEP ALL INVENTORY ROWS
# ══════════════════════════════════════════════════════════════════════════

amz_inv_df = amz_inv_df.drop_duplicates(

    subset=["ASIN", "Date"],

    keep="last"
)

# ══════════════════════════════════════════════════════════════════════════
# Instock Metrics
# ══════════════════════════════════════════════════════════════════════════

amz_inv_df = amz_inv_df.sort_values(

    ["ASIN", "Date"]
)

amz_inv_df["year_month"] = (

    amz_inv_df["Date"]

    .dt.to_period("M")
)

amz_inv_df["in_stock_flag"] = (

    amz_inv_df["Available Inventory"] > 0

).astype(int)

amz_inv_df["cumulative_instock_days"] = (

    amz_inv_df

    .groupby(["ASIN", "year_month"])["in_stock_flag"]

    .cumsum()
)

amz_inv_df["day_of_month"] = (

    amz_inv_df["Date"]

    .dt.day
)

amz_inv_df["Instock Rate"] = (

    amz_inv_df["cumulative_instock_days"]

    / amz_inv_df["day_of_month"]

    * 100
).round(2)

amz_inv_df.drop(

    columns=["year_month"],

    inplace=True
)

# ══════════════════════════════════════════════════════════════════════════
# LOAD HELPER FILE
# ══════════════════════════════════════════════════════════════════════════

print("\n📦 Loading Helper File...")

raw_helper = pd.read_excel(

    HELPER_FILE,

    header=None
)

# ══════════════════════════════════════════════════════════════════════════
# Detect Actual Header Row
# ══════════════════════════════════════════════════════════════════════════

header_row = None

possible_headers = [

    "Brand",

    "Amazon ASIN",

    "Amazon ASIN\n(Vendor Central)",

    "Product Code\n(Noon Minutes)",

    "Product Code\n(Noon Core)"
]

for i in range(len(raw_helper)):

    row_values = (

        raw_helper.iloc[i]

        .astype(str)

        .str.strip()

        .tolist()
    )

    if any(h in row_values for h in possible_headers):

        header_row = i
        break

# ══════════════════════════════════════════════════════════════════════════
# Read Helper Again Using Correct Header
# ══════════════════════════════════════════════════════════════════════════

if header_row is not None:

    print(f"✅ Helper Header Row Found: {header_row}")

    helper_df = pd.read_excel(

        HELPER_FILE,

        skiprows=header_row
    )

else:

    print("❌ Could Not Detect Header Row")

    helper_df = pd.read_excel(
        HELPER_FILE
    )

# ══════════════════════════════════════════════════════════════════════════
# Clean Helper Columns
# ══════════════════════════════════════════════════════════════════════════

helper_df.columns = (

    helper_df.columns

    .astype(str)

    .str.strip()
)

print("\n📌 Actual Helper Columns:")
print(helper_df.columns.tolist())

# ══════════════════════════════════════════════════════════════════════════
# AMAZON HELPER BUILD
# ══════════════════════════════════════════════════════════════════════════

amazon_helper_col = None

possible_amz_cols = [

    "Amazon ASIN\n(Vendor Central)",

    "Amazon ASIN (Vendor Central)",

    "Amazon ASIN"
]

for col in helper_df.columns:

    if str(col).strip() in possible_amz_cols:

        amazon_helper_col = col
        break

if amazon_helper_col:

    print(f"\n✅ Amazon Helper Key Found:")
    print(amazon_helper_col)

    helper_df.rename(

        columns={
            amazon_helper_col: "ASIN"
        },

        inplace=True
    )

else:

    print("\n❌ Amazon Helper Key Not Found")

    helper_df["ASIN"] = np.nan

# ══════════════════════════════════════════════════════════════════════════
# Clean ASIN
# ══════════════════════════════════════════════════════════════════════════

helper_df["ASIN"] = (

    helper_df["ASIN"]

    .astype(str)

    .str.upper()

    .str.strip()

    # .str.replace("-1", "", regex=False)

    .str.replace(".0", "", regex=False)
)

# ══════════════════════════════════════════════════════════════════════════
# Rename Helper Metadata Columns
# ══════════════════════════════════════════════════════════════════════════

helper_df.rename(columns={

    "Category"            : "Product Category",

    "Product Type"        : "Product Sub Category",

    "Product Description" : "Product Name",

    "Size"                : "Pack Size",

}, inplace=True)

# ══════════════════════════════════════════════════════════════════════════
# Keep Required Helper Columns
# ══════════════════════════════════════════════════════════════════════════

helper_cols = [

    "ASIN",

    "Brand",

    "Product Name",

    "Product Category",

    "Product Sub Category",

    "Pack Type",

    "Pack Size",

    "VC Ideal Price"
]

available_helper_cols = [

    c for c in helper_cols

    if c in helper_df.columns
]

helper_df = helper_df[
    available_helper_cols
]

# Remove Blank ASIN
helper_df = helper_df[
    helper_df["ASIN"].notna()
]

helper_df = helper_df[
    helper_df["ASIN"] != ""
]

helper_df = helper_df.drop_duplicates(
    subset=["ASIN"]
)

print("\n✅ Helper Final Shape:")
print(helper_df.shape)






# ══════════════════════════════════════════════════════════════════════════
# AMAZON HELPER MERGE
# FORCE HELPER TO OVERRIDE INVENTORY
# ══════════════════════════════════════════════════════════════════════════

def clean_merge_asin(series):
    # For Amazon ASINs: do NOT strip "-1" — Amazon ASINs are alphanumeric
    # and never end in -\d+. Only Noon NINs have that suffix pattern.
    return (
        series.astype(str)
        .str.upper()
        .str.strip()
        .str.replace("\xa0", "", regex=False)
        .str.replace(".0", "", regex=False)
        .str.replace(r"\s+", "", regex=True)
    )

# ----------------------------------------------------------
# CLEAN ASINS
# ----------------------------------------------------------

amz_inv_df["ASIN"] = clean_merge_asin(
    amz_inv_df["ASIN"]
)

helper_df["ASIN"] = clean_merge_asin(
    helper_df["ASIN"]
)

# ----------------------------------------------------------
# REMOVE BLANKS
# ----------------------------------------------------------

helper_df = helper_df[
    helper_df["ASIN"].notna()
]

helper_df = helper_df[
    helper_df["ASIN"] != ""
]

helper_df = helper_df.drop_duplicates(
    subset=["ASIN"],
    keep="first"
)

# ----------------------------------------------------------
# DEBUG MATCH RATE
# ----------------------------------------------------------

inventory_asins = set(
    amz_inv_df["ASIN"].dropna().unique()
)

helper_asins = set(
    helper_df["ASIN"].dropna().unique()
)

matched_asins = inventory_asins & helper_asins

print("\n════════════════════════════════════")
print("HELPER MATCH CHECK")
print("════════════════════════════════════")

print("Inventory ASINs :", len(inventory_asins))
print("Helper ASINs    :", len(helper_asins))
print("Matched ASINs   :", len(matched_asins))

# ----------------------------------------------------------
# MERGE
# ----------------------------------------------------------

amz_inv_df = amz_inv_df.merge(

    helper_df,

    on="ASIN",

    how="left",

    suffixes=("", "_helper")
)

# ----------------------------------------------------------
# HELPER COLUMNS
# ----------------------------------------------------------

metadata_cols = [

    "Brand",

    "Product Name",

    "Product Category",

    "Product Sub Category",

    "Pack Type",

    "Pack Size",

    "VC Ideal Price"
]

# ----------------------------------------------------------
# FORCE HELPER PRIORITY
# ----------------------------------------------------------

for col in metadata_cols:

    helper_col = f"{col}_helper"

    # If inventory doesn't have column
    if col not in amz_inv_df.columns:

        amz_inv_df[col] = np.nan

    # Helper column exists after merge
    if helper_col in amz_inv_df.columns:

        amz_inv_df[col] = np.where(

            amz_inv_df[helper_col].notna(),

            amz_inv_df[helper_col],

            amz_inv_df[col]
        )

        amz_inv_df.drop(
            columns=[helper_col],
            inplace=True
        )

# ----------------------------------------------------------
# CLEAN METADATA
# ----------------------------------------------------------

for col in metadata_cols:

    if col in amz_inv_df.columns:

        amz_inv_df[col] = (

            amz_inv_df[col]

            .astype(str)

            .replace("nan", np.nan)

            .replace("None", np.nan)

            .replace("", np.nan)

            .fillna("Unknown")
        )

# ----------------------------------------------------------
# REMOVE DUPLICATE SUFFIX COLS
# ----------------------------------------------------------

drop_cols = [

    c for c in amz_inv_df.columns

    if c.endswith("_x")
    or c.endswith("_y")
]

if drop_cols:

    amz_inv_df.drop(
        columns=drop_cols,
        inplace=True
    )

# ----------------------------------------------------------
# VALIDATION
# ----------------------------------------------------------

print("\n════════════════════════════════════")
print("HELPER MERGE COMPLETE")
print("════════════════════════════════════")

print("Rows :", len(amz_inv_df))

for col in metadata_cols:

    if col in amz_inv_df.columns:

        print(
            f"{col} Filled :",
            (amz_inv_df[col] != "Unknown").sum()
        )

# ----------------------------------------------------------
# TEST SPECIFIC ASIN
# ----------------------------------------------------------

TEST_ASIN = "B0G64MSXJS"

if TEST_ASIN in amz_inv_df["ASIN"].values:

    print("\nTEST ASIN FOUND")

    display(

        amz_inv_df.loc[
            amz_inv_df["ASIN"] == TEST_ASIN,
            [
                "ASIN",
                "Brand",
                "Product Name",
                "Product Category",
                "Product Sub Category",
                "Pack Type",
                "Pack Size",
                "VC Ideal Price"
            ]
        ].head()
    )

# ══════════════════════════════════════════════════════════════════════════
# Fill Metadata From Helper
# REMOVE DUPLICATE _helper COLUMNS
# ══════════════════════════════════════════════════════════════════════════

metadata_cols = [

    "Brand",

    "Product Name",

    "Product Category",

    "Product Sub Category",

    "Pack Type",

    "Pack Size",

    "VC Ideal Price"
]

# ══════════════════════════════════════════════════════════════════════
# HELPER PRIORITY LOGIC
# Helper should overwrite UNKNOWN / blank values
# ══════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════
# FORCE HELPER PRIORITY
# HELPER ALWAYS WINS
# ══════════════════════════════════════════════════════════════════════

for col in metadata_cols:

    helper_col = f"{col}_helper"

    if helper_col in amz_inv_df.columns:

        # FORCE helper overwrite
        amz_inv_df[col] = (

            amz_inv_df[helper_col]

            .combine_first(
                amz_inv_df.get(col)
            )
        )

        # Clean bad values
        amz_inv_df[col] = (

            amz_inv_df[col]

            .replace("", np.nan)

            .replace("nan", np.nan)

            .replace("None", np.nan)

            .fillna("Unknown")
        )

        # Drop duplicate helper column
        amz_inv_df.drop(

            columns=[helper_col],

            inplace=True
        )
# ══════════════════════════════════════════════════════════════════════════
# Final Metadata Fill
# ══════════════════════════════════════════════════════════════════════════

for col in metadata_cols:

    if col in amz_inv_df.columns:

        amz_inv_df[col] = (

            amz_inv_df[col]

            .fillna("Unknown")
        )

# ══════════════════════════════════════════════════════════════════════════
# Remove Any Remaining _x / _y Columns
# ══════════════════════════════════════════════════════════════════════════

drop_cols = [

    c for c in amz_inv_df.columns

    if c.endswith("_x") or c.endswith("_y")
]

if drop_cols:

    amz_inv_df.drop(

        columns=drop_cols,

        inplace=True
    )

# ══════════════════════════════════════════════════════════════════════════
# Final Validation
# ══════════════════════════════════════════════════════════════════════════

print("\n✅ AMAZON INVENTORY MASTER READY")

print("📊 Shape:", amz_inv_df.shape)

print("\n📅 Available Months:")

print(
    sorted(
        amz_inv_df["Date"]
        .dt.strftime("%Y-%m")
        .dropna()
        .unique()
    )
)

print("\n📦 Brand Fill Check:")

if "Brand" in amz_inv_df.columns:

    print(
        amz_inv_df["Brand"]
        .value_counts(dropna=False)
        .head(10)
    )

print("\n🧹 Duplicate Metadata Columns Removed")

display(
    amz_inv_df.head(10)
)

✅ Loading Inventory: may
✅ Loading Inventory: jun
✅ Loading Inventory: july
✅ Loading Inventory: aug

📦 Loading Helper File...
✅ Helper Header Row Found: 1

📌 Actual Helper Columns:
['S.No.', 'Brand', 'Priority', 'Category', 'Range', 'Product Type', 'Product Description', 'Product Description(Cleaned)', 'Focus Range/ NPD', 'Size', 'Variation', 'Barcode', 'Emami  SAP code', 'Truebell SAP Code', 'HS CODE', 'Origin', 'Case Size', 'Cost Price', 'Shelf Life', 'TB SOH', 'Noon eCom', 'Noon Minutes', 'Amazon', 'Amazon ASIN\n(Vendor Central)', 'Product Code\n(Noon Minutes)', 'Product Code\n(Noon Core)', 'Remarks', 'Pack Type', 'VC Ideal Price']

✅ Amazon Helper Key Found:
Amazon ASIN
(Vendor Central)

✅ Helper Final Shape:
(99, 8)

════════════════════════════════════
HELPER MATCH CHECK
════════════════════════════════════
Inventory ASINs : 116
Helper ASINs    : 99
Matched ASINs   : 95

════════════════════════════════════
HELPER MERGE COMPLETE
════════════════════════════════════
Rows : 9101
B

,ASIN,Brand,Product Name,Product Category,Product Sub Category,Pack Type,Pack Size,VC Ideal Price
6918,B0G64MSXJS,Creme 21,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown



✅ AMAZON INVENTORY MASTER READY
📊 Shape: (9101, 28)

📅 Available Months:
['2026-05', '2026-06', '2026-07', '2026-08']

📦 Brand Fill Check:
Brand
Creme 21                    5540
Emami 7 Oils in One         1702
Emami Smart And Handsome    1051
Himani Navratna              663
Emami                        111
Unknown                       34
Name: count, dtype: int64

🧹 Duplicate Metadata Columns Removed


,ASIN,Brand,Vendor Confirmation %,PO_received_amount,PO_received_Units,Open POs,Receive_Fill_Rate,Overall Vendor Lead Time (days),Aged 90+ Days Sellable Inventory,Aged 90+ Days Sellable Units,...,in_stock_flag,cumulative_instock_days,day_of_month,Instock Rate,Product Name,Product Category,Product Sub Category,Pack Type,Pack Size,VC Ideal Price
0,B00F93HFME,Himani Navratna,0.0,0.00,0,7.0,0.0,0.00,0.0,0.0,...,1,1,1,100.0,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.22376925836872
1,B00F93HFME,Himani Navratna,0.0,120.26,7,0.0,0.0,5.14,0.0,0.0,...,1,2,2,100.0,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.22376925836872
2,B00F93HFME,Himani Navratna,0.0,0.00,0,0.0,0.0,0.00,0.0,0.0,...,1,3,3,100.0,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.22376925836872
3,B00F93HFME,Himani Navratna,0.0,0.00,0,0.0,0.0,0.00,0.0,0.0,...,1,4,4,100.0,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.22376925836872
4,B00F93HFME,Himani Navratna,0.0,0.00,0,0.0,0.0,0.00,0.0,0.0,...,1,5,5,100.0,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.22376925836872
5,B00F93HFME,Himani Navratna,0.0,0.00,0,0.0,0.0,0.00,0.0,0.0,...,1,6,6,100.0,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.22376925836872
6,B00F93HFME,Himani Navratna,0.0,0.00,0,0.0,0.0,0.00,0.0,0.0,...,1,7,7,100.0,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.22376925836872
7,B00F93HFME,Himani Navratna,0.0,0.00,0,0.0,0.0,0.00,0.0,0.0,...,1,8,8,100.0,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.22376925836872
8,B00F93HFME,Himani Navratna,0.0,0.00,0,0.0,0.0,0.00,0.0,0.0,...,1,9,9,100.0,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.22376925836872
9,B00F93HFME,Himani Navratna,0.0,0.00,0,0.0,0.0,0.00,0.0,0.0,...,1,10,10,100.0,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.22376925836872


## ## 5 · Amazon – Revenue (`amz_rev_df`)

In [10]:
# ══════════════════════════════════════════════════════════════════════════
# AMAZON REVENUE — BUILD MONTHLY MASTER FROM DAILY FILES
# Mirrors the Amazon Inventory builder (Section 4) — reads every daily
# sales_amazon_emami_<dd>_<mon>_<year>.xlsx / .csv file, stacks them into
# one monthly master, and saves it to <month>_output/merged_revenue_<month>_<YEAR>.xlsx
# so the next cell can load it. THIS is the step that was missing — the
# revenue cell after this one was reading a merged file that nothing
# ever created, which is why totals were incomplete and the DOS merge
# downstream had nothing real to match against.
# ══════════════════════════════════════════════════════════════════════════

amz_revenue_parts = []

for process_month in MONTHS:

    input_folder = os.path.join(
        AMZ_REV_ROOT,
        process_month
    )

    output_folder = os.path.join(
        AMZ_REV_ROOT,
        f"{process_month}_output"
    )

    os.makedirs(output_folder, exist_ok=True)

    # Skip Missing Folder
    if not os.path.exists(input_folder):
        print(f"⚠️ Missing Revenue Folder: {process_month}")
        continue

    month_data = []

    # ══════════════════════════════════════════════════════════════════════
    # Read Daily Revenue Files
    # ══════════════════════════════════════════════════════════════════════

    for file in sorted(os.listdir(input_folder)):

        file_lower = file.lower()

        # Skip temp / lock files
        if file.startswith("~$"):
            continue

        # Only Excel or CSV
        if not (file_lower.endswith(".xlsx") or file_lower.endswith(".csv")):
            continue

        # Only Revenue Files
        if "sales_amazon_emami" not in file_lower:
            continue

        path = os.path.join(input_folder, file)

        print(f"📖 Reading: {file}")

        try:

            # ── Read Proper Data ───────────────────────────────────────
            if file_lower.endswith(".csv"):

                try:
                    df = pd.read_csv(
                        path,
                        dtype=str,
                        encoding="utf-8-sig",
                        encoding_errors="ignore",
                        on_bad_lines="skip"
                    )
                    df.columns = df.columns.astype(str).str.strip()
                    if "ASIN" not in df.columns:
                        raise ValueError("ASIN not in columns")
                except Exception:
                    # Fallback: scan up to 10 rows for ASIN header
                    raw = pd.read_csv(
                        path, header=None, dtype=str,
                        encoding="utf-8-sig", encoding_errors="ignore",
                        engine="python", sep=",", on_bad_lines="skip"
                    )
                    found = False
                    for _ri in range(min(10, len(raw))):
                        if "ASIN" in raw.iloc[_ri].astype(str).values:
                            df = pd.read_csv(
                                path, skiprows=_ri, dtype=str,
                                encoding="utf-8-sig", encoding_errors="ignore",
                                on_bad_lines="skip"
                            )
                            df.columns = df.columns.astype(str).str.strip()
                            found = True
                            break
                    if not found:
                        raise ValueError("ASIN header row not found in CSV")

            else:

                # Find actual header row in Excel (handles metadata banner rows)
                raw = pd.read_excel(path, header=None)

                header_matches = raw.eq("ASIN").any(axis=1)
                if not header_matches.any():
                    raise ValueError("ASIN header row not found")

                header_row = header_matches[header_matches].index[0]

                df = pd.read_excel(
                    path,
                    skiprows=header_row,
                    dtype=str
                )

            # ── Clean Columns ──────────────────────────────────────────
            df.columns = (
                df.columns
                .astype(str)
                .str.strip()
            )

            # ── Extract Date From Filename ────────────────────────────
            # match = re.search(r'(\d{1,2}_\w+_\d{4})', file)

            # if match:
            #     file_date = pd.to_datetime(
            #         match.group(1),
            #         format="%d_%b_%Y",
            #         errors="coerce"
            #     )
            #     df["Date"] = file_date
            # else:
            #     df["Date"] = pd.NaT

            match = re.search(
                r'(\d{1,2})_(\w+)_(\d{4})',
                file.lower()
            )

            if match:

                day = match.group(1)
                month = match.group(2)
                year = match.group(3)

                # Try abbreviated month (Jun, Jul)
                file_date = pd.to_datetime(
                    f"{day}_{month}_{year}",
                    format="%d_%b_%Y",
                    errors="coerce"
                )

                # Try full month (June, July)
                if pd.isna(file_date):
                    file_date = pd.to_datetime(
                        f"{day}_{month}_{year}",
                        format="%d_%B_%Y",
                        errors="coerce"
                    )

                df["Date"] = file_date

            else:
                df["Date"] = pd.NaT

            # ── Clean ASIN ────────────────────────────────────────────
            if "ASIN" in df.columns:
                df["ASIN"] = clean_asin(df["ASIN"])

            # ── Sales Channel ─────────────────────────────────────────
            df["Sales Channel"] = "Vendor Central"

            # ── Source Tracking ──────────────────────────────────────
            df["source_file"] = file

            month_data.append(df)

        except Exception as e:
            print(f"❌ Skipped: {file}")
            print("Reason:", e)

    # ══════════════════════════════════════════════════════════════════════
    # Merge Month Revenue
    # ══════════════════════════════════════════════════════════════════════

    if month_data:

        merged = pd.concat(month_data, ignore_index=True)

        # ── Clean Dates ───────────────────────────────────────────────
        merged["Date"] = clean_date(merged["Date"])

        # ── Remove Blank ASIN Rows ───────────────────────────────────
        if "ASIN" in merged.columns:
            merged = merged[merged["ASIN"].notna()]

        # ── Sort ─────────────────────────────────────────────────────
        sort_cols = ["Date"]
        if "ASIN" in merged.columns:
            sort_cols.append("ASIN")
        merged = merged.sort_values(sort_cols)

        # ── Remove Duplicate Rows ────────────────────────────────────
        merged = merged.drop_duplicates()

        # ── Save Monthly Master ──────────────────────────────────────
        out = os.path.join(
            output_folder,
            f"merged_revenue_{process_month}_{YEAR}.xlsx"
        )

        merged.to_excel(out, index=False)

        print(f"\n✅ Saved Monthly Revenue:")
        print(out)
        print(f"📊 Shape: {merged.shape}")

        amz_revenue_parts.append(merged)

    else:
        print(f"⚠️ No Revenue Files Found: {process_month}")

# ══════════════════════════════════════════════════════════════════════════
# Note: amz_revenue_parts is intentionally NOT combined into amz_rev_df here.
# The next cell re-reads the saved merged_revenue_<month>_<YEAR>.xlsx files
# from disk (same pattern as Amazon Inventory's Section 19), so the
# expensive daily-file parsing above only needs to re-run when new daily
# files are dropped into the revenue/<month> folder.
# ══════════════════════════════════════════════════════════════════════════

if amz_revenue_parts:
    print(f"\n✅ Built {len(amz_revenue_parts)} monthly revenue master(s) from daily files")
else:
    print("\n❌ No Amazon Revenue daily files found in any month folder")


📖 Reading: sales_amazon_emami_01_may_2026.xlsx
📖 Reading: sales_amazon_emami_02_may_2026.xlsx
📖 Reading: sales_amazon_emami_03_may_2026.xlsx
📖 Reading: sales_amazon_emami_04_may_2026.xlsx
📖 Reading: sales_amazon_emami_05_may_2026.xlsx
📖 Reading: sales_amazon_emami_06_may_2026.xlsx
📖 Reading: sales_amazon_emami_07_may_2026.xlsx
📖 Reading: sales_amazon_emami_08_may_2026.xlsx
📖 Reading: sales_amazon_emami_09_may_2026.xlsx
📖 Reading: sales_amazon_emami_10_may_2026.xlsx
📖 Reading: sales_amazon_emami_11_may_2026.xlsx
📖 Reading: sales_amazon_emami_12_may_2026.xlsx
📖 Reading: sales_amazon_emami_13_may_2026.xlsx
📖 Reading: sales_amazon_emami_14_may_2026.xlsx
📖 Reading: sales_amazon_emami_15_may_2026.xlsx
📖 Reading: sales_amazon_emami_16_may_2026.xlsx
📖 Reading: sales_amazon_emami_17_may_2026.xlsx
📖 Reading: sales_amazon_emami_18_may_2026.xlsx
📖 Reading: sales_amazon_emami_19_may_2026.xlsx
📖 Reading: sales_amazon_emami_20_may_2026.xlsx
📖 Reading: sales_amazon_emami_21_may_2026.xlsx
📖 Reading: sa

In [11]:
# ══════════════════════════════════════════════════════════════════════════
# AMAZON REVENUE MASTER
# REVENUE = PRIORITY BASE
# ══════════════════════════════════════════════════════════════════════════

rev_parts = []

# ══════════════════════════════════════════════════════════════════════════
# Load Monthly Revenue Files
# ══════════════════════════════════════════════════════════════════════════

for month in MONTHS:

    file_path = os.path.join(

        AMZ_REV_ROOT,

        f"{month}_output",

        f"merged_revenue_{month}_{YEAR}.xlsx"
    )

    if os.path.exists(file_path):

        print(f"✅ Loading Revenue: {month}")

        try:

            df = pd.read_excel(file_path)

            df["processed_month"] = month

            rev_parts.append(df)

        except Exception as e:

            print(f"❌ Error Loading: {file_path}")
            print("Reason:", e)

    else:

        print(f"⚠️ Revenue File Not Found: {month}")

# ══════════════════════════════════════════════════════════════════════════
# Combine Revenue
# ══════════════════════════════════════════════════════════════════════════

if rev_parts:

    amz_rev_df = pd.concat(

        rev_parts,

        ignore_index=True
    )

else:

    amz_rev_df = pd.DataFrame()

    print("❌ No Revenue Data Loaded")

# ══════════════════════════════════════════════════════════════════════════
# Clean Columns
# ══════════════════════════════════════════════════════════════════════════

amz_rev_df.columns = (

    amz_rev_df.columns

    .astype(str)

    .str.strip()
)

# Remove Unwanted
amz_rev_df.drop(

    columns=["Product Title"],

    errors="ignore",

    inplace=True
)

# ══════════════════════════════════════════════════════════════════════════
# Rename Revenue Columns
# ══════════════════════════════════════════════════════════════════════════

amz_rev_df.rename(columns={

    "Shipped Revenue" : "total_sales",

    "Shipped Units"   : "total_units",

}, inplace=True)

# ══════════════════════════════════════════════════════════════════════════
# Clean ASIN + Date
# ══════════════════════════════════════════════════════════════════════════

amz_rev_df["ASIN"] = (

    amz_rev_df["ASIN"]

    .astype(str)

    .str.upper()

    .str.strip()

    

    .str.replace(".0", "", regex=False)
)

amz_rev_df["Date"] = (

    pd.to_datetime(

        amz_rev_df["Date"],

        dayfirst=True,  # DD/MM/YYYY → correct year

        errors="coerce"
    )

    .dt.normalize()
)

# ══════════════════════════════════════════════════════════════════════════
# Revenue Numeric Cleaning
# ══════════════════════════════════════════════════════════════════════════

if "total_sales" in amz_rev_df.columns:

    amz_rev_df["total_sales"] = (

        amz_rev_df["total_sales"]

        .astype(str)

        .str.replace("AED", "", regex=False)

        .str.replace(",", "", regex=False)

        .str.strip()

        .pipe(pd.to_numeric, errors="coerce")

        .fillna(0)
    )

if "total_units" in amz_rev_df.columns:

    amz_rev_df["total_units"] = (

        pd.to_numeric(

            amz_rev_df["total_units"],

            errors="coerce"
        )

        .fillna(0)

        .astype(int)
    )

# ══════════════════════════════════════════════════════════════════════════
# Sales Channel
# ══════════════════════════════════════════════════════════════════════════

amz_rev_df["Sales Channel"] = "Vendor Central"

# ══════════════════════════════════════════════════════════════════════════
# REVENUE PRIORITY LOGIC
# KEEP ALL REVENUE ROWS
# ══════════════════════════════════════════════════════════════════════════

amz_rev_df = (

    amz_rev_df

    .groupby(

        ["ASIN", "Date"],

        as_index=False
    )

    .agg({

        "total_sales" : "sum",

        "total_units" : "sum"
    })
)

amz_rev_df["Sales Channel"] = "Vendor Central"

# ══════════════════════════════════════════════════════════════════════════
# Sort Revenue
# ══════════════════════════════════════════════════════════════════════════

amz_rev_df = amz_rev_df.sort_values(

    ["ASIN", "Date"]
)

# ══════════════════════════════════════════════════════════════════════════
# HELPER MERGE
# Helper Metadata = FINAL PRIORITY
# ══════════════════════════════════════════════════════════════════════════

amz_rev_df = amz_rev_df.merge(

    helper_df,

    on="ASIN",

    how="left",

    suffixes=("", "_helper")
)

# ══════════════════════════════════════════════════════════════════════════
# Fill Metadata From Helper
# REMOVE DUPLICATE _helper COLUMNS
# ══════════════════════════════════════════════════════════════════════════

metadata_cols = [

    "Brand",

    "Product Name",

    "Product Category",

    "Product Sub Category",

    "Pack Type",

    "Pack Size",

    "VC Ideal Price"
]

# ══════════════════════════════════════════════════════════════════════
# FORCE HELPER PRIORITY
# HELPER ALWAYS WINS
# ══════════════════════════════════════════════════════════════════════

for col in metadata_cols:

    helper_col = f"{col}_helper"

    if helper_col in amz_rev_df.columns:

        # FORCE helper overwrite
        amz_rev_df[col] = (

            amz_rev_df[helper_col]

            .combine_first(
                amz_rev_df.get(col)
            )
        )

        # Clean bad values
        amz_rev_df[col] = (

            amz_rev_df[col]

            .replace("", np.nan)

            .replace("nan", np.nan)

            .replace("None", np.nan)

            .fillna("Unknown")
        )

        # Remove duplicate helper column
        amz_rev_df.drop(

            columns=[helper_col],

            inplace=True
        )

# ══════════════════════════════════════════════════════════════════════════
# Final Metadata Fill
# ══════════════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════════
# HELPER PRIORITY LOGIC
# Helper should overwrite UNKNOWN / blank values
# ══════════════════════════════════════════════════════════════════════

for col in metadata_cols:

    helper_col = f"{col}_helper"

    if helper_col in amz_rev_df.columns:

        # Create base column if missing
        if col not in amz_rev_df.columns:

            amz_rev_df[col] = np.nan

        # Replace invalid values with NaN
        amz_rev_df[col] = (

            amz_rev_df[col]

            .replace("Unknown", np.nan)

            .replace("", np.nan)

            .replace("nan", np.nan)
        )

        # Helper gets priority
        amz_rev_df[col] = (

            amz_rev_df[col]

            .fillna(amz_rev_df[helper_col])
        )

        # Remove helper duplicate column
        amz_rev_df.drop(

            columns=[helper_col],

            inplace=True
        )

# ══════════════════════════════════════════════════════════════════════════
# Final Validation
# ══════════════════════════════════════════════════════════════════════════

print("\n✅ AMAZON REVENUE MASTER READY")

print("📊 Shape:", amz_rev_df.shape)

print("\n📅 Available Months:")

print(
    sorted(
        amz_rev_df["Date"]
        .dt.strftime("%Y-%m")
        .dropna()
        .unique()
    )
)

print("\n💰 Total Sales:")

print(
    round(
        amz_rev_df["total_sales"].sum(),
        2
    )
)

print("\n📦 Total Units:")

print(
    int(
        amz_rev_df["total_units"].sum()
    )
)

print("\n📦 Brand Fill Check:")

if "Brand" in amz_rev_df.columns:

    print(
        amz_rev_df["Brand"]
        .value_counts(dropna=False)
        .head(10)
    )

display(
    amz_rev_df.head(10)
)

✅ Loading Revenue: may
✅ Loading Revenue: jun
✅ Loading Revenue: july
✅ Loading Revenue: aug

✅ AMAZON REVENUE MASTER READY
📊 Shape: (2878, 12)

📅 Available Months:
['2026-05', '2026-06', '2026-07', '2026-08']

💰 Total Sales:
81957.03

📦 Total Units:
5803

📦 Brand Fill Check:
Brand
Creme 21                    2064
Emami 7 Oils in One          365
Emami Smart And Handsome     291
Himani Navratna              153
NaN                            5
Name: count, dtype: int64


,ASIN,Date,total_sales,total_units,Sales Channel,Brand,Product Name,Product Category,Product Sub Category,Pack Type,Pack Size,VC Ideal Price
0,B00F93HFME,2026-05-01,33.37,2,Vendor Central,Himani Navratna,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.223769
1,B00F93HFME,2026-05-02,47.43,3,Vendor Central,Himani Navratna,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.223769
2,B00F93HFME,2026-05-03,15.81,1,Vendor Central,Himani Navratna,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.223769
3,B00F93HFME,2026-05-04,108.60,6,Vendor Central,Himani Navratna,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.223769
4,B00F93HFME,2026-05-05,15.81,1,Vendor Central,Himani Navratna,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.223769
5,B00F93HFME,2026-05-07,31.62,2,Vendor Central,Himani Navratna,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.223769
6,B00F93HFME,2026-05-08,47.43,3,Vendor Central,Himani Navratna,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.223769
7,B00F93HFME,2026-05-11,19.51,1,Vendor Central,Himani Navratna,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.223769
8,B00F93HFME,2026-05-13,97.55,5,Vendor Central,Himani Navratna,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.223769
9,B00F93HFME,2026-05-15,56.53,2,Vendor Central,Himani Navratna,Himani Navratna Oil 300ml,Hair Care,Hair Oil,1.0,300ml,22.223769


# ## 6 · Amazon – Ads / SPAP (`amz_ads_df`)

In [12]:
# ══════════════════════════════════════════════════════════════════════════
# AMAZON ADS MASTER  (SP + SD)
# REWRITTEN: defensive renames, no shared-name dedup collisions, and a
# sales-checksum printed at every stage so a future SP/SD mismatch is
# caught immediately instead of silently disappearing.
# ══════════════════════════════════════════════════════════════════════════

def _clean_money(series):
    return (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace("AED", "", regex=False)
        .str.strip()
        .pipe(pd.to_numeric, errors="coerce")
        .fillna(0)
    )

def _rename_ads_cols(df):
    """One canonical rename map, applied identically to SP and SD."""
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    df.rename(columns={
        "Advertised ASIN"        : "ASIN",
        "14 Day Total Sales "    : "ads_sales",
        "14 Day Total Sales"     : "ads_sales",
        "14 Day Total Units (#)" : "ads_unit_sold",
        "Spend"                  : "ads_spend",
        "14 Day Conversion Rate" : "CVR",
        "Start Date"             : "Date",
    }, inplace=True)
    # Defensive: collapse any accidental duplicate columns AFTER rename,
    # keeping the first non-null value across the duplicates row-wise
    # (this is what protects us if two raw columns ever map to the same
    # canonical name at once).
    if df.columns.duplicated().any():
        dup_names = df.columns[df.columns.duplicated()].unique().tolist()
        print(f"⚠️  Duplicate columns after rename, coalescing: {dup_names}")
        for name in dup_names:
            cols = df.loc[:, df.columns == name]
            coalesced = cols.bfill(axis=1).iloc[:, 0]
            df = df.loc[:, df.columns != name]
            df[name] = coalesced
    return df

def _checksum(df, label, col="ads_sales"):
    total = pd.to_numeric(df.get(col, 0), errors="coerce").fillna(0).sum()
    print(f"   🔎 {label:<32} rows={len(df):>6}  {col}_sum={total:,.2f}")

# ══════════════════════════════════════════════════════════════════════════
# LOAD SP (SPONSORED PRODUCTS / SPAP)
# ══════════════════════════════════════════════════════════════════════════

spap_parts = []

for month in MONTHS:

    file_path = os.path.join(
        AMZ_ADS_ROOT, month,
        f"sponsored_products_advertised_product_emami_{month}_{YEAR}.xlsx"
    )

    if not os.path.exists(file_path):
        print(f"⚠️ SP Ads File Not Found: {month}")
        continue

    print(f"✅ Loading SP Ads: {month}")

    try:
        df = pd.read_excel(file_path)
        df = _rename_ads_cols(df)

        df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
        df["ASIN"] = (
            df["ASIN"].astype(str).str.upper().str.strip()
            .str.replace(".0", "", regex=False)
        )
        df["ads_sales"] = _clean_money(df["ads_sales"]) if "ads_sales" in df.columns else 0.0

        df["processed_month"] = month
        df["Sales Channel"] = "Vendor Central"
        df["Campaign Type"] = "SPAP"

        _checksum(df, f"SP raw file ({month})")
        spap_parts.append(df)

    except Exception as e:
        print(f"❌ Error Loading SP: {file_path}")
        print("Reason:", e)

# ══════════════════════════════════════════════════════════════════════════
# LOAD SD (SPONSORED DISPLAY)
# ══════════════════════════════════════════════════════════════════════════

sd_parts = []

for month in MONTHS:

    candidates = [
        f"Sponsored_Display_Advertised_product_emami_{month}_{YEAR}.xlsx",
        f"sponsored_display_advertised_product_emami_{month}_{YEAR}.xlsx",
    ]

    file_path = None
    for fname in candidates:
        p = os.path.join(AMZ_ADS_ROOT, month, fname)
        if os.path.exists(p):
            file_path = p
            break

    if file_path is None:
        print(f"⚠️ SD Ads File Not Found: {month}")
        continue

    print(f"✅ Loading SD Ads: {month}")

    try:
        df = pd.read_excel(file_path)
        df = _rename_ads_cols(df)

        df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
        df["ASIN"] = (
            df["ASIN"].astype(str).str.upper().str.strip()
            .str.replace(".0", "", regex=False)
        )
        df["ads_sales"] = _clean_money(df["ads_sales"]) if "ads_sales" in df.columns else 0.0

        df["processed_month"] = month
        df["Sales Channel"] = "Vendor Central"
        df["Campaign Type"] = "SD"

        _checksum(df, f"SD raw file ({month})")
        sd_parts.append(df)

    except Exception as e:
        print(f"❌ Error Loading SD: {file_path}")
        print("Reason:", e)

# ══════════════════════════════════════════════════════════════════════════
# COMBINE SP + SD
# ══════════════════════════════════════════════════════════════════════════

all_amz_ads_parts = spap_parts + sd_parts

if all_amz_ads_parts:
    amz_ads_df = pd.concat(all_amz_ads_parts, ignore_index=True)
    amz_ads_df.columns = amz_ads_df.columns.astype(str).str.strip()
    amz_ads_df = amz_ads_df.loc[:, ~amz_ads_df.columns.duplicated()]

    print(f"\n✅ SPAP rows : {sum(len(p) for p in spap_parts)}")
    print(f"   SD rows   : {sum(len(p) for p in sd_parts)}")
    print(f"   Total     : {len(amz_ads_df)}")

    _checksum(amz_ads_df[amz_ads_df["Campaign Type"] == "SPAP"], "Combined - SPAP slice")
    _checksum(amz_ads_df[amz_ads_df["Campaign Type"] == "SD"],   "Combined - SD slice")
else:
    amz_ads_df = pd.DataFrame()
    print("❌ No Amazon Ads Data Loaded (neither SPAP nor SD found)")

# ══════════════════════════════════════════════════════════════════════════
# NUMERIC CLEANING (all ads metrics, all campaign types, identical treatment)
# ══════════════════════════════════════════════════════════════════════════

numeric_cols = [
    "ads_sales", "ads_unit_sold", "ads_spend",
    "Impressions", "Clicks",
    "Click-Thru Rate (CTR)", "Cost Per Click (CPC)",
    "Total Advertising Cost of Sales (ACOS)",
    "Total Return on Advertising Spend (ROAS)", "CVR",
]

for col in numeric_cols:
    if col in amz_ads_df.columns:
        amz_ads_df[col] = _clean_money(amz_ads_df[col])
    else:
        amz_ads_df[col] = 0

_checksum(amz_ads_df[amz_ads_df["Campaign Type"] == "SD"], "After numeric cleaning - SD")

# ══════════════════════════════════════════════════════════════════════════
# DEDUP — FIXED: include Campaign Type AND Ad Group Name in the subset.
# The previous subset (ASIN, Date, Campaign Name) dropped a real SD row
# whenever an SP campaign happened to share the same Campaign Name as an
# SD campaign for the same ASIN+Date — silently zeroing out SD ads_sales.
# ══════════════════════════════════════════════════════════════════════════

before_rows = len(amz_ads_df)

amz_ads_df = amz_ads_df.drop_duplicates(
    subset=["ASIN", "Date", "Campaign Name","Ad Group Name",  "Campaign Type"],
    keep="last",
)

if len(amz_ads_df) != before_rows:
    print(f"⚠️  Dedup removed {before_rows - len(amz_ads_df)} true duplicate rows")

_checksum(amz_ads_df[amz_ads_df["Campaign Type"] == "SD"],   "After dedup - SD")
_checksum(amz_ads_df[amz_ads_df["Campaign Type"] == "SPAP"], "After dedup - SPAP")

# ══════════════════════════════════════════════════════════════════════════
# CAMPAIGN-LEVEL DATASET
# ══════════════════════════════════════════════════════════════════════════

campaign_cols = [
    "ASIN", "Date", "Sales Channel", "Campaign Name", 
    "Campaign Type", "Portfolio name", "Ad Group Name",
    "ads_sales", "ads_unit_sold", "ads_spend",
    "Impressions", "Clicks",
    "Click-Thru Rate (CTR)", "Cost Per Click (CPC)",
    "Total Advertising Cost of Sales (ACOS)",
    "Total Return on Advertising Spend (ROAS)", "CVR",
]
campaign_cols = [c for c in campaign_cols if c in amz_ads_df.columns]

amz_ads_campaign = amz_ads_df[campaign_cols].copy()

_checksum(amz_ads_campaign[amz_ads_campaign["Campaign Type"] == "SD"], "amz_ads_campaign - SD")

# ══════════════════════════════════════════════════════════════════════════
# AGGREGATED DATASET (ASIN x Date x Channel — used by revenue builder)
# ══════════════════════════════════════════════════════════════════════════

amz_ads_agg = (
    amz_ads_df
    .groupby(["ASIN", "Date", "Sales Channel"], as_index=False)
    .agg({
        "ads_sales"     : "sum",
        "ads_unit_sold" : "sum",
        "ads_spend"     : "sum",
        "Impressions"   : "sum",
        "Clicks"        : "sum",
    })
)

_checksum(amz_ads_agg, "amz_ads_agg (all types combined)")

# ══════════════════════════════════════════════════════════════════════════
# HELPER MERGE — metadata only, never touches ads_sales / ads_spend / etc.
# ══════════════════════════════════════════════════════════════════════════

amz_ads_campaign = amz_ads_campaign.merge(
    helper_df, on="ASIN", how="left", suffixes=("", "_helper")
)

metadata_cols = [
    "Brand", "Product Name", "Product Category",
    "Product Sub Category", "Pack Type", "Pack Size", "VC Ideal Price",
]

for col in metadata_cols:
    helper_col = f"{col}_helper"
    if helper_col in amz_ads_campaign.columns:
        if col not in amz_ads_campaign.columns:
            amz_ads_campaign[col] = np.nan
        amz_ads_campaign[col] = (
            amz_ads_campaign[helper_col].combine_first(amz_ads_campaign[col])
        )
        amz_ads_campaign[col] = (
            amz_ads_campaign[col]
            .replace(["", "nan", "None"], np.nan)
            .fillna("Unknown")
        )
        amz_ads_campaign.drop(columns=[helper_col], inplace=True)

drop_cols = [c for c in amz_ads_campaign.columns if c.endswith("_x") or c.endswith("_y")]
if drop_cols:
    amz_ads_campaign.drop(columns=drop_cols, inplace=True)

_checksum(amz_ads_campaign[amz_ads_campaign["Campaign Type"] == "SD"], "FINAL amz_ads_campaign - SD")
_checksum(amz_ads_campaign[amz_ads_campaign["Campaign Type"] == "SPAP"], "FINAL amz_ads_campaign - SPAP")

# ══════════════════════════════════════════════════════════════════════════
# FINAL VALIDATION
# ══════════════════════════════════════════════════════════════════════════

print("\n✅ AMAZON ADS READY")
print("\n📊 Campaign Shape:", amz_ads_campaign.shape)
print("📊 Aggregated Shape:", amz_ads_agg.shape)

print("\n💰 Total Ads Sales by Campaign Type:")
print(amz_ads_campaign.groupby("Campaign Type")["ads_sales"].sum())

print("\n💸 Total Ads Spend by Campaign Type:")
print(amz_ads_campaign.groupby("Campaign Type")["ads_spend"].sum())

print("\n📦 Brand Fill Check:")
if "Brand" in amz_ads_campaign.columns:
    print(amz_ads_campaign["Brand"].value_counts(dropna=False).head(10))

display(amz_ads_campaign.head(10))


✅ Loading SP Ads: may
   🔎 SP raw file (may)                rows=  3870  ads_sales_sum=10,492.40
✅ Loading SP Ads: jun
   🔎 SP raw file (jun)                rows=  3571  ads_sales_sum=9,940.32
✅ Loading SP Ads: july
   🔎 SP raw file (july)               rows=  4445  ads_sales_sum=12,481.85
✅ Loading SP Ads: aug
   🔎 SP raw file (aug)                rows=  3828  ads_sales_sum=11,069.29
⚠️ SD Ads File Not Found: may
✅ Loading SD Ads: jun
   🔎 SD raw file (jun)                rows=   679  ads_sales_sum=109.80
✅ Loading SD Ads: july
   🔎 SD raw file (july)               rows=   837  ads_sales_sum=81.48
✅ Loading SD Ads: aug
   🔎 SD raw file (aug)                rows=  1156  ads_sales_sum=81.25

✅ SPAP rows : 15714
   SD rows   : 2672
   Total     : 18386
   🔎 Combined - SPAP slice            rows= 15714  ads_sales_sum=43,983.86
   🔎 Combined - SD slice              rows=  2672  ads_sales_sum=272.53
   🔎 After numeric cleaning - SD      rows=  2672  ads_sales_sum=272.53
   🔎 After dedup - S

,ASIN,Date,Sales Channel,Campaign Name,Campaign Type,Portfolio name,Ad Group Name,ads_sales,ads_unit_sold,ads_spend,...,Total Advertising Cost of Sales (ACOS),Total Return on Advertising Spend (ROAS),CVR,Brand,Product Name,Product Category,Product Sub Category,Pack Type,Pack Size,VC Ideal Price
0,B0D7QBTN5N,2026-05-01,Vendor Central,Crème 21-Petroleum Jelly-SP-M-BD-(P),SPAP,Creme 21,Petroleum Jelly,0.0,0,3.28,...,0.0,0.0,0.0,Creme 21,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,Petroleum Jelly,1,300ml,16.065375
1,B0D7QBTN5N,2026-05-02,Vendor Central,Crème 21-Petroleum Jelly-SP-M-BD-(P),SPAP,Creme 21,Petroleum Jelly,0.0,0,0.00,...,0.0,0.0,0.0,Creme 21,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,Petroleum Jelly,1,300ml,16.065375
2,B0D7QBTN5N,2026-05-03,Vendor Central,Crème 21-Petroleum Jelly-SP-M-BD-(P),SPAP,Creme 21,Petroleum Jelly,0.0,0,1.73,...,0.0,0.0,0.0,Creme 21,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,Petroleum Jelly,1,300ml,16.065375
3,B0D7QBTN5N,2026-05-04,Vendor Central,Crème 21-Petroleum Jelly-SP-M-BD-(P),SPAP,Creme 21,Petroleum Jelly,0.0,0,0.00,...,0.0,0.0,0.0,Creme 21,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,Petroleum Jelly,1,300ml,16.065375
4,B0D7QBTN5N,2026-05-05,Vendor Central,Crème 21-Petroleum Jelly-SP-M-BD-(P),SPAP,Creme 21,Petroleum Jelly,0.0,0,0.00,...,0.0,0.0,0.0,Creme 21,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,Petroleum Jelly,1,300ml,16.065375
5,B0D7QBTN5N,2026-05-06,Vendor Central,Crème 21-Petroleum Jelly-SP-M-BD-(P),SPAP,Creme 21,Petroleum Jelly,0.0,0,0.00,...,0.0,0.0,0.0,Creme 21,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,Petroleum Jelly,1,300ml,16.065375
6,B0D7QBTN5N,2026-05-07,Vendor Central,Crème 21-Petroleum Jelly-SP-M-BD-(P),SPAP,Creme 21,Petroleum Jelly,0.0,0,0.00,...,0.0,0.0,0.0,Creme 21,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,Petroleum Jelly,1,300ml,16.065375
7,B0D7QBTN5N,2026-05-08,Vendor Central,Crème 21-Petroleum Jelly-SP-M-BD-(P),SPAP,Creme 21,Petroleum Jelly,0.0,0,0.00,...,0.0,0.0,0.0,Creme 21,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,Petroleum Jelly,1,300ml,16.065375
8,B0D7QBTN5N,2026-05-09,Vendor Central,Crème 21-Petroleum Jelly-SP-M-BD-(P),SPAP,Creme 21,Petroleum Jelly,0.0,0,0.00,...,0.0,0.0,0.0,Creme 21,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,Petroleum Jelly,1,300ml,16.065375
9,B0D7QBTN5N,2026-05-10,Vendor Central,Crème 21-Petroleum Jelly-SP-M-BD-(P),SPAP,Creme 21,Petroleum Jelly,0.0,0,0.00,...,0.0,0.0,0.0,Creme 21,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,Petroleum Jelly,1,300ml,16.065375


In [13]:
# ══════════════════════════════════════════════════════════════════════════
# EXPORT FOR AZURE PUSH
# "Ad Group Name" is kept INSIDE amz_ads_campaign (in memory) because
# build_marketing() needs it as a grouping key to avoid collapsing distinct
# ad groups under the same Campaign Name (see fix in the build_marketing
# cell). It is dropped ONLY from the exported CSV -- the file consumed by
# the Azure push does not need ad-group-level granularity, and this keeps
# the exported schema unchanged from before.
# ══════════════════════════════════════════════════════════════════════════

_export_df = amz_ads_campaign.drop(columns=["Ad Group Name"], errors="ignore")
_export_df.to_csv("amz_ads_campaign.csv", index=False)

## 7 · Noon – Marketing (`noon_ads_df`) — Daily Files → Monthly Masters

In [14]:
# ══════════════════════════════════════════════════════════════════════════
# NOON ADS DAILY FILES → MONTHLY MASTER
# MARKETING = PRIORITY BASE
# ══════════════════════════════════════════════════════════════════════════

for month in MONTHS:

    input_folder = os.path.join(

        NOON_ADS_ROOT,

        month
    )

    output_folder = os.path.join(

        NOON_ADS_ROOT,

        f"{month}_output"
    )

    # ══════════════════════════════════════════════════════════════════════
    # Skip Missing Folder
    # ══════════════════════════════════════════════════════════════════════

    if not os.path.exists(input_folder):

        print(f"⚠️ Folder Missing: {month}")

        continue

    os.makedirs(

        output_folder,

        exist_ok=True
    )

    all_data = []

    # ══════════════════════════════════════════════════════════════════════
    # Read Daily Ads Files
    # ══════════════════════════════════════════════════════════════════════

    for file in sorted(os.listdir(input_folder)):

        # Skip temp files
        if file.startswith("~$"):
            continue

        # Only Excel
        if not file.lower().endswith((".xlsx", ".xls")):
            continue

        # Only Noon Ads Files
        if "ads_noon" not in file.lower():
            continue

        file_path = os.path.join(

            input_folder,

            file
        )

        print(f"📖 Reading: {file}")

        try:

            # ══════════════════════════════════════════════════════════════
            # Read Sheet
            # ══════════════════════════════════════════════════════════════

            df = pd.read_excel(

                file_path,

                sheet_name="(Product) Sku"
            )

            # ══════════════════════════════════════════════════════════════
            # Clean Columns
            # ══════════════════════════════════════════════════════════════

            df.columns = (

                df.columns

                .astype(str)

                .str.strip()
            )

            # ══════════════════════════════════════════════════════════════
            # Extract Date From Filename
            # Example:
            # ads_noon_20_may_2026.xlsx
            # ══════════════════════════════════════════════════════════════

            # match = re.search(

            #     r'(\d{2}_\w+_\d{4})',

            #     file.lower()
            # )

            # if match:

            #     file_date = pd.to_datetime(

            #         match.group(1),

            #         format="%d_%b_%Y",

            #         errors="coerce"
            #     )

            #     df["Date"] = file_date

            # else:

            #     df["Date"] = pd.NaT

            match = re.search(
                r'(\d{2})_(\w+)_(\d{4})',
                file.lower()
            )

            if match:

                day = match.group(1)
                month_name = match.group(2)
                year = match.group(3)

                # Handle both short and full month names
                month_map = {
                    "jan": "Jan", "january": "January",
                    "feb": "Feb", "february": "February",
                    "mar": "Mar", "march": "March",
                    "apr": "Apr", "april": "April",
                    "may": "May",
                    "jun": "Jun", "june": "June",
                    "jul": "Jul", "july": "July",
                    "aug": "Aug", "august": "August",
                    "sep": "Sep", "sept": "Sep", "september": "September",
                    "oct": "Oct", "october": "October",
                    "nov": "Nov", "november": "November",
                    "dec": "Dec", "december": "December",
                }

                month_name = month_map.get(month_name, month_name)

                date_str = f"{day}-{month_name}-{year}"

                df["Date"] = pd.to_datetime(
                    date_str,
                    errors="coerce"
                )

            else:

                df["Date"] = pd.NaT

            # ══════════════════════════════════════════════════════════════
            # Sales Channel
            # ══════════════════════════════════════════════════════════════

            df["Sales Channel"] = "Noon"

            # ══════════════════════════════════════════════════════════════
            # Campaign Type
            # ══════════════════════════════════════════════════════════════

            df["Campaign Type"] = "Noon Ads"

            # ══════════════════════════════════════════════════════════════
            # Source File Tracking
            # ══════════════════════════════════════════════════════════════

            df["source_file"] = file

            # ══════════════════════════════════════════════════════════════
            # Clean SKU
            # ══════════════════════════════════════════════════════════════

            possible_sku_cols = [

                "Sku",

                "SKU",

                "sku",

                "(Product) Sku"
            ]

            sku_col = None

            for c in df.columns:

                if str(c).strip() in possible_sku_cols:

                    sku_col = c
                    break

            if sku_col:

                df.rename(

                    columns={
                        sku_col: "ASIN"
                    },

                    inplace=True
                )

                df["ASIN"] = (

                    df["ASIN"]

                    .astype(str)

                    .str.upper()

                    .str.strip()

                    # .str.replace("-1", "", regex=False)

                    .str.replace(".0", "", regex=False)
                )

            all_data.append(df)

            print(f"✅ {file} | Rows: {df.shape[0]}")

        except Exception as e:

            print(f"❌ Error Reading: {file}")

            print("Reason:", e)

    # ══════════════════════════════════════════════════════════════════════
    # Merge Monthly Files
    # ══════════════════════════════════════════════════════════════════════

    if all_data:

        merged = pd.concat(

            all_data,

            ignore_index=True
        )

        # Clean Date
        merged["Date"] = (

            pd.to_datetime(

                merged["Date"],

                errors="coerce"
            )

            .dt.normalize()
        )

        # Remove Blank ASIN
        if "ASIN" in merged.columns:

            merged = merged[
                merged["ASIN"].notna()
            ]

        # Sort
        merged = merged.sort_values(

            ["Date", "ASIN"]
        )

        # Remove Duplicates
        merged = merged.drop_duplicates()

        # ══════════════════════════════════════════════════════════════════
        # Save Monthly Master
        # ══════════════════════════════════════════════════════════════════

        out = os.path.join(

            output_folder,

            f"ads_noon_uae_{month}_master.xlsx"
        )

        merged.to_excel(

            out,

            index=False
        )

        print(f"\n✅ Saved Monthly Noon Ads Master:")

        print(out)

        print(f"📊 Shape: {merged.shape}")

    else:

        print(f"⚠️ No Noon Ads Files Found: {month}")

📖 Reading: ads_noon_09_may_2026.xlsx
✅ ads_noon_09_may_2026.xlsx | Rows: 5
📖 Reading: ads_noon_10_may_2026.xlsx
✅ ads_noon_10_may_2026.xlsx | Rows: 10
📖 Reading: ads_noon_11_may_2026.xlsx
✅ ads_noon_11_may_2026.xlsx | Rows: 10
📖 Reading: ads_noon_12_may_2026.xlsx
✅ ads_noon_12_may_2026.xlsx | Rows: 2
📖 Reading: ads_noon_13_may_2026.xlsx
✅ ads_noon_13_may_2026.xlsx | Rows: 15
📖 Reading: ads_noon_14_may_2026.xlsx
✅ ads_noon_14_may_2026.xlsx | Rows: 25
📖 Reading: ads_noon_15_may_2026.xlsx
✅ ads_noon_15_may_2026.xlsx | Rows: 43
📖 Reading: ads_noon_16_may_2026.xlsx
✅ ads_noon_16_may_2026.xlsx | Rows: 44
📖 Reading: ads_noon_17_may_2026.xlsx
✅ ads_noon_17_may_2026.xlsx | Rows: 47
📖 Reading: ads_noon_18_may_2026.xlsx
✅ ads_noon_18_may_2026.xlsx | Rows: 49

✅ Saved Monthly Noon Ads Master:
C:\Users\Shambhavi Dixit\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\Emami\input\noon\uae\marketing\2026\may_output\ads_noon_uae_may_master.xlsx
📊 Shape: (250, 17)
📖 Reading: ads_no

## 7b · Noon – Marketing Aggregation (`noon_ads_agg`)

In [15]:
# ══════════════════════════════════════════════════════════════════════════
# NOON ADS MASTER + HELPER MERGE (FINAL FIXED VERSION)
# ══════════════════════════════════════════════════════════════════════════

noon_ads_parts = []

# ══════════════════════════════════════════════════════════════════════════
# LOAD MONTHLY NOON ADS FILES
# ══════════════════════════════════════════════════════════════════════════

for month in MONTHS:

    file_path = os.path.join(

        NOON_ADS_ROOT,

        f"{month}_output",

        f"ads_noon_uae_{month}_master.xlsx"
    )

    if os.path.exists(file_path):

        try:

            df = pd.read_excel(file_path)

            df["source_month"] = month

            noon_ads_parts.append(df)

            print(f"✅ Noon Ads Loaded: {month}")

        except Exception as e:

            print(f"❌ Error Loading {month}: {e}")

    else:

        print(f"⚠️ Noon Ads File Not Found: {month}")

# ══════════════════════════════════════════════════════════════════════════
# COMBINE ALL MONTHS
# ══════════════════════════════════════════════════════════════════════════

if noon_ads_parts:

    noon_ads_df = pd.concat(

        noon_ads_parts,

        ignore_index=True
    )
    print("\n========== BEFORE DEDUP ==========")
    print(noon_ads_df["source_month"].value_counts(dropna=False))

else:

    noon_ads_df = pd.DataFrame()

    print("❌ No Noon Ads Data Found")

# ══════════════════════════════════════════════════════════════════════════
# CONTINUE ONLY IF DATA EXISTS
# ══════════════════════════════════════════════════════════════════════════

if not noon_ads_df.empty:

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN COLUMN NAMES
    # ══════════════════════════════════════════════════════════════════════

    noon_ads_df.columns = (

        noon_ads_df.columns

        .astype(str)

        .str.strip()
    )

    # ══════════════════════════════════════════════════════════════════════
    # RENAME COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    rename_map = {

        "Sku" : "ASIN",

        "SKU" : "ASIN",

        "(Product) Sku" : "ASIN",

        "Views" : "Impressions",

        "Orders" : "ads_unit_sold",

        "Order" : "ads_unit_sold",

        "Spend" : "ads_spend",

        "Spends" : "ads_spend",

        "Revenue" : "ads_sales",

        "Revenue (AED)" : "ads_sales",

        "Sales" : "ads_sales",

        "CTR" : "Click-Thru Rate (CTR)",

        "CPC" : "Cost Per Click (CPC)",

        "ROAS" : "Total Return on Advertising Spend (ROAS)",

        "ACOS" : "Total Advertising Cost of Sales (ACOS)"
    }

    noon_ads_df.rename(

        columns={
            k: v
            for k, v in rename_map.items()
            if k in noon_ads_df.columns
        },

        inplace=True
    )

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN ASIN
    # ══════════════════════════════════════════════════════════════════════

    if "ASIN" in noon_ads_df.columns:

        noon_ads_df["ASIN"] = (

            noon_ads_df["ASIN"]

            .astype(str)

            .str.upper()

            .str.strip()

            # .str.replace("-1", "", regex=False)

            .str.replace(".0", "", regex=False)
        )

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN DATE
    # ══════════════════════════════════════════════════════════════════════

    if "Date" in noon_ads_df.columns:

        noon_ads_df["Date"] = (

            pd.to_datetime(

                noon_ads_df["Date"],

                errors="coerce"
            )

            .dt.normalize()
        )

    # Remove invalid dates
    noon_ads_df = noon_ads_df[
        noon_ads_df["Date"].notna()
    ]

    # ══════════════════════════════════════════════════════════════════════
    # NUMERIC CLEANING
    # ══════════════════════════════════════════════════════════════════════

    numeric_cols = [

        "ads_sales",

        "ads_unit_sold",

        "ads_spend",

        "Impressions",

        "Clicks",

        "Click-Thru Rate (CTR)",

        "Cost Per Click (CPC)",

        "Total Advertising Cost of Sales (ACOS)",

        "Total Return on Advertising Spend (ROAS)"
    ]

    for col in numeric_cols:

        if col in noon_ads_df.columns:

            noon_ads_df[col] = (

                noon_ads_df[col]

                .astype(str)

                .str.replace(",", "", regex=False)

                .str.replace("AED", "", regex=False)

                .str.replace("%", "", regex=False)

                .str.strip()

                .pipe(pd.to_numeric, errors="coerce")

                .fillna(0)
            )

        else:

            noon_ads_df[col] = 0

    # ══════════════════════════════════════════════════════════════════════
    # ADD SALES CHANNEL
    # ══════════════════════════════════════════════════════════════════════

    noon_ads_df["Sales Channel"] = "Noon"

    # ══════════════════════════════════════════════════════════════════════
    # ADD CAMPAIGN TYPE
    # ══════════════════════════════════════════════════════════════════════

    if "Campaign Type" not in noon_ads_df.columns:

        noon_ads_df["Campaign Type"] = "Noon Ads"

    # ══════════════════════════════════════════════════════════════════════
    # REMOVE DUPLICATES
    # ══════════════════════════════════════════════════════════════════════

    dedupe_cols = [

        c for c in [

            "ASIN",

            "Date",

            "Campaign Name",

            "Campaign Type"

        ] if c in noon_ads_df.columns
    ]

    if dedupe_cols:

        noon_ads_df = noon_ads_df.drop_duplicates(

            subset=dedupe_cols,

            keep="last"
        )

    # ══════════════════════════════════════════════════════════════════════
    # CAMPAIGN OUTPUT
    # ══════════════════════════════════════════════════════════════════════

    campaign_cols = [

        "ASIN",

        "Date",

        "Sales Channel",

        "Campaign Name",

        "Campaign Type",

        "Portfolio name",

        "ads_sales",

        "ads_unit_sold",

        "ads_spend",

        "Impressions",

        "Clicks",

        "Click-Thru Rate (CTR)",

        "Cost Per Click (CPC)",

        "Total Advertising Cost of Sales (ACOS)",

        "Total Return on Advertising Spend (ROAS)"
    ]

    campaign_cols = [

        c for c in campaign_cols

        if c in noon_ads_df.columns
    ]

    noon_ads_campaign = noon_ads_df[
        campaign_cols
    ].copy()

    # ══════════════════════════════════════════════════════════════════════
    # AGGREGATED OUTPUT
    # ══════════════════════════════════════════════════════════════════════

    agg_dict = {

        "ads_sales"     : "sum",

        "ads_unit_sold" : "sum",

        "ads_spend"     : "sum"
    }

    if "Impressions" in noon_ads_df.columns:

        agg_dict["Impressions"] = "sum"

    if "Clicks" in noon_ads_df.columns:

        agg_dict["Clicks"] = "sum"

    noon_ads_agg = (

        noon_ads_df

        .groupby(

            ["ASIN", "Date"],

            as_index=False
        )

        .agg(agg_dict)
    )

    noon_ads_agg["Sales Channel"] = "Noon"

    # ══════════════════════════════════════════════════════════════════════
    # LOAD HELPER FILE
    # ══════════════════════════════════════════════════════════════════════

    print("\n📦 Loading Helper File...")

    helper_df = pd.read_excel(

        HELPER_FILE,

        header=1
    )

    helper_df.columns = (

        helper_df.columns

        .astype(str)

        .str.strip()
    )

    # ══════════════════════════════════════════════════════════════════════
    # FORCE NOON CORE COLUMN
    # ══════════════════════════════════════════════════════════════════════

    noon_helper_col = None

    for col in helper_df.columns:

        clean_col = (

            str(col)

            .replace("\n", " ")

            .strip()

            .lower()
        )

        # FORCE ONLY NOON CORE
        if "noon core" in clean_col:

            noon_helper_col = col

            break

    print("\n✅ Forced Noon Helper Column:")
    print(noon_helper_col)

    # ══════════════════════════════════════════════════════════════════════
    # RENAME HELPER KEY → ASIN
    # ══════════════════════════════════════════════════════════════════════

    helper_df.rename(

        columns={
            noon_helper_col: "ASIN"
        },

        inplace=True
    )

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN HELPER ASIN
    # ══════════════════════════════════════════════════════════════════════

    helper_df["ASIN"] = (

        helper_df["ASIN"]

        .astype(str)

        .str.upper()

        .str.strip()

        # .str.replace("-1", "", regex=False)

        .str.replace(".0", "", regex=False)
    )

    # ══════════════════════════════════════════════════════════════════════
    # RENAME HELPER METADATA COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    helper_df.rename(columns={

        "Product Description(Cleaned)" : "Product Name",

        "Category" : "Product Category",

        "Product Type" : "Product Sub Category",

        "Size" : "Pack Size"

    }, inplace=True)

    # ══════════════════════════════════════════════════════════════════════
    # KEEP REQUIRED HELPER COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    helper_cols = [

        "ASIN",

        "Brand",

        "Product Name",

        "Product Category",

        "Product Sub Category",

        "Pack Type",

        "Pack Size",

        "VC Ideal Price"
    ]

    helper_cols = [

        c for c in helper_cols

        if c in helper_df.columns
    ]

    helper_df = helper_df[
        helper_cols
    ]

    # Remove blank ASINs
    helper_df = helper_df[
        helper_df["ASIN"].notna()
    ]

    helper_df = helper_df[
        helper_df["ASIN"] != ""
    ]

    helper_df = helper_df.drop_duplicates(
        subset=["ASIN"]
    )

    print("\n✅ Helper Shape:")
    print(helper_df.shape)

    # ══════════════════════════════════════════════════════════════════════
    # MERGE HELPER
    # ══════════════════════════════════════════════════════════════════════

    noon_ads_campaign = noon_ads_campaign.merge(

        helper_df,

        on="ASIN",

        how="left"
    )

    # ══════════════════════════════════════════════════════════════════════
    # FILL BLANK VALUES
    # ══════════════════════════════════════════════════════════════════════

    metadata_cols = [

        "Brand",

        "Product Name",

        "Product Category",

        "Product Sub Category",

        "Pack Type",

        "Pack Size"
    ]

    for col in metadata_cols:

        if col in noon_ads_campaign.columns:

            noon_ads_campaign[col] = (

                noon_ads_campaign[col]

                .fillna("Unknown")
            )

    # ══════════════════════════════════════════════════════════════════════
    # REMOVE DUPLICATE _x _y COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    drop_cols = [

        c for c in noon_ads_campaign.columns

        if c.endswith("_x") or c.endswith("_y")
    ]

    if drop_cols:

        noon_ads_campaign.drop(

            columns=drop_cols,

            inplace=True
        )

else:

    noon_ads_campaign = pd.DataFrame()

    noon_ads_agg = pd.DataFrame()

# ══════════════════════════════════════════════════════════════════════════
# FINAL VALIDATION
# ══════════════════════════════════════════════════════════════════════════

print("\n✅ NOON ADS READY")

print("\n📊 Campaign Shape:")
print(noon_ads_campaign.shape)

print("\n📊 Aggregated Shape:")
print(noon_ads_agg.shape)

if not noon_ads_campaign.empty:

    print("\n💰 Total Ads Sales:")
    print(round(noon_ads_campaign["ads_sales"].sum(), 2))

    print("\n💸 Total Ads Spend:")
    print(round(noon_ads_campaign["ads_spend"].sum(), 2))

    print("\n📦 Brand Fill Check:")

    if "Brand" in noon_ads_campaign.columns:

        print(
            noon_ads_campaign["Brand"]
            .value_counts(dropna=False)
            .head(10)
        )

display(
    noon_ads_campaign.head(50)
)

✅ Noon Ads Loaded: may
✅ Noon Ads Loaded: jun
✅ Noon Ads Loaded: july
✅ Noon Ads Loaded: aug

========== BEFORE DEDUP ==========
source_month
jun     1348
july    1180
aug      613
may      250
Name: count, dtype: int64

📦 Loading Helper File...

✅ Forced Noon Helper Column:
Product Code
(Noon Core)

✅ Helper Shape:
(99, 8)

✅ NOON ADS READY

📊 Campaign Shape:
(3340, 21)

📊 Aggregated Shape:
(3026, 8)

💰 Total Ads Sales:
3560.39

💸 Total Ads Spend:
7705.38

📦 Brand Fill Check:
Brand
Creme 21                    1941
Emami Smart And Handsome     771
Emami 7 Oils in One          361
Himani Navratna              257
Unknown                       10
Name: count, dtype: int64


,ASIN,Date,Sales Channel,Campaign Name,Campaign Type,ads_sales,ads_unit_sold,ads_spend,Impressions,Clicks,...,Cost Per Click (CPC),Total Advertising Cost of Sales (ACOS),Total Return on Advertising Spend (ROAS),Brand,Product Name,Product Category,Product Sub Category,Pack Type,Pack Size,VC Ideal Price
0,Z1964954DB60ECED0DD94Z-1,2026-05-09,Noon,Crème 21-APC-SP-M-GKW-(P),Noon Ads,0.00,0,0.00,1,0,...,0.00,0,0.00,Creme 21,Creme 21 Smooth Moisturizing Cream,Skin Care,All-purpose cream,1.0,250ml,21.848910
1,Z26DB626569C62D2677B8Z-1,2026-05-09,Noon,Crème 21-Glycerin Cream-SP-M-GKW-(P),Noon Ads,0.00,0,0.00,1,0,...,0.00,0,0.00,Creme 21,Creme21 Glycerin Cream,Skin Care,Glycerin Cream,1.0,250ml,19.546207
2,ZA73E8B5D668F02855652Z-1,2026-05-09,Noon,Crème 21-APC-SP-M-GKW-(P),Noon Ads,0.00,0,0.00,2,0,...,0.00,0,0.00,Creme 21,Creme 21 Smooth Moisturizing Cream,Skin Care,All-purpose cream,1.0,150ml,15.904722
3,ZAA5A54B39F79F486D56EZ-1,2026-05-09,Noon,Crème 21-Body Lotion-SP-M-GKW-(P),Noon Ads,0.00,0,0.00,1,0,...,0.00,0,0.00,Creme 21,Creme 21 Body Lotion Ultra Dry Skin,Skin Care,Body Lotion,1.0,600ml,43.697821
4,ZD7FD03E72A66D849EAAFZ-1,2026-05-09,Noon,Crème 21-Body Lotion-SP-M-GKW-(P),Noon Ads,0.00,0,0.00,1,0,...,0.00,0,0.00,Creme 21,Creme 21 Brightening Insta Lotion,Skin Care,Body Lotion,1.0,250ml,18.742938
5,Z00FB7A1EB6F0E028BD15Z-1,2026-05-10,Noon,Crème 21-Body Lotion-SP-M-GKW-(P),Noon Ads,0.00,0,0.00,3,0,...,0.00,0,0.00,Creme 21,Creme 21 Brightening Insta Lotion,Skin Care,Body Lotion,1.0,400ml,27.846651
6,Z0DA878AA1AE47579FFFBZ-1,2026-05-10,Noon,Crème 21-APC-SP-M-GKW-(P),Noon Ads,0.00,0,0.00,2,0,...,0.00,0,0.00,Creme 21,Creme 21 Aqua Soft Cream,Skin Care,All-purpose cream,1.0,250ml,21.848910
7,Z26DB626569C62D2677B8Z-1,2026-05-10,Noon,Crème 21-Glycerin Cream-SP-M-GKW-(P),Noon Ads,0.00,0,0.00,1,0,...,0.00,0,0.00,Creme 21,Creme21 Glycerin Cream,Skin Care,Glycerin Cream,1.0,250ml,19.546207
8,Z7992434E0216F92970D9Z-1,2026-05-10,Noon,Crème 21-Petroleum Jelly-SP-M-GKW-(P),Noon Ads,0.00,0,0.00,1,0,...,0.00,0,0.00,Creme 21,Creme 21 Petroleum Jelly Coco Butter,Skin Care,Petroleum Jelly,1.0,300ml,16.065375
9,Z901D1156E8C9BE967CF8Z-1,2026-05-10,Noon,Crème 21-Soap-SP-M-GKW-(P),Noon Ads,0.00,0,0.00,3,0,...,0.00,0,0.00,Creme 21,Creme 21 Aquasoft Moisturizing Cream Soap,Skin Care,Soap,3.0,125gm,16.097506


In [16]:
noon_ads_campaign.to_csv("noon_ads_campaign.csv", index=False)


## 8 · Noon – Revenue (`noon_rev_df`)

In [17]:
# # ══════════════════════════════════════════════════════════════════════════
# # NOON REVENUE MASTER (FINAL FULL FIXED VERSION)
# # REVENUE = PRIORITY BASE
# # HELPER MERGE FIXED
# # DUPLICATE ASIN FIXED
# # APR + MAY DATA FIXED
# # ══════════════════════════════════════════════════════════════════════════

# noon_rev_parts = []

# # ══════════════════════════════════════════════════════════════════════════
# # LOAD MONTHLY FILES
# # ══════════════════════════════════════════════════════════════════════════

# for month in MONTHS:

#     month_folder = os.path.join(

#         NOON_REV_ROOT,

#         month
#     )

#     if not os.path.isdir(month_folder):

#         print(f"⚠️ Noon Revenue Folder Missing: {month}")

#         continue

#     # ─────────────────────────────────────────────────────────────────────
#     # READ FILES
#     # ─────────────────────────────────────────────────────────────────────

#     for file in sorted(os.listdir(month_folder)):

#         # Skip temp files
#         if file.startswith("~$"):

#             continue

#         # Only csv/excel
#         if not file.lower().endswith(

#             (".csv", ".xlsx", ".xls")
#         ):

#             continue

#         file_path = os.path.join(

#             month_folder,

#             file
#         )

#         try:

#             # ─────────────────────────────────────────────────────────────
#             # READ FILE
#             # ─────────────────────────────────────────────────────────────

#             if file.lower().endswith(".csv"):

#                 df = pd.read_csv(file_path)

#             else:

#                 df = pd.read_excel(file_path)

#             # ─────────────────────────────────────────────────────────────
#             # CLEAN COLUMNS
#             # ─────────────────────────────────────────────────────────────

#             df.columns = (

#                 df.columns

#                 .astype(str)

#                 .str.strip()
#             )

#             # Source Tracking
#             df["source_file"] = file

#             df["source_month"] = month

#             noon_rev_parts.append(df)

#             print(f"✅ Noon Revenue Loaded: {file}")

#         except Exception as e:

#             print(f"❌ Error Reading: {file}")

#             print("Reason:", e)

# # ══════════════════════════════════════════════════════════════════════════
# # COMBINE ALL MONTHS
# # ══════════════════════════════════════════════════════════════════════════

# if noon_rev_parts:

#     noon_rev_df = pd.concat(

#         noon_rev_parts,

#         ignore_index=True
#     )

# else:

#     noon_rev_df = pd.DataFrame()

#     print("❌ No Noon Revenue Data Found")

# # ══════════════════════════════════════════════════════════════════════════
# # CONTINUE ONLY IF DATA EXISTS
# # ══════════════════════════════════════════════════════════════════════════

# if not noon_rev_df.empty:

#     # ══════════════════════════════════════════════════════════════════════
#     # CLEAN COLUMN NAMES
#     # ══════════════════════════════════════════════════════════════════════

#     noon_rev_df.columns = (

#         noon_rev_df.columns

#         .astype(str)

#         .str.strip()
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # RENAME COLUMNS
#     # ══════════════════════════════════════════════════════════════════════

#     # ── DROP nsku/NSKU BEFORE rename ─────────────────────────────────────
#     # BUG FIX: Sales CSVs have BOTH 'nsku' (col 7) and 'ZSKU' (col 8).
#     # Old rename_map mapped both to 'ASIN'. Since nsku comes first,
#     # drop_duplicates kept nsku values (e.g. N28940734A) → WRONG ID.
#     # Correct ID for Noon is always ZSKU (e.g. Z1730364D9AC286FC839CZ-1).
#     # Fix: drop nsku/NSKU columns BEFORE renaming so only ZSKU → ASIN.
#     for _drop_col in ["nsku", "NSKU", "Nsku"]:
#         if _drop_col in noon_rev_df.columns:
#             noon_rev_df.drop(columns=[_drop_col], inplace=True)
#             print(f"  ✅ Dropped '{_drop_col}' — using ZSKU as ASIN only")

#     rename_map = {

#         "ZSKU" : "ASIN",

#         "zsku" : "ASIN",

#         "Sku"  : "ASIN",

#         "SKU"  : "ASIN",

#         "Qty" : "total_units",

#         "qty" : "total_units",

#         "Quantity" : "total_units",

#         "Revenue" : "total_sales",

#         "Sales" : "total_sales",

#         "Revenue (AED)" : "total_sales"
#     }

#     noon_rev_df.rename(

#         columns={

#             k: v

#             for k, v in rename_map.items()

#             if k in noon_rev_df.columns
#         },

#         inplace=True
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # REMOVE DUPLICATE COLUMNS
#     # ══════════════════════════════════════════════════════════════════════

#     noon_rev_df = noon_rev_df.loc[
#         :,
#         ~noon_rev_df.columns.duplicated()
#     ]

#     print("\n✅ Columns After Rename:")
#     print(noon_rev_df.columns.tolist())

#     # ══════════════════════════════════════════════════════════════════════
#     # VALIDATION
#     # ══════════════════════════════════════════════════════════════════════

#     required_cols = [

#         "ASIN",

#         "Date",

#         "total_sales",

#         "total_units"
#     ]

#     missing_cols = [

#         c for c in required_cols

#         if c not in noon_rev_df.columns
#     ]

#     if missing_cols:

#         print("\n❌ Missing Columns:")
#         print(missing_cols)

#     # ══════════════════════════════════════════════════════════════════════
#     # CLEAN ASIN
#     # ══════════════════════════════════════════════════════════════════════

#     noon_rev_df["ASIN"] = (

#         noon_rev_df["ASIN"]

#         .astype(str)

#         .str.upper()

#         .str.strip()

#         # .str.replace("-1", "", regex=False)

#         .str.replace(".0", "", regex=False)
#     )

#     # Remove blank ASIN
#     noon_rev_df = noon_rev_df[
#         noon_rev_df["ASIN"].notna()
#     ]

#     noon_rev_df = noon_rev_df[
#         noon_rev_df["ASIN"] != ""
#     ]

#     # ══════════════════════════════════════════════════════════════════════
#     # DATE PARSER
#     # ══════════════════════════════════════════════════════════════════════

#     def parse_mixed_dates(x):
#         """
#         Parse date strings safely for UAE data (DD-first convention).
#         %m/%d/%Y removed to prevent 01/06/2026 → Jan-6 misparse.
#         Dates outside 2020-2030 are rejected as invalid year artifacts.
#         """
#         if pd.isna(x):
#             return pd.NaT
#         x = str(x).strip()
#         formats = [
#             "%d-%b-%y",    # 01-Jun-26
#             "%d-%b-%Y",    # 01-Jun-2026
#             "%d/%m/%Y",    # 01/06/2026  (DD first — UAE standard)
#             "%Y-%m-%d",    # 2026-06-01  (ISO)
#             "%d-%m-%Y",    # 01-06-2026
#             "%Y/%m/%d",    # 2026/06/01
#             "%d.%m.%Y",    # 01.06.2026
#             "%d %b %Y",    # 01 Jun 2026
#             "%b %d %Y",    # Jun 01 2026
#         ]
#         for fmt in formats:
#             try:
#                 result = pd.to_datetime(x, format=fmt)
#                 if 2020 <= result.year <= 2030:
#                     return result
#             except Exception:
#                 continue
#         try:
#             result = pd.to_datetime(x, dayfirst=True, errors="coerce")
#             if pd.notna(result) and 2020 <= result.year <= 2030:
#                 return result
#         except Exception:
#             pass
#         return pd.NaT
#     # ══════════════════════════════════════════════════════════════════════
#     # CLEAN DATE
#     # ══════════════════════════════════════════════════════════════════════

#     noon_rev_df["Date"] = (

#         noon_rev_df["Date"]

#         .apply(parse_mixed_dates)

#         .dt.normalize()
#     )

#     # Remove invalid dates
#     noon_rev_df = noon_rev_df[
#         noon_rev_df["Date"].notna()
#     ]

#     # ══════════════════════════════════════════════════════════════════════
#     # CLEAN SALES
#     # ══════════════════════════════════════════════════════════════════════

#     noon_rev_df["total_sales"] = (

#         noon_rev_df["total_sales"]

#         .astype(str)

#         .str.replace(",", "", regex=False)

#         .str.replace("AED", "", regex=False)

#         .str.replace("aed", "", regex=False)

#         .str.strip()

#         .pipe(pd.to_numeric, errors="coerce")

#         .fillna(0)
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # CLEAN UNITS
#     # ══════════════════════════════════════════════════════════════════════

#     noon_rev_df["total_units"] = (

#         pd.to_numeric(

#             noon_rev_df["total_units"],

#             errors="coerce"
#         )

#         .fillna(0)

#         .astype(int)
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # REMOVE DUPLICATES
#     # ══════════════════════════════════════════════════════════════════════

#     # NOTE: 4-column dedup removed — it incorrectly drops rows where the same
#     # product genuinely sold the same units on two different days (rare but real).
#     # The .groupby().agg(sum) below handles true duplicates correctly.

#     # ══════════════════════════════════════════════════════════════════════
#     # AGGREGATE
#     # ══════════════════════════════════════════════════════════════════════

#     noon_rev_df = (

#         noon_rev_df

#         .groupby(

#             ["ASIN", "Date"],

#             as_index=False
#         )

#         .agg({

#             "total_sales" : "sum",

#             "total_units" : "sum"
#         })
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # SALES CHANNEL
#     # ══════════════════════════════════════════════════════════════════════

#     noon_rev_df["Sales Channel"] = "Noon"

#     # ══════════════════════════════════════════════════════════════════════
#     # SORT
#     # ══════════════════════════════════════════════════════════════════════

#     noon_rev_df = noon_rev_df.sort_values(

#         ["ASIN", "Date"]
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # LOAD HELPER FILE
#     # ══════════════════════════════════════════════════════════════════════

#     print("\n📦 Loading Helper File...")

#     helper_df = pd.read_excel(

#         HELPER_FILE,

#         header=1
#     )

#     helper_df.columns = (

#         helper_df.columns

#         .astype(str)

#         .str.strip()
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # FORCE NOON CORE COLUMN
#     # ══════════════════════════════════════════════════════════════════════

#     helper_key = None

#     for col in helper_df.columns:

#         clean_col = (

#             str(col)

#             .replace("\n", " ")

#             .strip()

#             .lower()
#         )

#         if "noon core" in clean_col:

#             helper_key = col

#             break

#     print("\n✅ Noon Helper Key:")
#     print(helper_key)

#     # ══════════════════════════════════════════════════════════════════════
#     # RENAME KEY
#     # ══════════════════════════════════════════════════════════════════════

#     helper_df.rename(

#         columns={
#             helper_key: "ASIN"
#         },

#         inplace=True
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # CLEAN HELPER ASIN
#     # ══════════════════════════════════════════════════════════════════════

#     helper_df["ASIN"] = (

#         helper_df["ASIN"]

#         .astype(str)

#         .str.upper()

#         .str.strip()

#         # .str.replace("-1", "", regex=False)

#         .str.replace(".0", "", regex=False)
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # RENAME HELPER METADATA
#     # ══════════════════════════════════════════════════════════════════════

#     helper_df.rename(columns={

#         "Product Description(Cleaned)" : "Product Name",

#         "Category" : "Product Category",

#         "Product Type" : "Product Sub Category",

#         "Size" : "Pack Size"

#     }, inplace=True)

#     # ══════════════════════════════════════════════════════════════════════
#     # KEEP REQUIRED HELPER COLUMNS
#     # ══════════════════════════════════════════════════════════════════════

#     helper_cols = [

#         "ASIN",

#         "Brand",

#         "Product Name",

#         "Product Category",

#         "Product Sub Category",

#         "Pack Type",

#         "Pack Size",

#         "VC Ideal Price"
#     ]

#     helper_cols = [

#         c for c in helper_cols

#         if c in helper_df.columns
#     ]

#     helper_df = helper_df[
#         helper_cols
#     ]

#     helper_df = helper_df.drop_duplicates(
#         subset=["ASIN"]
#     )

#     print("\n✅ Helper Shape:")
#     print(helper_df.shape)

#     # ══════════════════════════════════════════════════════════════════════
#     # MERGE HELPER
#     # ══════════════════════════════════════════════════════════════════════

#     noon_rev_df = noon_rev_df.merge(

#         helper_df,

#         on="ASIN",

#         how="left"
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # FILL BLANKS
#     # ══════════════════════════════════════════════════════════════════════

#     metadata_cols = [

#         "Brand",

#         "Product Name",

#         "Product Category",

#         "Product Sub Category",

#         "Pack Type",

#         "Pack Size"
#     ]

#     for col in metadata_cols:

#         if col in noon_rev_df.columns:

#             noon_rev_df[col] = (

#                 noon_rev_df[col]

#                 .fillna("Unknown")
#             )

# # ══════════════════════════════════════════════════════════════════════════
# # FINAL VALIDATION
# # ══════════════════════════════════════════════════════════════════════════

# print("\n✅ NOON REVENUE READY")

# print("\n📊 Shape:")
# print(noon_rev_df.shape)

# if not noon_rev_df.empty:

#     print("\n📅 Available Months:")

#     print(
#         sorted(
#             noon_rev_df["Date"]
#             .dt.strftime("%Y-%m")
#             .unique()
#         )
#     )

#     print("\n💰 Total Sales:")

#     print(
#         round(
#             noon_rev_df["total_sales"].sum(),
#             2
#         )
#     )

#     print("\n📦 Total Units:")

#     print(
#         noon_rev_df["total_units"].sum()
#     )

#     print("\n📦 Brand Fill Check:")

#     print(
#         noon_rev_df["Brand"]
#         .value_counts(dropna=False)
#         .head(10)
#     )

# display(
#     noon_rev_df.head(50)
# )

In [18]:
# ══════════════════════════════════════════════════════════════════════════
# NOON REVENUE MASTER (FINAL FULL FIXED VERSION)
# REVENUE = PRIORITY BASE
# HELPER MERGE FIXED
# DUPLICATE ASIN FIXED
# APR + MAY DATA FIXED
# ══════════════════════════════════════════════════════════════════════════

noon_rev_parts = []

# ══════════════════════════════════════════════════════════════════════════
# LOAD MONTHLY FILES
# FIX: STANDARDIZE SOURCE COLUMN NAMES BEFORE CONCAT
#
# July can have ZSKU while August can have zsku.
# If we concatenate first, Pandas creates both columns. Later renaming both
# to ASIN creates duplicate ASIN columns, and the populated August column
# can be discarded. Therefore, normalize every monthly file BEFORE concat.
# ══════════════════════════════════════════════════════════════════════════

noon_rev_parts = []

for month in MONTHS:

    month_folder = os.path.join(
        NOON_REV_ROOT,
        month
    )

    if not os.path.isdir(month_folder):

        print(f"⚠️ Noon Revenue Folder Missing: {month}")
        continue

    # ─────────────────────────────────────────────────────────────────────
    # READ FILES
    # ─────────────────────────────────────────────────────────────────────

    for file in sorted(os.listdir(month_folder)):

        # Skip temp files
        if file.startswith("~$"):
            continue

        # Only csv/excel
        if not file.lower().endswith(
            (".csv", ".xlsx", ".xls")
        ):
            continue

        file_path = os.path.join(
            month_folder,
            file
        )

        try:

            # ─────────────────────────────────────────────────────────────
            # READ FILE
            # ─────────────────────────────────────────────────────────────

            if file.lower().endswith(".csv"):
                df = pd.read_csv(file_path)
            else:
                df = pd.read_excel(file_path)

            # ─────────────────────────────────────────────────────────────
            # CLEAN COLUMN NAMES
            # ─────────────────────────────────────────────────────────────

            df.columns = (
                df.columns
                .astype(str)
                .str.strip()
            )

            # ─────────────────────────────────────────────────────────────
            # STANDARDIZE SOURCE COLUMN NAMES
            # ─────────────────────────────────────────────────────────────
            # ZSKU / zsku / Zsku -> ZSKU
            # nsku / NSKU       -> nsku
            # Qty / qty         -> Qty
            # Quantity          -> Qty
            # Revenue / Sales   -> Revenue
            # ─────────────────────────────────────────────────────────────

            normalized_columns = {}

            for col in df.columns:

                clean_col = (
                    str(col)
                    .strip()
                    .lower()
                    .replace("_", " ")
                )

                if clean_col == "zsku":
                    normalized_columns[col] = "ZSKU"

                elif clean_col == "nsku":
                    normalized_columns[col] = "nsku"

                elif clean_col in {"qty", "quantity"}:
                    normalized_columns[col] = "Qty"

                elif clean_col in {
                    "revenue",
                    "sales",
                    "revenue (aed)"
                }:
                    normalized_columns[col] = "Revenue"

                elif clean_col == "date":
                    normalized_columns[col] = "Date"

            df.rename(
                columns=normalized_columns,
                inplace=True
            )

            # ─────────────────────────────────────────────────────────────
            # SOURCE TRACKING
            # ─────────────────────────────────────────────────────────────

            df["source_file"] = file
            df["source_month"] = month

            # ─────────────────────────────────────────────────────────────
            # VALIDATE ZSKU
            # ─────────────────────────────────────────────────────────────

            if "ZSKU" not in df.columns:

                print(
                    f"⚠️ {month.upper()} | {file} | "
                    f"ZSKU column NOT FOUND"
                )

                print(
                    "   Available columns:",
                    df.columns.tolist()
                )

            else:

                zsku_count = (
                    df["ZSKU"]
                    .notna()
                    .sum()
                )

                print(
                    f"✅ {month.upper()} | {file} | "
                    f"Rows: {len(df):,} | "
                    f"ZSKU populated: {zsku_count:,}"
                )

            noon_rev_parts.append(df)

        except Exception as e:

            print(f"❌ Error Reading: {file}")
            print("Reason:", e)


# ══════════════════════════════════════════════════════════════════════════
# COMBINE ALL MONTHS
# ══════════════════════════════════════════════════════════════════════════

if noon_rev_parts:

    noon_rev_df = pd.concat(
        noon_rev_parts,
        ignore_index=True
    )

else:

    noon_rev_df = pd.DataFrame()

    print("❌ No Noon Revenue Data Found")

# ══════════════════════════════════════════════════════════════════════════
# CONTINUE ONLY IF DATA EXISTS
# ══════════════════════════════════════════════════════════════════════════

if not noon_rev_df.empty:

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN COLUMN NAMES
    # ══════════════════════════════════════════════════════════════════════

    noon_rev_df.columns = (

        noon_rev_df.columns

        .astype(str)

        .str.strip()
    )

    # ══════════════════════════════════════════════════════════════════════
    # RENAME COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    # ─────────────────────────────────────────────────────────────────────
    # DROP nsku BEFORE rename
    # ─────────────────────────────────────────────────────────────────────
    # nsku is NOT the Noon ASIN.
    # ZSKU is the correct Noon product identifier.
    # ZSKU/zsku was already standardized before concat.
    # ─────────────────────────────────────────────────────────────────────

    for _drop_col in ["nsku", "NSKU", "Nsku"]:

        if _drop_col in noon_rev_df.columns:

            noon_rev_df.drop(
                columns=[_drop_col],
                inplace=True
            )

            print(
                f"  ✅ Dropped '{_drop_col}' — "
                f"using ZSKU as ASIN only"
            )

    # ─────────────────────────────────────────────────────────────────────
    # RENAME COLUMNS
    # ─────────────────────────────────────────────────────────────────────

    rename_map = {

        "ZSKU": "ASIN",

        "Sku": "ASIN",

        "SKU": "ASIN",

        "Qty": "total_units",

        "Quantity": "total_units",

        "Revenue": "total_sales",

        "Sales": "total_sales",

        "Revenue (AED)": "total_sales"
    }

    noon_rev_df.rename(
        columns={
            k: v
            for k, v in rename_map.items()
            if k in noon_rev_df.columns
        },
        inplace=True
    )

    # ─────────────────────────────────────────────────────────────────────
    # REMOVE DUPLICATE COLUMNS SAFELY
    # ─────────────────────────────────────────────────────────────────────

    duplicate_columns = (
        noon_rev_df.columns[
            noon_rev_df.columns.duplicated()
        ]
        .tolist()
    )

    if duplicate_columns:

        print(
            f"⚠️ Duplicate columns found after rename: "
            f"{duplicate_columns}"
        )

        noon_rev_df = noon_rev_df.loc[
            :,
            ~noon_rev_df.columns.duplicated()
        ]

    print("\n✅ Columns After Rename:")
    print(noon_rev_df.columns.tolist())

    # ══════════════════════════════════════════════════════════════════════
    # VALIDATION
    # ══════════════════════════════════════════════════════════════════════

    required_cols = [

        "ASIN",

        "Date",

        "total_sales",

        "total_units"
    ]

    missing_cols = [

        c for c in required_cols

        if c not in noon_rev_df.columns
    ]

    if missing_cols:

        print("\n❌ Missing Columns:")
        print(missing_cols)

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN ASIN
    # ══════════════════════════════════════════════════════════════════════

    noon_rev_df["ASIN"] = (
        noon_rev_df["ASIN"]
        .astype("string")
        .str.upper()
        .str.strip()
        # Keep -1 because the helper uses the same Noon Core key format.
        .str.replace(".0", "", regex=False)
        .replace({
            "NAN": pd.NA,
            "NONE": pd.NA,
            "NULL": pd.NA,
            "": pd.NA
        })
    )

    # ASIN diagnostic by source month
    if "source_month" in noon_rev_df.columns:

        print("📌 ASIN CHECK BY MONTH:")

        asin_check = (
            noon_rev_df
            .groupby("source_month", dropna=False)
            .agg(
                rows=("ASIN", "size"),
                asin_present=("ASIN", lambda x: x.notna().sum()),
                asin_missing=("ASIN", lambda x: x.isna().sum())
            )
            .reset_index()
        )

        print(asin_check.to_string(index=False))

    # Remove blank ASIN
    before_asin_filter = len(noon_rev_df)

    noon_rev_df = noon_rev_df[
        noon_rev_df["ASIN"].notna()
    ].copy()

    removed_asin_rows = (
        before_asin_filter - len(noon_rev_df)
    )

    if removed_asin_rows:
        print(
            f"⚠️ Removed {removed_asin_rows:,} rows "
            f"with missing ASIN"
        )

    # ══════════════════════════════════════════════════════════════════════
    # DATE PARSER
    # ══════════════════════════════════════════════════════════════════════

    def parse_mixed_dates(x):
        """
        Parse date strings safely for UAE data (DD-first convention).
        %m/%d/%Y removed to prevent 01/06/2026 → Jan-6 misparse.
        Dates outside 2020-2030 are rejected as invalid year artifacts.
        """
        if pd.isna(x):
            return pd.NaT
        x = str(x).strip()
        formats = [
            "%d-%b-%y",    # 01-Jun-26
            "%d-%b-%Y",    # 01-Jun-2026
            "%d/%m/%Y",    # 01/06/2026  (DD first — UAE standard)
            "%Y-%m-%d",    # 2026-06-01  (ISO)
            "%d-%m-%Y",    # 01-06-2026
            "%Y/%m/%d",    # 2026/06/01
            "%d.%m.%Y",    # 01.06.2026
            "%d %b %Y",    # 01 Jun 2026
            "%b %d %Y",    # Jun 01 2026
        ]
        for fmt in formats:
            try:
                result = pd.to_datetime(x, format=fmt)
                if 2020 <= result.year <= 2030:
                    return result
            except Exception:
                continue
        try:
            result = pd.to_datetime(x, dayfirst=True, errors="coerce")
            if pd.notna(result) and 2020 <= result.year <= 2030:
                return result
        except Exception:
            pass
        return pd.NaT
    # ══════════════════════════════════════════════════════════════════════
    # CLEAN DATE
    # ══════════════════════════════════════════════════════════════════════

    noon_rev_df["Date"] = (

        noon_rev_df["Date"]

        .apply(parse_mixed_dates)

        .dt.normalize()
    )

    # Remove invalid dates
    noon_rev_df = noon_rev_df[
        noon_rev_df["Date"].notna()
    ]

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN SALES
    # ══════════════════════════════════════════════════════════════════════

    noon_rev_df["total_sales"] = (

        noon_rev_df["total_sales"]

        .astype(str)

        .str.replace(",", "", regex=False)

        .str.replace("AED", "", regex=False)

        .str.replace("aed", "", regex=False)

        .str.strip()

        .pipe(pd.to_numeric, errors="coerce")

        .fillna(0)
    )

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN UNITS
    # ══════════════════════════════════════════════════════════════════════

    noon_rev_df["total_units"] = (

        pd.to_numeric(

            noon_rev_df["total_units"],

            errors="coerce"
        )

        .fillna(0)

        .astype(int)
    )

    # ══════════════════════════════════════════════════════════════════════
    # REMOVE DUPLICATES
    # ══════════════════════════════════════════════════════════════════════

    # NOTE: 4-column dedup removed — it incorrectly drops rows where the same
    # product genuinely sold the same units on two different days (rare but real).
    # The .groupby().agg(sum) below handles true duplicates correctly.

    # ══════════════════════════════════════════════════════════════════════
    # AGGREGATE
    # ══════════════════════════════════════════════════════════════════════

    noon_rev_df = (

        noon_rev_df

        .groupby(

            ["ASIN", "Date"],

            as_index=False
        )

        .agg({

            "total_sales" : "sum",

            "total_units" : "sum"
        })
    )

    # ══════════════════════════════════════════════════════════════════════
    # SALES CHANNEL
    # ══════════════════════════════════════════════════════════════════════

    noon_rev_df["Sales Channel"] = "Noon"

    # ══════════════════════════════════════════════════════════════════════
    # SORT
    # ══════════════════════════════════════════════════════════════════════

    noon_rev_df = noon_rev_df.sort_values(

        ["ASIN", "Date"]
    )

    # ══════════════════════════════════════════════════════════════════════
    # LOAD HELPER FILE
    # ══════════════════════════════════════════════════════════════════════

    print("\n📦 Loading Helper File...")

    helper_df = pd.read_excel(

        HELPER_FILE,

        header=1
    )

    helper_df.columns = (

        helper_df.columns

        .astype(str)

        .str.strip()
    )

    # ══════════════════════════════════════════════════════════════════════
    # FORCE NOON CORE COLUMN
    # ══════════════════════════════════════════════════════════════════════

    helper_key = None

    for col in helper_df.columns:

        clean_col = (

            str(col)

            .replace("\n", " ")

            .strip()

            .lower()
        )

        if "noon core" in clean_col:

            helper_key = col

            break

    print("\n✅ Noon Helper Key:")
    print(helper_key)

    # ══════════════════════════════════════════════════════════════════════
    # RENAME KEY
    # ══════════════════════════════════════════════════════════════════════

    helper_df.rename(

        columns={
            helper_key: "ASIN"
        },

        inplace=True
    )

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN HELPER ASIN
    # ══════════════════════════════════════════════════════════════════════

    helper_df["ASIN"] = (

        helper_df["ASIN"]

        .astype(str)

        .str.upper()

        .str.strip()

        # .str.replace("-1", "", regex=False)

        .str.replace(".0", "", regex=False)
    )

    # ══════════════════════════════════════════════════════════════════════
    # RENAME HELPER METADATA
    # ══════════════════════════════════════════════════════════════════════

    helper_df.rename(columns={

        "Product Description(Cleaned)" : "Product Name",

        "Category" : "Product Category",

        "Product Type" : "Product Sub Category",

        "Size" : "Pack Size"

    }, inplace=True)

    # ══════════════════════════════════════════════════════════════════════
    # KEEP REQUIRED HELPER COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    helper_cols = [

        "ASIN",

        "Brand",

        "Product Name",

        "Product Category",

        "Product Sub Category",

        "Pack Type",

        "Pack Size",

        "VC Ideal Price"
    ]

    helper_cols = [

        c for c in helper_cols

        if c in helper_df.columns
    ]

    helper_df = helper_df[
        helper_cols
    ]

    helper_df = helper_df.drop_duplicates(
        subset=["ASIN"]
    )

    print("\n✅ Helper Shape:")
    print(helper_df.shape)

    # ══════════════════════════════════════════════════════════════════════
    # MERGE HELPER
    # ══════════════════════════════════════════════════════════════════════

    noon_rev_df = noon_rev_df.merge(

        helper_df,

        on="ASIN",

        how="left"
    )

    # ══════════════════════════════════════════════════════════════════════
    # FILL BLANKS
    # ══════════════════════════════════════════════════════════════════════

    metadata_cols = [

        "Brand",

        "Product Name",

        "Product Category",

        "Product Sub Category",

        "Pack Type",

        "Pack Size"
    ]

    for col in metadata_cols:

        if col in noon_rev_df.columns:

            noon_rev_df[col] = (

                noon_rev_df[col]

                .fillna("Unknown")
            )

# ══════════════════════════════════════════════════════════════════════════
# FINAL VALIDATION
# ══════════════════════════════════════════════════════════════════════════

print("\n✅ NOON REVENUE READY")

print("\n📊 Shape:")
print(noon_rev_df.shape)

if not noon_rev_df.empty:

    print("\n📅 Available Months:")

    print(
        sorted(
            noon_rev_df["Date"]
            .dt.strftime("%Y-%m")
            .unique()
        )
    )

    print("\n💰 Total Sales:")

    print(
        round(
            noon_rev_df["total_sales"].sum(),
            2
        )
    )

    print("\n📦 Total Units:")

    print(
        noon_rev_df["total_units"].sum()
    )

    print("\n📦 Brand Fill Check:")

    print(
        noon_rev_df["Brand"]
        .value_counts(dropna=False)
        .head(10)
    )

# ══════════════════════════════════════════════════════════════════════════
# AUGUST 2026 VALIDATION
# ══════════════════════════════════════════════════════════════════════════

if not noon_rev_df.empty:

    august_df = noon_rev_df[
        noon_rev_df["Date"].dt.year.eq(2026)
        & noon_rev_df["Date"].dt.month.eq(8)
    ].copy()

    print("\n📌 AUGUST 2026 CHECK:")

    if august_df.empty:

        print("❌ No August 2026 data found.")

    else:

        print(f"   August rows       : {len(august_df):,}")
        print(
            f"   August ASINs      : "
            f"{august_df['ASIN'].nunique():,}"
        )
        print(
            f"   August ASIN blank : "
            f"{august_df['ASIN'].isna().sum():,}"
        )
        print(
            f"   August sales      : "
            f"{august_df['total_sales'].sum():,.2f}"
        )
        print(
            f"   August units      : "
            f"{august_df['total_units'].sum():,}"
        )

        print("\n   August sample:")

        print(
            august_df[
                [
                    "ASIN",
                    "Date",
                    "total_sales",
                    "total_units",
                    "Sales Channel",
                    "Brand",
                    "Product Name"
                ]
            ]
            .head(20)
            .to_string(index=False)
        )


display(
    noon_rev_df.head(50)
)

✅ MAY | Noon_sales_may_2026.csv | Rows: 318 | ZSKU populated: 318
✅ JUN | Noon_sales_jun_2026.csv | Rows: 484 | ZSKU populated: 484
✅ JULY | Noon_sales_july_2026.csv | Rows: 569 | ZSKU populated: 569
✅ AUG | Noon_sales_aug_2026.csv | Rows: 383 | ZSKU populated: 383
  ✅ Dropped 'nsku' — using ZSKU as ASIN only

✅ Columns After Rename:
['Date', 'Year_Month', 'Country', 'platform', 'Warehouse', 'Warehouse Cod', 'Category', 'ASIN', 'Barcode', 'Title', 'Brand', 'Type', 'Sub-Type', 'total_units', 'total_sales', 'source_file', 'source_month']
📌 ASIN CHECK BY MONTH:
source_month  rows  asin_present  asin_missing
         aug   383           383             0
        july   569           569             0
         jun   484           484             0
         may   318           318             0

📦 Loading Helper File...

✅ Noon Helper Key:
Product Code
(Noon Core)

✅ Helper Shape:
(99, 8)

✅ NOON REVENUE READY

📊 Shape:
(1461, 12)

📅 Available Months:
['2026-05', '2026-06', '2026-07', '2026-

,ASIN,Date,total_sales,total_units,Sales Channel,Brand,Product Name,Product Category,Product Sub Category,Pack Type,Pack Size,VC Ideal Price
0,Z00EA9B233750B962B4BEZ-1,2026-05-16,14.45,1,Noon,Emami 7 Oils in One,Emami 7 in 1 Damage Control Hair Oil,Hair Care,Hair Oil,1,200ml,15.797619
1,Z00EA9B233750B962B4BEZ-1,2026-05-21,28.90,2,Noon,Emami 7 Oils in One,Emami 7 in 1 Damage Control Hair Oil,Hair Care,Hair Oil,1,200ml,15.797619
2,Z00EA9B233750B962B4BEZ-1,2026-05-30,14.45,1,Noon,Emami 7 Oils in One,Emami 7 in 1 Damage Control Hair Oil,Hair Care,Hair Oil,1,200ml,15.797619
3,Z00EA9B233750B962B4BEZ-1,2026-05-31,28.90,2,Noon,Emami 7 Oils in One,Emami 7 in 1 Damage Control Hair Oil,Hair Care,Hair Oil,1,200ml,15.797619
4,Z00EA9B233750B962B4BEZ-1,2026-06-01,14.20,1,Noon,Emami 7 Oils in One,Emami 7 in 1 Damage Control Hair Oil,Hair Care,Hair Oil,1,200ml,15.797619
5,Z00EA9B233750B962B4BEZ-1,2026-06-03,13.85,1,Noon,Emami 7 Oils in One,Emami 7 in 1 Damage Control Hair Oil,Hair Care,Hair Oil,1,200ml,15.797619
6,Z00EA9B233750B962B4BEZ-1,2026-06-06,16.70,1,Noon,Emami 7 Oils in One,Emami 7 in 1 Damage Control Hair Oil,Hair Care,Hair Oil,1,200ml,15.797619
7,Z00EA9B233750B962B4BEZ-1,2026-06-12,16.70,1,Noon,Emami 7 Oils in One,Emami 7 in 1 Damage Control Hair Oil,Hair Care,Hair Oil,1,200ml,15.797619
8,Z00EA9B233750B962B4BEZ-1,2026-06-16,16.70,1,Noon,Emami 7 Oils in One,Emami 7 in 1 Damage Control Hair Oil,Hair Care,Hair Oil,1,200ml,15.797619
9,Z00EA9B233750B962B4BEZ-1,2026-06-21,31.80,2,Noon,Emami 7 Oils in One,Emami 7 in 1 Damage Control Hair Oil,Hair Care,Hair Oil,1,200ml,15.797619


In [19]:
noon_rev_df.to_csv("noon_rev.csv", index=False)

## 9 · Noon Minutes – Revenue (`nms_rev_df`)

In [20]:
# # ══════════════════════════════════════════════════════════════════════════
# # NOON MINUTES REVENUE MASTER (FINAL HELPER FIXED VERSION)
# # PRODUCT CODE (NOON MINUTES) MERGE FIXED
# # BRAND / CATEGORY / PACK TYPE FIXED
# # ══════════════════════════════════════════════════════════════════════════

# nms_rev_parts = []

# # ══════════════════════════════════════════════════════════════════════════
# # LOAD MONTHLY FILES
# # ══════════════════════════════════════════════════════════════════════════

# for month in MONTHS:

#     month_folder = os.path.join(

#         NMS_REV_ROOT,

#         month
#     )

#     if not os.path.isdir(month_folder):

#         print(f"⚠️ NMS Revenue Folder Missing: {month}")

#         continue

#     # ─────────────────────────────────────────────────────────────────────
#     # READ FILES
#     # ─────────────────────────────────────────────────────────────────────

#     for file in sorted(os.listdir(month_folder)):

#         # Skip temp files
#         if file.startswith("~$"):

#             continue

#         # Only csv/excel
#         if not file.lower().endswith(

#             (".csv", ".xlsx", ".xls")
#         ):

#             continue

#         file_path = os.path.join(

#             month_folder,

#             file
#         )

#         try:

#             # ─────────────────────────────────────────────────────────────
#             # READ FILE
#             # ─────────────────────────────────────────────────────────────

#             if file.lower().endswith(".csv"):

#                 df = pd.read_csv(file_path)

#             else:

#                 df = pd.read_excel(file_path)

#             # ─────────────────────────────────────────────────────────────
#             # CLEAN COLUMNS
#             # ─────────────────────────────────────────────────────────────

#             df.columns = (

#                 df.columns

#                 .astype(str)

#                 .str.strip()
#             )

#             # Source Tracking
#             df["source_file"] = file

#             df["source_month"] = month

#             nms_rev_parts.append(df)

#             print(f"✅ NMS Revenue Loaded: {file}")

#         except Exception as e:

#             print(f"❌ Error Reading: {file}")

#             print("Reason:", e)

# # ══════════════════════════════════════════════════════════════════════════
# # COMBINE ALL MONTHS
# # ══════════════════════════════════════════════════════════════════════════

# if nms_rev_parts:

#     nms_rev_df = pd.concat(

#         nms_rev_parts,

#         ignore_index=True
#     )

# else:

#     nms_rev_df = pd.DataFrame()

#     print("❌ No Noon Minutes Revenue Data Found")

# # ══════════════════════════════════════════════════════════════════════════
# # CONTINUE ONLY IF DATA EXISTS
# # ══════════════════════════════════════════════════════════════════════════

# if not nms_rev_df.empty:

#     # ══════════════════════════════════════════════════════════════════════
#     # CLEAN COLUMN NAMES
#     # ══════════════════════════════════════════════════════════════════════

#     nms_rev_df.columns = (

#         nms_rev_df.columns

#         .astype(str)

#         .str.strip()
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # RENAME COLUMNS
#     # ══════════════════════════════════════════════════════════════════════

#         # ══════════════════════════════════════════════════════════════════════
#     # USE ONLY ZSKU AS PRODUCT CODE
#     # ══════════════════════════════════════════════════════════════════════

#     if "ZSKU" not in nms_rev_df.columns and "zsku" not in nms_rev_df.columns:

#         raise ValueError(
#             "ZSKU column not found in Noon Minutes Revenue files"
#         )

#     # ── DROP nsku/NSKU BEFORE rename ─────────────────────────────────────
#     # Same bug fix as Cell 32: drop nsku so only ZSKU → ASIN
#     for _drop_col in ["nsku", "NSKU", "Nsku"]:
#         if _drop_col in nms_rev_df.columns:
#             nms_rev_df.drop(columns=[_drop_col], inplace=True)
#             print(f"  ✅ Dropped '{_drop_col}' — using ZSKU as ASIN only")

#     rename_map = {

#         "ZSKU" : "ASIN",

#         "zsku" : "ASIN",

#         "Qty" : "total_units",

#         "qty" : "total_units",

#         "Quantity" : "total_units",

#         "Revenue" : "total_sales",

#         "Sales" : "total_sales",

#         "Revenue (AED)" : "total_sales"
#     }

#     nms_rev_df.rename(

#         columns={
#             k: v
#             for k, v in rename_map.items()
#             if k in nms_rev_df.columns
#         },

#         inplace=True
#     )

#     # Keep only required columns
#     required_cols = [

#         "ASIN",

#         "Date",

#         "total_sales",

#         "total_units"
#     ]

#     available_cols = [
#         c for c in required_cols
#         if c in nms_rev_df.columns
#     ]

#     nms_rev_df = nms_rev_df[available_cols]
#     nms_rev_df.rename(

#         columns={

#             k: v

#             for k, v in rename_map.items()

#             if k in nms_rev_df.columns
#         },

#         inplace=True
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # REMOVE DUPLICATE COLUMNS
#     # ══════════════════════════════════════════════════════════════════════

#     nms_rev_df = nms_rev_df.loc[
#         :,
#         ~nms_rev_df.columns.duplicated()
#     ]

#     print("\n✅ Columns After Rename:")
#     print(nms_rev_df.columns.tolist())

#     # ══════════════════════════════════════════════════════════════════════
#     # CLEAN ASIN
#     # ══════════════════════════════════════════════════════════════════════

#     nms_rev_df["ASIN"] = (

#         nms_rev_df["ASIN"]

#         .astype(str)

#         .str.upper()

#         .str.strip()

#         # .str.replace("-1", "", regex=False)

#         .str.replace(".0", "", regex=False)
#     )

#     # Remove blank ASIN
#     nms_rev_df = nms_rev_df[
#         nms_rev_df["ASIN"].notna()
#     ]

#     nms_rev_df = nms_rev_df[
#         nms_rev_df["ASIN"] != ""
#     ]

#     # ══════════════════════════════════════════════════════════════════════
#     # DATE PARSER
#     # ══════════════════════════════════════════════════════════════════════

#     def parse_mixed_dates(x):
#         """
#         Parse date strings safely for UAE data (DD-first convention).
#         %m/%d/%Y removed to prevent 01/06/2026 → Jan-6 misparse.
#         Dates outside 2020-2030 are rejected as invalid year artifacts.
#         """
#         if pd.isna(x):
#             return pd.NaT
#         x = str(x).strip()
#         formats = [
#             "%d-%b-%y",    # 01-Jun-26
#             "%d-%b-%Y",    # 01-Jun-2026
#             "%d/%m/%Y",    # 01/06/2026  (DD first — UAE standard)
#             "%Y-%m-%d",    # 2026-06-01  (ISO)
#             "%d-%m-%Y",    # 01-06-2026
#             "%Y/%m/%d",    # 2026/06/01
#             "%d.%m.%Y",    # 01.06.2026
#             "%d %b %Y",    # 01 Jun 2026
#             "%b %d %Y",    # Jun 01 2026
#         ]
#         for fmt in formats:
#             try:
#                 result = pd.to_datetime(x, format=fmt)
#                 if 2020 <= result.year <= 2030:
#                     return result
#             except Exception:
#                 continue
#         try:
#             result = pd.to_datetime(x, dayfirst=True, errors="coerce")
#             if pd.notna(result) and 2020 <= result.year <= 2030:
#                 return result
#         except Exception:
#             pass
#         return pd.NaT
#     # ══════════════════════════════════════════════════════════════════════
#     # CLEAN DATE
#     # ══════════════════════════════════════════════════════════════════════

#     nms_rev_df["Date"] = (

#         nms_rev_df["Date"]

#         .apply(parse_mixed_dates)

#         .dt.normalize()
#     )

#     # Remove invalid dates
#     nms_rev_df = nms_rev_df[
#         nms_rev_df["Date"].notna()
#     ]

#     # ══════════════════════════════════════════════════════════════════════
#     # CLEAN SALES
#     # ══════════════════════════════════════════════════════════════════════

#     nms_rev_df["total_sales"] = (

#         nms_rev_df["total_sales"]

#         .astype(str)

#         .str.replace(",", "", regex=False)

#         .str.replace("AED", "", regex=False)

#         .str.replace("aed", "", regex=False)

#         .str.strip()

#         .pipe(pd.to_numeric, errors="coerce")

#         .fillna(0)
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # CLEAN UNITS
#     # ══════════════════════════════════════════════════════════════════════

#     nms_rev_df["total_units"] = (

#         pd.to_numeric(

#             nms_rev_df["total_units"],

#             errors="coerce"
#         )

#         .fillna(0)

#         .astype(int)
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # REMOVE DUPLICATES
#     # ══════════════════════════════════════════════════════════════════════

#     # NOTE: 4-column dedup removed — groupby agg below handles true duplicates.

#     # ══════════════════════════════════════════════════════════════════════
#     # AGGREGATE
#     # ══════════════════════════════════════════════════════════════════════

#     nms_rev_df = (

#         nms_rev_df

#         .groupby(

#             ["ASIN", "Date"],

#             as_index=False
#         )

#         .agg({

#             "total_sales" : "sum",

#             "total_units" : "sum"
#         })
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # SALES CHANNEL
#     # ══════════════════════════════════════════════════════════════════════

#     nms_rev_df["Sales Channel"] = "Noon Minutes"

#     # ══════════════════════════════════════════════════════════════════════
#     # SORT
#     # ══════════════════════════════════════════════════════════════════════

#     nms_rev_df = nms_rev_df.sort_values(

#         ["ASIN", "Date"]
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # LOAD HELPER FILE
#     # ══════════════════════════════════════════════════════════════════════

#     print("\n📦 Loading Helper File...")

#     helper_df = pd.read_excel(

#         HELPER_FILE,

#         header=1
#     )

#     helper_df.columns = (

#         helper_df.columns

#         .astype(str)

#         .str.strip()
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # FIND PRODUCT CODE (NOON MINUTES)
#     # ══════════════════════════════════════════════════════════════════════

#     helper_key = None

#     for col in helper_df.columns:

#         clean_col = (

#             str(col)

#             .replace("\n", " ")

#             .replace("_", " ")

#             .strip()

#             .lower()
#         )

#         if (

#             "product code" in clean_col

#             and

#             "noon minutes" in clean_col
#         ):

#             helper_key = col

#             break

#     print("\n✅ NMS Helper Key:")
#     print(helper_key)

#     # ══════════════════════════════════════════════════════════════════════
#     # RENAME KEY
#     # ══════════════════════════════════════════════════════════════════════

#     helper_df.rename(

#         columns={
#             helper_key: "ASIN"
#         },

#         inplace=True
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # CLEAN HELPER ASIN
#     # ══════════════════════════════════════════════════════════════════════

#     helper_df["ASIN"] = (

#         helper_df["ASIN"]

#         .astype(str)

#         .str.upper()

#         .str.strip()

#         # .str.replace("-1", "", regex=False)

#         .str.replace(".0", "", regex=False)
#     )

#     print("\n📌 Helper ASIN Sample:")
#     print(helper_df["ASIN"].head())

#     print("\n📌 Revenue ASIN Sample:")
#     print(nms_rev_df["ASIN"].head())

#     # ══════════════════════════════════════════════════════════════════════
#     # RENAME HELPER METADATA
#     # ══════════════════════════════════════════════════════════════════════

#     helper_df.rename(columns={

#         "Product Description(Cleaned)" : "Product Name",

#         "Category" : "Product Category",

#         "Product Type" : "Product Sub Category",

#         "Size" : "Pack Size"

#     }, inplace=True)

#     # ══════════════════════════════════════════════════════════════════════
#     # KEEP REQUIRED HELPER COLUMNS
#     # ══════════════════════════════════════════════════════════════════════

#     helper_cols = [

#         "ASIN",

#         "Brand",

#         "Product Name",

#         "Product Category",

#         "Product Sub Category",

#         "Pack Type",

#         "Pack Size",

#         "VC Ideal Price"
#     ]

#     helper_cols = [

#         c for c in helper_cols

#         if c in helper_df.columns
#     ]

#     helper_df = helper_df[
#         helper_cols
#     ]

#     helper_df = helper_df.drop_duplicates(
#         subset=["ASIN"]
#     )

#     print("\n✅ Helper Shape:")
#     print(helper_df.shape)

#     # ══════════════════════════════════════════════════════════════════════
#     # MERGE HELPER
#     # ══════════════════════════════════════════════════════════════════════

#     nms_rev_df = nms_rev_df.merge(

#         helper_df,

#         on="ASIN",

#         how="left"
#     )

#     # ══════════════════════════════════════════════════════════════════════
#     # FILL BLANKS
#     # ══════════════════════════════════════════════════════════════════════

#     metadata_cols = [

#         "Brand",

#         "Product Name",

#         "Product Category",

#         "Product Sub Category",

#         "Pack Type",

#         "Pack Size"
#     ]

#     for col in metadata_cols:

#         if col in nms_rev_df.columns:

#             nms_rev_df[col] = (

#                 nms_rev_df[col]

#                 .fillna("Unknown")
#             )

# # ══════════════════════════════════════════════════════════════════════════
# # FINAL VALIDATION
# # ══════════════════════════════════════════════════════════════════════════

# print("\n✅ NOON MINUTES REVENUE READY")

# print("\n📊 Shape:")
# print(nms_rev_df.shape)

# if not nms_rev_df.empty:

#     print("\n📅 Available Months:")

#     print(
#         sorted(
#             nms_rev_df["Date"]
#             .dt.strftime("%Y-%m")
#             .unique()
#         )
#     )

#     print("\n💰 Total Sales:")

#     print(
#         round(
#             nms_rev_df["total_sales"].sum(),
#             2
#         )
#     )

#     print("\n📦 Total Units:")

#     print(
#         nms_rev_df["total_units"].sum()
#     )

#     print("\n📦 Brand Fill Check:")

#     print(
#         nms_rev_df["Brand"]
#         .value_counts(dropna=False)
#         .head(10)
#     )

# display(
#     nms_rev_df.head(50)
# )

In [21]:
# ══════════════════════════════════════════════════════════════════════════
# NOON MINUTES REVENUE MASTER (FINAL HELPER FIXED VERSION)
# PRODUCT CODE (NOON MINUTES) MERGE FIXED
# BRAND / CATEGORY / PACK TYPE FIXED
# ══════════════════════════════════════════════════════════════════════════

nms_rev_parts = []

# ══════════════════════════════════════════════════════════════════════════
# LOAD MONTHLY FILES
# FIX: STANDARDIZE SOURCE COLUMN NAMES BEFORE CONCAT
#
# July / older files may contain ZSKU while August may contain zsku.
# If we concatenate first, Pandas creates separate ZSKU and zsku columns.
# Renaming both afterwards can create duplicate ASIN columns and the
# populated column can be discarded. Therefore normalize each file BEFORE
# pd.concat().
# ══════════════════════════════════════════════════════════════════════════

nms_rev_parts = []

for month in MONTHS:

    month_folder = os.path.join(
        NMS_REV_ROOT,
        month
    )

    if not os.path.isdir(month_folder):

        print(f"⚠️ NMS Revenue Folder Missing: {month}")
        continue

    # ─────────────────────────────────────────────────────────────────────
    # READ FILES
    # ─────────────────────────────────────────────────────────────────────

    for file in sorted(os.listdir(month_folder)):

        # Skip temp files
        if file.startswith("~$"):
            continue

        # Only csv/excel
        if not file.lower().endswith(
            (".csv", ".xlsx", ".xls")
        ):
            continue

        file_path = os.path.join(
            month_folder,
            file
        )

        try:

            # ─────────────────────────────────────────────────────────────
            # READ FILE
            # ─────────────────────────────────────────────────────────────

            if file.lower().endswith(".csv"):
                df = pd.read_csv(file_path)
            else:
                df = pd.read_excel(file_path)

            # ─────────────────────────────────────────────────────────────
            # CLEAN COLUMN NAMES
            # ─────────────────────────────────────────────────────────────

            df.columns = (
                df.columns
                .astype(str)
                .str.strip()
            )

            # ─────────────────────────────────────────────────────────────
            # STANDARDIZE SOURCE COLUMN NAMES BEFORE CONCAT
            # ─────────────────────────────────────────────────────────────
            # ZSKU / zsku / Zsku -> ZSKU
            # nsku / NSKU       -> nsku
            # Qty / qty         -> Qty
            # Quantity          -> Qty
            # Revenue / Sales   -> Revenue
            # ─────────────────────────────────────────────────────────────

            normalized_columns = {}

            for col in df.columns:

                clean_col = (
                    str(col)
                    .strip()
                    .lower()
                    .replace("_", " ")
                )

                if clean_col == "zsku":
                    normalized_columns[col] = "ZSKU"

                elif clean_col == "nsku":
                    normalized_columns[col] = "nsku"

                elif clean_col in {"qty", "quantity"}:
                    normalized_columns[col] = "Qty"

                elif clean_col in {
                    "revenue",
                    "sales",
                    "revenue (aed)"
                }:
                    normalized_columns[col] = "Revenue"

                elif clean_col == "date":
                    normalized_columns[col] = "Date"

            df.rename(
                columns=normalized_columns,
                inplace=True
            )

            # ─────────────────────────────────────────────────────────────
            # SOURCE TRACKING
            # ─────────────────────────────────────────────────────────────

            df["source_file"] = file
            df["source_month"] = month

            # ─────────────────────────────────────────────────────────────
            # VALIDATE ZSKU
            # ─────────────────────────────────────────────────────────────

            if "ZSKU" not in df.columns:

                print(
                    f"⚠️ {month.upper()} | {file} | "
                    f"ZSKU column NOT FOUND"
                )

                print(
                    "   Available columns:",
                    df.columns.tolist()
                )

            else:

                zsku_count = (
                    df["ZSKU"]
                    .notna()
                    .sum()
                )

                print(
                    f"✅ {month.upper()} | {file} | "
                    f"Rows: {len(df):,} | "
                    f"ZSKU populated: {zsku_count:,}"
                )

            nms_rev_parts.append(df)

        except Exception as e:

            print(f"❌ Error Reading: {file}")
            print("Reason:", e)


# ══════════════════════════════════════════════════════════════════════════
# COMBINE ALL MONTHS
# ══════════════════════════════════════════════════════════════════════════

if nms_rev_parts:

    nms_rev_df = pd.concat(
        nms_rev_parts,
        ignore_index=True
    )

else:

    nms_rev_df = pd.DataFrame()

    print("❌ No Noon Minutes Revenue Data Found")

# ══════════════════════════════════════════════════════════════════════════
# CONTINUE ONLY IF DATA EXISTS
# ══════════════════════════════════════════════════════════════════════════

if not nms_rev_df.empty:

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN COLUMN NAMES
    # ══════════════════════════════════════════════════════════════════════

    nms_rev_df.columns = (

        nms_rev_df.columns

        .astype(str)

        .str.strip()
    )

    # ══════════════════════════════════════════════════════════════════════
    # RENAME COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    # ─────────────────────────────────────────────────────────────────────
    # VALIDATE ZSKU
    # ─────────────────────────────────────────────────────────────────────

    if "ZSKU" not in nms_rev_df.columns:

        raise ValueError(
            "❌ ZSKU column not found in Noon Minutes Revenue files"
        )

    # ─────────────────────────────────────────────────────────────────────
    # DROP nsku BEFORE RENAME
    # ─────────────────────────────────────────────────────────────────────
    # nsku is NOT the Noon Minutes product code.
    # ZSKU is the correct product identifier.
    # ─────────────────────────────────────────────────────────────────────

    for _drop_col in ["nsku", "NSKU", "Nsku"]:

        if _drop_col in nms_rev_df.columns:

            nms_rev_df.drop(
                columns=[_drop_col],
                inplace=True
            )

            print(
                f"  ✅ Dropped '{_drop_col}' — "
                f"using ZSKU as ASIN only"
            )

    rename_map = {

        "ZSKU": "ASIN",

        "Sku": "ASIN",

        "SKU": "ASIN",

        "Qty": "total_units",

        "Quantity": "total_units",

        "Revenue": "total_sales",

        "Sales": "total_sales",

        "Revenue (AED)": "total_sales"
    }

    nms_rev_df.rename(
        columns={
            k: v
            for k, v in rename_map.items()
            if k in nms_rev_df.columns
        },
        inplace=True
    )

    # ─────────────────────────────────────────────────────────────────────
    # VALIDATE REQUIRED COLUMNS
    # ─────────────────────────────────────────────────────────────────────

    required_cols = [
        "ASIN",
        "Date",
        "total_sales",
        "total_units"
    ]

    missing_cols = [
        c
        for c in required_cols
        if c not in nms_rev_df.columns
    ]

    if missing_cols:

        raise ValueError(
            f"❌ Required Noon Minutes Revenue columns are missing: "
            f"{missing_cols}"
        )

    # ─────────────────────────────────────────────────────────────────────
    # DUPLICATE COLUMN CHECK
    # ─────────────────────────────────────────────────────────────────────

    duplicate_columns = (
        nms_rev_df.columns[
            nms_rev_df.columns.duplicated()
        ]
        .tolist()
    )

    if duplicate_columns:

        print(
            f"⚠️ Duplicate columns found after rename: "
            f"{duplicate_columns}"
        )

        nms_rev_df = nms_rev_df.loc[
            :,
            ~nms_rev_df.columns.duplicated()
        ]

    print("\n✅ Columns After Rename:")
    print(nms_rev_df.columns.tolist())

    # ─────────────────────────────────────────────────────────────────────
    # KEEP REQUIRED COLUMNS + SOURCE MONTH FOR DIAGNOSTICS
    # ─────────────────────────────────────────────────────────────────────

    keep_cols = [
        "ASIN",
        "Date",
        "total_sales",
        "total_units",
        "source_month",
        "source_file"
    ]

    available_cols = [
        c
        for c in keep_cols
        if c in nms_rev_df.columns
    ]

    nms_rev_df = nms_rev_df[available_cols].copy()

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN ASIN
    # ══════════════════════════════════════════════════════════════════════

    nms_rev_df["ASIN"] = (
        nms_rev_df["ASIN"]
        .astype("string")
        .str.upper()
        .str.strip()
        # Keep -1 because the helper uses the same Noon Minutes key format.
        .str.replace(".0", "", regex=False)
        .replace({
            "NAN": pd.NA,
            "NONE": pd.NA,
            "NULL": pd.NA,
            "": pd.NA
        })
    )

    # ─────────────────────────────────────────────────────────────────────
    # ASIN CHECK BY MONTH
    # ─────────────────────────────────────────────────────────────────────

    if "source_month" in nms_rev_df.columns:

        print("\n📌 NMS ASIN CHECK BY MONTH:")

        asin_check = (
            nms_rev_df
            .groupby("source_month", dropna=False)
            .agg(
                rows=("ASIN", "size"),
                asin_present=("ASIN", lambda x: x.notna().sum()),
                asin_missing=("ASIN", lambda x: x.isna().sum())
            )
            .reset_index()
        )

        print(
            asin_check.to_string(index=False)
        )

    # Remove blank ASIN
    before_asin_filter = len(nms_rev_df)

    nms_rev_df = nms_rev_df[
        nms_rev_df["ASIN"].notna()
    ].copy()

    removed_asin_rows = (
        before_asin_filter - len(nms_rev_df)
    )

    if removed_asin_rows:

        print(
            f"⚠️ Removed {removed_asin_rows:,} rows "
            f"with missing ASIN"
        )

    # ══════════════════════════════════════════════════════════════════════
    # DATE PARSER
    # ══════════════════════════════════════════════════════════════════════

    def parse_mixed_dates(x):
        """
        Parse date strings safely for UAE data (DD-first convention).
        %m/%d/%Y removed to prevent 01/06/2026 → Jan-6 misparse.
        Dates outside 2020-2030 are rejected as invalid year artifacts.
        """
        if pd.isna(x):
            return pd.NaT
        x = str(x).strip()
        formats = [
            "%d-%b-%y",    # 01-Jun-26
            "%d-%b-%Y",    # 01-Jun-2026
            "%d/%m/%Y",    # 01/06/2026  (DD first — UAE standard)
            "%Y-%m-%d",    # 2026-06-01  (ISO)
            "%d-%m-%Y",    # 01-06-2026
            "%Y/%m/%d",    # 2026/06/01
            "%d.%m.%Y",    # 01.06.2026
            "%d %b %Y",    # 01 Jun 2026
            "%b %d %Y",    # Jun 01 2026
        ]
        for fmt in formats:
            try:
                result = pd.to_datetime(x, format=fmt)
                if 2020 <= result.year <= 2030:
                    return result
            except Exception:
                continue
        try:
            result = pd.to_datetime(x, dayfirst=True, errors="coerce")
            if pd.notna(result) and 2020 <= result.year <= 2030:
                return result
        except Exception:
            pass
        return pd.NaT
    # ══════════════════════════════════════════════════════════════════════
    # CLEAN DATE
    # ══════════════════════════════════════════════════════════════════════

    nms_rev_df["Date"] = (

        nms_rev_df["Date"]

        .apply(parse_mixed_dates)

        .dt.normalize()
    )

    # Remove invalid dates
    nms_rev_df = nms_rev_df[
        nms_rev_df["Date"].notna()
    ]

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN SALES
    # ══════════════════════════════════════════════════════════════════════

    nms_rev_df["total_sales"] = (

        nms_rev_df["total_sales"]

        .astype(str)

        .str.replace(",", "", regex=False)

        .str.replace("AED", "", regex=False)

        .str.replace("aed", "", regex=False)

        .str.strip()

        .pipe(pd.to_numeric, errors="coerce")

        .fillna(0)
    )

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN UNITS
    # ══════════════════════════════════════════════════════════════════════

    nms_rev_df["total_units"] = (

        pd.to_numeric(

            nms_rev_df["total_units"],

            errors="coerce"
        )

        .fillna(0)

        .astype(int)
    )

    # ══════════════════════════════════════════════════════════════════════
    # REMOVE DUPLICATES
    # ══════════════════════════════════════════════════════════════════════

    # NOTE: 4-column dedup removed — groupby agg below handles true duplicates.

    # ══════════════════════════════════════════════════════════════════════
    # AGGREGATE
    # ══════════════════════════════════════════════════════════════════════

    nms_rev_df = (

        nms_rev_df

        .groupby(

            ["ASIN", "Date"],

            as_index=False
        )

        .agg({

            "total_sales" : "sum",

            "total_units" : "sum"
        })
    )

    # ══════════════════════════════════════════════════════════════════════
    # SALES CHANNEL
    # ══════════════════════════════════════════════════════════════════════

    nms_rev_df["Sales Channel"] = "Noon Minutes"

    # ══════════════════════════════════════════════════════════════════════
    # SORT
    # ══════════════════════════════════════════════════════════════════════

    nms_rev_df = nms_rev_df.sort_values(

        ["ASIN", "Date"]
    )

    # ══════════════════════════════════════════════════════════════════════
    # LOAD HELPER FILE
    # ══════════════════════════════════════════════════════════════════════

    print("\n📦 Loading Helper File...")

    helper_df = pd.read_excel(

        HELPER_FILE,

        header=1
    )

    helper_df.columns = (

        helper_df.columns

        .astype(str)

        .str.strip()
    )

    # ══════════════════════════════════════════════════════════════════════
    # FIND PRODUCT CODE (NOON MINUTES)
    # ══════════════════════════════════════════════════════════════════════

    helper_key = None

    for col in helper_df.columns:

        clean_col = (

            str(col)

            .replace("\n", " ")

            .replace("_", " ")

            .strip()

            .lower()
        )

        if (

            "product code" in clean_col

            and

            "noon minutes" in clean_col
        ):

            helper_key = col

            break

    print("\n✅ NMS Helper Key:")
    print(helper_key)

    if helper_key is None:

        raise ValueError(
            "❌ Could not find the Noon Minutes Product Code "
            "column in the helper file."
        )

    # ══════════════════════════════════════════════════════════════════════
    # RENAME KEY
    # ══════════════════════════════════════════════════════════════════════

    helper_df.rename(

        columns={
            helper_key: "ASIN"
        },

        inplace=True
    )

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN HELPER ASIN
    # ══════════════════════════════════════════════════════════════════════

    helper_df["ASIN"] = (

        helper_df["ASIN"]

        .astype("string")

        .str.upper()

        .str.strip()

        # Keep -1 because the helper uses the same Noon Minutes key format.

        .str.replace(".0", "", regex=False)

        .replace({
            "NAN": pd.NA,
            "NONE": pd.NA,
            "NULL": pd.NA,
            "": pd.NA
        })
    )

    print("\n📌 Helper ASIN Sample:")
    print(helper_df["ASIN"].head())

    print("\n📌 Revenue ASIN Sample:")
    print(nms_rev_df["ASIN"].head())

    # ══════════════════════════════════════════════════════════════════════
    # RENAME HELPER METADATA
    # ══════════════════════════════════════════════════════════════════════

    helper_df.rename(columns={

        "Product Description(Cleaned)" : "Product Name",

        "Category" : "Product Category",

        "Product Type" : "Product Sub Category",

        "Size" : "Pack Size"

    }, inplace=True)

    # ══════════════════════════════════════════════════════════════════════
    # KEEP REQUIRED HELPER COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    helper_cols = [

        "ASIN",

        "Brand",

        "Product Name",

        "Product Category",

        "Product Sub Category",

        "Pack Type",

        "Pack Size",

        "VC Ideal Price"
    ]

    helper_cols = [

        c for c in helper_cols

        if c in helper_df.columns
    ]

    helper_df = helper_df[
        helper_cols
    ]

    helper_df = helper_df.drop_duplicates(
        subset=["ASIN"]
    )

    print("\n✅ Helper Shape:")
    print(helper_df.shape)

    # ══════════════════════════════════════════════════════════════════════
    # MERGE HELPER
    # ══════════════════════════════════════════════════════════════════════

    nms_rev_df = nms_rev_df.merge(

        helper_df,

        on="ASIN",

        how="left"
    )

    # ══════════════════════════════════════════════════════════════════════
    # FILL BLANKS
    # ══════════════════════════════════════════════════════════════════════

    metadata_cols = [

        "Brand",

        "Product Name",

        "Product Category",

        "Product Sub Category",

        "Pack Type",

        "Pack Size"
    ]

    for col in metadata_cols:

        if col in nms_rev_df.columns:

            nms_rev_df[col] = (

                nms_rev_df[col]

                .fillna("Unknown")
            )

# ══════════════════════════════════════════════════════════════════════════
# FINAL VALIDATION
# ══════════════════════════════════════════════════════════════════════════

print("\n✅ NOON MINUTES REVENUE READY")

print("\n📊 Shape:")
print(nms_rev_df.shape)

if not nms_rev_df.empty:

    print("\n📅 Available Months:")

    print(
        sorted(
            nms_rev_df["Date"]
            .dt.strftime("%Y-%m")
            .unique()
        )
    )

    print("\n💰 Total Sales:")

    print(
        round(
            nms_rev_df["total_sales"].sum(),
            2
        )
    )

    print("\n📦 Total Units:")

    print(
        nms_rev_df["total_units"].sum()
    )

    print("\n📦 Brand Fill Check:")

    print(
        nms_rev_df["Brand"]
        .value_counts(dropna=False)
        .head(10)
    )

# ══════════════════════════════════════════════════════════════════════════
# AUGUST 2026 VALIDATION
# ══════════════════════════════════════════════════════════════════════════

if not nms_rev_df.empty:

    august_df = nms_rev_df[
        nms_rev_df["Date"].dt.year.eq(2026)
        & nms_rev_df["Date"].dt.month.eq(8)
    ].copy()

    print("\n📌 AUGUST 2026 NMS CHECK:")

    if august_df.empty:

        print("❌ No August 2026 Noon Minutes data found.")

    else:

        print(
            f"   August rows       : {len(august_df):,}"
        )

        print(
            f"   August ASINs      : "
            f"{august_df['ASIN'].nunique():,}"
        )

        print(
            f"   August ASIN blank : "
            f"{august_df['ASIN'].isna().sum():,}"
        )

        print(
            f"   August sales      : "
            f"{august_df['total_sales'].sum():,.2f}"
        )

        print(
            f"   August units      : "
            f"{august_df['total_units'].sum():,}"
        )

        print("\n   August sample:")

        sample_cols = [
            "ASIN",
            "Date",
            "total_sales",
            "total_units",
            "Sales Channel",
            "Brand",
            "Product Name"
        ]

        sample_cols = [
            c
            for c in sample_cols
            if c in august_df.columns
        ]

        print(
            august_df[
                sample_cols
            ]
            .head(20)
            .to_string(index=False)
        )


display(
    nms_rev_df.head(50)
)

✅ MAY | Noon Minutes_sales_may_2026.csv | Rows: 1,202 | ZSKU populated: 1,202
✅ JUN | Noon Minutes_sales_jun_2026.csv | Rows: 1,018 | ZSKU populated: 1,018
✅ JULY | Noon Minutes_sales_july_2026.csv | Rows: 1,165 | ZSKU populated: 1,165
✅ AUG | Noon Minutes_sales_aug_2026.csv | Rows: 1,105 | ZSKU populated: 1,105
  ✅ Dropped 'nsku' — using ZSKU as ASIN only

✅ Columns After Rename:
['Date', 'Year_Month', 'Country', 'platform', 'Warehouse', 'Warehouse Cod', 'Category', 'ASIN', 'Barcode', 'Title', 'Brand', 'Type', 'Sub-Type', 'total_units', 'total_sales', 'source_file', 'source_month']

📌 NMS ASIN CHECK BY MONTH:
source_month  rows  asin_present  asin_missing
         aug  1105          1105             0
        july  1165          1165             0
         jun  1018          1018             0
         may  1202          1202             0

📦 Loading Helper File...

✅ NMS Helper Key:
Product Code
(Noon Minutes)

📌 Helper ASIN Sample:
0                           -
1    ZA4EA14149E7DB09

,ASIN,Date,total_sales,total_units,Sales Channel,Brand,Product Name,Product Category,Product Sub Category,Pack Type,Pack Size,VC Ideal Price
0,Z045B32EE54200CC7E00FZ-1,2026-07-25,18.40,1,Noon Minutes,Emami 7 Oils in One,Emami 7 in 1 Hair Oil Black Seed,Hair Care,Hair Oil,1,300ml,20.081719
1,Z045B32EE54200CC7E00FZ-1,2026-07-27,13.05,1,Noon Minutes,Emami 7 Oils in One,Emami 7 in 1 Hair Oil Black Seed,Hair Care,Hair Oil,1,300ml,20.081719
2,Z045B32EE54200CC7E00FZ-1,2026-07-28,26.10,2,Noon Minutes,Emami 7 Oils in One,Emami 7 in 1 Hair Oil Black Seed,Hair Care,Hair Oil,1,300ml,20.081719
3,Z045B32EE54200CC7E00FZ-1,2026-07-29,52.20,4,Noon Minutes,Emami 7 Oils in One,Emami 7 in 1 Hair Oil Black Seed,Hair Care,Hair Oil,1,300ml,20.081719
4,Z045B32EE54200CC7E00FZ-1,2026-07-30,39.15,3,Noon Minutes,Emami 7 Oils in One,Emami 7 in 1 Hair Oil Black Seed,Hair Care,Hair Oil,1,300ml,20.081719
5,Z045B32EE54200CC7E00FZ-1,2026-07-31,36.80,2,Noon Minutes,Emami 7 Oils in One,Emami 7 in 1 Hair Oil Black Seed,Hair Care,Hair Oil,1,300ml,20.081719
6,Z045B32EE54200CC7E00FZ-1,2026-08-01,16.45,1,Noon Minutes,Emami 7 Oils in One,Emami 7 in 1 Hair Oil Black Seed,Hair Care,Hair Oil,1,300ml,20.081719
7,Z045B32EE54200CC7E00FZ-1,2026-08-02,16.45,1,Noon Minutes,Emami 7 Oils in One,Emami 7 in 1 Hair Oil Black Seed,Hair Care,Hair Oil,1,300ml,20.081719
8,Z045B32EE54200CC7E00FZ-1,2026-08-03,16.45,1,Noon Minutes,Emami 7 Oils in One,Emami 7 in 1 Hair Oil Black Seed,Hair Care,Hair Oil,1,300ml,20.081719
9,Z045B32EE54200CC7E00FZ-1,2026-08-04,26.90,2,Noon Minutes,Emami 7 Oils in One,Emami 7 in 1 Hair Oil Black Seed,Hair Care,Hair Oil,1,300ml,20.081719


In [22]:
nms_rev_df.to_csv("noon.csv", index=False)

## 10 · Noon Minutes – Marketing (`nms_ads_df`)

In [23]:
# ══════════════════════════════════════════════════════════════════════════
# NOON MINUTES ADS MASTER (FINAL FIXED VERSION)
# HELPER MERGE FIXED
# BRAND / CATEGORY / PACK TYPE FIXED
# DUPLICATE ASIN FIXED
# ══════════════════════════════════════════════════════════════════════════

nms_ads_parts = []

# ══════════════════════════════════════════════════════════════════════════
# LOAD MONTHLY FILES
# ══════════════════════════════════════════════════════════════════════════

for month in MONTHS:

    file_path = os.path.join(

        NMS_ADS_ROOT,

        month,

        f"noon_minuts_ads_{month}_{YEAR}.csv"
    )

    if os.path.exists(file_path):

        try:

            df = pd.read_csv(file_path)
            print(f"\n===== {month.upper()} =====")
            print("Rows:", len(df))

            if "Date" in df.columns:
                print("Sample dates:")
                print(df["Date"].head())

            df["processed_month"] = month

            nms_ads_parts.append(df)

            print(f"✅ NMS Ads Loaded: {month}")

        except Exception as e:

            print(f"❌ {month}: {e}")

    else:

        print(f"⚠️ NMS Ads Not Found: {month}")

# ══════════════════════════════════════════════════════════════════════════
# COMBINE ALL MONTHS
# ══════════════════════════════════════════════════════════════════════════

if nms_ads_parts:

    nms_ads_df = pd.concat(

        nms_ads_parts,

        ignore_index=True
    )

else:

    nms_ads_df = pd.DataFrame()

    print("❌ No NMS Ads Data Found")

# ══════════════════════════════════════════════════════════════════════════
# CONTINUE ONLY IF DATA EXISTS
# ══════════════════════════════════════════════════════════════════════════

if not nms_ads_df.empty:

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN COLUMN NAMES
    # ══════════════════════════════════════════════════════════════════════

    nms_ads_df.columns = (

        nms_ads_df.columns

        .astype(str)

        .str.strip()
    )

    # ══════════════════════════════════════════════════════════════════════
    # RENAME COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    rename_map = {

        "SKU" : "ASIN",

        "Sku" : "ASIN",

        "sku" : "ASIN",

        "Views" : "Impressions",

        "Spends" : "ads_spend",

        "Spend" : "ads_spend",

        "Revenue" : "ads_sales",

        "Revenue (AED)" : "ads_sales",

        "Orders" : "ads_unit_sold",

        "Portfolio Name" : "Portfolio name"
    }

    nms_ads_df.rename(

        columns={

            k: v

            for k, v in rename_map.items()

            if k in nms_ads_df.columns
        },

        inplace=True
    )

    # ══════════════════════════════════════════════════════════════════════
    # REMOVE DUPLICATE COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    nms_ads_df = nms_ads_df.loc[
        :,
        ~nms_ads_df.columns.duplicated()
    ]

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN ASIN
    # ══════════════════════════════════════════════════════════════════════

    nms_ads_df["ASIN"] = (

        nms_ads_df["ASIN"]

        .astype(str)

        .str.upper()

        .str.strip()

        # .str.replace("-1", "", regex=False)

        .str.replace(".0", "", regex=False)
    )

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN DATE
    # ══════════════════════════════════════════════════════════════════════

    # nms_ads_df["Date"] = (

    #     pd.to_datetime(

    #         nms_ads_df["Date"],
    #         format="%d-%m-%Y",
    #         errors="coerce"
    #     )

    #     .dt.normalize()
    # )

    # # Remove invalid dates
    # nms_ads_df = nms_ads_df[
    #     nms_ads_df["Date"].notna()
    # ]
     
    # ══════════════════════════════════════════════════════════════
# CLEAN DATE (Supports Multiple Formats)
# ══════════════════════════════════════════════════════════════

    def parse_date(x):

        if pd.isna(x):
            return pd.NaT

        x = str(x).strip()

        formats = [

            "%Y-%m-%d",      # 2026-05-01
            "%d-%m-%Y",      # 05-06-2026
            "%d/%m/%Y",      # 05/06/2026
            "%Y/%m/%d",      # 2026/06/05
            "%d-%b-%Y",      # 05-Jun-2026
            "%d-%B-%Y",      # 05-June-2026
            "%d %b %Y",      # 05 Jun 2026
            "%d %B %Y",      # 05 June 2026
            "%m/%d/%Y",      # 06/05/2026
            "%m-%d-%Y",      # 06-05-2026
        ]

        for fmt in formats:

            try:
                return pd.to_datetime(x, format=fmt)
            except:
                pass

        # Final fallback
        return pd.to_datetime(x, errors="coerce", dayfirst=True)


    nms_ads_df["Date"] = (
        nms_ads_df["Date"]
            .apply(parse_date)
            .dt.normalize()
    )

    # Remove invalid dates
    nms_ads_df = nms_ads_df[
        nms_ads_df["Date"].notna()
    ]

    # ══════════════════════════════════════════════════════════════════════
    # SALES CHANNEL
    # ══════════════════════════════════════════════════════════════════════

    nms_ads_df["Sales Channel"] = "Noon Minutes"

    # ══════════════════════════════════════════════════════════════════════
    # DEFAULT COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    if "Campaign Type" not in nms_ads_df.columns:

        nms_ads_df["Campaign Type"] = "Noon Minutes Ads"

    if "Portfolio name" not in nms_ads_df.columns:

        nms_ads_df["Portfolio name"] = ""

    # ══════════════════════════════════════════════════════════════════════
    # NUMERIC CLEANUP
    # ══════════════════════════════════════════════════════════════════════

    numeric_cols = [

        "ads_sales",

        "ads_unit_sold",

        "ads_spend",

        "Impressions",

        "Clicks"
    ]

    for col in numeric_cols:

        if col in nms_ads_df.columns:

            nms_ads_df[col] = (

                nms_ads_df[col]

                .astype(str)

                .str.replace(",", "", regex=False)

                .str.replace("%", "", regex=False)

                .str.replace("AED", "", regex=False)

                .str.strip()

                .pipe(pd.to_numeric, errors="coerce")

                .fillna(0)
            )

        else:

            nms_ads_df[col] = 0

            # # ══════════════════════════════════════════════════════════════════════
        # KPI CALCULATIONS (FIXED)
        # ══════════════════════════════════════════════════════════════════════

        # CTR
        if "CTR" in nms_ads_df.columns:

            nms_ads_df["Click-Thru Rate (CTR)"] = (
                pd.to_numeric(
                    nms_ads_df["CTR"],
                    errors="coerce"
                ).fillna(0)
            )

        else:

            nms_ads_df["Click-Thru Rate (CTR)"] = 0


        # CPC
        if "CPC" in nms_ads_df.columns:

            nms_ads_df["Cost Per Click (CPC)"] = (
                pd.to_numeric(
                    nms_ads_df["CPC"],
                    errors="coerce"
                ).fillna(0)
            )

        else:

            nms_ads_df["Cost Per Click (CPC)"] = 0


        # ROAS
        if "ROAS" in nms_ads_df.columns:

            nms_ads_df["Total Return on Advertising Spend (ROAS)"] = (
                pd.to_numeric(
                    nms_ads_df["ROAS"],
                    errors="coerce"
                ).fillna(0)
            )

        else:

            nms_ads_df["Total Return on Advertising Spend (ROAS)"] = 0


        # ACOS
        nms_ads_df["Total Advertising Cost of Sales (ACOS)"] = np.where(

            nms_ads_df["ads_sales"] > 0,

            (
                nms_ads_df["ads_spend"]
                / nms_ads_df["ads_sales"]
                * 100
            ).round(2),

            0
        )

        # CVR
        nms_ads_df["CVR"] = np.where(

            nms_ads_df["Clicks"] > 0,

            (
                nms_ads_df["ads_unit_sold"]
                / nms_ads_df["Clicks"]
                * 100
            ).round(2),

            0
        )

    # ══════════════════════════════════════════════════════════════════════
    # LOAD HELPER FILE
    # ══════════════════════════════════════════════════════════════════════

    print("\n📦 Loading Helper File...")

    helper_df = pd.read_excel(

        HELPER_FILE,

        header=1
    )

    helper_df.columns = (

        helper_df.columns

        .astype(str)

        .str.strip()
    )

    # ══════════════════════════════════════════════════════════════════════
    # FIND PRODUCT CODE (NOON MINUTES)
    # ══════════════════════════════════════════════════════════════════════

    helper_key = None

    for col in helper_df.columns:

        clean_col = (

            str(col)

            .replace("\n", " ")

            .replace("_", " ")

            .strip()

            .lower()
        )

        if (

            "product code" in clean_col

            and

            "noon minutes" in clean_col
        ):

            helper_key = col

            break

    print("\n✅ NMS Helper Key:")
    print(helper_key)

    # ══════════════════════════════════════════════════════════════════════
    # RENAME KEY
    # ══════════════════════════════════════════════════════════════════════

    helper_df.rename(

        columns={
            helper_key: "ASIN"
        },

        inplace=True
    )

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN HELPER ASIN
    # ══════════════════════════════════════════════════════════════════════

    helper_df["ASIN"] = (

        helper_df["ASIN"]

        .astype(str)

        .str.upper()

        .str.strip()

        # .str.replace("-1", "", regex=False)

        .str.replace(".0", "", regex=False)
    )

    # ══════════════════════════════════════════════════════════════════════
    # RENAME HELPER METADATA
    # ══════════════════════════════════════════════════════════════════════

    helper_df.rename(columns={

        "Product Description(Cleaned)" : "Product Name",

        "Category" : "Product Category",

        "Product Type" : "Product Sub Category",

        "Size" : "Pack Size"

    }, inplace=True)

    # ══════════════════════════════════════════════════════════════════════
    # KEEP REQUIRED HELPER COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    helper_cols = [

        "ASIN",

        "Brand",

        "Product Name",

        "Product Category",

        "Product Sub Category",

        "Pack Type",

        "Pack Size",

        "VC Ideal Price"
    ]

    helper_cols = [

        c for c in helper_cols

        if c in helper_df.columns
    ]

    helper_df = helper_df[
        helper_cols
    ]

    helper_df = helper_df.drop_duplicates(
        subset=["ASIN"]
    )

    print("\n✅ Helper Shape:")
    print(helper_df.shape)

    # ══════════════════════════════════════════════════════════════════════
    # MERGE HELPER
    # ══════════════════════════════════════════════════════════════════════

    nms_ads_df = nms_ads_df.merge(

        helper_df,

        on="ASIN",

        how="left"
    )

    # ══════════════════════════════════════════════════════════════════════
    # FILL BLANKS
    # ══════════════════════════════════════════════════════════════════════

    metadata_cols = [

        "Brand",

        "Product Name",

        "Product Category",

        "Product Sub Category",

        "Pack Type",

        "Pack Size"
    ]

    for col in metadata_cols:

        if col in nms_ads_df.columns:

            nms_ads_df[col] = (

                nms_ads_df[col]

                .fillna("Unknown")
            )

    # ══════════════════════════════════════════════════════════════════════
    # CAMPAIGN OUTPUT
    # ══════════════════════════════════════════════════════════════════════

    _camp_cols = [

        "ASIN",

        "Date",

        "Sales Channel",

        "Campaign Name",

        "Campaign Type",

        "Portfolio name",

        "ads_sales",

        "ads_unit_sold",

        "ads_spend",

        "Impressions",

        "Clicks",

        "Click-Thru Rate (CTR)",

        "Cost Per Click (CPC)",

        "Total Advertising Cost of Sales (ACOS)",

        "Total Return on Advertising Spend (ROAS)",

        "CVR",

        "Brand",

        "Product Name",

        "Product Category",

        "Product Sub Category",

        "Pack Type",

        "Pack Size",

        "VC Ideal Price"
    ]

    _camp_cols = [

        c for c in _camp_cols

        if c in nms_ads_df.columns
    ]

    nms_ads_campaign = nms_ads_df[
        _camp_cols
    ].copy()

    # ══════════════════════════════════════════════════════════════════════
    # AGGREGATED OUTPUT
    # ══════════════════════════════════════════════════════════════════════

    nms_ads_agg = (

        nms_ads_df

        .groupby(

            ["ASIN", "Date", "Sales Channel"],

            as_index=False
        )

        .agg({

            "ads_sales" : "sum",

            "ads_unit_sold" : "sum",

            "ads_spend" : "sum"
        })
    )

else:

    nms_ads_campaign = pd.DataFrame()

    nms_ads_agg = pd.DataFrame()

# ══════════════════════════════════════════════════════════════════════════
# FINAL VALIDATION
# ══════════════════════════════════════════════════════════════════════════

print("\n✅ NMS ADS READY")

print("\n📊 Campaign Shape:")
print(nms_ads_campaign.shape)

print("\n📊 Aggregated Shape:")
print(nms_ads_agg.shape)

if not nms_ads_campaign.empty:

    print("\n💰 Total Ads Sales:")
    print(round(nms_ads_campaign["ads_sales"].sum(), 2))

    print("\n📦 Brand Fill Check:")

    print(
        nms_ads_campaign["Brand"]
        .value_counts(dropna=False)
        .head(10)
    )

display(
    nms_ads_campaign.head(50)
)


===== MAY =====
Rows: 849
Sample dates:
0    2026-05-01
1    2026-05-01
2    2026-05-01
3    2026-05-01
4    2026-05-01
Name: Date, dtype: object
✅ NMS Ads Loaded: may

===== JUN =====
Rows: 596
Sample dates:
0    2026-06-05
1    2026-06-05
2    2026-06-05
3    2026-06-05
4    2026-06-05
Name: Date, dtype: object
✅ NMS Ads Loaded: jun

===== JULY =====
Rows: 937
Sample dates:
0    2026-07-01
1    2026-07-01
2    2026-07-01
3    2026-07-01
4    2026-07-01
Name: Date, dtype: object
✅ NMS Ads Loaded: july

===== AUG =====
Rows: 950
Sample dates:
0    2026-08-01
1    2026-08-01
2    2026-08-01
3    2026-08-01
4    2026-08-01
Name: Date, dtype: object
✅ NMS Ads Loaded: aug

📦 Loading Helper File...

✅ NMS Helper Key:
Product Code
(Noon Minutes)

✅ Helper Shape:
(19, 8)

✅ NMS ADS READY

📊 Campaign Shape:
(3332, 22)

📊 Aggregated Shape:
(1152, 6)

💰 Total Ads Sales:
34959.9

📦 Brand Fill Check:
Brand
Creme 21                    1539
Emami Smart And Handsome     978
Emami 7 Oils in One      

,ASIN,Date,Sales Channel,Campaign Name,Campaign Type,Portfolio name,ads_sales,ads_unit_sold,ads_spend,Impressions,...,Cost Per Click (CPC),Total Advertising Cost of Sales (ACOS),Total Return on Advertising Spend (ROAS),CVR,Brand,Product Category,Product Sub Category,Pack Type,Pack Size,VC Ideal Price
0,Z5540F7088905D2C64FFBZ-1,2026-05-01,Noon Minutes,SAH-Face Wash-SP-M-BO (P),Noon Minutes Ads,,0.00,0,0.00,37,...,0.00,0.00,0.00,0.00,Emami Smart And Handsome,Men's Grooming,Face Wash,1,100gm,24.633576
1,ZBA36704959668354C95AZ-1,2026-05-01,Noon Minutes,SAH-Face Cream-SP-M-GKW LT (E),Noon Minutes Ads,,0.00,0,0.00,3,...,0.00,0.00,0.00,0.00,Emami Smart And Handsome,Men's Grooming,Face Cream,1,50gm,15.797619
2,Z3472F77ACFFA103E8ED6Z-1,2026-05-01,Noon Minutes,Crème 21-Glycerin Oil-SP-M-GKW LT (E),Noon Minutes Ads,,0.00,0,0.00,7,...,0.00,0.00,0.00,0.00,Creme 21,Skin Care,Glycerin Oil,1,200ml,16.600888
3,Z5540F7088905D2C64FFBZ-1,2026-05-01,Noon Minutes,SAH-All-SP-M-CT\t\t,Noon Minutes Ads,,0.00,0,2.51,40,...,2.51,0.00,0.00,0.00,Emami Smart And Handsome,Men's Grooming,Face Wash,1,100gm,24.633576
4,ZB925B53F738D1172A5DFZ-1,2026-05-01,Noon Minutes,SAH-Beard Wash-SP-M-CT,Noon Minutes Ads,,0.00,0,4.01,81,...,2.01,0.00,0.00,0.00,Emami Smart And Handsome,Men's Grooming,Beard Wash,1,100gm,29.988701
5,ZF4F3189FA13EFE7A2689Z-1,2026-05-01,Noon Minutes,NRO-Hair Oil-SP-M-BD (P),Noon Minutes Ads,,0.00,0,0.00,1,...,0.00,0.00,0.00,0.00,Himani Navratna,Hair Care,Hair Oil,1,200ml,16.065375
6,ZBA36704959668354C95AZ-1,2026-05-01,Noon Minutes,SAH-Face Cream-SP-M-BD (P),Noon Minutes Ads,,0.00,0,0.00,1,...,0.00,0.00,0.00,0.00,Emami Smart And Handsome,Men's Grooming,Face Cream,1,50gm,15.797619
7,ZBA36704959668354C95AZ-1,2026-05-01,Noon Minutes,SAH-Face Cream-SP-M-CT,Noon Minutes Ads,,0.00,0,0.00,34,...,0.00,0.00,0.00,0.00,Emami Smart And Handsome,Men's Grooming,Face Cream,1,50gm,15.797619
8,Z38685AD13D6A297BDF7DZ-1,2026-05-01,Noon Minutes,Creme 21-Face Serum-SP-M-GKW-(P),Noon Minutes Ads,,0.00,0,0.00,118,...,0.00,0.00,0.00,0.00,Creme 21,Face Care,Face Serum,1,30ml,52.480226
9,ZBA36704959668354C95AZ-1,2026-05-01,Noon Minutes,SAH-Face Cream-SP-M-BO (P),Noon Minutes Ads,,11.40,1,14.85,35,...,2.97,130.26,0.77,20.00,Emami Smart And Handsome,Men's Grooming,Face Cream,1,50gm,15.797619


In [24]:
nms_ads_campaign.to_csv("nms_ads_campaign_output.csv", index=False)

# helper processing

In [25]:
# ══════════════════════════════════════════════════════════════════════════
# HELPER FILE PROCESSING  ─  MASTER SHEET
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  HELPER FILE PROCESSING")
print("="*60)

xls = pd.ExcelFile(HELPER_FILE)
print("\n📌 Available Sheets:", xls.sheet_names)

helper_raw = pd.read_excel(HELPER_FILE, sheet_name="Master Sheet", header=1)
helper_raw.columns = helper_raw.columns.astype(str).str.strip()

print("\n📌 Raw Helper Columns:")
print(helper_raw.columns.tolist())

# ── Robust column rename: match on normalised lowercase ────────────────────
def _match_col(cols, *keywords):
    """Return first column whose normalised name contains ALL keywords."""
    for c in cols:
        norm = str(c).replace("\n"," ").replace("_"," ").strip().lower()
        if all(k in norm for k in keywords):
            return c
    return None

_asin_col  = _match_col(helper_raw.columns, "amazon", "asin")
_nc_col    = (_match_col(helper_raw.columns, "product code", "noon core")
              or _match_col(helper_raw.columns, "noon core"))
_nms_col   = (_match_col(helper_raw.columns, "product code", "noon minutes")
              or _match_col(helper_raw.columns, "noon minutes"))
_name_col  = _match_col(helper_raw.columns, "product description")
_cat_col   = _match_col(helper_raw.columns, "category")
_sub_col   = _match_col(helper_raw.columns, "product type")
_size_col  = _match_col(helper_raw.columns, "size")
_pt_col    = _match_col(helper_raw.columns, "pack type")
_vc_col    = _match_col(helper_raw.columns, "vc ideal price")

rename_map = {}
if _asin_col : rename_map[_asin_col]  = "ASIN"
if _nc_col   : rename_map[_nc_col]    = "NC_NIN"
if _nms_col  : rename_map[_nms_col]   = "NMs_NIN"
if _name_col : rename_map[_name_col]  = "Product Name"
if _cat_col  : rename_map[_cat_col]   = "Product Category"
if _sub_col  : rename_map[_sub_col]   = "Product Sub Category"
if _size_col : rename_map[_size_col]  = "Pack Size"
if _pt_col   : rename_map[_pt_col]    = "Pack Type"
if _vc_col   : rename_map[_vc_col]    = "VC Ideal Price"

helper_raw.rename(columns=rename_map, inplace=True)

if "SC Ideal Price" not in helper_raw.columns:
    helper_raw["SC Ideal Price"] = 0

print("\n✅ Renamed Columns:")
print(helper_raw.columns.tolist())

# ── Clean keys ─────────────────────────────────────────────────────────────
def _clean_asin_key(series):
    """Clean Amazon ASINs — strips trailing digit suffix like -1 (not part of Amazon ASIN)."""
    return (series.astype(str).str.upper().str.strip()
            .str.replace("\xa0","",regex=False)
            .str.replace(".0","",regex=False)
            .str.replace(r"\s+","",regex=True)
            .str.replace(r"-\d+$","",regex=True)   # e.g. B07H8Q2JGG-1 → B07H8Q2JGG
            .replace({"NAN":np.nan,"":np.nan,"NONE":np.nan,"-":np.nan}))

def _clean_nin_key(series):
    """Clean Noon NINs — KEEP digit suffix like -1 (it IS part of the Noon NIN, e.g. Z...Z-1)."""
    return (series.astype(str).str.upper().str.strip()
            .str.replace("\xa0","",regex=False)
            .str.replace(".0","",regex=False)
            .str.replace(r"\s+","",regex=True)
            .replace({"NAN":np.nan,"":np.nan,"NONE":np.nan,"-":np.nan}))

# Keep _clean_key as alias (used in Section A functions)
def _clean_key(series):
    return _clean_nin_key(series)

if "ASIN" in helper_raw.columns:
    helper_raw["ASIN"] = _clean_asin_key(helper_raw["ASIN"])
if "NC_NIN" in helper_raw.columns:
    helper_raw["NC_NIN"] = _clean_nin_key(helper_raw["NC_NIN"])
if "NMs_NIN" in helper_raw.columns:
    helper_raw["NMs_NIN"] = _clean_nin_key(helper_raw["NMs_NIN"])

# ── Brand clean ────────────────────────────────────────────────────────────
if "Brand" in helper_raw.columns:
    helper_raw["Brand"] = (helper_raw["Brand"].astype(str).str.strip()
                           .replace({"":np.nan,"nan":np.nan,"None":np.nan})
                           .fillna("Emami"))

# ── Pack Size numeric ──────────────────────────────────────────────────────
if "Pack Size" in helper_raw.columns:
    helper_raw["Pack Size"] = (pd.to_numeric(
        helper_raw["Pack Size"].astype(str).str.extract(r"(\d+)")[0],
        errors="coerce").fillna(0))

# ── VC Ideal Price numeric ─────────────────────────────────────────────────
if "VC Ideal Price" in helper_raw.columns:
    helper_raw["VC Ideal Price"] = (pd.to_numeric(
        helper_raw["VC Ideal Price"].astype(str).str.replace(",","",regex=False).str.strip(),
        errors="coerce").fillna(0))

INFO_COLS = [c for c in [
    "Brand","Product Category","Product Sub Category","Product Name",
    "Pack Type","Pack Size","VC Ideal Price","SC Ideal Price"
] if c in helper_raw.columns]

# ── Amazon helper ──────────────────────────────────────────────────────────
h_asin = (helper_raw[helper_raw["ASIN"].notna()]
          .sort_values("Brand", na_position="last")
          .drop_duplicates(subset=["ASIN"], keep="first")
          [["ASIN"] + INFO_COLS].copy())

# ── Noon Core helper ───────────────────────────────────────────────────────
h_nc = (helper_raw[helper_raw["NC_NIN"].notna()]
        .sort_values("Brand", na_position="last")
        .drop_duplicates(subset=["NC_NIN"], keep="first")
        [["NC_NIN"] + INFO_COLS].copy())

# ── Noon Minutes helper ────────────────────────────────────────────────────
h_nms = (helper_raw[helper_raw["NMs_NIN"].notna()]
         .sort_values("Brand", na_position="last")
         .drop_duplicates(subset=["NMs_NIN"], keep="first")
         [["NMs_NIN"] + INFO_COLS].copy())

print(f"\n✅ HELPER READY")
print(f"   Amazon helper rows     : {len(h_asin)}")
print(f"   Noon Core helper rows  : {len(h_nc)}")
print(f"   Noon Minutes helper rows: {len(h_nms)}")
print(f"\n   Sample Amazon ASIN    :")
print(h_asin["ASIN"].head(5).tolist())
print(f"\n   Sample NC NIN         :")
print(h_nc["NC_NIN"].head(5).tolist())
print(f"\n   Sample NMs NIN        :")
print(h_nms["NMs_NIN"].head(5).tolist())
print("\nDetected Columns:")
print("ASIN      :", _asin_col)
print("NC_NIN    :", _nc_col)
print("NMs_NIN   :", _nms_col)

print("Brand     :", "Brand" in helper_raw.columns)


  HELPER FILE PROCESSING

📌 Available Sheets: ['Master Sheet']

📌 Raw Helper Columns:
['S.No.', 'Brand', 'Priority', 'Category', 'Range', 'Product Type', 'Product Description', 'Product Description(Cleaned)', 'Focus Range/ NPD', 'Size', 'Variation', 'Barcode', 'Emami  SAP code', 'Truebell SAP Code', 'HS CODE', 'Origin', 'Case Size', 'Cost Price', 'Shelf Life', 'TB SOH', 'Noon eCom', 'Noon Minutes', 'Amazon', 'Amazon ASIN\n(Vendor Central)', 'Product Code\n(Noon Minutes)', 'Product Code\n(Noon Core)', 'Remarks', 'Pack Type', 'VC Ideal Price']

✅ Renamed Columns:
['S.No.', 'Brand', 'Priority', 'Product Category', 'Range', 'Product Sub Category', 'Product Name', 'Product Description(Cleaned)', 'Focus Range/ NPD', 'Pack Size', 'Variation', 'Barcode', 'Emami  SAP code', 'Truebell SAP Code', 'HS CODE', 'Origin', 'Case Size', 'Cost Price', 'Shelf Life', 'TB SOH', 'Noon eCom', 'Noon Minutes', 'Amazon', 'ASIN', 'NMs_NIN', 'NC_NIN', 'Remarks', 'Pack Type', 'VC Ideal Price', 'SC Ideal Price']

✅

## 12 · Noon Inventory Dashboard (Noon & Noon Minutes)

In [26]:
# ══════════════════════════════════════════════════════════════════════════
# ██  SECTION A — INVENTORY PROCESSING (Amazon · Noon Core · Noon Minutes)
#
#  ORDER:
#   A-0  Shared helper functions
#   A-1  Noon Core inventory   → nc_inv_df
#   A-2  Noon Minutes inventory→ nms_inv_df
#   A-3  Amazon inventory DOS  → amz_inv_df  (needs amz_rev_df from Cell 10)
#   A-4  Summary
# ══════════════════════════════════════════════════════════════════════════

# ── A-0  SHARED HELPERS ────────────────────────────────────────────────────

_MONTH_NUM = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
              "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}

def _read_file(fp):
    if fp.lower().endswith(".csv"):
        return pd.read_csv(fp)
    return pd.read_excel(fp)

def _clean_num(series):
    return (pd.to_numeric(
        series.astype(str)
        .str.replace("AED","",regex=False).str.replace(",","",regex=False)
        .str.replace("\xa0","",regex=False).str.strip(),
        errors="coerce").fillna(0))
def _clean_key(obj):
    """
    Clean Noon NINs.
    Handles both Series and DataFrame.
    """

    if isinstance(obj, pd.DataFrame):
        print("⚠️ Duplicate NIN columns detected. Using first column.")
        obj = obj.iloc[:, 0]

    return (
        obj.astype(str)
           .str.upper()
           .str.strip()
           .str.replace("\xa0", "", regex=False)
           .str.replace(".0", "", regex=False)
           .str.replace(r"\s+", "", regex=True)
           .replace({
               "NAN": np.nan,
               "": np.nan,
               "NONE": np.nan,
               "-": np.nan
           })
    )
# ═══════════════════════════════════════════════════════════════════════
# LOAD PRE-MERGED NOON / NMS INVENTORY FILES
# ═══════════════════════════════════════════════════════════════════════

def load_monthly_merged_files(root, months, prefix, label):

    dfs = []

    for month in months:

        fp = os.path.join(
            root,
            month,
            f"{prefix}_{month}_{YEAR}.xlsx"
        )

        if not os.path.exists(fp):

            print(f"⚠️ [{label}] Missing : {fp}")
            continue

        try:

            df = pd.read_excel(
                fp,
                dtype=str
            )

            df.columns = (
                df.columns
                .astype(str)
                .str.strip()
            )

            dfs.append(df)

            print(
                f"✅ [{label}] {month} "
                f"({len(df):,} rows)"
            )

        except Exception as e:

            print(
                f"❌ [{label}] Failed : {fp}"
            )

            print(e)

    if dfs:

        final_df = pd.concat(
            dfs,
            ignore_index=True
        )

        print(
            f"\n✅ [{label}] Total Rows : "
            f"{len(final_df):,}"
        )

        return final_df

    return pd.DataFrame()

def _resolve_noon_date(df, label=""):
    """
    Deterministically compute ONE clean 'Date' Series for noon/NMs inventory rows,
    independent of glob() order, concat column order, or which files were present.

    Priority per row (NOT per column — every row picks its own best source):
      1. A genuine "Date"-named column (already-parsed dates, e.g. from a
         pre-saved merged_*.xlsx file built by the consolidation step).
      2. "Refresh Time" / "Snapshot Date" / "Report Date" text columns
         (e.g. "17 Jun 2026, 11:05:00" from raw daily exports).
      3. Derived from the day-folder name (_day / _month columns), for files
         that carry no date information of their own.

    Any resulting date outside 2020–2035 is treated as invalid (NaT) — this
    catches stray old files / bad parses instead of silently mixing them in.
    """
    cols_lower = {c.strip().lower(): c for c in df.columns}

    # Candidate source columns, in priority order, matched by exact lowercase name
    date_priority = ["date", "snapshot date", "report date", "snapshot_date", "refresh time"]
    candidates = [cols_lower[name] for name in date_priority if name in cols_lower]

    if candidates:
        # Build a per-row coalesced raw value: first non-null candidate, in priority order
        raw_value = pd.Series(pd.NA, index=df.index, dtype=object)
        for col_name in candidates:
            col = df[col_name]
            if isinstance(col, pd.DataFrame):   # safety: duplicate-named column slipped through
                col = col.iloc[:, 0]
            still_missing = raw_value.isna()
            raw_value.loc[still_missing] = col.loc[still_missing]

        # Parse: datetimes pass straight through; strings get comma-stripped + dayfirst parse
        def _parse_one(v):
    
            if pd.isna(v):
                return pd.NaT

            if isinstance(v, (pd.Timestamp, np.datetime64)):
                return pd.Timestamp(v).normalize()

            s = (
                str(v)
                .replace("\u202f", " ")
                .replace("\xa0", " ")
                .strip()
            )

            if not s:
                return pd.NaT

            return pd.to_datetime(
                s,
                errors="coerce",
                format="mixed"
            )

        parsed = raw_value.map(_parse_one)
    else:
        parsed = pd.Series(pd.NaT, index=df.index)

    # ── Fallback to day-folder name for any row still missing a date ───────
    missing = parsed.isna()
    if missing.any() and "_day" in df.columns and "_month" in df.columns:
        def _from_day_folder(r):
            s = str(r["_day"]).strip()
            month_num = _MONTH_NUM.get(str(r["_month"]).strip().lower(), 1)
            try:
                ts = pd.to_datetime(s, dayfirst=True, errors="coerce")
                if pd.notna(ts) and 2020 <= ts.year <= 2035:
                    return ts.normalize()
            except Exception:
                pass
            m = re.search(r"(\d+)", s)
            day = int(m.group(1)) if m else 1
            if day < 1 or day > 31:
                day = 1
            try:
                return pd.Timestamp(year=YEAR, month=month_num, day=day)
            except Exception:
                return pd.NaT
        parsed.loc[missing] = df.loc[missing].apply(_from_day_folder, axis=1)

    parsed = pd.to_datetime(parsed, errors="coerce").dt.normalize()

    # ── Year sanity filter — reject anything outside a plausible range ─────
    valid = parsed.dt.year.between(2020, 2035)
    bad_n = int((~valid & parsed.notna()).sum())
    if bad_n > 0:
        bad_years = sorted(parsed[~valid & parsed.notna()].dt.year.unique().tolist())
        print(f"  ⚠️  [{label}] Rejected {bad_n} rows with implausible year {bad_years}")
    parsed = parsed.where(valid, other=pd.NaT)
    print(f"\n[{label}] Parsed Date Range:")
    print(parsed.min(), "→", parsed.max())

    print(f"\n[{label}] Last 20 Dates:")
    print(
        sorted(
            parsed.dropna()
            .dt.strftime("%Y-%m-%d")
            .unique()
        )[-20:]
    )

    return parsed
def standardise_noon_inv(raw, stock_col, label):
    
    if raw.empty:
        return pd.DataFrame()

    df = raw.copy()

    col_map = {}
    stock_mapped = False

    for c in df.columns:

        cl = c.strip().lower()

        if cl in [
            "zsku",
            "sku",
            "nin",
            "nins",
            "product code",
            "product_code"
        ]:
            col_map[c] = "NINs"

        elif cl in [
            "barcode",
            "ean"
        ]:
            col_map[c] = "Barcode"

        elif (
            cl == stock_col.lower()
            and not stock_mapped
        ):
            col_map[c] = "Available Inventory"
            stock_mapped = True

    df.rename(
        columns=col_map,
        inplace=True
    )

    df = df.loc[
        :,
        ~df.columns.duplicated()
    ]

    if "NINs" not in df.columns:

        raise KeyError(
            f"[{label}] NIN column not found"
        )

    df["NINs"] = _clean_key(
        df["NINs"]
    )

    if "Barcode" not in df.columns:
        df["Barcode"] = ""

    df["Barcode"] = (
        df["Barcode"]
        .astype(str)
        .str.strip()
    )

    df["Available Inventory"] = _clean_num(
        df["Available Inventory"]
    )

    df["Date"] = (
        pd.to_datetime(
            df["Date"],
            errors="coerce"
        )
        .dt.normalize()
    )

    df = df[
        df["Date"].notna()
    ]

    df = df[
        df["NINs"].notna()
    ]

    print(
        f"\n[{label}] "
        f"{df['Date'].min().date()} → "
        f"{df['Date'].max().date()}"
    )
    
    agg = (
        df.groupby(
            ["NINs","Date"],
            as_index=False,
            dropna=False
        )
        .agg({
            "Available Inventory":"sum",
            "Barcode":"first"
        })
    )

    return agg


def compute_l30_dos(inv_df, rev_df, id_col, inventory_is_current=False):
    """
    30-calendar-day Last 30 Days Sales + DOS, with channel-aware date anchoring.

    T-2 RULE — applies to ALL channels (Noon, Noon Minutes, Amazon):
    ════════════════════════════════════════════════════════════════════
    For EVERY inventory snapshot date D, the 30-day window is:
        t_end   = D - 2 days          (T-2: latest confirmed sales)
        t_start = t_end - 29 days     (30 days inclusive)

    Verified correct against noon_dos_calculator.py:
        Inventory 19-Jun → T-2=17-Jun → window 19-May to 17-Jun = 609 units ✅

    NOTE on ASIN IDs for Noon/NMs:
        Sales CSVs have BOTH 'nsku' (col 7) and 'ZSKU' (col 8).
        ALWAYS use ZSKU — it is the correct Noon product identifier.
        nsku (e.g. N28940734A) is an internal warehouse SKU, NOT a NIN.
        Cells 32/34 drop nsku before rename so only ZSKU → ASIN.

    inv_df : [id_col, Date, Available Inventory]
    rev_df : [id_col, Date, total_units]  ← must use ZSKU as id_col for Noon/NMs
    inventory_is_current : True  → Noon/NMs (inv snapshot = T-0 current stock)
                           False → Amazon   (inv snapshot already at T-2)
    Both use the same T-2 formula — inventory_is_current kept for clarity.
    """
    df = inv_df.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df[df["Date"].notna()].copy()

    # ── Build daily revenue series ────────────────────────────────────────
    if rev_df is not None and not rev_df.empty:
        rev = rev_df.copy()
        if "total_units" not in rev.columns:
            rev["total_units"] = 0
        rev["total_units"] = _clean_num(rev["total_units"])
        rev["Date"] = (
            pd.to_datetime(
                rev["Date"],
                errors="coerce"
            )
            .dt.normalize()
        )
        rev = rev[rev["Date"].notna()]
        daily_rev = rev.groupby([id_col, "Date"], as_index=False)["total_units"].sum()
    else:
        daily_rev = pd.DataFrame(columns=[id_col, "Date", "total_units"])

    df = df.sort_values([id_col, "Date"])

    merged_parts = []

    for asin, inv_grp in df.groupby(id_col):
        inv_grp = inv_grp.copy()

        # Revenue series for this product
        sales_grp = daily_rev[daily_rev[id_col] == asin][["Date", "total_units"]].copy()

        if sales_grp.empty:
            inv_grp["Last 30 Days Sales"] = 0.0
        else:
            sales_grp["Date"] = pd.to_datetime(
                sales_grp["Date"],
                errors="coerce"
            ).dt.normalize()

            sales_grp = (
                sales_grp
                .set_index("Date")
                .sort_index()
            )["total_units"]
            last_sales_date = sales_grp.index.max()

            # T-2 RULE — same for ALL channels (verified vs noon_dos_calculator.py):
            #   t_end   = inv_date - 2 days      (T-2: latest confirmed sales)
            #   t_start = t_end    - 29 days      (30 days inclusive)
            #   e.g. Inventory 19-Jun → T-2=17-Jun → window 19-May→17-Jun = 609 ✅
            #
            # inventory_is_current=True  → Noon/NMs (inventory is T-0 snapshot)
            # inventory_is_current=False → Amazon   (inventory already at T-2)
            # Both use the same T-2 window formula — only difference is
            # that Amazon inv_date IS already the T-2 date itself.
            l30_list = []
            for inv_date in inv_grp["Date"]:
                if inventory_is_current:
                    # Noon/NMs: apply T-2 offset from the inventory snapshot date
                    t_end   = inv_date - pd.Timedelta(days=2)
                    t_start = t_end    - pd.Timedelta(days=29)
                else:
                    # Amazon: inventory snapshot is already at T-2
                    # Apply T-2 from inventory date (same formula, same result)
                    t_end   = inv_date - pd.Timedelta(days=2)
                    t_start = t_end    - pd.Timedelta(days=29)
                # Cap t_end to last available sales date
                if t_end > last_sales_date:
                    t_end   = last_sales_date
                    t_start = t_end - pd.Timedelta(days=29)
                mask = (sales_grp.index >= t_start) & (sales_grp.index <= t_end)
                l30_list.append(float(sales_grp.loc[mask].sum()))

            inv_grp["Last 30 Days Sales"] = l30_list

        merged_parts.append(inv_grp)

    df = pd.concat(merged_parts, ignore_index=True) if merged_parts else df

    # ── Daily Sales Rate + DOS ────────────────────────────────────────────
    df["Daily Sales Rate"] = np.where(
        df["Last 30 Days Sales"] > 0,
        df["Last 30 Days Sales"] / 30,
        0
    )

    dos_raw = np.where(
        df["Daily Sales Rate"] > 0,
        df["Available Inventory"] / df["Daily Sales Rate"],
        np.nan
    )

    df["DOS"] = (
        pd.Series(dos_raw, dtype=float)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .round(0)
        .clip(upper=3650)
        .astype("Int64")
    )

    df.drop(columns=["Daily Sales Rate"], inplace=True, errors="ignore")
    return df


def _apply_helper_map(inv_df, h_df, h_key, id_col, info_cols):
    """Map helper metadata onto inv_df[id_col] → h_df[h_key]."""
    inv_df = inv_df.copy()
    # Drop any pre-existing info cols to avoid _x/_y
    inv_df.drop(columns=[c for c in info_cols if c in inv_df.columns], inplace=True, errors="ignore")

    h = (h_df[[h_key] + [c for c in info_cols if c in h_df.columns]]
         .dropna(subset=[h_key])
         .drop_duplicates(subset=[h_key])
         .rename(columns={h_key: id_col}))

    inv_df = inv_df.merge(h, on=id_col, how="left")

    # Clean up any _x/_y
    for col in list(inv_df.columns):
        if col.endswith("_x"):
            base = col[:-2]
            y = base + "_y"
            inv_df[base] = inv_df[col].combine_first(inv_df.get(y, pd.Series(dtype=object)))
            inv_df.drop(columns=[col, y], inplace=True, errors="ignore")

    for col in info_cols:
        if col not in inv_df.columns:
            inv_df[col] = np.nan
        inv_df[col] = (inv_df[col]
                       .replace({"":np.nan,"nan":np.nan,"None":np.nan})
                       .fillna("Unknown"))
    return inv_df


INFO_COLS = [c for c in [
    "Brand","Product Category","Product Sub Category","Product Name",
    "Pack Type","Pack Size","VC Ideal Price","SC Ideal Price"
] if c in (list(h_asin.columns) + list(h_nc.columns) + list(h_nms.columns))]

VENDOR_ONLY = ["Vendor Confirmation %","PO_received_amount","PO_received_Units",
               "Receive_Fill_Rate","Overall Vendor Lead Time (days)",
               "Aged 90+ Days Sellable Inventory","Aged 90+ Days Sellable Units",
               "Sellable Inventory Amount","Unsellable On Hand Inventory Amount",
               "Unsellable On Hand Units","Open POs"]


# ══════════════════════════════════════════════════════════════════════════
# A-1  NOON CORE INVENTORY
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*65)
print("  A-1  NOON CORE – Inventory")
print("="*65)

nc_raw = load_monthly_merged_files(
    NOON_INV_ROOT,
    MONTHS,
    "merged_noon",
    "NC"
)
if not nc_raw.empty:
    nc_clean = standardise_noon_inv(nc_raw, "Quantity", "NC")
    print(f"\n  Aggregated: {len(nc_clean):,} rows | {nc_clean['NINs'].nunique()} NINs | "
          f"{nc_clean['Date'].min().date()} → {nc_clean['Date'].max().date()}")

    # Revenue for L30: use noon_rev_df if already built, else build here
    if 'noon_rev_df' in dir() and not noon_rev_df.empty:
        _nc_rev = noon_rev_df[["ASIN","Date","total_units"]].rename(columns={"ASIN":"NINs"})
    else:
        _nc_rev = pd.DataFrame(columns=["NINs","Date","total_units"])
        print("noon_rev_df not yet available — L30 Sales = 0 (re-run after Cell 18)")

    nc_clean = compute_l30_dos(nc_clean, _nc_rev, "NINs", inventory_is_current=True)
    

    
    # Apply helper via NC_NIN
    
    # Helper match check
    h_nc["NC_NIN"] = _clean_key(h_nc["NC_NIN"])

    nc_keys = set(nc_clean["NINs"].dropna())
    helper_nc_keys = set(h_nc["NC_NIN"].dropna())

    print("\nNoon Core Merge Check")
    print("Inventory NINs :", len(nc_keys))
    print("Helper NINs    :", len(helper_nc_keys))
    print("Matched NINs   :", len(nc_keys & helper_nc_keys))

    nc_clean = _apply_helper_map(
        nc_clean,
        h_nc,
        "NC_NIN",
        "NINs",
        INFO_COLS
    )

    print("\nPOST MERGE DEBUG")

    print(
        nc_clean[
            ["NINs","Brand","Product Name"]
        ]
        .head(20)
    )

    print("\nBrand Null Count:")
    print(
        nc_clean["Brand"].isna().sum()
    )

    print("\nBrand Values:")
    print(
        nc_clean["Brand"]
        .value_counts(dropna=False)
        .head(10)
    )    

    nc_inv_df = nc_clean.rename(columns={"NINs":"ASIN"}).copy()
    nc_inv_df["Sales Channel"] = "Noon"
    for col in VENDOR_ONLY:
        if col not in nc_inv_df.columns:
            nc_inv_df[col] = 0
    nc_inv_df = nc_inv_df.sort_values(["ASIN","Date"])

    print(f"\n✅ NOON CORE READY")
    print(f"   Shape           : {nc_inv_df.shape}")
    print(f"   Unique ASINs    : {nc_inv_df['ASIN'].nunique()}")
    print(f"   Date range      : {nc_inv_df['Date'].min().date()} → {nc_inv_df['Date'].max().date()}")
    print(f"   Avg DOS         : {nc_inv_df['DOS'].mean():.1f}")
    print(f"   Avg L30 Sales   : {nc_inv_df['Last 30 Days Sales'].mean():.1f}")
    print(f"\n   Brand fill:")
    print(nc_inv_df["Brand"].value_counts(dropna=False).head(6).to_string())
    print(f"\n   Category fill:")
    print(nc_inv_df["Product Category"].value_counts(dropna=False).head(6).to_string())
    print(f"\n   Sample rows:")
    print(nc_inv_df[["ASIN","Date","Available Inventory","Last 30 Days Sales","DOS","Brand","Product Name"]].head(8).to_string(index=False))
else:
    nc_inv_df = pd.DataFrame()
    print("\n❌ No Noon Core inventory files found")


# ══════════════════════════════════════════════════════════════════════════
# A-2  NOON MINUTES INVENTORY
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*65)
print("  A-2  NOON MINUTES – Inventory")
print("="*65)

nms_raw = load_monthly_merged_files(
    NMS_INV_ROOT,
    MONTHS,
    "merged_nms",
    "NMs"
)

print("\nRAW NMS DATE RANGE")
print(
    pd.to_datetime(
        nms_raw["Date"],
        errors="coerce"
    ).min()
)

print(
    pd.to_datetime(
        nms_raw["Date"],
        errors="coerce"
    ).max()
)

print(
    sorted(
        pd.to_datetime(
            nms_raw["Date"],
            errors="coerce"
        )
        .dt.strftime("%Y-%m-%d")
        .dropna()
        .unique()
    )[-20:]
)

if not nms_raw.empty:
    nms_clean = standardise_noon_inv(nms_raw, "Stock", "NMs")
    print(f"\n  Aggregated: {len(nms_clean):,} rows | {nms_clean['NINs'].nunique()} NINs | "
          f"{nms_clean['Date'].min().date()} → {nms_clean['Date'].max().date()}")

    if 'nms_rev_df' in dir() and not nms_rev_df.empty:
        _nms_rev = nms_rev_df[["ASIN","Date","total_units"]].rename(columns={"ASIN":"NINs"})
    else:
        _nms_rev = pd.DataFrame(columns=["NINs","Date","total_units"])
        print("  ℹ️  nms_rev_df not yet available — L30 Sales = 0 (re-run after Cell 20)")

    nms_clean = compute_l30_dos(nms_clean, _nms_rev, "NINs", inventory_is_current=True)
    h_nms["NMs_NIN"] = _clean_key(h_nms["NMs_NIN"])

    nms_keys = set(nms_clean["NINs"].dropna())
    helper_nms_keys = set(h_nms["NMs_NIN"].dropna())

    print("\nNoon Minutes Merge Check")
    print("Inventory NINs :", len(nms_keys))
    print("Helper NINs    :", len(helper_nms_keys))
    print("Matched NINs   :", len(nms_keys & helper_nms_keys))

    nms_clean = _apply_helper_map(
        nms_clean,
        h_nms,
        "NMs_NIN",
        "NINs",
        INFO_COLS
    )

    nms_inv_df = nms_clean.rename(columns={"NINs":"ASIN"}).copy()
    nms_inv_df["Sales Channel"] = "Noon Minutes"
    for col in VENDOR_ONLY:
        if col not in nms_inv_df.columns:
            nms_inv_df[col] = 0
    nms_inv_df = nms_inv_df.sort_values(["ASIN","Date"])

    print(f"\n✅ NOON MINUTES READY")
    print(f"   Shape           : {nms_inv_df.shape}")
    print(f"   Unique ASINs    : {nms_inv_df['ASIN'].nunique()}")
    print(f"   Date range      : {nms_inv_df['Date'].min().date()} → {nms_inv_df['Date'].max().date()}")
    print(f"   Avg DOS         : {nms_inv_df['DOS'].mean():.1f}")
    print(f"   Avg L30 Sales   : {nms_inv_df['Last 30 Days Sales'].mean():.1f}")
    print(f"\n   Brand fill:")
    print(nms_inv_df["Brand"].value_counts(dropna=False).head(6).to_string())
    print(f"\n   Sample rows:")
    print(nms_inv_df[["ASIN","Date","Available Inventory","Last 30 Days Sales","DOS","Brand","Product Name"]].head(8).to_string(index=False))
else:
    nms_inv_df = pd.DataFrame()
    print("\n❌ No Noon Minutes inventory files found")


# ══════════════════════════════════════════════════════════════════════════
# A-3  AMAZON – DOS / Last 30 Days Sales
# NOTE: amz_rev_df is built in Cell 10.
#       If Cell 10 ran first → DOS is computed now.
#       If not → DOS = 0 (re-run this cell after Cell 10).
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*65)
print("  A-3  AMAZON – DOS / Last 30 Days Sales")
print("="*65)

if 'amz_inv_df' in dir() and not amz_inv_df.empty:

    if "Available Inventory" not in amz_inv_df.columns:
        print("  ❌  'Available Inventory' not in amz_inv_df")
    else:
        # Clean ASIN with robust function
        amz_inv_df["ASIN"] = _clean_key(amz_inv_df["ASIN"].fillna(""))

        _amz_rev = pd.DataFrame()
        if 'amz_rev_df' in dir() and not amz_rev_df.empty:
            _amz_rev = amz_rev_df[["ASIN","Date","total_units"]].copy()
            _amz_rev["ASIN"] = _clean_key(_amz_rev["ASIN"].fillna(""))
        else:
            print("  ℹ️  amz_rev_df not yet — L30 = 0 (re-run after Cell 10)")

        # Drop old L30/DOS if re-running
        amz_inv_df.drop(columns=["Last 30 Days Sales","DOS","total_units"],
                        inplace=True, errors="ignore")

        amz_inv_df = compute_l30_dos(amz_inv_df, _amz_rev, "ASIN", inventory_is_current=False)

        # Re-apply Amazon helper (also with robust key cleaning)
        # Clean helper keys again before merge
        h_asin["ASIN"] = _clean_key(h_asin["ASIN"])

        inventory_asins = set(amz_inv_df["ASIN"].dropna())
        helper_asins = set(h_asin["ASIN"].dropna())

        print("\nAmazon Merge Check")
        print("Inventory ASINs :", len(inventory_asins))
        print("Helper ASINs    :", len(helper_asins))
        print("Matched ASINs   :", len(inventory_asins & helper_asins))

        amz_inv_df = _apply_helper_map(
            amz_inv_df,
            h_asin,
            "ASIN",
            "ASIN",
            INFO_COLS
        )

        print(f"\n✅ AMAZON DOS COMPUTED")
        print(f"   Shape          : {amz_inv_df.shape}")
        print(f"   Avg DOS        : {amz_inv_df['DOS'].mean():.1f}")
        print(f"   Avg L30 Sales  : {amz_inv_df['Last 30 Days Sales'].mean():.1f}")
        print(f"\n   Brand fill:")
        print(amz_inv_df["Brand"].value_counts(dropna=False).head(6).to_string())
        print(f"\n   Sample rows:")
        print(amz_inv_df[["ASIN","Date","Available Inventory","Last 30 Days Sales","DOS","Brand","Product Name"]].head(8).to_string(index=False))
        print(f"\n   Test ASIN B0G64MSXJS:")
        _t = amz_inv_df[amz_inv_df["ASIN"]=="B0G64MSXJS"]
        if not _t.empty:
            print(_t[["ASIN","Brand","Product Name","Product Category","Product Sub Category"]].head(3).to_string(index=False))
        else:
            print("   Not found in inventory.")
else:
    print("\n⚠️  amz_inv_df not available — run Cell 7 first")


# ══════════════════════════════════════════════════════════════════════════
# A-4  INVENTORY SUMMARY
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*65)
print("  A-4  INVENTORY SUMMARY")
print("="*65)

for _lbl, _df in [
    ("Amazon        (Vendor Central)", amz_inv_df  if 'amz_inv_df'  in dir() else pd.DataFrame()),
    ("Noon Core     (NC)             ", nc_inv_df   if 'nc_inv_df'   in dir() else pd.DataFrame()),
    ("Noon Minutes  (NMs)            ", nms_inv_df  if 'nms_inv_df'  in dir() else pd.DataFrame()),
]:
    if _df is not None and not _df.empty:
        print(f"\n  {_lbl}")
        print(f"    Rows          : {len(_df):,}")
        print(f"    Unique IDs    : {_df['ASIN'].nunique()}")
        _dates = pd.to_datetime(_df['Date'], errors='coerce').dropna()
        print(f"    Date range    : {_dates.min().date()} → {_dates.max().date()}")
        if "DOS" in _df.columns:
            print(f"    Avg DOS       : {_df['DOS'].mean():.1f}")
        if "Last 30 Days Sales" in _df.columns:
            print(f"    Avg L30       : {_df['Last 30 Days Sales'].mean():.1f}")
        if "Available Inventory" in _df.columns:
            print(f"    Total SOH     : {int(_df['Available Inventory'].sum()):,}")
        if "Brand" in _df.columns:
            unk = (_df["Brand"]=="Unknown").sum()

            print(f"    Brand Unknown : {unk} / {len(_df)}")

    else:        print(f"\n  {_lbl} : ❌ No data")


  A-1  NOON CORE – Inventory
✅ [NC] may (1,162 rows)
✅ [NC] jun (8,592 rows)
✅ [NC] july (11,092 rows)
✅ [NC] aug (7,825 rows)

✅ [NC] Total Rows : 28,671

[NC] 2026-05-18 → 2026-08-24

  Aggregated: 8,123 rows | 100 NINs | 2026-05-18 → 2026-08-24

Noon Core Merge Check
Inventory NINs : 100
Helper NINs    : 99
Matched NINs   : 94

POST MERGE DEBUG
                        NINs                Brand  \
0   Z00EA9B233750B962B4BEZ-1  Emami 7 Oils in One   
1   Z00EA9B233750B962B4BEZ-1  Emami 7 Oils in One   
2   Z00EA9B233750B962B4BEZ-1  Emami 7 Oils in One   
3   Z00EA9B233750B962B4BEZ-1  Emami 7 Oils in One   
4   Z00EA9B233750B962B4BEZ-1  Emami 7 Oils in One   
5   Z00EA9B233750B962B4BEZ-1  Emami 7 Oils in One   
6   Z00EA9B233750B962B4BEZ-1  Emami 7 Oils in One   
7   Z00EA9B233750B962B4BEZ-1  Emami 7 Oils in One   
8   Z00EA9B233750B962B4BEZ-1  Emami 7 Oils in One   
9   Z00EA9B233750B962B4BEZ-1  Emami 7 Oils in One   
10  Z00EA9B233750B962B4BEZ-1  Emami 7 Oils in One   
11  Z00EA9B23

In [27]:
inv_ids = set(
    nms_inv_df["ASIN"]
    .dropna()
    .astype(str)
)

rev_ids = set(
    nms_rev_df["ASIN"]
    .dropna()
    .astype(str)
)

print("Inventory:", len(inv_ids))
print("Revenue:", len(rev_ids))
print("Matched:", len(inv_ids & rev_ids))

Inventory: 16
Revenue: 16
Matched: 16


In [28]:
print(
    nms_inv_df["ASIN"].head()
)

print(
    nms_rev_df["ASIN"].head()
)

0    Z045B32EE54200CC7E00FZ-1
1    Z045B32EE54200CC7E00FZ-1
2    Z045B32EE54200CC7E00FZ-1
3    Z045B32EE54200CC7E00FZ-1
4    Z045B32EE54200CC7E00FZ-1
Name: ASIN, dtype: object
0    Z045B32EE54200CC7E00FZ-1
1    Z045B32EE54200CC7E00FZ-1
2    Z045B32EE54200CC7E00FZ-1
3    Z045B32EE54200CC7E00FZ-1
4    Z045B32EE54200CC7E00FZ-1
Name: ASIN, dtype: string


In [29]:
nms_inv_df.to_csv("nms_inventory.csv", index=False)

In [30]:
nms_inv_df

,ASIN,Date,Available Inventory,Barcode,Last 30 Days Sales,DOS,Brand,Product Category,Product Sub Category,Product Name,...,PO_received_amount,PO_received_Units,Receive_Fill_Rate,Overall Vendor Lead Time (days),Aged 90+ Days Sellable Inventory,Aged 90+ Days Sellable Units,Sellable Inventory Amount,Unsellable On Hand Inventory Amount,Unsellable On Hand Units,Open POs
0,Z045B32EE54200CC7E00FZ-1,2026-07-25,472.0,8901248451239,0.0,0,Emami 7 Oils in One,Hair Care,Hair Oil,Emami 7 in 1 Hair Oil Black Seed 300ml,...,0,0,0,0,0,0,0,0,0,0
1,Z045B32EE54200CC7E00FZ-1,2026-07-26,374.0,8901248451239,0.0,0,Emami 7 Oils in One,Hair Care,Hair Oil,Emami 7 in 1 Hair Oil Black Seed 300ml,...,0,0,0,0,0,0,0,0,0,0
2,Z045B32EE54200CC7E00FZ-1,2026-07-27,478.0,8901248451239,1.0,3650,Emami 7 Oils in One,Hair Care,Hair Oil,Emami 7 in 1 Hair Oil Black Seed 300ml,...,0,0,0,0,0,0,0,0,0,0
3,Z045B32EE54200CC7E00FZ-1,2026-07-28,476.0,8901248451239,1.0,3650,Emami 7 Oils in One,Hair Care,Hair Oil,Emami 7 in 1 Hair Oil Black Seed 300ml,...,0,0,0,0,0,0,0,0,0,0
4,Z045B32EE54200CC7E00FZ-1,2026-07-29,475.0,8901248451239,2.0,3650,Emami 7 Oils in One,Hair Care,Hair Oil,Emami 7 in 1 Hair Oil Black Seed 300ml,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1066,ZF4F3189FA13EFE7A2689Z-1,2026-08-16,208.0,8901248104043,143.0,44,Himani Navratna,Hair Care,Hair Oil,Himani Navratna Oil 200ml,...,0,0,0,0,0,0,0,0,0,0
1067,ZF4F3189FA13EFE7A2689Z-1,2026-08-17,238.0,8901248104043,136.0,52,Himani Navratna,Hair Care,Hair Oil,Himani Navratna Oil 200ml,...,0,0,0,0,0,0,0,0,0,0
1068,ZF4F3189FA13EFE7A2689Z-1,2026-08-18,273.0,8901248104043,139.0,59,Himani Navratna,Hair Care,Hair Oil,Himani Navratna Oil 200ml,...,0,0,0,0,0,0,0,0,0,0
1069,ZF4F3189FA13EFE7A2689Z-1,2026-08-19,265.0,8901248104043,136.0,58,Himani Navratna,Hair Care,Hair Oil,Himani Navratna Oil 200ml,...,0,0,0,0,0,0,0,0,0,0


In [31]:
# sell_through is now computed once, centrally, inside add_metrics()
# (Last 30 Days Sales / 30, per ASIN x Date, same formula for every
# channel) -- no longer pre-computed here since it gets overwritten
# there anyway, and "Last 30 Days Sales" isn't finalized until after
# the DOS merge happens later in the pipeline.

In [32]:
# # Dashboard helpers: prefer already-processed inventory frames to avoid duplication/errors
# import pandas as pd

# # Noon (Noon Core)
# if 'noon_inv_df' in globals():
#     noon_dashboard_noon = noon_inv_df
# elif 'nc_clean' in globals():
#     noon_dashboard_noon = nc_clean.copy()
# else:
#     noon_dashboard_noon = pd.DataFrame()

# # Noon Minutes
# if 'nms_inv_df' in globals():
#     noon_dashboard_nms = nms_inv_df
# elif 'nms_clean' in globals():
#     noon_dashboard_nms = nms_clean.copy()
# else:
#     noon_dashboard_nms = pd.DataFrame()

# print(f"Dashboard sources: noon rows={len(noon_dashboard_noon):,}, nms rows={len(noon_dashboard_nms):,}")


## 13 · Amazon Sell-Through (90-day file)

In [33]:
# ══════════════════════════════════════════════════════════════════════════
# AMAZON SELL-THROUGH — REMOVED
# Sell-through is no longer computed from a separate 90-day Sales_ASIN
# export merged at the ASIN level. It is now computed simply as:
#     sell_through = Last 30 Days Sales / 30
# directly inside add_metrics(), per ASIN x Date, for every channel
# (Amazon, Noon, Noon Minutes) -- see the SELL THROUGH block in the
# add_metrics() function below. This cell (and the amz_st dataframe it
# used to produce) is intentionally left empty.
# ══════════════════════════════════════════════════════════════════════════


## 14 · Inventory Metrics Builder
> **NEW BASE (helper-ASIN-priority)**: every output below now starts from the
> full `<helper ASIN> × <every date seen in that channel>` matrix, built from
> the channel's helper slice (`h_asin` / `h_nc` / `h_nms`). Inventory, revenue,
> and DOS are LEFT-merged ONTO this base.
>
> **This fixes the dropped-ASIN bug**: previously, an ASIN with sales or ad
> spend on a given date but NO inventory row that date was silently dropped,
> because the old base was `inv.copy()` (inventory only). Now it's never
> dropped — it just gets `Available Inventory = 0` for that date, while its
> real `total_sales` / `total_units` / `ads_*` values are preserved.
>
> **DOS is now computed IN-NOTEBOOK** (ported from `dos_calculator.py` — same
> T-2 / 30-day window logic) instead of being loaded from 3 separate pre-built
> Excel files. `Available Inventory` is overwritten with the DOS table's
> Inventory value wherever a match exists (it's the aggregated, de-duplicated
> figure).

In [34]:
# ══════════════════════════════════════════════════════════════════════════
# DOS (Days of Stock) CALCULATOR — IN-NOTEBOOK VERSION
# (replaces dos_calculator.py — same logic, now runs inline so the
#  notebook is self-contained and doesn't depend on 3 separate Excel
#  outputs being generated and saved by a standalone script first)
# ──────────────────────────────────────────────────────────────────────────
# WINDOW LOGIC (same for all 3 channels, per inventory date):
#   T-2   = inventory_date - 2 days     (latest available sales)
#   Start = T-2 - 29 days               (30 days inclusive)
# Keys that had sales in the window but are NOT in the inventory on that
# date (sold out / delisted) are included with Inventory = 0, DOS = 0 —
# this keeps Last 30 Days Sales totals matching actual sales data.
# ══════════════════════════════════════════════════════════════════════════

def compute_dos_table(inventory_df, sales_df, key_col,
                       inv_qty_col="Available Inventory",
                       sales_qty_col="total_units"):
    """
    For every Key × inventory Date:
      1. Aggregate stock across all rows for that key/date.
      2. Calculate Last 30 Days Sales in window [T-2-29, T-2].
      3. Add any keys that had sales in the window but are missing from
         the inventory on that date (sold out / delisted) -> Inventory = 0.
      4. DOS = Inventory / (L30Sales / 30), 0 if no sales, capped at 3650.

    Returns: <key_col> | Date | Inventory | Last 30 Days Sales | DOS
    """

    if inventory_df is None or inventory_df.empty:
        return pd.DataFrame(columns=[key_col, "Date", "Inventory",
                                      "Last 30 Days Sales", "DOS"])

    inv = inventory_df[[key_col, "Date", inv_qty_col]].copy()
    inv["Date"] = pd.to_datetime(inv["Date"], errors="coerce").dt.normalize()
    inv = inv[inv["Date"].notna()]

    inv = (
        inv.groupby([key_col, "Date"], as_index=False)[inv_qty_col]
        .sum()
        .rename(columns={inv_qty_col: "Inventory"})
    )

    if sales_df is not None and not sales_df.empty and sales_qty_col in sales_df.columns:
        sales = sales_df[[key_col, "Date", sales_qty_col]].copy()
        sales["Date"] = pd.to_datetime(sales["Date"], errors="coerce").dt.normalize()
        sales = sales[sales["Date"].notna()]
        sales[sales_qty_col] = pd.to_numeric(sales[sales_qty_col], errors="coerce").fillna(0)
    else:
        sales = pd.DataFrame(columns=[key_col, "Date", sales_qty_col])

    all_inv_dates = sorted(inv["Date"].unique())
    records = []

    for inv_date in all_inv_dates:

        available_sales_dates = sales.loc[sales["Date"] <= inv_date, "Date"]
        if available_sales_dates.empty:
            continue

        t_end = available_sales_dates.max()
        t_start = t_end - pd.Timedelta(days=29)

        win_sales = sales[(sales["Date"] >= t_start) & (sales["Date"] <= t_end)]

        key_sales = (
            win_sales.groupby(key_col)[sales_qty_col]
            .sum()
            .reset_index()
            .rename(columns={sales_qty_col: "L30Sales"})
        )

        inv_today = inv[inv["Date"] == inv_date][[key_col, "Inventory"]].copy()

        merged = inv_today.merge(key_sales, on=key_col, how="outer")
        merged["Inventory"] = merged["Inventory"].fillna(0)
        merged["L30Sales"] = merged["L30Sales"].fillna(0)

        # Drop rows where neither inventory nor sales ever existed
        merged = merged[(merged["Inventory"] > 0) | (merged["L30Sales"] > 0)]

        for _, row in merged.iterrows():
            inventory = row["Inventory"]
            l30 = row["L30Sales"]
            daily_avg = l30 / 30
            dos = round(inventory / daily_avg, 1) if daily_avg > 0 else 0

            records.append({
                key_col: row[key_col],
                "Date": inv_date,
                "Inventory": inventory,
                "Last 30 Days Sales": l30,
                "DOS": dos,
            })

    result = pd.DataFrame(records)
    if result.empty:
        return pd.DataFrame(columns=[key_col, "Date", "Inventory",
                                      "Last 30 Days Sales", "DOS"])

    result["DOS"] = (
        pd.to_numeric(result["DOS"], errors="coerce")
        .fillna(0).round(0).clip(upper=3650).astype("Int64")
    )
    result = result.sort_values(["Date", key_col]).reset_index(drop=True)
    return result


print("📥 Computing DOS tables in-notebook (Amazon / Noon / Noon Minutes)...")

amz_dos_df = compute_dos_table(
    amz_inv_df, amz_rev_df, "ASIN",
    inv_qty_col="Available Inventory", sales_qty_col="total_units"
) if 'amz_inv_df' in dir() and not amz_inv_df.empty else pd.DataFrame()

noon_dos_df = compute_dos_table(
    nc_inv_df, noon_rev_df, "ASIN",
    inv_qty_col="Available Inventory", sales_qty_col="total_units"
) if 'nc_inv_df' in dir() and not nc_inv_df.empty else pd.DataFrame()

nms_dos_df = compute_dos_table(
    nms_inv_df, nms_rev_df, "ASIN",
    inv_qty_col="Available Inventory", sales_qty_col="total_units"
) if 'nms_inv_df' in dir() and not nms_inv_df.empty else pd.DataFrame()

print(f"  ✅ Amazon DOS rows       : {len(amz_dos_df):,}")
print(f"  ✅ Noon DOS rows         : {len(noon_dos_df):,}")
print(f"  ✅ Noon Minutes DOS rows : {len(nms_dos_df):,}")


# ══════════════════════════════════════════════════════════════════════════
# HELPER-ASIN-PRIORITY BASE  (fixes: ASINs with sales/ads but no inventory
# row on a given date used to get silently dropped at the final merge)
# ──────────────────────────────────────────────────────────────────────────
# The base for every output is now the full <helper ASIN> × <every date
# seen anywhere for that channel> matrix — built from the channel's helper
# slice (h_asin / h_nc / h_nms) crossed with the union of dates seen in
# that channel's inventory, revenue, and ads data. Inventory, revenue, DOS,
# and ads are then LEFT-merged ONTO this base, so a row is kept (with 0s)
# even when an ASIN has revenue/ads but is missing from inventory that day.
# ══════════════════════════════════════════════════════════════════════════

def build_asin_priority_base(helper_df, helper_key_col, *signal_dfs):
    """
    helper_df       : h_asin / h_nc / h_nms  (one row per known ASIN/NIN)
    helper_key_col  : 'ASIN' / 'NC_NIN' / 'NMs_NIN'
    signal_dfs      : any number of DataFrames that each have a 'Date' col
                       (inventory_df, revenue_df, ads_df, ...) — every date
                       seen in ANY of them becomes a row for EVERY ASIN.

    Returns: ASIN | Date   (one row per known ASIN per known date)
    """
    asins = (
        helper_df[[helper_key_col]]
        .dropna()
        .drop_duplicates()
        .rename(columns={helper_key_col: "ASIN"})
    )
    asins["ASIN"] = asins["ASIN"].astype(str).str.upper().str.strip()
    asins = asins[asins["ASIN"] != ""].drop_duplicates()

    all_dates = set()
    for d in signal_dfs:
        if d is not None and not d.empty and "Date" in d.columns:
            dates = pd.to_datetime(d["Date"], errors="coerce").dropna().dt.normalize()
            all_dates.update(dates.unique())

    if not all_dates:
        return pd.DataFrame(columns=["ASIN", "Date"])

    dates_df = pd.DataFrame({"Date": sorted(all_dates)})

    base = (
        asins.assign(_k=1)
        .merge(dates_df.assign(_k=1), on="_k")
        .drop(columns="_k")
    )
    return base


print("\n📐 Building helper-ASIN-priority base (ASIN × Date) per channel...")

amz_asin_base = build_asin_priority_base(
    h_asin, "ASIN", amz_inv_df, amz_rev_df,
    amz_ads_agg if 'amz_ads_agg' in dir() else None
)
nc_asin_base = build_asin_priority_base(
    h_nc, "NC_NIN", nc_inv_df, noon_rev_df,
    noon_ads_agg if 'noon_ads_agg' in dir() else None
)
nms_asin_base = build_asin_priority_base(
    h_nms, "NMs_NIN", nms_inv_df, nms_rev_df,
    nms_ads_agg if 'nms_ads_agg' in dir() else None
)

print(f"  Amazon base rows       : {len(amz_asin_base):,}  "
      f"({amz_asin_base['ASIN'].nunique() if not amz_asin_base.empty else 0} ASINs)")
print(f"  Noon base rows         : {len(nc_asin_base):,}  "
      f"({nc_asin_base['ASIN'].nunique() if not nc_asin_base.empty else 0} NINs)")
print(f"  Noon Minutes base rows : {len(nms_asin_base):,}  "
      f"({nms_asin_base['ASIN'].nunique() if not nms_asin_base.empty else 0} NINs)")


# ══════════════════════════════════════════════════════════════════════════
# FINAL INVENTORY METRICS BUILDER
# BASE = helper-ASIN-priority matrix (NOT raw inventory) — every known ASIN
# for every known date is kept, even if inventory/revenue/ads is missing
# for that ASIN+Date (those rows just get 0s instead of being dropped).
# Inventory, revenue (for total_sales/total_units/sell_through), and DOS
# (Inventory / Last 30 Days Sales / DOS) are all LEFT-merged onto it.
# ══════════════════════════════════════════════════════════════════════════

def add_metrics(asin_base, inv, rev_df, dos_df, channel=""):

    # ══════════════════════════════════════════════════════════════════════
    # BASE = ASIN × DATE (helper priority — never dropped downstream)
    # ══════════════════════════════════════════════════════════════════════

    df = asin_base.copy()
    df["ASIN"] = df["ASIN"].astype(str).str.upper().str.strip()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
    df = df[df["Date"].notna()]

    # ══════════════════════════════════════════════════════════════════════
    # INVENTORY MERGE  (left-merge inventory's own columns onto the base)
    # ══════════════════════════════════════════════════════════════════════

    if inv is not None and not inv.empty:

        inv_clean = inv.copy()
        inv_clean = inv_clean.loc[:, ~inv_clean.columns.duplicated()]

        inv_clean["ASIN"] = inv_clean["ASIN"].astype(str).str.upper().str.strip()
        inv_clean["Date"] = pd.to_datetime(inv_clean["Date"], errors="coerce").dt.normalize()
        inv_clean = inv_clean[inv_clean["Date"].notna()]
        inv_clean = inv_clean.drop_duplicates(subset=["ASIN", "Date"], keep="first")

        df = df.merge(inv_clean, on=["ASIN", "Date"], how="left", suffixes=("", "_invsrc"))

        drop_dupe_cols = [c for c in df.columns if c.endswith("_invsrc")]
        if drop_dupe_cols:
            df.drop(columns=drop_dupe_cols, inplace=True)

    if "Available Inventory" not in df.columns:
        df["Available Inventory"] = 0
    df["Available Inventory"] = pd.to_numeric(
        df["Available Inventory"], errors="coerce"
    ).fillna(0)

    # ══════════════════════════════════════════════════════════════════════
    # REVENUE MERGE  (still used for total_sales / total_units / sell_through)
    # ══════════════════════════════════════════════════════════════════════

    if rev_df is not None and not rev_df.empty:

        rev = rev_df.copy()
        rev["ASIN"] = rev["ASIN"].astype(str).str.upper().str.strip()
        rev["Date"] = pd.to_datetime(rev["Date"], errors="coerce").dt.normalize()

        keep_cols = [c for c in ["ASIN", "Date", "total_sales", "total_units"] if c in rev.columns]
        rev = rev[keep_cols].drop_duplicates(subset=["ASIN", "Date"])

        df = df.merge(rev, on=["ASIN", "Date"], how="left")

    if "total_sales" not in df.columns:
        df["total_sales"] = 0
    if "total_units" not in df.columns:
        df["total_units"] = 0

    df["total_sales"] = pd.to_numeric(df["total_sales"], errors="coerce").fillna(0)
    df["total_units"] = pd.to_numeric(df["total_units"], errors="coerce").fillna(0).astype(int)

    # ══════════════════════════════════════════════════════════════════════
    # INVENTORY / LAST 30 DAYS SALES / DOS — FROM IN-NOTEBOOK DOS TABLES
    # Available Inventory is REPLACED by the DOS table's Inventory value
    # wherever a match exists (it's the authoritative aggregated number).
    # ══════════════════════════════════════════════════════════════════════

    df.drop(columns=["Last 30 Days Sales", "DOS"], inplace=True, errors="ignore")

    if dos_df is not None and not dos_df.empty:

        dos = dos_df.copy()
        dos["ASIN"] = dos["ASIN"].astype(str).str.upper().str.strip()
        dos["Date"] = pd.to_datetime(dos["Date"], errors="coerce").dt.normalize()
        dos = dos.drop_duplicates(subset=["ASIN", "Date"], keep="first")
        dos = dos.rename(columns={"Inventory": "_dos_inventory"})

        df = df.merge(
            dos[["ASIN", "Date", "_dos_inventory", "Last 30 Days Sales", "DOS"]],
            on=["ASIN", "Date"], how="left",
        )

        df["Available Inventory"] = np.where(
            df["_dos_inventory"].notna(),
            df["_dos_inventory"],
            df["Available Inventory"],
        )
        df.drop(columns=["_dos_inventory"], inplace=True, errors="ignore")
        df["Available Inventory"] = pd.to_numeric(
            df["Available Inventory"], errors="coerce"
        ).fillna(0)

    else:
        df["Last 30 Days Sales"] = 0.0
        df["DOS"] = 0

    df["Last 30 Days Sales"] = pd.to_numeric(df["Last 30 Days Sales"], errors="coerce").fillna(0)
    df["DOS"] = (
        pd.to_numeric(df["DOS"], errors="coerce")
        .fillna(0).round(0).clip(upper=3650).astype("Int64")
    )

    # ══════════════════════════════════════════════════════════════════════
    # SELL THROUGH
    # NEW FORMULA (replaces both the old 90-day-file ASIN-level lookup and
    # the old total_units / Available Inventory ratio): simply
    #     sell_through = Last 30 Days Sales / 30
    # computed per ASIN x Date, identically for every channel.
    # ══════════════════════════════════════════════════════════════════════

    df["sell_through"] = np.where(
        df["Last 30 Days Sales"] > 0,
        df["Last 30 Days Sales"] / 30,
        0
    )
    df["sell_through"] = (
        pd.to_numeric(df["sell_through"], errors="coerce")
        .fillna(0).round(4)
    )

    # ══════════════════════════════════════════════════════════════════════
    # OOS DATE
    # ══════════════════════════════════════════════════════════════════════

    today_dt = date.today()

    def calc_oos(group):
        group = group.sort_values("Date").copy()
        current_month_df = group[
            (group["Date"].dt.month == today_dt.month)
            & (group["Date"].dt.year == today_dt.year)
        ]
        if current_month_df.empty:
            return pd.Series(["Instock"] * len(group), index=group.index)

        latest = current_month_df.iloc[-1]

        if latest["Available Inventory"] == 0:
            prev_stock = current_month_df[
                (current_month_df["Date"] < latest["Date"])
                & (current_month_df["Available Inventory"] > 0)
            ]
            if not prev_stock.empty:
                oos_date = prev_stock.iloc[-1]["Date"] + pd.Timedelta(days=1)
            else:
                oos_date = current_month_df.iloc[0]["Date"]
        else:
            st = latest["sell_through"]
            if pd.notna(st) and st > 0:
                projected_days = int(latest["Available Inventory"] / st)
                if projected_days > 3650:
                    projected_days = 3650
                oos_date = latest["Date"] + pd.Timedelta(days=projected_days)
            else:
                oos_date = "Instock"

        return pd.Series([oos_date] * len(group), index=group.index)

    df["OOS Date"] = (
        df.groupby("ASIN", group_keys=False).apply(calc_oos)
    )

    # ══════════════════════════════════════════════════════════════════════
    # STOCK STATUS
    # ══════════════════════════════════════════════════════════════════════

    df["min_available_qty"] = (df["sell_through"] * 30).round(2)
    df["max_available_qty"] = (df["sell_through"] * 60).round(2)

    conditions = [
        ((df["Available Inventory"] > df["min_available_qty"])
         & (df["Available Inventory"] <= df["max_available_qty"])),
        (df["Available Inventory"] > df["max_available_qty"]),
        (df["Available Inventory"] <= df["min_available_qty"]),
    ]
    choices = ["Healthy", "Overstock", "Understock"]
    df["Stock_Status"] = np.select(conditions, choices, default="Unknown")

    # ══════════════════════════════════════════════════════════════════════
    # OPEN PO FLAGS
    # ══════════════════════════════════════════════════════════════════════

    if "Open POs" not in df.columns:
        df["Open POs"] = 0
    df["Open POs"] = pd.to_numeric(df["Open POs"], errors="coerce").fillna(0)

    df["No/Low Stock wt Open POs"] = (
        (df["Stock_Status"] == "Understock") & (df["Open POs"] > 0)
    ).astype(int)
    df["No/Low Stock wt no Open POs"] = (
        (df["Stock_Status"] == "Understock") & (df["Open POs"] <= 0)
    ).astype(int)

    # ══════════════════════════════════════════════════════════════════════
    # INSTOCK RATE
    # ══════════════════════════════════════════════════════════════════════

    df = df.sort_values(["ASIN", "Date"])
    df["year_month"] = df["Date"].dt.to_period("M")
    df["in_stock_flag"] = (df["Available Inventory"] > 0).astype(int)
    df["cumulative_instock_days"] = (
        df.groupby(["ASIN", "year_month"])["in_stock_flag"].cumsum()
    )
    df["day_of_month"] = df["Date"].dt.day
    df["Instock Rate"] = (
        df["cumulative_instock_days"] / df["day_of_month"] * 100
    ).round(2)
    df.drop(columns=["year_month"], inplace=True)

    # ══════════════════════════════════════════════════════════════════════
    # METADATA FROM HELPER (the base only has ASIN/Date — bring metadata in)
    # ══════════════════════════════════════════════════════════════════════

    df["Sales Channel"] = channel

    # ══════════════════════════════════════════════════════════════════════
    # REMOVE DUPLICATES + FINAL SORT
    # ══════════════════════════════════════════════════════════════════════

    df = df.drop_duplicates(subset=["ASIN", "Date"], keep="first")
    df = df.sort_values(["ASIN", "Date"])

    return df


# ══════════════════════════════════════════════════════════════════════════
# BUILD FINAL INVENTORY OUTPUTS  (ASIN-priority base — nothing dropped)
# ══════════════════════════════════════════════════════════════════════════

amz_inv_final = add_metrics(
    amz_asin_base, amz_inv_df, amz_rev_df, amz_dos_df, "Vendor Central"
)

nms_inv_final = add_metrics(
    nms_asin_base, nms_inv_df, nms_rev_df, nms_dos_df, "Noon Minutes"
)

nc_inv_final = add_metrics(
    nc_asin_base, nc_inv_df, noon_rev_df, noon_dos_df, "Noon"
)

# ── Re-attach helper metadata (Brand/Product Name/Category/etc.) ──────────
# The ASIN-priority base only carries ASIN+Date, so product metadata needs
# to be merged back in from the same helper slices used everywhere else.

amz_inv_final = _apply_helper_map(amz_inv_final, h_asin, "ASIN", "ASIN", INFO_COLS)
nc_inv_final  = _apply_helper_map(nc_inv_final,  h_nc,   "NC_NIN",  "ASIN", INFO_COLS)
nms_inv_final = _apply_helper_map(nms_inv_final, h_nms,  "NMs_NIN", "ASIN", INFO_COLS)

# ══════════════════════════════════════════════════════════════════════════
# VALIDATION
# ══════════════════════════════════════════════════════════════════════════

print("\n✅ FINAL INVENTORY OUTPUT READY (helper-ASIN-priority base)")

print("\n📊 Amazon:")
print(amz_inv_final.shape)

print("\n📊 Noon Minutes:")
print(nms_inv_final.shape)

print("\n📊 Noon:")
print(nc_inv_final.shape)

print("\n💰 Amazon Sales:")
print(round(amz_inv_final["total_sales"].sum(), 2))

print("\n💰 Noon Sales:")
print(round(nc_inv_final["total_sales"].sum(), 2))

print("\n💰 Noon Minutes Sales:")
print(round(nms_inv_final["total_sales"].sum(), 2))


📥 Computing DOS tables in-notebook (Amazon / Noon / Noon Minutes)...
  ✅ Amazon DOS rows       : 9,099
  ✅ Noon DOS rows         : 8,368
  ✅ Noon Minutes DOS rows : 1,106

📐 Building helper-ASIN-priority base (ASIN × Date) per channel...
  Amazon base rows       : 11,270  (98 ASINs)
  Noon base rows         : 11,484  (99 NINs)
  Noon Minutes base rows : 2,070  (18 NINs)


C:\Users\Shambhavi Dixit\AppData\Local\Temp\ipykernel_35500\1867321793.py:370: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("ASIN", group_keys=False).apply(calc_oos)
C:\Users\Shambhavi Dixit\AppData\Local\Temp\ipykernel_35500\1867321793.py:370: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("ASIN", group_keys=False).apply(calc_oos)
C:\Users\Shambhavi Dixit\AppData\Local\Temp\ipykernel_35500\1867


✅ FINAL INVENTORY OUTPUT READY (helper-ASIN-priority base)

📊 Amazon:
(11270, 40)

📊 Noon Minutes:
(2070, 39)

📊 Noon:
(11484, 39)

💰 Amazon Sales:
81877.28

💰 Noon Sales:
33677.06

💰 Noon Minutes Sales:
75799.45


## 15 · Build Inventory Output (All 3 channels stacked)

In [35]:
# ══════════════════════════════════════════════════════════════════════════
# FINAL INVENTORY OUTPUT
# DUPLICATE COLUMN SAFE
# BRAND FILL FIXED
# SORTING FIXED
# ══════════════════════════════════════════════════════════════════════════

def align_cols(df, cols):

    # ─────────────────────────────────────────────────────────────────────
    # EMPTY DF
    # ─────────────────────────────────────────────────────────────────────

    if df is None or df.empty:

        return pd.DataFrame(columns=cols)

    out = df.copy()

    # ══════════════════════════════════════════════════════════════════════
    # REMOVE DUPLICATE COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    out = out.loc[
        :,
        ~out.columns.duplicated()
    ]

    # ══════════════════════════════════════════════════════════════════════
    # ADD MISSING COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    for col in cols:

        if col not in out.columns:

            out[col] = np.nan

    # ══════════════════════════════════════════════════════════════════════
    # KEEP ONLY REQUIRED COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    out = out[cols].copy()

    return out

# ══════════════════════════════════════════════════════════════════════════
# FINAL INVENTORY COLUMN ORDER
# ══════════════════════════════════════════════════════════════════════════

INVENTORY_COLS = [

    "ASIN",

    "Brand",

    "Product Category",

    "Product Sub Category",

    "Product Name",

    "Pack Type",

    "Pack Size",

    "SC Ideal Price",

    "VC Ideal Price",

    "Available Inventory",

    "Sellable Inventory Amount",

    "Unsellable On Hand Inventory Amount",

    "Unsellable On Hand Units",

    "Vendor Confirmation %",

    "PO_received_amount",

    "PO_received_Units",

    "Open POs",

    "Receive_Fill_Rate",

    "Overall Vendor Lead Time (days)",

    "Aged 90+ Days Sellable Inventory",

    "Aged 90+ Days Sellable Units",

    "Date",

    "Sales Channel",

    "in_stock_flag",

    "cumulative_instock_days",

    "day_of_month",

    "Instock Rate",

    "total_sales",

    "total_units",

    "sell_through",

    "Last 30 Days Sales",

    "DOS",

    "OOS Date",

    "min_available_qty",

    "max_available_qty",

    "Stock_Status",

    "No/Low Stock wt Open POs",

    "No/Low Stock wt no Open POs",
]

# ══════════════════════════════════════════════════════════════════════════
# CONCAT ALL CHANNELS
# INVENTORY = PRIORITY OUTPUT
# ══════════════════════════════════════════════════════════════════════════

inventory_output = pd.concat([

    align_cols(

        amz_inv_final,

        INVENTORY_COLS
    ),

    align_cols(

        nms_inv_final,

        INVENTORY_COLS
    ),

    align_cols(

        nc_inv_final,

        INVENTORY_COLS
    ),

], ignore_index=True)

# ══════════════════════════════════════════════════════════════════════════
# REMOVE DUPLICATES
# ══════════════════════════════════════════════════════════════════════════

inventory_output = inventory_output.drop_duplicates(

    subset=[

        "ASIN",

        "Date",

        "Sales Channel"
    ],

    keep="first"
)

# ══════════════════════════════════════════════════════════════════════════
# CLEAN DATE
# ══════════════════════════════════════════════════════════════════════════

inventory_output["Date"] = (

    pd.to_datetime(

        inventory_output["Date"],

        errors="coerce"
    )

    .dt.normalize()
)

# ══════════════════════════════════════════════════════════════════════════
# FILL BRAND
# ══════════════════════════════════════════════════════════════════════════

if "Brand" in inventory_output.columns:

    inventory_output["Brand"] = (

        inventory_output["Brand"]

        .replace("", np.nan)

        .fillna("Emami")
    )

# ══════════════════════════════════════════════════════════════════════════
# FILL TEXT COLUMNS
# ══════════════════════════════════════════════════════════════════════════

text_cols = [

    "Product Category",

    "Product Sub Category",

    "Product Name",

    "Pack Type",

    "Stock_Status"
]

for col in text_cols:

    if col in inventory_output.columns:

        inventory_output[col] = (

            inventory_output[col]

            .replace("", np.nan)

            .fillna("Unknown")
        )

# ══════════════════════════════════════════════════════════════════════════
# FILL NUMERIC COLUMNS
# ══════════════════════════════════════════════════════════════════════════

numeric_cols = inventory_output.select_dtypes(

    include=[

        "float",

        "int",

        "Int64"
    ]
).columns

inventory_output[numeric_cols] = (

    inventory_output[numeric_cols]

    .fillna(0)
)

# ══════════════════════════════════════════════════════════════════════════
# SORT
# ══════════════════════════════════════════════════════════════════════════

inventory_output = inventory_output.sort_values(

    [

        "Sales Channel",

        "ASIN",

        "Date"
    ]
)

inventory_output = inventory_output.reset_index(

    drop=True
)

# ══════════════════════════════════════════════════════════════════════════
# FINAL VALIDATION
# ══════════════════════════════════════════════════════════════════════════

print("\n✅ INVENTORY OUTPUT READY")

print("\n📊 Shape:")
print(inventory_output.shape)

print("\n📦 Sales Channel Split:")
print(
    inventory_output["Sales Channel"]
    .value_counts(dropna=False)
)

print("\n📦 Brand Check:")
print(
    inventory_output["Brand"]
    .value_counts(dropna=False)
    .head(10)
)

print("\n💰 Total Sales:")
print(
    round(
        inventory_output["total_sales"].sum(),
        2
    )
)

display(
    inventory_output.head(50)
)


✅ INVENTORY OUTPUT READY

📊 Shape:
(24824, 38)

📦 Sales Channel Split:
Sales Channel
Noon              11484
Vendor Central    11270
Noon Minutes       2070
Name: count, dtype: int64

📦 Brand Check:
Brand
Creme 21                    15242
Emami 7 Oils in One          4387
Emami Smart And Handsome     3117
Himani Navratna              2078
Name: count, dtype: int64

💰 Total Sales:
191353.79


,ASIN,Brand,Product Category,Product Sub Category,Product Name,Pack Type,Pack Size,SC Ideal Price,VC Ideal Price,Available Inventory,...,total_units,sell_through,Last 30 Days Sales,DOS,OOS Date,min_available_qty,max_available_qty,Stock_Status,No/Low Stock wt Open POs,No/Low Stock wt no Open POs
0,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0,0.0,0.0,0,2026-08-01 00:00:00,0.0,0.0,Understock,0,1
1,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0,0.0,0.0,0,2026-08-01 00:00:00,0.0,0.0,Understock,0,1
2,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0,0.0,0.0,0,2026-08-01 00:00:00,0.0,0.0,Understock,0,1
3,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0,0.0,0.0,0,2026-08-01 00:00:00,0.0,0.0,Understock,0,1
4,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0,0.0,0.0,0,2026-08-01 00:00:00,0.0,0.0,Understock,0,1
5,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0,0.0,0.0,0,2026-08-01 00:00:00,0.0,0.0,Understock,0,1
6,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0,0.0,0.0,0,2026-08-01 00:00:00,0.0,0.0,Understock,0,1
7,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0,0.0,0.0,0,2026-08-01 00:00:00,0.0,0.0,Understock,0,1
8,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0,0.0,0.0,0,2026-08-01 00:00:00,0.0,0.0,Understock,0,1
9,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0,0.0,0.0,0,2026-08-01 00:00:00,0.0,0.0,Understock,0,1


In [36]:
# ══════════════════════════════════════════════════════════════════════════
# VALIDATION — confirm the helper-ASIN-priority fix is working
# (no ASIN with sales/ads should be missing from the final output anymore)
# ══════════════════════════════════════════════════════════════════════════

print("\n" + "="*65)
print("  ASIN-PRIORITY FIX CHECK")
print("="*65)

def _check_channel(label, helper_df, helper_key, rev_df, ads_agg_df, final_df):
    helper_asins = set(
        helper_df[helper_key].dropna().astype(str).str.upper().str.strip()
    )

    signal_asins = set()
    if rev_df is not None and not rev_df.empty and "ASIN" in rev_df.columns:
        signal_asins |= set(rev_df.loc[rev_df.get("total_units", 0) > 0, "ASIN"]
                             .dropna().astype(str).str.upper().str.strip())
    if ads_agg_df is not None and not ads_agg_df.empty and "ASIN" in ads_agg_df.columns:
        signal_asins |= set(ads_agg_df["ASIN"].dropna().astype(str).str.upper().str.strip())

    output_asins = set(
        final_df.loc[final_df["Sales Channel"] == label, "ASIN"]
        .dropna().astype(str).str.upper().str.strip()
    ) if final_df is not None and not final_df.empty else set()

    missing = (signal_asins & helper_asins) - output_asins

    print(f"\n  {label}")
    print(f"    Helper ASINs            : {len(helper_asins)}")
    print(f"    ASINs w/ sales or ads    : {len(signal_asins)}")
    print(f"    Present in final output : {len(signal_asins & output_asins)}")
    print(f"    Still missing           : {len(missing)}"
          + (f"  ⚠️  {sorted(list(missing))[:10]}" if missing else "  ✅"))

_check_channel("Vendor Central", h_asin, "ASIN", amz_rev_df,
                amz_ads_agg if 'amz_ads_agg' in dir() else None, inventory_output)
_check_channel("Noon", h_nc, "NC_NIN", noon_rev_df,
                noon_ads_agg if 'noon_ads_agg' in dir() else None, inventory_output)
_check_channel("Noon Minutes", h_nms, "NMs_NIN", nms_rev_df,
                nms_ads_agg if 'nms_ads_agg' in dir() else None, inventory_output)



  ASIN-PRIORITY FIX CHECK

  Vendor Central
    Helper ASINs            : 98
    ASINs w/ sales or ads    : 92
    Present in final output : 91
    Still missing           : 0  ✅

  Noon
    Helper ASINs            : 99
    ASINs w/ sales or ads    : 102
    Present in final output : 93
    Still missing           : 0  ✅

  Noon Minutes
    Helper ASINs            : 18
    ASINs w/ sales or ads    : 17
    Present in final output : 17
    Still missing           : 0  ✅


In [37]:
inventory_output.to_csv("final_inventory_output.csv", index=False)

In [38]:
inventory_output.columns

Index(['ASIN', 'Brand', 'Product Category', 'Product Sub Category',
       'Product Name', 'Pack Type', 'Pack Size', 'SC Ideal Price',
       'VC Ideal Price', 'Available Inventory', 'Sellable Inventory Amount',
       'Unsellable On Hand Inventory Amount', 'Unsellable On Hand Units',
       'Vendor Confirmation %', 'PO_received_amount', 'PO_received_Units',
       'Open POs', 'Receive_Fill_Rate', 'Overall Vendor Lead Time (days)',
       'Aged 90+ Days Sellable Inventory', 'Aged 90+ Days Sellable Units',
       'Date', 'Sales Channel', 'in_stock_flag', 'cumulative_instock_days',
       'day_of_month', 'Instock Rate', 'total_sales', 'total_units',
       'sell_through', 'Last 30 Days Sales', 'DOS', 'OOS Date',
       'min_available_qty', 'max_available_qty', 'Stock_Status',
       'No/Low Stock wt Open POs', 'No/Low Stock wt no Open POs'],
      dtype='object')

In [39]:
duplicate_rows = inventory_output[
    inventory_output.duplicated(
        subset=["ASIN", "Date", "Sales Channel"],
        keep=False
    )
].sort_values(["ASIN", "Date", "Sales Channel"])

print(f"Total duplicate rows: {len(duplicate_rows)}")
display(duplicate_rows)

Total duplicate rows: 0


,ASIN,Brand,Product Category,Product Sub Category,Product Name,Pack Type,Pack Size,SC Ideal Price,VC Ideal Price,Available Inventory,...,total_units,sell_through,Last 30 Days Sales,DOS,OOS Date,min_available_qty,max_available_qty,Stock_Status,No/Low Stock wt Open POs,No/Low Stock wt no Open POs


In [40]:
# ══════════════════════════════════════════════════════════════════════════
# apply_helper  ← FIXED
# Rules:
#  • how='left'  → inventory rows are backbone; helper only fills info cols
#  • Drop existing helper cols before merging to avoid _x/_y
#  • Amazon  → merge on ASIN
#  • Noon    → merge on ASIN == NC_NIN
#  • NMS     → merge on ASIN == NMs_NIN
#  • Unmapped rows get "Unknown" / 0
# ══════════════════════════════════════════════════════════════════════════

def apply_helper(df):

    INFO_COLS_LOCAL = [
        "Brand", "Product Category", "Product Sub Category",
        "Product Name", "Pack Type", "Pack Size",
        "VC Ideal Price", "SC Ideal Price"
    ]
    INFO_COLS_LOCAL = [c for c in INFO_COLS_LOCAL if c in h_asin.columns or c in h_nc.columns or c in h_nms.columns]

    out = df.copy()

    # Clean ASIN key per channel:
    # Amazon (Vendor Central): standard ASIN cleaning
    # Noon / Noon Minutes: NIN cleaning — KEEP -\d+ suffix
    def _clean_asin_for_channel(row):
        s = str(row["ASIN"]).upper().strip().replace("\xa0","").replace(".0","")
        import re as _re
        s = _re.sub(r"\s+","",s)
        # For Amazon only: strip trailing digit variant suffix like -1
        if row.get("Sales Channel") == "Vendor Central":
            s = _re.sub(r"-\d+$","",s)
        return s
    out["ASIN"] = out.apply(_clean_asin_for_channel, axis=1)

    # ── Drop existing helper columns to prevent _x / _y on re-run ─────────
    cols_to_drop = [c for c in INFO_COLS_LOCAL if c in out.columns]
    out.drop(columns=cols_to_drop, inplace=True, errors="ignore")

    # ── Split by channel ───────────────────────────────────────────────────
    amz_mask = out["Sales Channel"] == "Vendor Central"
    nms_mask = out["Sales Channel"] == "Noon Minutes"
    nc_mask  = out["Sales Channel"] == "Noon"

    amz = out[amz_mask].copy()
    nms = out[nms_mask].copy()
    nc  = out[nc_mask].copy()
    other = out[~(amz_mask | nms_mask | nc_mask)].copy()

    # ── Merge Amazon ───────────────────────────────────────────────────────
    if not amz.empty and not h_asin.empty:
        h = h_asin[[c for c in ["ASIN"] + INFO_COLS_LOCAL if c in h_asin.columns]].copy()
        amz = amz.merge(h, on="ASIN", how="left")

    # ── Merge Noon Core ───────────────────────────────────────────────────
    if not nc.empty and not h_nc.empty:
        h = h_nc[[c for c in ["NC_NIN"] + INFO_COLS_LOCAL if c in h_nc.columns]].copy()
        nc = nc.merge(h, left_on="ASIN", right_on="NC_NIN", how="left")
        nc.drop(columns=["NC_NIN"], inplace=True, errors="ignore")

    # ── Merge Noon Minutes ────────────────────────────────────────────────
    if not nms.empty and not h_nms.empty:
        h = h_nms[[c for c in ["NMs_NIN"] + INFO_COLS_LOCAL if c in h_nms.columns]].copy()
        nms = nms.merge(h, left_on="ASIN", right_on="NMs_NIN", how="left")
        nms.drop(columns=["NMs_NIN"], inplace=True, errors="ignore")

    # ── Recombine ─────────────────────────────────────────────────────────
    final = pd.concat([amz, nms, nc, other], ignore_index=True)

    # ── Remove any leftover _x / _y ───────────────────────────────────────
    for col in list(final.columns):
        if col.endswith("_x"):
            base = col[:-2]
            y_col = base + "_y"
            final[base] = final[col].combine_first(final.get(y_col))
            final.drop(columns=[col, y_col], inplace=True, errors="ignore")

    # ── Fill unknowns ──────────────────────────────────────────────────────
    text_info = ["Brand","Product Category","Product Sub Category","Product Name","Pack Type"]
    for col in text_info:
        if col not in final.columns:
            final[col] = "Unknown"
        final[col] = final[col].fillna("Unknown")

    if "Brand" in final.columns:
        final["Brand"] = final["Brand"].replace("", "Unknown").replace("Unknown", "Emami")

    num_info = ["Pack Size", "VC Ideal Price", "SC Ideal Price"]
    for col in num_info:
        if col not in final.columns:
            final[col] = 0
        final[col] = pd.to_numeric(final[col], errors="coerce").fillna(0)

    return final


# ── Apply to inventory_output ─────────────────────────────────────────────
inventory_output = apply_helper(inventory_output)

print("\n✅ Helper applied to inventory_output")
print("  Shape:", inventory_output.shape)
print("  Brand value counts:")
print(inventory_output["Brand"].value_counts(dropna=False).head(8))
print("\n  Product Category fill rate:")
filled = (inventory_output["Product Category"] != "Unknown").sum()
print(f"  {filled}/{len(inventory_output)} rows have Product Category")



✅ Helper applied to inventory_output
  Shape: (24824, 38)
  Brand value counts:
Brand
Creme 21                    15242
Emami 7 Oils in One          4387
Emami Smart And Handsome     3117
Himani Navratna              2078
Name: count, dtype: int64

  Product Category fill rate:
  24824/24824 rows have Product Category


In [41]:
# ==========================================================
# RENAME EXISTING COLUMNS
# ==========================================================

rename_columns = {
    "ASIN": "asin",
    "Available Inventory": "available_inventory",
    "Sellable Inventory Amount": "sellable_inventory_amount",
    "Unsellable On Hand Inventory Amount": "unsellable_on_hand_inventory_amount",
    "Unsellable On Hand Units": "unsellable_on_hand_units",
    "Vendor Confirmation %": "vendor_confirmation_pct",
    "PO_received_amount": "po_received_amount",
    "PO_received_Units": "po_received_units",
    "Open Pos": "open_pos",
    "Open POs": "open_pos",
    "Receive_Fill_Rate": "receive_fill_rate",
    "Overall Vendor Lead Time (days)": "overall_vendor_lead_time_days",
    "Aged 90+ Days Sellable Inventory": "aged_90_plus_days_sellable_inventory",
    "Aged 90+ Days Sellable Units": "aged_90_plus_days_sellable_units",
    "Date": "date",
    "Sales Channel": "sales_channel",
    "Instock Rate": "instock_rate",
    "Last 30 Days Sales": "units_shipped_t30",
    "DOS": "dos",
    "OOS Date": "oos_date",
    "Stock_Status": "stock_status",
    "No/Low Stock wt Open Pos": "no_low_stock_wt_open_pos",
    "No/Low Stock wt Open POs": "no_low_stock_wt_open_pos",
    "No/Low Stock wt no Open Pos": "no_low_stock_wt_no_open_pos",
    "No/Low Stock wt no Open POs": "no_low_stock_wt_no_open_pos",
    "Brand": "brand",
    "Product Category": "product_category",
    "Product Sub Category": "product_sub_category",
    "Product Name": "product_name",
    "Pack Type": "pack_type",
    "Pack Size": "pack_size",
    "VC Ideal Price": "vc_ideal_price",
    "SC Ideal Price": "sc_ideal_price",
}

inventory_output.rename(columns=rename_columns, inplace=True)

# ==========================================================
# REQUIRED COLUMNS
# ==========================================================

required_columns = [
    "sku",
    "fnsku",
    "asin",
    "condition",
    "your_price",
    "mfn_listing_exists",
    "mfn_fulfillable_quantity",
    "afn_listing_exists",
    "afn_warehouse_quantity",
    "afn_fulfillable_quantity",
    "afn_unsellable_quantity",
    "afn_reserved_quantity",
    "afn_total_quantity",
    "per_unit_volume",
    "afn_inbound_working_quantity",
    "afn_inbound_shipped_quantity",
    "afn_inbound_receiving_quantity",
    "afn_researching_quantity",
    "afn_reserved_future_supply",
    "afn_future_supply_buyable",
    "store",
    "date",
    "reserved_qty",
    "reserved_customerorders",
    "reserved_fc_transfers",
    "reserved_fc_processing",
    "available_inventory",
    "units_shipped_t30",
    "units_shipped_t60",
    "units_shipped_t90",
    "open_pos",
    "sell_through",
    "dos",
    "min_available_qty",
    "max_available_qty",
    "stock_status",
    "instock_rate",
    "no_low_stock_wt_open_pos",
    "no_low_stock_wt_no_open_pos",
    "total_sales",
    "total_units",
    "product_name",
    "product_category",
    "pack_type",
    "pack_size",
    "sc_ideal_price",
    "vc_ideal_price",
    "sales_channel",
    "brand",
    "product_sub_category",
    "vendor_confirmation_pct",
    "po_received_amount",
    "po_received_units",
    "receive_fill_rate",
    "overall_vendor_lead_time_days",
    "aged_90_plus_days_sellable_inventory",
    "aged_90_plus_days_sellable_units",
    "sellable_inventory_amount",
    "unsellable_on_hand_inventory_amount",
    "unsellable_on_hand_units",
    "in_stock_flag",
    "cumulative_instock_days",
    "day_of_month",
    "oos_date",
]

# ==========================================================
# TEXT COLUMNS
# ==========================================================

text_columns = [
    "sku",
    "fnsku",
    "asin",
    "condition",
    "store",
    "product_name",
    "product_category",
    "pack_type",
    "pack_size",
    "sales_channel",
    "brand",
    "product_sub_category",
    "stock_status",
    "oos_date",
]

# ==========================================================
# ADD ONLY MISSING COLUMNS
# ==========================================================

missing_columns = []

for col in required_columns:

    if col not in inventory_output.columns:

        missing_columns.append(col)

        if col in text_columns:
            inventory_output[col] = ""
        else:
            inventory_output[col] = 0

print(f"Added {len(missing_columns)} missing columns:")
print(missing_columns)

# ==========================================================
# REORDER COLUMNS
# ==========================================================

extra_columns = [
    col for col in inventory_output.columns
    if col not in required_columns
]

inventory_output = inventory_output[
    required_columns + extra_columns
]

print("\nFinal Shape :", inventory_output.shape)
print("Total Columns :", len(inventory_output.columns))
print("\nFinal Columns:")
print(inventory_output.columns.tolist())

Added 26 missing columns:
['sku', 'fnsku', 'condition', 'your_price', 'mfn_listing_exists', 'mfn_fulfillable_quantity', 'afn_listing_exists', 'afn_warehouse_quantity', 'afn_fulfillable_quantity', 'afn_unsellable_quantity', 'afn_reserved_quantity', 'afn_total_quantity', 'per_unit_volume', 'afn_inbound_working_quantity', 'afn_inbound_shipped_quantity', 'afn_inbound_receiving_quantity', 'afn_researching_quantity', 'afn_reserved_future_supply', 'afn_future_supply_buyable', 'store', 'reserved_qty', 'reserved_customerorders', 'reserved_fc_transfers', 'reserved_fc_processing', 'units_shipped_t60', 'units_shipped_t90']

Final Shape : (24824, 64)
Total Columns : 64

Final Columns:
['sku', 'fnsku', 'asin', 'condition', 'your_price', 'mfn_listing_exists', 'mfn_fulfillable_quantity', 'afn_listing_exists', 'afn_warehouse_quantity', 'afn_fulfillable_quantity', 'afn_unsellable_quantity', 'afn_reserved_quantity', 'afn_total_quantity', 'per_unit_volume', 'afn_inbound_working_quantity', 'afn_inbound_shi

In [42]:
inventory_output.to_csv("final_inventory_output.csv", index=False)

In [43]:
inventory_output.columns

Index(['sku', 'fnsku', 'asin', 'condition', 'your_price', 'mfn_listing_exists',
       'mfn_fulfillable_quantity', 'afn_listing_exists',
       'afn_warehouse_quantity', 'afn_fulfillable_quantity',
       'afn_unsellable_quantity', 'afn_reserved_quantity',
       'afn_total_quantity', 'per_unit_volume', 'afn_inbound_working_quantity',
       'afn_inbound_shipped_quantity', 'afn_inbound_receiving_quantity',
       'afn_researching_quantity', 'afn_reserved_future_supply',
       'afn_future_supply_buyable', 'store', 'date', 'reserved_qty',
       'reserved_customerorders', 'reserved_fc_transfers',
       'reserved_fc_processing', 'available_inventory', 'units_shipped_t30',
       'units_shipped_t60', 'units_shipped_t90', 'open_pos', 'sell_through',
       'dos', 'min_available_qty', 'max_available_qty', 'stock_status',
       'instock_rate', 'no_low_stock_wt_open_pos',
       'no_low_stock_wt_no_open_pos', 'total_sales', 'total_units',
       'product_name', 'product_category', 'pack_typ

In [44]:
print(h_asin["ASIN"].head(5))

print(h_nms["NMs_NIN"].head(5))

print(h_nc["NC_NIN"].head(5))

0     B07H8Q2JGG
34    B096F51K8B
35    B096F6NR5L
36    B096F4ZYSR
37    B096F4NXF3
Name: ASIN, dtype: object
1     ZA4EA14149E7DB0926923Z-1
58    Z38685AD13D6A297BDF7DZ-1
55    ZD1D7ED3DBE3F11D792B9Z-1
31    Z3472F77ACFFA103E8ED6Z-1
60    ZE48CCD14111D3C0B6E27Z-1
Name: NMs_NIN, dtype: object
0     ZEEBC06BA70278269A891Z-1
33    ZBE7FDC403E1EF26CF8CAZ-1
34    ZF111DB442ACEF30A7609Z-1
35    Z901D1156E8C9BE967CF8Z-1
36    ZD54CE6C516414DA6EED4Z-1
Name: NC_NIN, dtype: object


## 16 · Build Revenue Output
> Revenue output = Inventory base + Ads merged OUTER.
>
> **Note**: `inv_final` (Section 14) is now built on the helper-ASIN-priority
> base, so an ASIN with revenue/ads on a date but no inventory row that date
> is already present here with `Available Inventory = 0` — it is no longer
> dropped at this step.

In [45]:
# ══════════════════════════════════════════════════════════════════════════
# FINAL REVENUE OUTPUT BUILDER
# INVENTORY = PRIORITY
# ADS LEFT MERGED
# ORGANIC SALES FIXED
# DUPLICATE SAFE
# ══════════════════════════════════════════════════════════════════════════

def build_revenue_channel(inv_final, ads_agg):

    # ══════════════════════════════════════════════════════════════════════
    # BASE
    # ══════════════════════════════════════════════════════════════════════

    df = inv_final.copy()

    # Remove duplicate columns
    df = df.loc[
        :,
        ~df.columns.duplicated()
    ]

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN KEYS
    # ══════════════════════════════════════════════════════════════════════

    df["ASIN"] = (

        df["ASIN"]

        .astype(str)

        .str.upper()

        .str.strip()
    )

    df["Date"] = (

        pd.to_datetime(

            df["Date"],

            errors="coerce"
        )

        .dt.normalize()
    )

    # ══════════════════════════════════════════════════════════════════════
    # ADS MERGE
    # outer MERGE
    # INVENTORY IS PRIORITY
    # ══════════════════════════════════════════════════════════════════════

    if ads_agg is not None and not ads_agg.empty:

        ads = ads_agg.copy()

        # Remove duplicate cols
        ads = ads.loc[
            :,
            ~ads.columns.duplicated()
        ]

        keep_cols = [

            "ASIN",

            "Date",

            "Sales Channel",

            "ads_sales",

            "ads_unit_sold",

            "ads_spend"
        ]

        keep_cols = [

            c for c in keep_cols

            if c in ads.columns
        ]

        ads = ads[
            keep_cols
        ]

        # ─────────────────────────────────────────────────────────────────
        # CLEAN
        # ─────────────────────────────────────────────────────────────────

        ads["ASIN"] = (

            ads["ASIN"]

            .astype(str)

            .str.upper()

            .str.strip()
        )

        ads["Date"] = (

            pd.to_datetime(

                ads["Date"],

                errors="coerce"
            )

            .dt.normalize()
        )

        # ─────────────────────────────────────────────────────────────────
        # NUMERIC
        # ─────────────────────────────────────────────────────────────────

        for col in [

            "ads_sales",

            "ads_unit_sold",

            "ads_spend"
        ]:

            if col in ads.columns:

                ads[col] = (

                    pd.to_numeric(

                        ads[col],

                        errors="coerce"
                    )

                    .fillna(0)
                )

        # ─────────────────────────────────────────────────────────────────
        # REMOVE DUPLICATES
        # ─────────────────────────────────────────────────────────────────

        ads = ads.drop_duplicates(

            subset=[

                "ASIN",

                "Date",

                "Sales Channel"
            ]
        )

        # ─────────────────────────────────────────────────────────────────
        # MERGE
        # ─────────────────────────────────────────────────────────────────

        merge_keys = [

            "ASIN",

            "Date"
        ]

        if "Sales Channel" in ads.columns:

            merge_keys.append("Sales Channel")

        df = df.merge(

            ads,

            on=merge_keys,

            how="left"  # ← FIXED: inventory is priority
        )

    # ══════════════════════════════════════════════════════════════════════
    # FILL ADS NULLS
    # ══════════════════════════════════════════════════════════════════════

    for col in [

        "ads_sales",

        "ads_unit_sold",

        "ads_spend"
    ]:

        if col not in df.columns:

            df[col] = 0

        df[col] = (

            pd.to_numeric(

                df[col],

                errors="coerce"
            )

            .fillna(0)
        )

    # ══════════════════════════════════════════════════════════════════════
    # ORGANIC SALES
    # ══════════════════════════════════════════════════════════════════════

    df["organic_sale"] = (

        df["total_sales"]

        - df["ads_sales"]
    )

    # No negative organic sales
    df["organic_sale"] = (

        df["organic_sale"]

        .clip(lower=0)

        .round(2)
    )

    # ══════════════════════════════════════════════════════════════════════
    # FINAL SORT
    # ══════════════════════════════════════════════════════════════════════

    df = df.sort_values(

        [

            "Sales Channel",

            "ASIN",

            "Date"
        ]
    )

    return df

# ══════════════════════════════════════════════════════════════════════════
# BUILD CHANNEL OUTPUTS
# ══════════════════════════════════════════════════════════════════════════

amz_rev_final = build_revenue_channel(

    amz_inv_final,

    amz_ads_agg
)

nc_rev_final = build_revenue_channel(

    nc_inv_final,

    noon_ads_agg
)

nms_rev_final = build_revenue_channel(

    nms_inv_final,

    nms_ads_agg
)

# ══════════════════════════════════════════════════════════════════════════
# FINAL COLUMN ORDER
# ══════════════════════════════════════════════════════════════════════════

REVENUE_COLS = INVENTORY_COLS + [

    "ads_sales",

    "ads_unit_sold",

    "ads_spend",

    "organic_sale"
]

# ══════════════════════════════════════════════════════════════════════════
# CONCAT ALL CHANNELS
# ══════════════════════════════════════════════════════════════════════════

revenue_output = pd.concat([

    align_cols(

        amz_rev_final,

        REVENUE_COLS
    ),

    align_cols(

        nc_rev_final,

        REVENUE_COLS
    ),

    align_cols(

        nms_rev_final,

        REVENUE_COLS
    ),

], ignore_index=True)

# ══════════════════════════════════════════════════════════════════════════
# REMOVE DUPLICATES
# ══════════════════════════════════════════════════════════════════════════

revenue_output = revenue_output.drop_duplicates(

    subset=[

        "ASIN",

        "Date",

        "Sales Channel"
    ],

    keep="first"
)

# ══════════════════════════════════════════════════════════════════════════
# CLEAN DATE
# ══════════════════════════════════════════════════════════════════════════

revenue_output["Date"] = (

    pd.to_datetime(

        revenue_output["Date"],

        errors="coerce"
    )

    .dt.normalize()
)

# ══════════════════════════════════════════════════════════════════════════
# FILL TEXT NULLS
# ══════════════════════════════════════════════════════════════════════════

text_cols = [

    "Brand",

    "Product Category",

    "Product Sub Category",

    "Product Name",

    "Pack Type",

    "Stock_Status"
]

for col in text_cols:

    if col in revenue_output.columns:

        revenue_output[col] = (

            revenue_output[col]

            .replace("", np.nan)

            .fillna("Unknown")
        )

# ══════════════════════════════════════════════════════════════════════════
# FILL NUMERIC NULLS
# ══════════════════════════════════════════════════════════════════════════

numeric_cols = revenue_output.select_dtypes(

    include=[

        "float",

        "int",

        "Int64"
    ]
).columns

revenue_output[numeric_cols] = (

    revenue_output[numeric_cols]

    .fillna(0)
)

# ══════════════════════════════════════════════════════════════════════════
# SORT
# ══════════════════════════════════════════════════════════════════════════

revenue_output = revenue_output.sort_values(

    [

        "Sales Channel",

        "ASIN",

        "Date"
    ]
)

revenue_output = revenue_output.reset_index(

    drop=True
)

# ══════════════════════════════════════════════════════════════════════════
# FINAL VALIDATION
# ══════════════════════════════════════════════════════════════════════════

print("\n✅ REVENUE OUTPUT READY")

print("\n📊 Shape:")
print(revenue_output.shape)

print("\n📦 Sales Channel Split:")
print(
    revenue_output["Sales Channel"]
    .value_counts(dropna=False)
)

print("\n💰 Total Sales:")
print(
    round(
        revenue_output["total_sales"].sum(),
        2
    )
)

print("\n💰 Ads Sales:")
print(
    round(
        revenue_output["ads_sales"].sum(),
        2
    )
)

print("\n💰 Organic Sales:")
print(
    round(
        revenue_output["organic_sale"].sum(),
        2
    )
)

# ══════════════════════════════════════════════════════════════════════════
# END-TO-END ADS CHECKSUM (same safety net as build_marketing)
# Compares each channel's pre-merge amz_ads_agg total against what
# survived into revenue_output, so a future regression is caught here
# instead of being discovered downstream.
# ══════════════════════════════════════════════════════════════════════════

print("\n🔎 ADS CHECKSUM (pre-merge agg total vs final revenue_output total):")

_rev_checksum_pairs = [
    ("Vendor Central", amz_ads_agg),
    ("Noon",           noon_ads_agg),
    ("Noon Minutes",   nms_ads_agg),
]

for _channel_name, _agg_df in _rev_checksum_pairs:
    if _agg_df is None or _agg_df.empty:
        continue
    _pre_total = pd.to_numeric(_agg_df.get("ads_sales", 0), errors="coerce").fillna(0).sum()
    _post_total = pd.to_numeric(
        revenue_output.loc[revenue_output["Sales Channel"] == _channel_name, "ads_sales"],
        errors="coerce"
    ).fillna(0).sum()
    _diff = _pre_total - _post_total
    flag = "⚠️ MISMATCH" if abs(_diff) > 0.01 else "✅ OK"
    print(f"   {flag}  {_channel_name:<16} pre={_pre_total:,.2f}  post={_post_total:,.2f}  diff={_diff:,.2f}")

display(
    revenue_output.head(50)
)



✅ REVENUE OUTPUT READY

📊 Shape:
(24824, 42)

📦 Sales Channel Split:
Sales Channel
Noon              11484
Vendor Central    11270
Noon Minutes       2070
Name: count, dtype: int64

💰 Total Sales:
191353.79

💰 Ads Sales:
82776.68

💰 Organic Sales:
128548.71

🔎 ADS CHECKSUM (pre-merge agg total vs final revenue_output total):
   ✅ OK  Vendor Central   pre=44,256.39  post=44,256.39  diff=-0.00
   ✅ OK  Noon             pre=3,560.39  post=3,560.39  diff=-0.00
   ✅ OK  Noon Minutes     pre=34,959.90  post=34,959.90  diff=-0.00


,ASIN,Brand,Product Category,Product Sub Category,Product Name,Pack Type,Pack Size,SC Ideal Price,VC Ideal Price,Available Inventory,...,OOS Date,min_available_qty,max_available_qty,Stock_Status,No/Low Stock wt Open POs,No/Low Stock wt no Open POs,ads_sales,ads_unit_sold,ads_spend,organic_sale
0,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,2026-08-01 00:00:00,0.0,0.0,Understock,0,1,0.0,0.0,0.0,0.0
1,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,2026-08-01 00:00:00,0.0,0.0,Understock,0,1,0.0,0.0,0.0,0.0
2,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,2026-08-01 00:00:00,0.0,0.0,Understock,0,1,0.0,0.0,0.0,0.0
3,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,2026-08-01 00:00:00,0.0,0.0,Understock,0,1,0.0,0.0,0.0,0.0
4,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,2026-08-01 00:00:00,0.0,0.0,Understock,0,1,0.0,0.0,0.0,0.0
5,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,2026-08-01 00:00:00,0.0,0.0,Understock,0,1,0.0,0.0,0.0,0.0
6,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,2026-08-01 00:00:00,0.0,0.0,Understock,0,1,0.0,0.0,0.0,0.0
7,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,2026-08-01 00:00:00,0.0,0.0,Understock,0,1,0.0,0.0,0.0,0.0
8,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,2026-08-01 00:00:00,0.0,0.0,Understock,0,1,0.0,0.0,0.0,0.0
9,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,2026-08-01 00:00:00,0.0,0.0,Understock,0,1,0.0,0.0,0.0,0.0


In [46]:
# ==========================================================
# RENAME EXISTING COLUMNS
# ==========================================================

rename_columns = {
    "ASIN": "asin",
    "Brand": "brand",
    "Product Category": "product_category",
    "Product Sub Category": "product_sub_category",
    "Product Name": "product_name",
    "Pack Type": "pack_type",
    "Pack Size": "pack_size",
    "SC Ideal Price": "sc_ideal_price",
    "VC Ideal Price": "vc_ideal_price",
    "Available Inventory": "available_inventory",
    "Sellable Inventory Amount": "sellable_inventory_amount",
    "Unsellable On Hand Inventory Amount": "unsellable_on_hand_inventory_amount",
    "Unsellable On Hand Units": "unsellable_on_hand_units",
    "Vendor Confirmation %": "vendor_confirmation_pct",
    "PO_received_amount": "po_received_amount",
    "PO_received_Units": "po_received_units",
    "Open Pos": "open_pos",
    "Open POs": "open_pos",
    "Receive_Fill_Rate": "receive_fill_rate",
    "Overall Vendor Lead Time (days)": "overall_vendor_lead_time_days",
    "Aged 90+ Days Sellable Inventory": "aged_90_plus_days_sellable_inventory",
    "Aged 90+ Days Sellable Units": "aged_90_plus_days_sellable_units",
    "Date": "date",
    "Sales Channel": "sales_channel",
    "Instock Rate": "instock_rate",
    "Last 30 Days Sales": "units_shipped_t30",
    "DOS": "dos",
    "OOS Date": "oos_date",
    "Stock_Status": "stock_status",
    "No/Low Stock wt Open Pos": "no_low_stock_wt_open_pos",
    "No/Low Stock wt Open POs": "no_low_stock_wt_open_pos",
    "No/Low Stock wt no Open Pos": "no_low_stock_wt_no_open_pos",
    "No/Low Stock wt no Open POs": "no_low_stock_wt_no_open_pos",
}

revenue_output.rename(columns=rename_columns, inplace=True)

# ==========================================================
# REQUIRED COLUMN ORDER
# ==========================================================

required_columns = [
    "asin",
    "brand",
    "product_sub_category",
    "vendor_confirmation_pct",
    "po_received_amount",
    "po_received_units",
    "open_pos",
    "receive_fill_rate",
    "overall_vendor_lead_time_days",
    "aged_90_plus_days_sellable_inventory",
    "aged_90_plus_days_sellable_units",
    "sellable_inventory_amount",
    "available_inventory",
    "unsellable_on_hand_inventory_amount",
    "unsellable_on_hand_units",
    "date",
    "sales_channel",
    "in_stock_flag",
    "cumulative_instock_days",
    "day_of_month",
    "instock_rate",
    "total_sales",
    "total_units",
    "sell_through",
    "dos",
    "oos_date",
    "min_available_qty",
    "max_available_qty",
    "stock_status",
    "no_low_stock_wt_open_pos",
    "no_low_stock_wt_no_open_pos",
    "ads_sales",
    "ads_unit_sold",
    "ads_spend",
    # "organic_sale",
    "product_name",
    "product_category",
    "pack_type",
    "pack_size",
    "sc_ideal_price",
    "vc_ideal_price",
    "sku",
    "fnsku",
    "condition",
    "your_price",
    "mfn_listing_exists",
    "mfn_fulfillable_quantity",
    "afn_listing_exists",
    "afn_warehouse_quantity",
    "afn_fulfillable_quantity",
    "afn_unsellable_quantity",
    "afn_reserved_quantity",
    "afn_total_quantity",
    "per_unit_volume",
    "afn_inbound_working_quantity",
    "afn_inbound_shipped_quantity",
    "afn_inbound_receiving_quantity",
    "afn_researching_quantity",
    "afn_reserved_future_supply",
    "afn_future_supply_buyable",
    "store",
    "reserved_qty",
    "reserved_customerorders",
    "reserved_fc_transfers",
    "reserved_fc_processing",
    "units_shipped_t30",
    "units_shipped_t60",
    "units_shipped_t90",
    # "year_month",
]

# ==========================================================
# TEXT COLUMNS
# ==========================================================

text_columns = [
    "asin",
    "brand",
    "product_sub_category",
    "sales_channel",
    "product_name",
    "product_category",
    "pack_type",
    "pack_size",
    "stock_status",
    "oos_date",
    "sku",
    "fnsku",
    "condition",
    "store",
    # "year_month",
]

# ==========================================================
# ADD MISSING COLUMNS
# ==========================================================

missing_columns = []

for col in required_columns:

    if col not in revenue_output.columns:

        missing_columns.append(col)

        if col in text_columns:
            revenue_output[col] = ""
        else:
            revenue_output[col] = 0

print("=" * 80)
print(f"Added {len(missing_columns)} Missing Columns")
print("=" * 80)

for col in missing_columns:
    print(col)

# ==========================================================
# REORDER COLUMNS
# ==========================================================

extra_columns = [
    col for col in revenue_output.columns
    if col not in required_columns
]

revenue_output = revenue_output[
    required_columns + extra_columns
]

# ==========================================================
# FINAL CHECK
# ==========================================================

print("\nFinal Shape :", revenue_output.shape)
print("Total Columns :", len(revenue_output.columns))
print("\nColumns:")
print(revenue_output.columns.tolist())

Added 26 Missing Columns
sku
fnsku
condition
your_price
mfn_listing_exists
mfn_fulfillable_quantity
afn_listing_exists
afn_warehouse_quantity
afn_fulfillable_quantity
afn_unsellable_quantity
afn_reserved_quantity
afn_total_quantity
per_unit_volume
afn_inbound_working_quantity
afn_inbound_shipped_quantity
afn_inbound_receiving_quantity
afn_researching_quantity
afn_reserved_future_supply
afn_future_supply_buyable
store
reserved_qty
reserved_customerorders
reserved_fc_transfers
reserved_fc_processing
units_shipped_t60
units_shipped_t90

Final Shape : (24824, 68)
Total Columns : 68

Columns:
['asin', 'brand', 'product_sub_category', 'vendor_confirmation_pct', 'po_received_amount', 'po_received_units', 'open_pos', 'receive_fill_rate', 'overall_vendor_lead_time_days', 'aged_90_plus_days_sellable_inventory', 'aged_90_plus_days_sellable_units', 'sellable_inventory_amount', 'available_inventory', 'unsellable_on_hand_inventory_amount', 'unsellable_on_hand_units', 'date', 'sales_channel', 'in_sto

In [47]:
revenue_output.to_csv("emami_uae_revenue_test.csv", index=False)

In [48]:
revenue_output.columns

Index(['asin', 'brand', 'product_sub_category', 'vendor_confirmation_pct',
       'po_received_amount', 'po_received_units', 'open_pos',
       'receive_fill_rate', 'overall_vendor_lead_time_days',
       'aged_90_plus_days_sellable_inventory',
       'aged_90_plus_days_sellable_units', 'sellable_inventory_amount',
       'available_inventory', 'unsellable_on_hand_inventory_amount',
       'unsellable_on_hand_units', 'date', 'sales_channel', 'in_stock_flag',
       'cumulative_instock_days', 'day_of_month', 'instock_rate',
       'total_sales', 'total_units', 'sell_through', 'dos', 'oos_date',
       'min_available_qty', 'max_available_qty', 'stock_status',
       'no_low_stock_wt_open_pos', 'no_low_stock_wt_no_open_pos', 'ads_sales',
       'ads_unit_sold', 'ads_spend', 'product_name', 'product_category',
       'pack_type', 'pack_size', 'sc_ideal_price', 'vc_ideal_price', 'sku',
       'fnsku', 'condition', 'your_price', 'mfn_listing_exists',
       'mfn_fulfillable_quantity', 'afn_l

## 17 · Build Marketing Output
> Campaign-level ads merged OUTER into inventory base.
>
> **Note**: same as Section 16 — because `inv_final` already carries every
> helper ASIN for every date, ASINs with ad spend/sales but no inventory row
> that day are preserved instead of being dropped.

In [49]:
# ══════════════════════════════════════════════════════════════════════════
# FINAL MARKETING OUTPUT BUILDER
# INVENTORY = PRIORITY
# CAMPAIGN ADS OUTER MERGED
# DUPLICATE SAFE
# ORGANIC SALES FIXED
# ══════════════════════════════════════════════════════════════════════════

def build_marketing(inv_final, ads_campaign, channel):

    # ══════════════════════════════════════════════════════════════════════
    # BASE
    # ══════════════════════════════════════════════════════════════════════

    df = inv_final.copy()

    # Remove duplicate columns
    df = df.loc[
        :,
        ~df.columns.duplicated()
    ]

    # ══════════════════════════════════════════════════════════════════════
    # CLEAN KEYS
    # ══════════════════════════════════════════════════════════════════════

    df["ASIN"] = (

        df["ASIN"]

        .astype(str)

        .str.upper()

        .str.strip()

        

        .str.replace(".0", "", regex=False)
    )

    df["Date"] = (

        pd.to_datetime(

            df["Date"],

            errors="coerce"
        )

        .dt.normalize()
    )

    df["Sales Channel"] = channel

    # ══════════════════════════════════════════════════════════════════════
    # ADS MERGE
    # INVENTORY IS PRIORITY
    # ══════════════════════════════════════════════════════════════════════

    if ads_campaign is not None and not ads_campaign.empty:

        a = ads_campaign.copy()

        # Remove duplicate columns
        a = a.loc[
            :,
            ~a.columns.duplicated()
        ]

        # ─────────────────────────────────────────────────────────────────
        # CLEAN KEYS
        # ─────────────────────────────────────────────────────────────────

        a["ASIN"] = (

            a["ASIN"]

            .astype(str)

            .str.upper()

            .str.strip()

            

            .str.replace(".0", "", regex=False)
        )

        a["Date"] = (

            pd.to_datetime(

                a["Date"],

                errors="coerce"
            )

            .dt.normalize()
        )

        # ─────────────────────────────────────────────────────────────────
        # SALES CHANNEL
        # ─────────────────────────────────────────────────────────────────

        if "Sales Channel" not in a.columns:

            a["Sales Channel"] = channel

        # ─────────────────────────────────────────────────────────────────
        # NUMERIC CLEANUP
        # ─────────────────────────────────────────────────────────────────

        numeric_ads_cols = [

            "ads_sales",

            "ads_unit_sold",

            "ads_spend",

            "Impressions",

            "Clicks",

            "Click-Thru Rate (CTR)",

            "Cost Per Click (CPC)",

            "Total Advertising Cost of Sales (ACOS)",

            "Total Return on Advertising Spend (ROAS)",

            "CVR"
        ]

        for col in numeric_ads_cols:

            if col in a.columns:

                a[col] = (

                    pd.to_numeric(

                        a[col],

                        errors="coerce"
                    )

                    .fillna(0)
                )

        # ─────────────────────────────────────────────────────────────────
        # KEEP REQUIRED COLS
        # ─────────────────────────────────────────────────────────────────

        keep_cols = [

            "ASIN",

            "Date",

            "Sales Channel",

            "ads_sales",

            "ads_unit_sold",

            "ads_spend",

            "Impressions",

            "Clicks",

            "Click-Thru Rate (CTR)",

            "Cost Per Click (CPC)",

            "Total Advertising Cost of Sales (ACOS)",

            "Total Return on Advertising Spend (ROAS)",

            "CVR",

            "Campaign Name",

            "Ad Group Name",

            "Campaign Type",

            "Portfolio name"
        ]

        keep_cols = [

            c for c in keep_cols

            if c in a.columns
        ]

        a = a[
            keep_cols
        ]

        # ─────────────────────────────────────────────────────────────────
        # REMOVE DUPLICATES
        # FIXED: subset now includes # "Ad Group Name" and "Campaign Type".
        # The old subset (ASIN, Date, Sales Channel, Campaign Name) treated
        # two different ad groups under the SAME campaign name as the SAME
        # row, and silently dropped one of them (keep="first") BEFORE the
        # sum-aggregation below ever ran -- losing real ads_sales/spend for
        # that ad group. A single campaign legitimately has multiple ad
        # groups (e.g. "Generic" + "Almond Oil" under one campaign name),
        # each with its own sales figure, so the subset must be specific
        # enough to only catch true duplicate rows, never distinct ad
        # groups or distinct campaign types (SPAP vs SD) that share a name.
        # ─────────────────────────────────────────────────────────────────

        dedup_subset = [
            c for c in ["ASIN", "Date", "Sales Channel",
                        "Campaign Name", "Ad Group Name", "Campaign Type"]
            if c in a.columns
        ]

        before_dedup_rows = len(a)
        before_dedup_sales = pd.to_numeric(a.get("ads_sales", 0), errors="coerce").fillna(0).sum()

        a = a.drop_duplicates(
            subset=dedup_subset,
            keep="first"
        )

        after_dedup_sales = pd.to_numeric(a.get("ads_sales", 0), errors="coerce").fillna(0).sum()
        if abs(before_dedup_sales - after_dedup_sales) > 0.01:
            print(
                f"⚠️ build_marketing[{channel}] dedup dropped "
                f"{before_dedup_rows - len(a)} row(s) AND "
                f"{before_dedup_sales - after_dedup_sales:,.2f} in ads_sales "
                f"-- check for true duplicate rows in ads_campaign input."
            )

        # ─────────────────────────────────────────────────────────────────
        # AGGREGATE ADS TO ONE ROW PER ASIN + DATE before merging
        # Without this, multiple campaigns per ASIN+Date cause inventory
        # rows to duplicate (one copy per campaign = fan-out).
        # ─────────────────────────────────────────────────────────────────

        sum_cols  = ["ads_sales","ads_unit_sold","ads_spend",
                     "Impressions","Clicks","CVR"]
        mean_cols = ["Click-Thru Rate (CTR)","Cost Per Click (CPC)",
                     "Total Advertising Cost of Sales (ACOS)",
                     "Total Return on Advertising Spend (ROAS)"]
        text_cols_ads = ["Campaign Name", "Ad Group Name", "Campaign Type","Portfolio name"]

        agg_dict = {}
        for c in sum_cols:
            if c in a.columns:
                agg_dict[c] = "sum"
        for c in mean_cols:
            if c in a.columns:
                agg_dict[c] = "mean"
        for c in text_cols_ads:
            if c in a.columns:
                # Join multiple campaign names with " | "
                agg_dict[c] = lambda x: " | ".join(x.dropna().astype(str).unique())

        if agg_dict:
            a_agg = (a.groupby(["ASIN","Date","Sales Channel"], as_index=False)
                      .agg(agg_dict))
        else:
            a_agg = a.drop_duplicates(subset=["ASIN","Date","Sales Channel"])

        # ─────────────────────────────────────────────────────────────────
        # MERGE (one-to-one: inventory row gets one aggregated ads row)
        # ─────────────────────────────────────────────────────────────────

        df = df.merge(

            a_agg,

            on=[

                "ASIN",

                "Date",

                "Sales Channel"
            ],

            how="left"  # inventory is priority; ads fills in where available
        )

    # ══════════════════════════════════════════════════════════════════════
    # FILL ADS COLUMNS
    # ══════════════════════════════════════════════════════════════════════

    ads_cols = [

        "ads_sales",

        "ads_unit_sold",

        "ads_spend",

        "Impressions",

        "Clicks",

        "Click-Thru Rate (CTR)",

        "Cost Per Click (CPC)",

        "Total Advertising Cost of Sales (ACOS)",

        "Total Return on Advertising Spend (ROAS)",

        "CVR"
    ]

    for col in ads_cols:

        if col not in df.columns:

            df[col] = 0

        df[col] = (

            pd.to_numeric(

                df[col],

                errors="coerce"
            )

            .fillna(0)
        )

    # ══════════════════════════════════════════════════════════════════════
    # TEXT ADS COLS
    # ══════════════════════════════════════════════════════════════════════

    text_cols = [

        "Campaign Name",

        "Ad Group Name",

        "Campaign Type",

        "Portfolio name"
    ]

    for col in text_cols:

        if col not in df.columns:

            df[col] = ""

        df[col] = (

            df[col]

            .fillna("")
        )

    # ══════════════════════════════════════════════════════════════════════
    # ORGANIC SALES
    # ══════════════════════════════════════════════════════════════════════

    df["organic_sale"] = (

        df["total_sales"]

        - df["ads_sales"]
    )

    df["organic_sale"] = (

        df["organic_sale"]

        .clip(lower=0)

        .round(2)
    )

    # ══════════════════════════════════════════════════════════════════════
    # FINAL SORT
    # ══════════════════════════════════════════════════════════════════════

    df = df.sort_values(

        [

            "Sales Channel",

            "ASIN",

            "Date"
        ]
    )

    return df

# ══════════════════════════════════════════════════════════════════════════
# BUILD FINAL MARKETING OUTPUTS
# ══════════════════════════════════════════════════════════════════════════

amz_mkt_final = build_marketing(

    amz_inv_final,

    amz_ads_campaign,

    "Vendor Central"
)

nms_mkt_final = build_marketing(

    nms_inv_final,

    nms_ads_campaign,

    "Noon Minutes"
)

nc_mkt_final = build_marketing(

    nc_inv_final,

    noon_ads_campaign,

    "Noon"
)

# ══════════════════════════════════════════════════════════════════════════
# FINAL COLUMN ORDER
# ══════════════════════════════════════════════════════════════════════════

MARKETING_COLS = INVENTORY_COLS + [

    "ads_sales",

    "ads_unit_sold",

    "ads_spend",

    "organic_sale",

    "Impressions",

    "Clicks",

    "Click-Thru Rate (CTR)",

    "Cost Per Click (CPC)",

    "Total Advertising Cost of Sales (ACOS)",

    "Total Return on Advertising Spend (ROAS)",

    "CVR",

    "Campaign Name",

    # 

    "Campaign Type",

    "Portfolio name"
]

# ══════════════════════════════════════════════════════════════════════════
# CONCAT ALL CHANNELS
# ══════════════════════════════════════════════════════════════════════════

marketing_output = pd.concat([

    align_cols(

        amz_mkt_final,

        MARKETING_COLS
    ),

    align_cols(

        nms_mkt_final,

        MARKETING_COLS
    ),

    align_cols(

        nc_mkt_final,

        MARKETING_COLS
    ),

], ignore_index=True)

# ══════════════════════════════════════════════════════════════════════════
# REMOVE DUPLICATES
# FIXED: subset is now ASIN + Date + Sales Channel only -- that's the true
# grain of marketing_output (one row per ASIN per Date per Channel, with
# ads already aggregated/joined into a single "Campaign Name" string by
# build_marketing). Including "Campaign Name" here added no protection
# and risked dropping a legitimate row if two different ASIN/Date/Channel
# combinations happened to end up with the same joined campaign-name text.
# ══════════════════════════════════════════════════════════════════════════

before_final_dedup_sales = pd.to_numeric(marketing_output.get("ads_sales", 0), errors="coerce").fillna(0).sum()

marketing_output = marketing_output.drop_duplicates(

    subset=[

        "ASIN",

        "Date",

        "Sales Channel"
    ],

    keep="first"
)

after_final_dedup_sales = pd.to_numeric(marketing_output.get("ads_sales", 0), errors="coerce").fillna(0).sum()
if abs(before_final_dedup_sales - after_final_dedup_sales) > 0.01:
    print(
        f"⚠️ Final marketing_output dedup dropped "
        f"{before_final_dedup_sales - after_final_dedup_sales:,.2f} in ads_sales "
        f"-- there were true ASIN+Date+Channel duplicates in the concatenated output."
    )

# ══════════════════════════════════════════════════════════════════════════
# CLEAN DATE
# ══════════════════════════════════════════════════════════════════════════

marketing_output["Date"] = (

    pd.to_datetime(

        marketing_output["Date"],

        errors="coerce"
    )

    .dt.normalize()
)

# ══════════════════════════════════════════════════════════════════════════
# FILL TEXT NULLS
# ══════════════════════════════════════════════════════════════════════════

text_fill_cols = [

    "Brand",

    "Product Category",

    "Product Sub Category",

    "Product Name",

    "Pack Type",

    "Campaign Name",

    # 

    "Campaign Type",

    "Portfolio name"
]

for col in text_fill_cols:

    if col in marketing_output.columns:

        marketing_output[col] = (

            marketing_output[col]

            .replace("", np.nan)

            .fillna("Unknown")
        )

# ══════════════════════════════════════════════════════════════════════════
# FILL NUMERIC NULLS
# ══════════════════════════════════════════════════════════════════════════

numeric_cols = marketing_output.select_dtypes(

    include=[

        "float",

        "int",

        "Int64"
    ]
).columns

marketing_output[numeric_cols] = (

    marketing_output[numeric_cols]

    .fillna(0)
)

# ══════════════════════════════════════════════════════════════════════════
# SORT
# ══════════════════════════════════════════════════════════════════════════

marketing_output = marketing_output.sort_values(

    [

        "Sales Channel",

        "ASIN",

        "Date"
    ]
)

marketing_output = marketing_output.reset_index(

    drop=True
)

# ══════════════════════════════════════════════════════════════════════════
# FINAL VALIDATION
# ══════════════════════════════════════════════════════════════════════════

print("\n✅ MARKETING OUTPUT READY")

print("\n📊 Shape:")
print(marketing_output.shape)

print("\n📦 Sales Channel Split:")
print(
    marketing_output["Sales Channel"]
    .value_counts(dropna=False)
)

print("\n💰 Total Ads Sales:")
print(
    round(
        marketing_output["ads_sales"].sum(),
        2
    )
)

print("\n💰 Total Organic Sales:")
print(
    round(
        marketing_output["organic_sale"].sum(),
        2
    )
)

# ══════════════════════════════════════════════════════════════════════════
# END-TO-END ADS CHECKSUM
# Compares each channel's pre-merge campaign-level ads_sales total against
# what survived into marketing_output. Any nonzero diff means a row got
# lost somewhere in build_marketing -- this will catch it immediately
# instead of it being discovered downstream in a report.
# ══════════════════════════════════════════════════════════════════════════

print("\n🔎 ADS CHECKSUM (pre-merge campaign total vs final marketing_output total):")

_checksum_pairs = [
    ("Vendor Central", amz_ads_campaign),
    ("Noon Minutes",   nms_ads_campaign),
    ("Noon",           noon_ads_campaign),
]

for _channel_name, _campaign_df in _checksum_pairs:
    if _campaign_df is None or _campaign_df.empty:
        continue
    _pre_total = pd.to_numeric(_campaign_df.get("ads_sales", 0), errors="coerce").fillna(0).sum()
    _post_total = pd.to_numeric(
        marketing_output.loc[marketing_output["Sales Channel"] == _channel_name, "ads_sales"],
        errors="coerce"
    ).fillna(0).sum()
    _diff = _pre_total - _post_total
    flag = "⚠️ MISMATCH" if abs(_diff) > 0.01 else "✅ OK"
    print(f"   {flag}  {_channel_name:<16} pre={_pre_total:,.2f}  post={_post_total:,.2f}  diff={_diff:,.2f}")

display(
    marketing_output.head(50)
)



✅ MARKETING OUTPUT READY

📊 Shape:
(24824, 52)

📦 Sales Channel Split:
Sales Channel
Noon              11484
Vendor Central    11270
Noon Minutes       2070
Name: count, dtype: int64

💰 Total Ads Sales:
82776.68

💰 Total Organic Sales:
128548.71

🔎 ADS CHECKSUM (pre-merge campaign total vs final marketing_output total):
   ✅ OK  Vendor Central   pre=44,256.39  post=44,256.39  diff=0.00
   ✅ OK  Noon Minutes     pre=34,959.90  post=34,959.90  diff=0.00
   ✅ OK  Noon             pre=3,560.39  post=3,560.39  diff=0.00


,ASIN,Brand,Product Category,Product Sub Category,Product Name,Pack Type,Pack Size,SC Ideal Price,VC Ideal Price,Available Inventory,...,Impressions,Clicks,Click-Thru Rate (CTR),Cost Per Click (CPC),Total Advertising Cost of Sales (ACOS),Total Return on Advertising Spend (ROAS),CVR,Campaign Name,Campaign Type,Portfolio name
0,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,Unknown,Unknown
1,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,Unknown,Unknown
2,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,Unknown,Unknown
3,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,Unknown,Unknown
4,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,Unknown,Unknown
5,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,Unknown,Unknown
6,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,Unknown,Unknown
7,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,Unknown,Unknown
8,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,Unknown,Unknown
9,Z00BAEEDE51FE817AFE40Z-1,Creme 21,Skin Care,Body Lotion,Creme 21 Body Lotion Dry Skin 600ml,1,600,0,43.697821,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown,Unknown,Unknown


In [50]:
marketing_output.columns

Index(['ASIN', 'Brand', 'Product Category', 'Product Sub Category',
       'Product Name', 'Pack Type', 'Pack Size', 'SC Ideal Price',
       'VC Ideal Price', 'Available Inventory', 'Sellable Inventory Amount',
       'Unsellable On Hand Inventory Amount', 'Unsellable On Hand Units',
       'Vendor Confirmation %', 'PO_received_amount', 'PO_received_Units',
       'Open POs', 'Receive_Fill_Rate', 'Overall Vendor Lead Time (days)',
       'Aged 90+ Days Sellable Inventory', 'Aged 90+ Days Sellable Units',
       'Date', 'Sales Channel', 'in_stock_flag', 'cumulative_instock_days',
       'day_of_month', 'Instock Rate', 'total_sales', 'total_units',
       'sell_through', 'Last 30 Days Sales', 'DOS', 'OOS Date',
       'min_available_qty', 'max_available_qty', 'Stock_Status',
       'No/Low Stock wt Open POs', 'No/Low Stock wt no Open POs', 'ads_sales',
       'ads_unit_sold', 'ads_spend', 'organic_sale', 'Impressions', 'Clicks',
       'Click-Thru Rate (CTR)', 'Cost Per Click (CPC)',
  

In [51]:
# ==========================================================
# RENAME EXISTING COLUMNS
# ==========================================================

rename_columns = {

    "ASIN": "asin",
    "Brand": "brand",
    "Product Category": "product_category",
    "Product Sub Category": "product_sub_category",
    "Product Name": "product_name",
    "Pack Type": "pack_type",
    "Pack Size": "pack_size",
    "SC Ideal Price": "sc_ideal_price",
    "VC Ideal Price": "vc_ideal_price",
    "CVR": "cvr",

    "Available Inventory": "available_inventory",
    "Sellable Inventory Amount": "sellable_inventory_amount",
    "Unsellable On Hand Inventory Amount": "unsellable_on_hand_inventory_amount",
    "Unsellable On Hand Units": "unsellable_on_hand_units",

    "Vendor Confirmation %": "vendor_confirmation_pct",
    "PO_received_amount": "po_received_amount",
    "PO_received_Units": "po_received_units",

    "Open Pos": "open_pos",
    "Open POs": "open_pos",

    "Receive_Fill_Rate": "receive_fill_rate",

    "Overall Vendor Lead Time (days)": "overall_vendor_lead_time_days",

    "Aged 90+ Days Sellable Inventory": "aged_90_plus_days_sellable_inventory",
    "Aged 90+ Days Sellable Units": "aged_90_plus_days_sellable_units",

    "Date": "date",
    "Sales Channel": "sales_channel",

    "Instock Rate": "instock_rate",

    "Last 30 Days Sales": "units_shipped_t30",

    "DOS": "dos",
    "OOS Date": "oos_date",

    "Stock_Status": "stock_status",

    "No/Low Stock wt Open Pos": "no_low_stock_wt_no_open_pos",
    "No/Low Stock wt Open POs": "no_low_stock_wt_open_pos",

    "No/Low Stock wt no Open Pos": "no_low_stock_wt_no_open_pos",
    "No/Low Stock wt no Open POs": "no_low_stock_wt_no_open_pos",

    "Impressions": "impressions",
    "Clicks": "clicks",
    "Click-Thru Rate (CTR)": "click_thru_rate_ctr",
    "Cost Per Click (CPC)": "cost_per_click_cpc",
    "Total Advertising Cost of Sales (ACOS)": "total_advertising_cost_of_sales_acos",
    "Total Return on Advertising Spend (ROAS)": "total_return_on_advertising_spend_roas",
    "Campaign Name": "campaign_name",
    "Campaign Type": "campaign_type",
    "Portfolio name": "portfolio_name",

}

marketing_output.rename(columns=rename_columns, inplace=True)

# ==========================================================
# REQUIRED COLUMN ORDER
# ==========================================================

required_columns = [
    "sku",
    "fnsku",
    "asin",
    "condition",
    "your_price",
    "mfn_listing_exists",
    "mfn_fulfillable_quantity",
    "afn_listing_exists",
    "afn_warehouse_quantity",
    "afn_fulfillable_quantity",
    "afn_unsellable_quantity",
    "afn_reserved_quantity",
    "afn_total_quantity",
    "per_unit_volume",
    "afn_inbound_working_quantity",
    "afn_inbound_shipped_quantity",
    "afn_inbound_receiving_quantity",
    "afn_researching_quantity",
    "afn_reserved_future_supply",
    "afn_future_supply_buyable",
    "store",
    "date",
    "reserved_qty",
    "reserved_customerorders",
    "reserved_fc_transfers",
    "reserved_fc_processing",
    "available_inventory",
    "units_shipped_t30",
    "units_shipped_t60",
    "units_shipped_t90",
    "open_pos",
    "sell_through",
    "dos",
    "min_available_qty",
    "max_available_qty",
    "stock_status",
    # "year_month",
    "instock_rate",
    "no_low_stock_wt_open_pos",
    "no_low_stock_wt_no_open_pos",
    "total_sales",
    "total_units",
    "sales_channel",
    "brand",
    "ads_sales",
    "ads_unit_sold",
    "ads_spend",
    "impressions",
    "clicks",
    "click_thru_rate_ctr",
    "cost_per_click_cpc",
    "total_advertising_cost_of_sales_acos",
    "total_return_on_advertising_spend_roas",
    "cvr",
    "campaign_name",
    "campaign_type",
    "portfolio_name",
    "product_name",
    "product_category",
    "pack_type",
    "pack_size",
    "sc_ideal_price",
    "vc_ideal_price",
    "product_sub_category",
    "vendor_confirmation_pct",
    "po_received_amount",
    "po_received_units",
    "receive_fill_rate",
    "overall_vendor_lead_time_days",
    "aged_90_plus_days_sellable_inventory",
    "aged_90_plus_days_sellable_units",
    "sellable_inventory_amount",
    "unsellable_on_hand_inventory_amount",
    "unsellable_on_hand_units",
    "in_stock_flag",
    "cumulative_instock_days",
    "day_of_month",
    "oos_date",
]

# ==========================================================
# TEXT COLUMNS
# ==========================================================

text_columns = [
    "sku",
    "fnsku",
    "asin",
    "condition",
    "store",
    "sales_channel",
    "brand",
    "campaign_name",
    "campaign_type",
    "portfolio_name",
    "product_name",
    "product_category",
    "product_sub_category",
    "pack_type",
    "pack_size",
    "stock_status",
    # "year_month",
    "oos_date",
]

# ==========================================================
# ADD MISSING COLUMNS
# ==========================================================

missing_columns = []

for col in required_columns:

    if col not in marketing_output.columns:

        missing_columns.append(col)

        if col in text_columns:
            marketing_output[col] = ""
        else:
            marketing_output[col] = 0

print("=" * 80)
print(f"Added {len(missing_columns)} Missing Columns")
print("=" * 80)

for col in missing_columns:
    print(col)

# ==========================================================
# REORDER COLUMNS
# ==========================================================

extra_columns = [
    col for col in marketing_output.columns
    if col not in required_columns
]

marketing_output = marketing_output[
    required_columns + extra_columns
]

# ==========================================================
# FINAL CHECK
# ==========================================================

print("\nFinal Shape :", marketing_output.shape)
print("Total Columns :", len(marketing_output.columns))
print("\nColumns:")
print(marketing_output.columns.tolist())

Added 26 Missing Columns
sku
fnsku
condition
your_price
mfn_listing_exists
mfn_fulfillable_quantity
afn_listing_exists
afn_warehouse_quantity
afn_fulfillable_quantity
afn_unsellable_quantity
afn_reserved_quantity
afn_total_quantity
per_unit_volume
afn_inbound_working_quantity
afn_inbound_shipped_quantity
afn_inbound_receiving_quantity
afn_researching_quantity
afn_reserved_future_supply
afn_future_supply_buyable
store
reserved_qty
reserved_customerorders
reserved_fc_transfers
reserved_fc_processing
units_shipped_t60
units_shipped_t90

Final Shape : (24824, 78)
Total Columns : 78

Columns:
['sku', 'fnsku', 'asin', 'condition', 'your_price', 'mfn_listing_exists', 'mfn_fulfillable_quantity', 'afn_listing_exists', 'afn_warehouse_quantity', 'afn_fulfillable_quantity', 'afn_unsellable_quantity', 'afn_reserved_quantity', 'afn_total_quantity', 'per_unit_volume', 'afn_inbound_working_quantity', 'afn_inbound_shipped_quantity', 'afn_inbound_receiving_quantity', 'afn_researching_quantity', 'afn_res

In [52]:
marketing_output.to_csv("emami_uae_marketing_test.csv", index=False)

In [53]:
marketing_output.columns

Index(['sku', 'fnsku', 'asin', 'condition', 'your_price', 'mfn_listing_exists',
       'mfn_fulfillable_quantity', 'afn_listing_exists',
       'afn_warehouse_quantity', 'afn_fulfillable_quantity',
       'afn_unsellable_quantity', 'afn_reserved_quantity',
       'afn_total_quantity', 'per_unit_volume', 'afn_inbound_working_quantity',
       'afn_inbound_shipped_quantity', 'afn_inbound_receiving_quantity',
       'afn_researching_quantity', 'afn_reserved_future_supply',
       'afn_future_supply_buyable', 'store', 'date', 'reserved_qty',
       'reserved_customerorders', 'reserved_fc_transfers',
       'reserved_fc_processing', 'available_inventory', 'units_shipped_t30',
       'units_shipped_t60', 'units_shipped_t90', 'open_pos', 'sell_through',
       'dos', 'min_available_qty', 'max_available_qty', 'stock_status',
       'instock_rate', 'no_low_stock_wt_open_pos',
       'no_low_stock_wt_no_open_pos', 'total_sales', 'total_units',
       'sales_channel', 'brand', 'ads_sales', 'ads_u

In [54]:
import re
 
def clean_columns(df):

    def to_snake_case(col):

        col = str(col).strip()
 
        # Replace common symbols

        col = col.replace("%", "pct")

        col = col.replace("&", "and")

        col = col.replace("@", "at")
 
        # Replace separators with underscore

        col = re.sub(r"[\/\\\-\s]+", "_", col)
 
        # Remove brackets

        col = re.sub(r"[\(\)\[\]\{\}]", "", col)
 
        # Remove all remaining special characters except underscore

        col = re.sub(r"[^A-Za-z0-9_]", "", col)
 
        # Convert CamelCase to snake_case (optional)

        col = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", col)
 
        # Replace multiple underscores with a single one

        col = re.sub(r"_+", "_", col)
 
        # Remove leading/trailing underscores

        col = col.strip("_")
 
        return col.lower()
 
    df.columns = [to_snake_case(col) for col in df.columns]

    return df
 

In [55]:
import time
import pandas as pd
from io import BytesIO
from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import ServiceResponseError
import pandas as pd


df = marketing_output.copy()
 
# -------- STEP 1: CLEAN COLUMN NAMES --------
def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()              # remove leading/trailing spaces
        .str.lower()              # convert to lowercase
        .str.replace(r'[\s\-]+', '_', regex=True)  # replace spaces with underscore
    )
    return df
 
# Apply cleaning
df = clean_columns(df)
 
# ====================================================================
# AZURE CONFIG
# ====================================================================
AZURE_CONNECTION_STRING = (
    "DefaultEndpointsProtocol=https;AccountName=thriveclientdatabase;AccountKey=t3YQjkSXWXafdSIzp48QxNgmuW8BSh+6FzHaEy6VhGbgINYgo7Sica0BZw9sAKEPbwBcjqqhK5p6+AStuxDzSA==;EndpointSuffix=core.windows.net"
)
 
AZURE_CONTAINER = "thrive-client-emami"   # must be lowercase, valid format
 
# ====================================================================
# UPLOAD WITH RETRY
# ====================================================================
def upload_blob_with_retry(blob_client, data, blob_path, max_retries=3, max_concurrency=4, timeout=600):
    """Upload blob with retry + exponential backoff."""
    for attempt in range(max_retries):
        try:
            data.seek(0)
 
            blob_client.upload_blob(
                data,
                overwrite=True,
                max_concurrency=max_concurrency,
                timeout=timeout
            )
 
            print(f"✅ Uploaded: {blob_path}")
            return
 
        except ServiceResponseError as e:
            wait = (2 ** attempt) * 5
            print(f"⚠️ Azure upload failed (Attempt {attempt+1}/{max_retries}) for {blob_path}")
            print(f"   Error: {str(e)}")
            print(f"   Retrying in {wait} sec...\n")
            time.sleep(wait)
 
        except Exception as e:
            print(f"❌ Unexpected error for {blob_path}: {str(e)}")
            raise
 
    print(f"❌ Final Failure: Could NOT upload {blob_path} after {max_retries} attempts.")
    raise RuntimeError("Upload Failed")
 
 
def upload_to_azure_csv(df, blob_path, connection_string, container_name):
    df = df.copy()
 
    # Fix all object columns
    object_cols = df.select_dtypes(include='object').columns
    df[object_cols] = df[object_cols].astype(str)
 
    # Replace NaN with None
    df = df.where(pd.notnull(df), None)
 
 
    buffer = BytesIO()
    df.to_csv(buffer, index=False)
    buffer.seek(0)
 
    blob_service_client = BlobServiceClient.from_connection_string(
        connection_string,
        connection_timeout=600,
        read_timeout=600
    )
 
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_path)
    upload_blob_with_retry(blob_client, buffer, blob_path)
 
upload_to_azure_csv(df,"sanity_data/datahive/marketing_output.csv", AZURE_CONNECTION_STRING, AZURE_CONTAINER)
 
 

✅ Uploaded: sanity_data/datahive/marketing_output.csv


In [56]:
marketing_output.columns

Index(['sku', 'fnsku', 'asin', 'condition', 'your_price', 'mfn_listing_exists',
       'mfn_fulfillable_quantity', 'afn_listing_exists',
       'afn_warehouse_quantity', 'afn_fulfillable_quantity',
       'afn_unsellable_quantity', 'afn_reserved_quantity',
       'afn_total_quantity', 'per_unit_volume', 'afn_inbound_working_quantity',
       'afn_inbound_shipped_quantity', 'afn_inbound_receiving_quantity',
       'afn_researching_quantity', 'afn_reserved_future_supply',
       'afn_future_supply_buyable', 'store', 'date', 'reserved_qty',
       'reserved_customerorders', 'reserved_fc_transfers',
       'reserved_fc_processing', 'available_inventory', 'units_shipped_t30',
       'units_shipped_t60', 'units_shipped_t90', 'open_pos', 'sell_through',
       'dos', 'min_available_qty', 'max_available_qty', 'stock_status',
       'instock_rate', 'no_low_stock_wt_open_pos',
       'no_low_stock_wt_no_open_pos', 'total_sales', 'total_units',
       'sales_channel', 'brand', 'ads_sales', 'ads_u

## 18 · BuyBox – Daily Scrape + Save
> Run this section daily. Saves one file per day under `emami_amazon_buybox/{year}/emami/ae/{month}/{day}/ae_BUYBOX_{day}.xlsx`

In [57]:
import time, random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import selenium.common.exceptions as se
try:
    from webdriver_manager.chrome import ChromeDriverManager
except ImportError:
    pass

DOMAIN   = "ae"
CURRENCY = "AED"
now      = datetime.datetime.now()
run_date = now.strftime("%Y-%m-%d")
run_time = now.strftime("%H:%M:%S")

# ── Dynamic output path ────────────────────────────────────────────────────
year_str  = now.strftime("%Y")
month_str = now.strftime("%b").lower()   # may, apr, etc.
day_str   = now.strftime("%d")           # 01, 22, etc.

base_output_path = os.path.join(
    BASE, "output", "emami_amazon_buybox",
    year_str, "emami", DOMAIN, month_str, day_str
)
os.makedirs(base_output_path, exist_ok=True)

# ── ASINs from helper file ─────────────────────────────────────────────────
asins_bb = h_asin["ASIN"].dropna().unique()
print("Total ASINs for BuyBox:", len(asins_bb))

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

results = []

def get_price():
    try:
        return (driver.find_element(By.CLASS_NAME, "a-price-whole").text + "." +
                driver.find_element(By.CLASS_NAME, "a-price-fraction").text)
    except:
        return ""

def get_moq():
    try:
        text = driver.find_element(By.XPATH, "//*[contains(text(),'Minimum order quantity')]").text
        return text.split(":")[-1].strip()
    except:
        return "1"

for asin in asins_bb:
    print("\nProcessing:", asin)
    url = f"https://www.amazon.{DOMAIN}/dp/{asin}"

    try:
        driver.get(url)
    except se.TimeoutException:
        print(f"[TIMEOUT] {asin}"); continue

    time.sleep(random.uniform(4, 6))

    try:
        title = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.ID, "productTitle"))
        ).text.strip()
    except:
        print(f"[NO TITLE] {asin}"); continue

    price  = get_price()
    seller = "Amazon.ae"
    moq    = get_moq()

    # ── Rating, Reviews, BSR Rank ──────────────────────────────────────────
    rating   = ""
    reviews  = ""
    bsr_main = ""     # e.g. "#11,708 in Beauty"
    bsr_sub  = ""     # e.g. "#196 in Body Creams"

    try:
        rating = driver.find_element(
            By.XPATH, "//span[@data-hook='rating-out-of-text']"
        ).text.strip().split()[0]
    except:
        try:
            rating = driver.find_element(
                By.XPATH, "//span[@id='acrPopover']"
            ).get_attribute("title").strip().split()[0]
        except:
            pass

    try:
        reviews = driver.find_element(
            By.XPATH, "//span[@id='acrCustomerReviewText']"
        ).text.strip().replace(",", "").split()[0]
    except:
        pass

    try:
        # BSR block — look for the product information table
        bsr_elements = driver.find_elements(
            By.XPATH,
            "//th[contains(text(),'Best Sellers Rank')]/following-sibling::td//span"
        )
        if not bsr_elements:
            bsr_elements = driver.find_elements(
                By.XPATH,
                "//*[@id='productDetails_detailBullets_sections1']//tr[contains(.,'Best Sellers Rank')]//span"
            )
        if bsr_elements and len(bsr_elements) >= 1:
            bsr_main = bsr_elements[0].text.strip()
        if bsr_elements and len(bsr_elements) >= 2:
            bsr_sub  = bsr_elements[1].text.strip()
    except:
        pass

    print(f"  Rating={rating} | Reviews={reviews} | BSR={bsr_main} | Sub={bsr_sub}")

    # Seller detection (3 strategies)
    for sel_id in ["sellerProfileTriggerId"]:
        try:
            t = driver.find_element(By.ID, sel_id).text.strip()
            if t: seller = t; break
        except: pass

    if seller == "Amazon.ae":
        try:
            t = driver.find_element(By.XPATH,
                "//*[contains(normalize-space(text()),'Sold by')]/following::a[1]").text.strip()
            if t: seller = t
        except: pass

    if seller == "Amazon.ae":
        try:
            t = driver.find_element(By.XPATH,
                "//div[@id='tabular-buybox']//span[contains(@class,'tabular-buybox-text') and contains(.,'Sold by')]/following::a[1]"
            ).text.strip()
            if t: seller = t
        except: pass

    seller = seller.strip() or "Amazon.ae"

    results.append({"Date": run_date, "Time": run_time, "ASIN": asin, "Title": title,
                    "Seller": seller, "Price": price, "MOQ": moq, "BuyBox": "YES", "Link": url,
                    "Rating": rating, "Reviews": reviews,
                    "BSR_Main": bsr_main, "BSR_Sub": bsr_sub})
    buybox_seller = seller

    # ── Offer listing ──────────────────────────────────────────────────────
    offer_url   = f"https://www.amazon.{DOMAIN}/gp/offer-listing/{asin}"
    offers_loaded = False
    for attempt in range(1, 4):
        try:
            driver.get(offer_url); time.sleep(5); offers_loaded = True; break
        except (se.TimeoutException, se.WebDriverException) as e:
            print(f"[WARN] Offer page attempt {attempt}/3: {e}")
            if attempt < 3: time.sleep(5)

    if not offers_loaded:
        continue

    seen = set()
    for offer in driver.find_elements(By.XPATH, "//div[contains(@id,'aod-offer')]"):
        try:
            osel  = ""
            links = offer.find_elements(By.XPATH, ".//div[contains(@id,'aod-offer-soldBy')]//a")
            if links:
                osel = links[0].text.strip() or (links[0].get_attribute("title") or "").strip()

            if not osel:
                for s in offer.find_elements(By.XPATH, ".//div[contains(@id,'aod-offer-soldBy')]//span"):
                    t = s.text.strip()
                    if t and "sold by" not in t.lower():
                        osel = t; break

            if not osel:
                try:
                    div = offer.find_element(By.XPATH, ".//div[contains(@id,'aod-offer-soldBy')]")
                    lines = [(l.strip()) for l in (div.get_attribute("innerText") or "").split("\n") if l.strip()]
                    if lines: osel = lines[-1]
                except: pass

            osel = osel.strip()
            if not osel or osel in seen or osel == buybox_seller: continue
            seen.add(osel)

            try:
                oprice = (offer.find_element(By.CSS_SELECTOR, ".a-price-whole").text + "." +
                          offer.find_element(By.CSS_SELECTOR, ".a-price-fraction").text)
            except:
                oprice = ""
            if not oprice: continue

            results.append({"Date": run_date, "Time": run_time, "ASIN": asin,
                            "Title": title, "Seller": osel, "Price": oprice,
                            "MOQ": "1", "BuyBox": "NO", "Link": url})
            print(f"  Offer: {osel} | {oprice}")
        except:
            pass

driver.quit()

# ── Standardise raw output ─────────────────────────────────────────────────
df_out = pd.DataFrame(results)

if not df_out.empty:
    df_out.columns = df_out.columns.str.lower().str.strip()
    df_out.rename(columns={
        "title": "product_name", "seller": "sold_by",
        "Rating": "rating", "Reviews": "reviews",
        "BSR_Main": "bsr_main", "BSR_Sub": "bsr_sub"
    }, inplace=True)

    df_out["region"]  = DOMAIN.upper()
    df_out["buybox"]  = df_out["buybox"].astype(str).str.upper()
    df_out["moq"]     = df_out["moq"].astype(str).str.extract(r'(\d+)')[0].fillna("1")
    df_out["price"]   = pd.to_numeric(df_out["price"], errors="coerce")

    standard_cols = ["date","time","region","asin","buybox","link","product_name","rating","reviews","bsr_main","bsr_sub","price","sold_by","moq"]
    for col in standard_cols:
        if col not in df_out.columns:
            df_out[col] = None
    df_out = df_out[standard_cols]

    # ── Save daily file ────────────────────────────────────────────────────
    file_name = f"{DOMAIN}_BUYBOX_{day_str}.xlsx"
    file_path_daily = os.path.join(base_output_path, file_name)
    df_out.to_excel(file_path_daily, index=False)
    print(f"\n✅ Daily BuyBox saved: {file_path_daily}")
else:
    print("❌ No BuyBox results")

Total ASINs for BuyBox: 98

Processing: B07H8Q2JGG
  Rating=4.2 | Reviews=(400) | BSR= | Sub=
  Offer: ALKABAYEL AE | 25.00
  Offer: PEAK NEST FZ LLE | 17.99
  Offer: Al Amin Beauty Store | 18.50
  Offer: Bara Trading | 42.77

Processing: B096F51K8B
  Rating=4.2 | Reviews=(39) | BSR= | Sub=
  Offer: PEAK NEST FZ LLE | 17.99

Processing: B096F6NR5L
  Rating=4.2 | Reviews=(39) | BSR= | Sub=
  Offer: Day To Day Hypermarket LLC | 33.35

Processing: B096F4ZYSR
  Rating=4.2 | Reviews=(39) | BSR= | Sub=
  Offer: PEAK NEST FZ LLE | 14.99

Processing: B096F4NXF3
  Rating=4.2 | Reviews=(39) | BSR= | Sub=

Processing: B096F56YHT
  Rating=4.2 | Reviews=(39) | BSR= | Sub=
  Offer: PEAK NEST FZ LLE | 17.99

Processing: B096F7J9ZF
  Rating=4.2 | Reviews=(39) | BSR= | Sub=
  Offer: Day To Day Hypermarket LLC | 13.99

Processing: B0G64JZJN9
  Rating=5 | Reviews=(1) | BSR= | Sub=

Processing: B0CNQ6JWGL
  Rating=4.7 | Reviews=(6) | BSR= | Sub=
  Offer: Bara Trading | 39.00

Processing: B0D7ZPCZNF
  Rati

## 19 · BuyBox – Merge Historical Files + Build Final Output
> Reads ALL daily files from all month sub-folders. Merges INTO inventory output (Outer join on ASIN + Date).

In [58]:
# ── 19a. Load all historical daily BuyBox files ────────────────────────────
standard_cols = ["date","time","region","asin","buybox","link","product_name","rating","reviews","bsr_main","bsr_sub","price","sold_by","moq"]

dfs = []

base_month_path = os.path.join(
    BASE, "output", "emami_amazon_buybox", str(YEAR), "emami", "ae"
)

if os.path.isdir(base_month_path):
    for month_folder in os.listdir(base_month_path):
        month_path = os.path.join(base_month_path, month_folder)
        if not os.path.isdir(month_path): continue

        for day_folder in os.listdir(month_path):
            day_path = os.path.join(month_path, day_folder)
            if not os.path.isdir(day_path): continue

            for file in os.listdir(day_path):
                if not file.endswith(".xlsx"): continue
                fp = os.path.join(day_path, file)
                try:
                    df = pd.read_excel(fp)
                    df.columns = df.columns.str.lower().str.strip()
                    df.rename(columns={"title": "product_name", "seller": "sold_by", "type": "buybox"}, inplace=True)

                    if "buybox" in df.columns:
                        df["buybox"] = (
                            df["buybox"].astype(str).str.lower()
                            .replace({"yes": "YES", "main": "YES"})
                            .apply(lambda x: "YES" if x == "YES" else "NO")
                        )
                    else:
                        df["buybox"] = "NO"

                    if "region" not in df.columns: df["region"] = "AE"
                    if "moq"    in df.columns:
                        df["moq"] = df["moq"].astype(str).str.extract(r'(\d+)')[0].fillna("1")
                    else:
                        df["moq"] = "1"
                    if "price" in df.columns:
                        df["price"] = pd.to_numeric(df["price"], errors="coerce")
                    else:
                        df["price"] = None

                    for col in standard_cols:
                        if col not in df.columns: df[col] = None
                    df = df[standard_cols]
                    dfs.append(df)
                except Exception as e:
                    print(f"❌ Error reading {fp}: {e}")
else:
    print("⚠️ BuyBox base folder not found")

if dfs:
    buybox_df = pd.concat(dfs, ignore_index=True)
    buybox_df["date"] = pd.to_datetime(buybox_df["date"], errors="coerce")
    buybox_df["asin"] = buybox_df["asin"].astype(str).str.upper().str.strip()
    print(f"\n✅ Historical BuyBox loaded | Shape: {buybox_df.shape}")
else:
    buybox_df = pd.DataFrame(columns=standard_cols)
    print("❌ No BuyBox data found")


# ── 19b. Transpose to wide format (one row per ASIN + Date) ───────────────
rows = []
for (asin, dt), grp in buybox_df.groupby(["asin", "date"]):
    row = {"ASIN": asin, "Date": dt,
           "time":         grp["time"].iloc[0],
           "product_name": grp["product_name"].iloc[0],
           "link":         grp["link"].iloc[0],
           "rating":       grp["rating"].iloc[0]  if "rating"   in grp.columns else "",
           "reviews":      grp["reviews"].iloc[0] if "reviews"  in grp.columns else "",
           "bsr_main":     grp["bsr_main"].iloc[0] if "bsr_main" in grp.columns else "",
           "bsr_sub":      grp["bsr_sub"].iloc[0]  if "bsr_sub"  in grp.columns else ""}

    owner = grp[grp["buybox"] == "YES"]
    if not owner.empty:
        row["current_owner"]       = owner.iloc[0]["sold_by"]
        row["current_owner_price"] = owner.iloc[0]["price"]
        row["current_owner_moq"]   = owner.iloc[0]["moq"]

    hijackers = grp[grp["buybox"] != "YES"]
    for i, (_, r) in enumerate(hijackers.iterrows(), start=1):
        row[f"hijacker_{i}"]       = r["sold_by"]
        row[f"hijacker_{i}_price"] = r["price"]
        row[f"hijacker_{i}_moq"]   = r["moq"]

    rows.append(row)

buybox_transposed = pd.DataFrame(rows)

if not buybox_transposed.empty:
    buybox_transposed["Date"] = pd.to_datetime(buybox_transposed["Date"], errors="coerce").dt.normalize()
    buybox_transposed["ASIN"] = buybox_transposed["ASIN"].astype(str).str.upper().str.strip()
    buybox_transposed = buybox_transposed.drop_duplicates(subset=["ASIN", "Date"])

print("✅ BuyBox Transposed | Shape:", buybox_transposed.shape)


# ── 19c. Merge BuyBox INTO Inventory Output (priority = inventory) ─────────
# Use VENDOR CENTRAL inventory base only (buybox is Amazon)
vc_inv_base = align_cols(
    amz_inv_final, INVENTORY_COLS
).copy()
vc_inv_base["Date"] = pd.to_datetime(vc_inv_base["Date"], errors="coerce").dt.normalize()
vc_inv_base["ASIN"] = vc_inv_base["ASIN"].astype(str).str.upper().str.strip()

buybox_merged = vc_inv_base.merge(
    buybox_transposed,
    on=["ASIN", "Date"],
    how="left"   # inventory dates are the backbone; buybox fills in where available
)

# ── 19d. Analytics ────────────────────────────────────────────────────────
hijacker_name_cols  = [c for c in buybox_merged.columns if c.startswith("hijacker_") and "_price" not in c and "_moq" not in c]
hijacker_price_cols = [c for c in buybox_merged.columns if "hijacker_" in c and "_price" in c]

buybox_merged["hijacker_count"]     = buybox_merged[hijacker_name_cols].notna().sum(axis=1)
buybox_merged["min_hijacker_price"] = buybox_merged[hijacker_price_cols].min(axis=1) if hijacker_price_cols else np.nan
buybox_merged["price_gap"]          = (buybox_merged["current_owner_price"] - buybox_merged["min_hijacker_price"])
buybox_merged["is_hijacked"]         = buybox_merged["hijacker_count"].apply(lambda x: "YES" if x > 0 else "NO")

# ── 19e. Fill text / numeric nulls ────────────────────────────────────────
buybox_merged = buybox_merged.fillna(0)

buybox_output = buybox_merged.copy()

print("\n✅ BuyBox Output Ready | Shape:", buybox_output.shape)


✅ Historical BuyBox loaded | Shape: (16832, 14)
✅ BuyBox Transposed | Shape: (8351, 42)

✅ BuyBox Output Ready | Shape: (11270, 82)


C:\Users\Shambhavi Dixit\AppData\Local\Temp\ipykernel_35500\3730348160.py:125: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  buybox_merged = buybox_merged.fillna(0)


In [59]:
# ==========================================================
# RENAME EXISTING COLUMNS
# ==========================================================

rename_columns = {

    "ASIN": "asin",
    "Available Inventory": "available_inventory",
    "Sellable Inventory Amount": "sellable_inventory_amount",
    "Unsellable On Hand Inventory Amount": "unsellable_on_hand_inventory_amount",
    "Unsellable On Hand Units": "unsellable_on_hand_units",

    "Vendor Confirmation %": "vendor_confirmation_pct",
    "PO_received_amount": "po_received_amount",
    "PO_received_Units": "po_received_units",

    "Open Pos": "open_pos",
    "Open POs": "open_pos",

    "Receive_Fill_Rate": "receive_fill_rate",

    "Overall Vendor Lead Time (days)": "overall_vendor_lead_time_days",

    "Aged 90+ Days Sellable Inventory": "aged_90_days_sellable_inventory",
    "Aged 90+ Days Sellable Units": "aged_90_days_sellable_units",

    "Date": "date",

    "Sales Channel": "sales_channel",

    "Instock Rate": "instock_rate",

    "Last 30 Days Sales": "units_shipped_t30",

    "DOS": "dos",

    "OOS Date": "oos_date",

    "Stock_Status": "stock_status",

    "No/Low Stock wt Open Pos": "no_low_stock_wt_open_pos",
    "No/Low Stock wt Open POs": "no_low_stock_wt_open_pos",

    "No/Low Stock wt no Open Pos": "no_low_stock_wt_no_open_pos",
    "No/Low Stock wt no Open POs": "no_low_stock_wt_no_open_pos",

    "Brand": "brand",
    "Product Category": "product_category",
    "Product Sub Category": "product_sub_category",
    "Product Name": "product_name",
    "Pack Type": "pack_type",
    "Pack Size": "pack_size",

    "SC Ideal Price": "sc_ideal_price",
    "VC Ideal Price": "vc_ideal_price",

    "time": "date_temp",

    "link": "current_owner",
    "rating": "current_owner_price",
    "reviews": "current_owner_moq",

    "bsr_main": "min_hijacker_price",
    "bsr_sub": "price_gap",

}

buybox_output.rename(columns=rename_columns, inplace=True)

# ==========================================================
# REQUIRED COLUMN ORDER
# ==========================================================

required_columns = [
    "asin",
    "brand",
    "product_sub_category",
    "vendor_confirmation_pct",
    "po_received_amount",
    "po_received_units",
    "open_pos",
    "receive_fill_rate",
    "overall_vendor_lead_time_days",
    "aged_90_days_sellable_inventory",
    "aged_90_days_sellable_units",
    "sellable_inventory_amount",
    "available_inventory",
    "unsellable_on_hand_inventory_amount",
    "unsellable_on_hand_units",
    "date",
    "product_name",
    "product_category",
    "pack_type",
    "pack_size",
    "sc_ideal_price",
    "vc_ideal_price",
    "sales_channel",
    "in_stock_flag",
    "cumulative_instock_days",
    "day_of_month",
    "instock_rate",
    "total_sales",
    "total_units",
    "sell_through",
    "dos",
    "oos_date",
    "min_available_qty",
    "max_available_qty",
    "stock_status",
    "no_low_stock_wt_open_pos",
    "no_low_stock_wt_no_open_pos",

    "date_temp",
    "current_owner",
    "current_owner_price",
    "current_owner_moq",

    "hijacker_1","hijacker_1_price","hijacker_1_moq",
    "hijacker_2","hijacker_2_price","hijacker_2_moq",
    "hijacker_3","hijacker_3_price","hijacker_3_moq",
    "hijacker_4","hijacker_4_price","hijacker_4_moq",
    "hijacker_5","hijacker_5_price","hijacker_5_moq",
    "hijacker_6","hijacker_6_price","hijacker_6_moq",
    "hijacker_7","hijacker_7_price","hijacker_7_moq",
    "hijacker_8","hijacker_8_price","hijacker_8_moq",
    "hijacker_9","hijacker_9_price","hijacker_9_moq",
    "hijacker_10","hijacker_10_price","hijacker_10_moq",
    "hijacker_11","hijacker_11_price","hijacker_11_moq",
    "hijacker_12","hijacker_12_price","hijacker_12_moq",
    "hijacker_13","hijacker_13_price","hijacker_13_moq",
    "hijacker_14","hijacker_14_price","hijacker_14_moq",
    "hijacker_15","hijacker_15_price","hijacker_15_moq",
    "hijacker_16","hijacker_16_price","hijacker_16_moq",
    "hijacker_17","hijacker_17_price","hijacker_17_moq",
    "hijacker_18","hijacker_18_price","hijacker_18_moq",
    "hijacker_19","hijacker_19_price","hijacker_19_moq",
    "hijacker_20","hijacker_20_price","hijacker_20_moq",

    "sku",
    "fnsku",
    "condition",
    "your_price",
    "mfn_listing_exists",
    "mfn_fulfillable_quantity",
    "afn_listing_exists",
    "afn_warehouse_quantity",
    "afn_fulfillable_quantity",
    "afn_unsellable_quantity",
    "afn_reserved_quantity",
    "afn_total_quantity",
    "per_unit_volume",
    "afn_inbound_working_quantity",
    "afn_inbound_shipped_quantity",
    "afn_inbound_receiving_quantity",
    "afn_researching_quantity",
    "afn_reserved_future_supply",
    "afn_future_supply_buyable",
    "store",
    "reserved_qty",
    "reserved_customerorders",
    "reserved_fc_transfers",
    "reserved_fc_processing",
    "units_shipped_t30",
    "units_shipped_t60",
    "units_shipped_t90",
    # "year_month",
]

# ==========================================================
# TEXT COLUMNS
# ==========================================================

text_columns = [
    c for c in required_columns
    if any(x in c for x in [
        "asin","brand","product","pack","sales_channel",
        "stock_status","condition","store","sku","fnsku",
        "date_temp","current_owner",
        # "year_month",
        "oos_date",
        "hijacker"
    ])
]

# ==========================================================
# ADD MISSING COLUMNS
# ==========================================================

missing_columns = []

for col in required_columns:

    if col not in buybox_output.columns:

        missing_columns.append(col)

        if col in text_columns:
            buybox_output[col] = ""
        else:
            buybox_output[col] = 0

print("=" * 80)
print(f"Added {len(missing_columns)} Missing Columns")
print("=" * 80)
print(missing_columns)

# ==========================================================
# REORDER
# ==========================================================

extra_columns = [
    c for c in buybox_output.columns
    if c not in required_columns
]

buybox_output = buybox_output[
    required_columns + extra_columns
]

print("\nFinal Shape :", buybox_output.shape)
print("Total Columns :", len(buybox_output.columns))

Added 56 Missing Columns
['hijacker_11', 'hijacker_11_price', 'hijacker_11_moq', 'hijacker_12', 'hijacker_12_price', 'hijacker_12_moq', 'hijacker_13', 'hijacker_13_price', 'hijacker_13_moq', 'hijacker_14', 'hijacker_14_price', 'hijacker_14_moq', 'hijacker_15', 'hijacker_15_price', 'hijacker_15_moq', 'hijacker_16', 'hijacker_16_price', 'hijacker_16_moq', 'hijacker_17', 'hijacker_17_price', 'hijacker_17_moq', 'hijacker_18', 'hijacker_18_price', 'hijacker_18_moq', 'hijacker_19', 'hijacker_19_price', 'hijacker_19_moq', 'hijacker_20', 'hijacker_20_price', 'hijacker_20_moq', 'sku', 'fnsku', 'condition', 'your_price', 'mfn_listing_exists', 'mfn_fulfillable_quantity', 'afn_listing_exists', 'afn_warehouse_quantity', 'afn_fulfillable_quantity', 'afn_unsellable_quantity', 'afn_reserved_quantity', 'afn_total_quantity', 'per_unit_volume', 'afn_inbound_working_quantity', 'afn_inbound_shipped_quantity', 'afn_inbound_receiving_quantity', 'afn_researching_quantity', 'afn_reserved_future_supply', 'afn_f

In [60]:
buybox_output

,asin,brand,product_sub_category,vendor_confirmation_pct,po_received_amount,po_received_units,open_pos,receive_fill_rate,overall_vendor_lead_time_days,aged_90_days_sellable_inventory,...,min_hijacker_price,min_hijacker_price,price_gap,price_gap,hijacker_count,min_hijacker_price,min_hijacker_price,price_gap,price_gap,is_hijacked
0,B00F93HFME,Himani Navratna,Hair Oil,0.0,0.00,0.0,7.0,0.0,0.00,0.0,...,0,0.0,0,0.0,0,0,0.0,0,0.0,NO
1,B00F93HFME,Himani Navratna,Hair Oil,0.0,120.26,7.0,0.0,0.0,5.14,0.0,...,0,0.0,0,0.0,0,0,0.0,0,0.0,NO
2,B00F93HFME,Himani Navratna,Hair Oil,0.0,0.00,0.0,0.0,0.0,0.00,0.0,...,0,0.0,0,0.0,0,0,0.0,0,0.0,NO
3,B00F93HFME,Himani Navratna,Hair Oil,0.0,0.00,0.0,0.0,0.0,0.00,0.0,...,0,0.0,0,0.0,0,0,0.0,0,0.0,NO
4,B00F93HFME,Himani Navratna,Hair Oil,0.0,0.00,0.0,0.0,0.0,0.00,0.0,...,0,0.0,0,0.0,0,0,0.0,0,0.0,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11265,B0GH9ZDVZX,Creme 21,All-purpose cream,0.00%,0.00,0.0,96.0,0.0,0.00,AED 0.00,...,0,0.0,0,0.0,0,0,0.0,0,0.0,NO
11266,B0GH9ZDVZX,Creme 21,All-purpose cream,0.00%,0.00,1.0,95.0,0.0,10.00,AED 0.00,...,0,0.0,0,0.0,0,0,0.0,0,0.0,NO
11267,B0GH9ZDVZX,Creme 21,All-purpose cream,0.00%,0.00,7.0,55.0,0.0,11.14,AED 0.00,...,0,0.0,0,0.0,0,0,0.0,0,0.0,NO
11268,B0GH9ZDVZX,Creme 21,All-purpose cream,0.00%,0.00,29.0,26.0,0.0,5.38,AED 0.00,...,0,0.0,0,0.0,0,0,0.0,0,0.0,NO


In [61]:
buybox_output.to_csv("emami_uae_buybox_test.csv", index=False)

In [62]:
print("Total Columns :", len(buybox_output.columns))
print("Unique Columns:", len(set(buybox_output.columns)))

duplicates = buybox_output.columns[buybox_output.columns.duplicated()].tolist()

print("Duplicate Columns:")
print(duplicates)

Total Columns : 142
Unique Columns: 132
Duplicate Columns:
['product_name', 'current_owner', 'current_owner_price', 'current_owner_moq', 'min_hijacker_price', 'price_gap', 'min_hijacker_price', 'min_hijacker_price', 'price_gap', 'price_gap']


In [63]:
print("Before :", buybox_output.shape)

buybox_output = buybox_output.loc[:, ~buybox_output.columns.duplicated()]

print("After  :", buybox_output.shape)

Before : (11270, 142)
After  : (11270, 132)


In [64]:
print("Duplicates:",
      buybox_output.columns[buybox_output.columns.duplicated()].tolist())

Duplicates: []


In [65]:
# import time
# import pandas as pd
# from io import BytesIO
# from azure.storage.blob import BlobServiceClient
# from azure.core.exceptions import ServiceResponseError
# import pandas as pd
# df = marketing_output.copy()
 
# # -------- STEP 1: CLEAN COLUMN NAMES --------
# def clean_columns(df):
#     df.columns = (
#         df.columns
#         .str.strip()              # remove leading/trailing spaces
#         .str.lower()              # convert to lowercase
#         .str.replace(r'[\s\-]+', '_', regex=True)  # replace spaces with underscore
#     )
#     return df
 
# # Apply cleaning
# df = clean_columns(df)
 
# # ====================================================================
# # AZURE CONFIG
# # ====================================================================
# AZURE_CONNECTION_STRING = (
#     "DefaultEndpointsProtocol=https;AccountName=thriveclientdatabase;AccountKey=t3YQjkSXWXafdSIzp48QxNgmuW8BSh+6FzHaEy6VhGbgINYgo7Sica0BZw9sAKEPbwBcjqqhK5p6+AStuxDzSA==;EndpointSuffix=core.windows.net"
# )
 
# AZURE_CONTAINER = "thrive-client-emami"   # must be lowercase, valid format
 
# # ====================================================================
# # UPLOAD WITH RETRY
# # ====================================================================
# def upload_blob_with_retry(blob_client, data, blob_path, max_retries=3, max_concurrency=4, timeout=600):
#     """Upload blob with retry + exponential backoff."""
#     for attempt in range(max_retries):
#         try:
#             data.seek(0)
 
#             blob_client.upload_blob(
#                 data,
#                 overwrite=True,
#                 max_concurrency=max_concurrency,
#                 timeout=timeout
#             )
 
#             print(f"✅ Uploaded: {blob_path}")
#             return
 
#         except ServiceResponseError as e:
#             wait = (2 ** attempt) * 5
#             print(f"⚠️ Azure upload failed (Attempt {attempt+1}/{max_retries}) for {blob_path}")
#             print(f"   Error: {str(e)}")
#             print(f"   Retrying in {wait} sec...\n")
#             time.sleep(wait)
 
#         except Exception as e:
#             print(f"❌ Unexpected error for {blob_path}: {str(e)}")
#             raise
 
#     print(f"❌ Final Failure: Could NOT upload {blob_path} after {max_retries} attempts.")
#     raise RuntimeError("Upload Failed")
 
 
# def upload_to_azure_csv(df, blob_path, connection_string, container_name):
#     df = df.copy()
 
#     # Fix all object columns
#     object_cols = df.select_dtypes(include='object').columns
#     df[object_cols] = df[object_cols].astype(str)
 
#     # Replace NaN with None
#     df = df.where(pd.notnull(df), None)
 
 
#     buffer = BytesIO()
#     df.to_csv(buffer, index=False)
#     buffer.seek(0)
 
#     blob_service_client = BlobServiceClient.from_connection_string(
#         connection_string,
#         connection_timeout=600,
#         read_timeout=600
#     )
 
#     blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_path)
#     upload_blob_with_retry(blob_client, buffer, blob_path)
 
# upload_to_azure_csv(df,"sanity_data/datahive/emami/emami_amazon_buybox_test.csv", AZURE_CONNECTION_STRING, AZURE_CONTAINER)
 
 

## 19b - Amazon - Separate Per-File Exports

In [66]:
# ══════════════════════════════════════════════════════════════════════════
# AMAZON (VENDOR CENTRAL) — SEPARATE OUTPUT FILES
# Drop this in AFTER your existing Amazon sections have run:
#   amz_inv_final   (Section 14/15 — inventory metrics builder)
#   amz_ads_campaign (Section 6 — has helper metadata already merged in)
#   amz_rev_df       (Section 5 — raw revenue, PLACEHOLDER, see note below)
#   buybox_output    (Section 19 — already in the right shape)
#
# This section does NOT re-parse any raw files. It only selects, renames,
# and exports what your existing pipeline already builds — as 4 separate
# CSVs pushed to 4 separate Azure blob paths, instead of merged into
# inventory_output / marketing_output / revenue_output.
# ══════════════════════════════════════════════════════════════════════════

CLIENT_ID = "TH-1783660475608"
COUNTRY   = "AE"
DATE_FMT  = "%d-%b-%Y"   # e.g. 22-Jul-2026

def finalize_export(df, rename_map, final_cols, date_cols=()):
    """Select -> rename -> add country/client_id -> format dates -> reorder."""
    out = df.copy()
    out.columns = out.columns.astype(str).str.strip()
    out = out.rename(columns=rename_map)

    for col in final_cols:
        if col not in out.columns:
            out[col] = np.nan

    out["country"]   = COUNTRY
    out["client_id"] = CLIENT_ID

    for col in date_cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.strftime(DATE_FMT)

    return out[final_cols]


# ──────────────────────────────────────────────────────────────────────────
# 1) AMAZON INVENTORY
# Source: amz_inv_final (already has every one of these fields under its
# Title Case name — this is a pure rename, no new logic).
# ──────────────────────────────────────────────────────────────────────────

AMZ_INV_RENAME = {
    "ASIN": "asin",
    "Brand": "brand",
    "Product Sub Category": "product_sub_category",
    "Vendor Confirmation %": "vendor_confirmation_pct",
    "PO_received_amount": "po_received_amount",
    "PO_received_Units": "po_received_units",
    "Open POs": "open_pos",
    "Receive_Fill_Rate": "receive_fill_rate",
    "Overall Vendor Lead Time (days)": "overall_vendor_lead_time_days",
    "Aged 90+ Days Sellable Inventory": "aged_90_plus_days_sellable_inventory",
    "Aged 90+ Days Sellable Units": "aged_90_plus_days_sellable_units",
    "Sellable Inventory Amount": "sellable_inventory_amount",
    "Available Inventory": "available_inventory",
    "Unsellable On Hand Inventory Amount": "unsellable_on_hand_inventory_amount",
    "Unsellable On Hand Units": "unsellable_on_hand_units",
    "Date": "date",
    "Product Name": "product_name",
    "Product Category": "product_category",
    "Pack Type": "pack_type",
    "Pack Size": "pack_size",
    "SC Ideal Price": "sc_ideal_price",
    "VC Ideal Price": "vc_ideal_price",
    "Sales Channel": "sales_channel",
    "cumulative_instock_days": "cumulative_instock_days",
    "day_of_month": "day_of_month",
    "Instock Rate": "instock_rate",
}

AMZ_INV_COLS = [
    "asin", "brand", "product_sub_category", "vendor_confirmation_pct",
    "po_received_amount", "po_received_units", "open_pos", "receive_fill_rate",
    "overall_vendor_lead_time_days", "aged_90_plus_days_sellable_inventory",
    "aged_90_plus_days_sellable_units", "sellable_inventory_amount",
    "available_inventory", "unsellable_on_hand_inventory_amount",
    "unsellable_on_hand_units", "date", "product_name", "product_category",
    "pack_type", "pack_size", "sc_ideal_price", "vc_ideal_price",
    "sales_channel", "in_stock_flag", "cumulative_instock_days",
    "day_of_month", "instock_rate", "country", "client_id",
]
# ------------------------------------------------------------------
# Ensure metadata exists in Ads dataset
# ------------------------------------------------------------------

# Sales Channel
if "Sales Channel" not in amz_ads_campaign.columns:
    amz_ads_campaign = amz_ads_campaign.merge(
        helper_df[["ASIN", "Sales Channel"]],
        on="ASIN",
        how="left"
    )

# VC Ideal Price
if "VC Ideal Price" not in amz_ads_campaign.columns:
    amz_ads_campaign = amz_ads_campaign.merge(
        helper_df[["ASIN", "VC Ideal Price"]],
        on="ASIN",
        how="left"
    )


amz_inventory_export = finalize_export(
    amz_inv_final[amz_inv_final["Sales Channel"] == "Vendor Central"],
    AMZ_INV_RENAME, AMZ_INV_COLS, date_cols=["date"],
)


# ──────────────────────────────────────────────────────────────────────────
# 2) AMAZON ADS
# Source: amz_ads_campaign. Helper metadata (Brand, Product Name, Category,
# Pack Type, Pack Size, VC Ideal Price) is ALREADY merged in by Section 6.
#
# ⚠️  TWO THINGS I COULD NOT VERIFY WITHOUT A REAL FILE — PLEASE CONFIRM:
#   1. "orders": your current ads pipeline never reads an orders column from
#      the raw Amazon Ads export. Amazon's Advertised Product report usually
#      has a column called "14 Day Total Orders (#)" alongside the "14 Day
#      Total Sales" / "14 Day Total Units (#)" columns you already map —
#      I've added it below, but please open one raw
#      sponsored_products_advertised_product_emami_<month>_<year>.xlsx file
#      and confirm the exact column header (it may be "7 Day..." depending
#      on your report's attribution window).
#   2. "sc_ideal_price": Section 6's helper merge only pulls in VC Ideal
#      Price, not SC Ideal Price. I've added SC Ideal Price to the merge
#      below so it comes through.
# ──────────────────────────────────────────────────────────────────────────

# Add SC Ideal Price + the orders column to the raw ads rename map
# (put this inside _rename_ads_cols's dict in Section 6, or patch here):
ORDERS_COL_CANDIDATES = ["14 Day Total Orders (#)", "7 Day Total Orders (#)", "Total Orders (#)"]
for cand in ORDERS_COL_CANDIDATES:
    if cand in amz_ads_df.columns:
        amz_ads_campaign["orders"] = amz_ads_df.loc[amz_ads_campaign.index, cand] \
            if cand in amz_ads_df.columns else np.nan
        break
else:
    amz_ads_campaign["orders"] = np.nan
    print("⚠️  No 'orders' column found in amz_ads_df — check ORDERS_COL_CANDIDATES above")

# if "SC Ideal Price" not in amz_ads_campaign.columns:
#     amz_ads_campaign = amz_ads_campaign.merge(
#         helper_df[["ASIN", "VC Ideal Price"]], on="ASIN", how="left"
#     )

AMZ_ADS_RENAME = {
    "ASIN": "asin",
    "Date": "date",
    "Impressions": "impressions",
    "Clicks": "clicks",
    "ads_spend": "ads_spend",
    "ads_sales": "ads_sale",
    "ads_unit_sold": "units",
    "CVR": "conversion_rate",
    "Cost Per Click (CPC)": "cpc",
    "Total Return on Advertising Spend (ROAS)": "roas",
    "Total Advertising Cost of Sales (ACOS)": "acos",
    "Campaign Name": "campaign_name",
    "Portfolio name": "portfolio_name",
    "Campaign Type": "report_type",
    "Product Name": "product_name",
    "Product Category": "product_category",
    "Pack Type": "pack_type",
    "Pack Size": "pack_size",
    "SC Ideal Price": "sc_ideal_price",
    "VC Ideal Price": "vc_ideal_price",
}

AMZ_ADS_COLS = [
    "asin", "date", "impressions", "clicks", "ads_spend", "ads_sale",
    "orders", "units", "conversion_rate", "cpc", "roas", "acos",
    "campaign_name", "portfolio_name", "report_type", "product_name",
    "product_category", "pack_type", "pack_size", "sc_ideal_price", "vc_ideal_price","sales_channel",
    "country", "client_id",
]

# amz_ads_export = finalize_export(amz_ads_campaign, AMZ_ADS_RENAME, AMZ_ADS_COLS, date_cols=["date"])
amz_ads_export = finalize_export(
    amz_ads_campaign[amz_ads_campaign["Sales Channel"] == "Vendor Central"],
    AMZ_ADS_RENAME, AMZ_ADS_COLS, date_cols=["date"],
)



# ──────────────────────────────────────────────────────────────────────────
# 3) AMAZON ORDERS / REVENUE
# ⚠️  BLOCKED — needs your input before this can be trusted.
#
# Your required schema (brand_code, parent_asin, shipped_cogs,
# customer_returns, confirmed_units, sales_discount) matches an Amazon
# Vendor "Sales Diagnostic" / Retail Analytics export. But Section 5's
# amz_rev_df pipeline currently:
#   - only keeps "Shipped Revenue" -> total_sales and "Shipped Units" -> total_units
#   - immediately groups by [ASIN, Date] and SUMS, discarding every other
#     raw column (brand code, parent ASIN, COGS, returns, etc. are read
#     from the file then thrown away before amz_rev_df is even created)
#
# So those 6 fields aren't available anywhere in the current pipeline yet.
# I don't want to invent column names for a live client Azure feed — can
# you open one raw sales_amazon_emami_<dd>_<mon>_<year> file and tell me
# the exact header names for: Brand Code, Parent ASIN, Shipped COGS,
# Customer Returns, Confirmed Units, Sales Discount (or paste the full
# header row)? Once I have those I'll rewrite Section 5 to carry them
# through instead of discarding them at the groupby step.
#
# Placeholder below so the file still exports something today — every
# field in ORDERS_COL_CANDIDATES-style TODO stays blank until confirmed.
# ──────────────────────────────────────────────────────────────────────────

AMZ_ORDERS_RENAME = {
    "ASIN": "asin",
    "Product Name": "product_name",
    "Brand": "brand",
    "Product Category": "product_category",
    "Product Sub Category": "product_sub_category",
    "total_sales": "total_sales",
    "total_units": "total_units",
    "Date": "date",
    # NOT YET AVAILABLE — see note above:
    # "<raw header>": "brand_code",
    # "<raw header>": "parent_asin",
    # "<raw header>": "shipped_cogs",
    # "<raw header>": "customer_returns",
    # "<raw header>": "confirmed_units",
    # "<raw header>": "sales_discount",
}

AMZ_ORDERS_COLS = [
    "asin", "product_name", "brand_code", "brand", "product_category",
    "product_sub_category", "parent_asin", "total_sales", "shipped_cogs",
    "total_units", "customer_returns", "confirmed_units", "sales_discount",
    "date", "country", "client_id",
]

amz_orders_export = finalize_export(amz_rev_df, AMZ_ORDERS_RENAME, AMZ_ORDERS_COLS, date_cols=["date"])

amz_orders_export = finalize_export(
    amz_rev_df[amz_rev_df["Sales Channel"] == "Vendor Central"],
    AMZ_ORDERS_RENAME, AMZ_ORDERS_COLS, date_cols=["date"],
)





# ──────────────────────────────────────────────────────────────────────────
# 4) AMAZON BUYBOX
# Source: buybox_output. Already built in exactly this shape by Section 19 —
# just adding country/client_id and confirming column order.
# ──────────────────────────────────────────────────────────────────────────

AMZ_BUYBOX_COLS = [
    "asin", "date", "current_owner", "current_owner_price", "current_owner_moq",
] + [f"hijacker_{i}{suf}" for i in range(1, 21) for suf in ("", "_price", "_moq")] + [
    "product_name", "category", "pack_type", "pack_size",
    "sc_ideal_price", "vc_ideal_price", "brand", "sales_channel",
    "client_id", "country",
]

amz_buybox_export = finalize_export(buybox_output, {}, AMZ_BUYBOX_COLS, date_cols=["date"])


In [67]:
amz_orders_export.fillna(0, inplace=True)
amz_ads_export.fillna(0, inplace=True)
amz_inventory_export.fillna(0, inplace=True)

In [68]:
amz_ads_export.to_csv("emami_uae_amazon_ads.csv", index=False)
amz_orders_export.to_csv("emami_uae_amazon_orders.csv", index=False)
amz_inventory_export.to_csv("emami_uae_amazon_inventory.csv", index=False)
amz_buybox_export.to_csv("emami_uae_amazon_buybox.csv", index=False)


In [69]:
amz_inventory_export

,asin,brand,product_sub_category,vendor_confirmation_pct,po_received_amount,po_received_units,open_pos,receive_fill_rate,overall_vendor_lead_time_days,aged_90_plus_days_sellable_inventory,...,pack_size,sc_ideal_price,vc_ideal_price,sales_channel,in_stock_flag,cumulative_instock_days,day_of_month,instock_rate,country,client_id
0,B00F93HFME,Himani Navratna,Hair Oil,0.0,0.00,0.0,7.0,0.0,0.00,0.0,...,300,0,22.223769,Vendor Central,1,1,1,100.00,AE,TH-1783660475608
1,B00F93HFME,Himani Navratna,Hair Oil,0.0,120.26,7.0,0.0,0.0,5.14,0.0,...,300,0,22.223769,Vendor Central,1,2,2,100.00,AE,TH-1783660475608
2,B00F93HFME,Himani Navratna,Hair Oil,0.0,0.00,0.0,0.0,0.0,0.00,0.0,...,300,0,22.223769,Vendor Central,1,3,3,100.00,AE,TH-1783660475608
3,B00F93HFME,Himani Navratna,Hair Oil,0.0,0.00,0.0,0.0,0.0,0.00,0.0,...,300,0,22.223769,Vendor Central,1,4,4,100.00,AE,TH-1783660475608
4,B00F93HFME,Himani Navratna,Hair Oil,0.0,0.00,0.0,0.0,0.0,0.00,0.0,...,300,0,22.223769,Vendor Central,1,5,5,100.00,AE,TH-1783660475608
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11265,B0GH9ZDVZX,Creme 21,All-purpose cream,0.00%,0.00,0.0,96.0,0.0,0.00,AED 0.00,...,150,0,15.904722,Vendor Central,1,19,19,100.00,AE,TH-1783660475608
11266,B0GH9ZDVZX,Creme 21,All-purpose cream,0.00%,0.00,1.0,95.0,0.0,10.00,AED 0.00,...,150,0,15.904722,Vendor Central,1,20,20,100.00,AE,TH-1783660475608
11267,B0GH9ZDVZX,Creme 21,All-purpose cream,0.00%,0.00,7.0,55.0,0.0,11.14,AED 0.00,...,150,0,15.904722,Vendor Central,1,21,21,100.00,AE,TH-1783660475608
11268,B0GH9ZDVZX,Creme 21,All-purpose cream,0.00%,0.00,29.0,26.0,0.0,5.38,AED 0.00,...,150,0,15.904722,Vendor Central,1,22,22,100.00,AE,TH-1783660475608


In [70]:
amz_orders_export

,asin,product_name,brand_code,brand,product_category,product_sub_category,parent_asin,total_sales,shipped_cogs,total_units,customer_returns,confirmed_units,sales_discount,date,country,client_id
0,B00F93HFME,Himani Navratna Oil 300ml,0.0,Himani Navratna,Hair Care,Hair Oil,0.0,33.37,0.0,2,0.0,0.0,0.0,01-May-2026,AE,TH-1783660475608
1,B00F93HFME,Himani Navratna Oil 300ml,0.0,Himani Navratna,Hair Care,Hair Oil,0.0,47.43,0.0,3,0.0,0.0,0.0,02-May-2026,AE,TH-1783660475608
2,B00F93HFME,Himani Navratna Oil 300ml,0.0,Himani Navratna,Hair Care,Hair Oil,0.0,15.81,0.0,1,0.0,0.0,0.0,03-May-2026,AE,TH-1783660475608
3,B00F93HFME,Himani Navratna Oil 300ml,0.0,Himani Navratna,Hair Care,Hair Oil,0.0,108.60,0.0,6,0.0,0.0,0.0,04-May-2026,AE,TH-1783660475608
4,B00F93HFME,Himani Navratna Oil 300ml,0.0,Himani Navratna,Hair Care,Hair Oil,0.0,15.81,0.0,1,0.0,0.0,0.0,05-May-2026,AE,TH-1783660475608
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2873,B0GH9ZDVZX,Creme21 Cocoa Butter Soft Cream 150ml,0.0,Creme 21,Skin Care,All-purpose cream,0.0,11.39,0.0,1,0.0,0.0,0.0,15-Aug-2026,AE,TH-1783660475608
2874,B0GH9ZDVZX,Creme21 Cocoa Butter Soft Cream 150ml,0.0,Creme 21,Skin Care,All-purpose cream,0.0,22.78,0.0,2,0.0,0.0,0.0,16-Aug-2026,AE,TH-1783660475608
2875,B0GH9ZDVZX,Creme21 Cocoa Butter Soft Cream 150ml,0.0,Creme 21,Skin Care,All-purpose cream,0.0,11.96,0.0,1,0.0,0.0,0.0,20-Aug-2026,AE,TH-1783660475608
2876,B0GH9ZDVZX,Creme21 Cocoa Butter Soft Cream 150ml,0.0,Creme 21,Skin Care,All-purpose cream,0.0,34.17,0.0,3,0.0,0.0,0.0,21-Aug-2026,AE,TH-1783660475608


In [71]:
amz_ads_export

,asin,date,impressions,clicks,ads_spend,ads_sale,orders,units,conversion_rate,cpc,...,report_type,product_name,product_category,pack_type,pack_size,sc_ideal_price,vc_ideal_price,sales_channel,country,client_id
0,B0D7QBTN5N,01-May-2026,2,1,3.28,0.0,0,0,0.0,3.280,...,SPAP,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,1,300ml,0.0,16.065375,0.0,AE,TH-1783660475608
1,B0D7QBTN5N,02-May-2026,2,0,0.00,0.0,0,0,0.0,0.000,...,SPAP,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,1,300ml,0.0,16.065375,0.0,AE,TH-1783660475608
2,B0D7QBTN5N,03-May-2026,14,2,1.73,0.0,0,0,0.0,0.865,...,SPAP,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,1,300ml,0.0,16.065375,0.0,AE,TH-1783660475608
3,B0D7QBTN5N,04-May-2026,11,0,0.00,0.0,0,0,0.0,0.000,...,SPAP,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,1,300ml,0.0,16.065375,0.0,AE,TH-1783660475608
4,B0D7QBTN5N,05-May-2026,2,0,0.00,0.0,0,0,0.0,0.000,...,SPAP,Creme 21 Petroleum Jelly Coco Butter 300ml,Skin Care,1,300ml,0.0,16.065375,0.0,AE,TH-1783660475608
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18381,B0G644TDP5,19-Aug-2026,5,0,0.00,0.0,0,0,0.0,0.000,...,SD,Creme21 Papaya Brightening Body Lotion 400ml,Skin Care,1,400ml,0.0,27.793099,0.0,AE,TH-1783660475608
18382,B0G644TDP5,20-Aug-2026,7,0,0.00,0.0,0,0,0.0,0.000,...,SD,Creme21 Papaya Brightening Body Lotion 400ml,Skin Care,1,400ml,0.0,27.793099,0.0,AE,TH-1783660475608
18383,B0G644TDP5,21-Aug-2026,1,0,0.00,0.0,0,0,0.0,0.000,...,SD,Creme21 Papaya Brightening Body Lotion 400ml,Skin Care,1,400ml,0.0,27.793099,0.0,AE,TH-1783660475608
18384,B0G644TDP5,22-Aug-2026,5,0,0.00,0.0,0,0,0.0,0.000,...,SD,Creme21 Papaya Brightening Body Lotion 400ml,Skin Care,1,400ml,0.0,27.793099,0.0,AE,TH-1783660475608


## 19c - Noon Core + Noon Minutes - Separate Per-File Exports

In [72]:
# ══════════════════════════════════════════════════════════════════════════
# NOON CORE + NOON MINUTES — SEPARATE PER-FILE EXPORTS (ADS / INVENTORY / REVENUE)
# Reuses CLIENT_ID / COUNTRY / DATE_FMT / finalize_export defined in the
# Amazon export section above. Needs (already built earlier in the notebook):
#   noon_ads_campaign, nc_inv_final, nc_rev_final   (Noon Core)
#   nms_ads_campaign,  nms_inv_final, nms_rev_final  (Noon Minutes)
# Pricing files for both are handled separately, right after the scrape
# cells that build them (see the NOON PRICING EXPORTS section near the end).
# ══════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────────
# Shared rename maps (same field meanings across both channels)
# ──────────────────────────────────────────────────────────────────────────

NOON_ADS_RENAME = {
    "ASIN": "asin",
    "Date": "date",
    "Sales Channel": "sales_channel",
    "Campaign Name": "campaign_name",
    "Campaign Type": "campaign_type",
    
    "ads_sales": "ads_sales",
    "ads_unit_sold": "ads_unit_sold",
    "ads_spend": "ads_spend",
    "Impressions": "impressions",
    "Clicks": "clicks",
    "Click-Thru Rate (CTR)": "ctr",
    "Cost Per Click (CPC)": "cpc",
    "Total Advertising Cost of Sales (ACOS)": "acos",
    "Total Return on Advertising Spend (ROAS)": "roas",
    "CVR": "cvr",
    "Brand": "brand",
    "Product Name": "product_name",
    "Product Category": "product_category",
    "Product Sub Category": "product_sub_category",
    "Pack Type": "pack_type",
    "Pack Size": "pack_size",
    "VC Ideal Price": "vc_ideal_price",
}

# Noon Core's ads sheet has no Portfolio name / CVR columns in your current
# pipeline (only Noon Minutes does) — finalize_export fills missing columns
# with blank rather than erroring, so the two channels can share one map.
NOON_CORE_ADS_COLS = [
    "asin", "date", "sales_channel", "campaign_name", "campaign_type",
    "ads_sales", "ads_unit_sold", "ads_spend", "impressions", "clicks",
    "ctr", "cpc", "acos", "roas", "brand", "product_name", "product_category",
    "product_sub_category", "pack_type", "pack_size", "vc_ideal_price",
    "country", "client_id",
]

NMS_ADS_COLS = [
    "asin", "date", "sales_channel", "campaign_name", "campaign_type",
    "ads_sales", "ads_unit_sold", "ads_spend",
    "impressions", "clicks", "ctr", "cpc", "acos", "roas", "cvr",
    "brand", "product_category", "product_sub_category", "pack_type",
    "pack_size", "vc_ideal_price", "country", "client_id",
]

noon_ads_export = finalize_export(noon_ads_campaign, NOON_ADS_RENAME, NOON_CORE_ADS_COLS, date_cols=["date"])
nms_ads_export  = finalize_export(nms_ads_campaign,  NOON_ADS_RENAME, NMS_ADS_COLS,       date_cols=["date"])


# ──────────────────────────────────────────────────────────────────────────
# Inventory — nc_inv_final / nms_inv_final only carry INVENTORY_COLS
# (Section 14's shared schema), which does NOT include Barcode. Barcode is
# a static per-ASIN attribute that lives on the raw nc_inv_df / nms_inv_df
# instead, so it's merged in here by ASIN (first non-null wins).
# ──────────────────────────────────────────────────────────────────────────

NOON_INV_RENAME = {
    "ASIN": "asin",
    "Date": "date",
    "Available Inventory": "available_inventory",
    
    "Last 30 Days Sales": "last_30_day_sale_qty",
    "DOS": "dos",
    "Brand": "brand",
    "Product Category": "product_category",
    "Product Sub Category": "product_sub_category",
    "Product Name": "product_name",
    "Pack Type": "pack_type",
    "Pack Size": "pack_size",
    "VC Ideal Price": "vc_ideal_price",
    
    "Sales Channel": "sales_channel",
}

NOON_INV_COLS = [
    "asin", "date", "available_inventory", "last_30_day_sale_qty",
    "dos", "brand", "product_category", "product_sub_category",
    "product_name", "pack_type", "pack_size", "vc_ideal_price",
    "sales_channel", "country",'client_id'
]

def _with_barcode(inv_final, raw_inv_df):
    """Merge the static Barcode field from the raw inventory df (deduped by ASIN)."""
    if "Barcode" not in raw_inv_df.columns:
        out = inv_final.copy()
        out["Barcode"] = np.nan
        return out
    barcode_lookup = (
        raw_inv_df[["ASIN", "Barcode"]]
        .dropna(subset=["ASIN"])
        .drop_duplicates(subset=["ASIN"], keep="first")
    )
    return inv_final.merge(barcode_lookup, on="ASIN", how="left")

nc_inventory_export  = finalize_export(_with_barcode(nc_inv_final,  nc_inv_df),  NOON_INV_RENAME, NOON_INV_COLS, date_cols=["date"])
nms_inventory_export = finalize_export(_with_barcode(nms_inv_final, nms_inv_df), NOON_INV_RENAME, NOON_INV_COLS, date_cols=["date"])

# ──────────────────────────────────────────────────────────────────────────
# Revenue Export (Noon Core + Noon Minutes)
# Only keep the required columns
# ──────────────────────────────────────────────────────────────────────────

NOON_REV_RENAME = {
    "Date": "date",
    "ASIN": "asin",
    "total_units": "total_units",
    "total_sales": "total_sales",
    "Product Name": "product_name",
    "Product Category": "product_category",
    "Pack Size": "pack_size",
    "Brand": "brand",
    "Product Sub Category": "product_sub_category",
    "Pack Type": "pack_type",
    "Sales Channel": "sales_channel",
    "VC Ideal Price": "vc_ideal_price",
}

NOON_REV_COLS = [
    "date",
    "asin",
    "total_units",
    "total_sales",
    "product_name",
    "product_category",
    "pack_size",
    "country",
    "client_id",
    "brand",
    "product_sub_category",
    "pack_type",
    "sales_channel",
    "vc_ideal_price",
]

nc_revenue_export = finalize_export(
    _with_barcode(nc_rev_final, nc_inv_df),
    NOON_REV_RENAME,
    NOON_REV_COLS,
    date_cols=["date"],
)

nms_revenue_export = finalize_export(
    _with_barcode(nms_rev_final, nms_inv_df),
    NOON_REV_RENAME,
    NOON_REV_COLS,
    date_cols=["date"],
)

In [73]:
noon_ads_export.fillna(0, inplace=True)
nc_revenue_export.fillna(0, inplace=True)
nms_ads_export.fillna(0, inplace=True)
nc_inventory_export.fillna(0, inplace=True) 
nms_inventory_export.fillna(0, inplace=True)    
nms_revenue_export.fillna(0, inplace=True)
nms_ads_export.fillna(0, inplace=True)

In [74]:
noon_ads_export.to_csv("emami_uae_noon_ads.csv", index=False)
nc_inventory_export.to_csv("emami_uae_noon_inventory.csv", index=False)         
nms_inventory_export.to_csv("emami_uae_noon_minutes_inventory.csv", index=False)
nc_revenue_export.to_csv("emami_uae_noon_revenue.csv", index=False)
nms_revenue_export.to_csv("emami_uae_noon_minutes_revenue.csv", index=False)
nms_ads_export.to_csv("emami_uae_noon_minutes_ads.csv", index=False)


## HELPER FILE (AMAZON,NOON,NOON_MINUTES)

In [75]:
# # apply_helper function and its application to all outputs
# # is defined and called in Cell 35 above (HELPER FILE PROCESSING section).
# # This cell applies helper to revenue, marketing and buybox outputs too.

# # ── Apply helper to revenue_output ────────────────────────────────────────
# revenue_output = apply_helper(revenue_output)
# print("✅ Helper applied to revenue_output | Shape:", revenue_output.shape)

# # ── Apply helper to marketing_output ──────────────────────────────────────
# marketing_output = apply_helper(marketing_output)
# print("✅ Helper applied to marketing_output | Shape:", marketing_output.shape)

# # ── Apply helper to buybox_output ─────────────────────────────────────────
# if "Sales Channel" not in buybox_output.columns:
#     buybox_output["Sales Channel"] = "Vendor Central"
# buybox_output = apply_helper(buybox_output)
# print("✅ Helper applied to buybox_output | Shape:", buybox_output.shape)

# # ── Sanity check all outputs ──────────────────────────────────────────────
# for name, df in [
#     ("inventory_output", inventory_output),
#     ("revenue_output",   revenue_output),
#     ("marketing_output", marketing_output),
#     ("buybox_output",    buybox_output),
# ]:
#     brand_blank = (
#         df["Brand"].isna().sum() + 
#         (df["Brand"].astype(str).str.strip().isin(["","Unknown","nan"])).sum()
#         if "Brand" in df.columns else len(df)
#     )
#     cat_blank = (
#         df["Product Category"].isna().sum() +
#         (df["Product Category"].astype(str).str.strip().isin(["","Unknown","nan"])).sum()
#         if "Product Category" in df.columns else len(df)
#     )
#     channel_counts = df["Sales Channel"].value_counts().to_dict() if "Sales Channel" in df.columns else {}
#     print(f"\n{name}")
#     print(f"  Shape           : {df.shape}")
#     print(f"  Brand blank     : {brand_blank}")
#     print(f"  Cat blank       : {cat_blank}")
#     print(f"  Channels        : {channel_counts}")
#     if "Date" in df.columns:
#         dates = pd.to_datetime(df["Date"], errors="coerce").dropna()
#         print(f"  Date range      : {dates.min().date()} → {dates.max().date()}")


In [76]:
inventory_output.columns

Index(['sku', 'fnsku', 'asin', 'condition', 'your_price', 'mfn_listing_exists',
       'mfn_fulfillable_quantity', 'afn_listing_exists',
       'afn_warehouse_quantity', 'afn_fulfillable_quantity',
       'afn_unsellable_quantity', 'afn_reserved_quantity',
       'afn_total_quantity', 'per_unit_volume', 'afn_inbound_working_quantity',
       'afn_inbound_shipped_quantity', 'afn_inbound_receiving_quantity',
       'afn_researching_quantity', 'afn_reserved_future_supply',
       'afn_future_supply_buyable', 'store', 'date', 'reserved_qty',
       'reserved_customerorders', 'reserved_fc_transfers',
       'reserved_fc_processing', 'available_inventory', 'units_shipped_t30',
       'units_shipped_t60', 'units_shipped_t90', 'open_pos', 'sell_through',
       'dos', 'min_available_qty', 'max_available_qty', 'stock_status',
       'instock_rate', 'no_low_stock_wt_open_pos',
       'no_low_stock_wt_no_open_pos', 'total_sales', 'total_units',
       'product_name', 'product_category', 'pack_typ

In [77]:
# ============================================================================
# COLUMN MAPPING
# ============================================================================

to_old = {
    "asin": "ASIN",
    "brand": "Brand",
    "product_category": "Product Category",
    "product_sub_category": "Product Sub Category",
    "product_name": "Product Name",
    "pack_type": "Pack Type",
    "pack_size": "Pack Size",
    "sales_channel": "Sales Channel",
    "date": "Date",
}

to_new = {v: k for k, v in to_old.items()}


def run_helper(df):

    temp = df.copy()

    # Rename only if column exists
    temp.rename(
        columns={k: v for k, v in to_old.items() if k in temp.columns},
        inplace=True,
    )

    if "Sales Channel" not in temp.columns:
        temp["Sales Channel"] = "Vendor Central"

    temp = apply_helper(temp)

    temp.rename(
        columns={k: v for k, v in to_new.items() if k in temp.columns},
        inplace=True,
    )

    return temp


# ============================================================================
# APPLY HELPER
# ============================================================================

inventory_output = run_helper(inventory_output)
print("✅ Helper applied to inventory_output | Shape:", inventory_output.shape)

revenue_output = run_helper(revenue_output)
print("✅ Helper applied to revenue_output | Shape:", revenue_output.shape)

marketing_output = run_helper(marketing_output)
print("✅ Helper applied to marketing_output | Shape:", marketing_output.shape)

buybox_output = run_helper(buybox_output)
print("✅ Helper applied to buybox_output | Shape:", buybox_output.shape)


# ============================================================================
# SANITY CHECK
# ============================================================================

for name, df in [
    ("inventory_output", inventory_output),
    ("revenue_output", revenue_output),
    ("marketing_output", marketing_output),
    ("buybox_output", buybox_output),
]:

    brand_blank = (
        df["brand"].fillna("").astype(str).str.strip()
        .isin(["", "Unknown", "nan"]).sum()
        if "brand" in df.columns else len(df)
    )

    cat_blank = (
        df["product_category"].fillna("").astype(str).str.strip()
        .isin(["", "Unknown", "nan"]).sum()
        if "product_category" in df.columns else len(df)
    )

    channel_counts = (
        df["sales_channel"].value_counts(dropna=False).to_dict()
        if "sales_channel" in df.columns else {}
    )

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("Shape          :", df.shape)
    print("Brand Blank    :", brand_blank)
    print("Category Blank :", cat_blank)
    print("Channels       :", channel_counts)

    if "date" in df.columns:
        dates = pd.to_datetime(df["date"], errors="coerce").dropna()

        if not dates.empty:
            print(f"Date Range     : {dates.min().date()} → {dates.max().date()}")
        else:
            print("Date Range     : No valid dates")

✅ Helper applied to inventory_output | Shape: (24824, 66)
✅ Helper applied to revenue_output | Shape: (24824, 70)
✅ Helper applied to marketing_output | Shape: (24824, 80)
✅ Helper applied to buybox_output | Shape: (11270, 134)

inventory_output
Shape          : (24824, 66)
Brand Blank    : 0
Category Blank : 0
Channels       : {'Noon': 11484, 'Vendor Central': 11270, 'Noon Minutes': 2070}
Date Range     : 2026-05-01 → 2026-08-24

revenue_output
Shape          : (24824, 70)
Brand Blank    : 0
Category Blank : 0
Channels       : {'Noon': 11484, 'Vendor Central': 11270, 'Noon Minutes': 2070}
Date Range     : 2026-05-01 → 2026-08-24

marketing_output
Shape          : (24824, 80)
Brand Blank    : 0
Category Blank : 0
Channels       : {'Noon': 11484, 'Vendor Central': 11270, 'Noon Minutes': 2070}
Date Range     : 2026-05-01 → 2026-08-24

buybox_output
Shape          : (11270, 134)
Brand Blank    : 0
Category Blank : 0
Channels       : {'Vendor Central': 11270}
Date Range     : 2026-05-01 → 

## 20 · Save All Outputs

In [78]:
def save_output(df, path, label):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    out = df.copy()

    out.columns = (
        out.columns.astype(str).str.strip().str.lower()
        .str.replace(" ", "_", regex=False).str.replace("-", "_", regex=False)
        .str.replace("(", "", regex=False).str.replace(")", "", regex=False)
        .str.replace("%", "pct", regex=False).str.replace("/", "_", regex=False)
        .str.replace(".", "", regex=False)
    )
    out = out.loc[:, ~out.columns.duplicated()]

    if "date" in out.columns:
        # Keep as proper datetime in Excel (enables Date Filters)
        # Convert to string only for CSV
        if path.lower().endswith(".csv"):
            out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.strftime("%d-%b-%y")
        else:
            out["date"] = pd.to_datetime(out["date"], errors="coerce")

    if "oos_date" in out.columns:
        out["oos_date"] = np.where(
            out["oos_date"].astype(str).str.lower().isin(["instock", "nan", "nat"]),
            "instock",
            pd.to_datetime(out["oos_date"], errors="coerce").dt.strftime("%d-%b-%y").str.lower()
        )

    numeric_cols = out.select_dtypes(include=["float", "int"]).columns
    out[numeric_cols] = out[numeric_cols].fillna(0)
    out = out.fillna("")

    if path.lower().endswith(".csv"):
        out.to_csv(path, index=False, encoding="utf-8-sig")
    else:
        out.to_excel(path, index=False)

    print(f"\n✅ {label} Saved | {path} | Shape: {out.shape}")


save_output(inventory_output, OUT_INV,    "Inventory")
save_output(revenue_output,   OUT_REV,    "Revenue")
save_output(marketing_output, OUT_MARK,   "Marketing")
save_output(buybox_output,    OUT_BUYBOX, "BuyBox")


✅ Inventory Saved | C:\Users\Shambhavi Dixit\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\client_mamangement_team\EMAMI\inventory_output_2026.xlsx | Shape: (24824, 64)

✅ Revenue Saved | C:\Users\Shambhavi Dixit\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\client_mamangement_team\EMAMI\revenue_output_2026.xlsx | Shape: (24824, 68)

✅ Marketing Saved | C:\Users\Shambhavi Dixit\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\client_mamangement_team\EMAMI\marketing_output_2026.xlsx | Shape: (24824, 78)

✅ BuyBox Saved | C:\Users\Shambhavi Dixit\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\client_mamangement_team\EMAMI\buybox_output_2026.xlsx | Shape: (11270, 132)


In [79]:
marketing_output.columns

Index(['sku', 'fnsku', 'asin', 'condition', 'your_price', 'mfn_listing_exists',
       'mfn_fulfillable_quantity', 'afn_listing_exists',
       'afn_warehouse_quantity', 'afn_fulfillable_quantity',
       'afn_unsellable_quantity', 'afn_reserved_quantity',
       'afn_total_quantity', 'per_unit_volume', 'afn_inbound_working_quantity',
       'afn_inbound_shipped_quantity', 'afn_inbound_receiving_quantity',
       'afn_researching_quantity', 'afn_reserved_future_supply',
       'afn_future_supply_buyable', 'store', 'date', 'reserved_qty',
       'reserved_customerorders', 'reserved_fc_transfers',
       'reserved_fc_processing', 'available_inventory', 'units_shipped_t30',
       'units_shipped_t60', 'units_shipped_t90', 'open_pos', 'sell_through',
       'dos', 'min_available_qty', 'max_available_qty', 'stock_status',
       'instock_rate', 'no_low_stock_wt_open_pos',
       'no_low_stock_wt_no_open_pos', 'total_sales', 'total_units',
       'sales_channel', 'ads_sales', 'ads_unit_sold'

In [80]:
import time
from io import BytesIO
import pandas as pd

from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import ServiceResponseError

# ====================================================================
# CLEAN COLUMN NAMES
# ====================================================================

def clean_columns(df):
    df = df.copy()

    # Clean column names
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-]+", "_", regex=True)
        .str.replace(r"[^\w]", "", regex=True)
    )

    return df


# ====================================================================
# MAKE DUPLICATE COLUMN NAMES UNIQUE
# ====================================================================

def make_unique_columns(df):

    cols = pd.Series(df.columns)

    if cols.duplicated().any():

        print("\n⚠ Duplicate columns detected:")

        duplicates = cols[cols.duplicated()].unique()

        for d in duplicates:
            print(f"   - {d}")

        new_cols = []
        counts = {}

        for col in cols:

            if col in counts:
                counts[col] += 1
                new_cols.append(f"{col}_{counts[col]}")
            else:
                counts[col] = 0
                new_cols.append(col)

        df.columns = new_cols

        print("\n✅ Duplicate columns renamed.")

    return df


# ====================================================================
# AZURE CONFIG
# ====================================================================

AZURE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=thriveclientdatabase;AccountKey=t3YQjkSXWXafdSIzp48QxNgmuW8BSh+6FzHaEy6VhGbgINYgo7Sica0BZw9sAKEPbwBcjqqhK5p6+AStuxDzSA==;EndpointSuffix=core.windows.net"

AZURE_CONTAINER = "thrive-client-emami"


# ====================================================================
# UPLOAD WITH RETRY
# ====================================================================

def upload_blob_with_retry(
    blob_client,
    data,
    blob_path,
    max_retries=3,
    max_concurrency=4,
    timeout=600,
):

    for attempt in range(max_retries):

        try:

            data.seek(0)

            blob_client.upload_blob(
                data,
                overwrite=True,
                max_concurrency=max_concurrency,
                timeout=timeout,
            )

            print(f"✅ Uploaded : {blob_path}")
            return

        except ServiceResponseError as e:

            wait = (2 ** attempt) * 5

            print(
                f"⚠ Upload failed ({attempt+1}/{max_retries}) : {blob_path}"
            )
            print(e)
            print(f"Retrying in {wait} seconds...\n")

            time.sleep(wait)

        except Exception as e:

            print(f"❌ Unexpected Error : {blob_path}")
            print(e)
            raise

    raise RuntimeError(f"Upload failed for {blob_path}")


# ====================================================================
# UPLOAD DATAFRAME
# ====================================================================

def upload_to_azure_csv(
    df,
    blob_path,
    connection_string,
    container_name,
):

    print("\nCleaning dataframe...")

    # Clean column names
    df = clean_columns(df)
    df["country"] = "AE"


    # Make duplicate columns unique
    df = make_unique_columns(df)

    print(f"Rows    : {len(df):,}")
    print(f"Columns : {len(df.columns)}")
    print(f"Unique Columns : {len(set(df.columns))}")

    # Convert object columns safely
    object_cols = df.select_dtypes(include=["object"]).columns

    for col in object_cols:
        df[col] = df[col].astype(str)

    # Replace NaN with None
    df = df.where(pd.notnull(df), None)

    # Create CSV
    buffer = BytesIO()

    df.to_csv(
        buffer,
        index=False,
        encoding="utf-8-sig"
    )

    buffer.seek(0)

    # Azure Client
    blob_service_client = BlobServiceClient.from_connection_string(
        connection_string,
        connection_timeout=600,
        read_timeout=600,
    )

    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=blob_path,
    )

    upload_blob_with_retry(
        blob_client,
        buffer,
        blob_path,
    )


# PUSH AZURE

In [81]:
marketing_output.columns

Index(['sku', 'fnsku', 'asin', 'condition', 'your_price', 'mfn_listing_exists',
       'mfn_fulfillable_quantity', 'afn_listing_exists',
       'afn_warehouse_quantity', 'afn_fulfillable_quantity',
       'afn_unsellable_quantity', 'afn_reserved_quantity',
       'afn_total_quantity', 'per_unit_volume', 'afn_inbound_working_quantity',
       'afn_inbound_shipped_quantity', 'afn_inbound_receiving_quantity',
       'afn_researching_quantity', 'afn_reserved_future_supply',
       'afn_future_supply_buyable', 'store', 'date', 'reserved_qty',
       'reserved_customerorders', 'reserved_fc_transfers',
       'reserved_fc_processing', 'available_inventory', 'units_shipped_t30',
       'units_shipped_t60', 'units_shipped_t90', 'open_pos', 'sell_through',
       'dos', 'min_available_qty', 'max_available_qty', 'stock_status',
       'instock_rate', 'no_low_stock_wt_open_pos',
       'no_low_stock_wt_no_open_pos', 'total_sales', 'total_units',
       'sales_channel', 'ads_sales', 'ads_unit_sold'

# NOON_CORE 

In [82]:
import pandas as pd
from datetime import datetime

# ============================================================
# LOAD FILE
# ============================================================

input_file = r"C:\Users\Shambhavi Dixit\Downloads\combined_result (12).csv"

df = pd.read_csv(
    input_file,
    low_memory=False
)

# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
)

# ============================================================
# PRICING
# 
# First priority  -> offer_sale_price
# If blank        -> offer_price
# ============================================================

df["Pricing"] = (
    pd.to_numeric(
        df["offer_sale_price"],
        errors="coerce"
    )
    .fillna(
        pd.to_numeric(
            df["offer_price"],
            errors="coerce"
        )
    )
)

# ============================================================
# FINAL COLUMN MAPPING
# ============================================================

final_df = df.rename(
    columns={
        "sku": "NINs",
        "product_title": "Product Name",
        "seller_rating": "Rating",
        "seller_rating_count": "Reviews",
    }
)

# ============================================================
# ADD DATE
# ============================================================

final_df["Date"] = datetime.today().strftime("%d-%b-%Y")

# ============================================================
# KEEP ONLY REQUIRED COLUMNS
# ============================================================

final_df = final_df[
    [
        "NINs",
        "Product Name",
        "Pricing",
        "Rating",
        "Reviews",
        "Date",
    ]
].copy()

# ============================================================
# CLEAN VALUES
# ============================================================

final_df["NINs"] = final_df["NINs"].astype(str).str.strip()

final_df["Product Name"] = (
    final_df["Product Name"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# ============================================================
# REMOVE DUPLICATE NINs IF REQUIRED
# ============================================================

final_df = final_df.drop_duplicates(
    subset=["NINs"],
    keep="last"
).reset_index(drop=True)

# ============================================================
# CHECK
# ============================================================

print("Final shape:", final_df.shape)

print("\nFinal columns:")
print(final_df.columns.tolist())

print("\nFinal data:")
print(final_df.head(20))

Final shape: (75, 6)

Final columns:
['NINs', 'Product Name', 'Pricing', 'Rating', 'Reviews', 'Date']

Final data:
                      NINs                                       Product Name  \
0   Z34B3C5734B848A9B5264Z  Creme 21 Healing Musk Body Lotion - 400ml | In...   
1   Z1A6ADD1F53ABC7A67822Z  Creme 21 Cocoa Butter Body Cream - 150ml | Dee...   
2   Z490E0F7063C325D62654Z  Emami 7 Oils in One Rosemary Hair Oil – 300ml ...   
3   Z9AF27D9B0C0CD2A5A926Z  Emami Smart And Handsome Men Radiant Glow Crea...   
4   Z09FBE4CB0B7B62B70D88Z  Emami 7 Oils in One Shea Butter Hair Oil – 200...   
5   ZFB22A2EAB9698654056BZ  Creme 21 Aqua Soft Lotion – 400ml | Face, Body...   
6   ZBE7FDC403E1EF26CF8CAZ  Creme 21 Antibacterial Moisturising Cream Soap...   
7   Z9EE60C4E995A165ABFABZ  Creme 21 Soft Care Moisturising Cream Soap – N...   
8   Z57EFB0A2FF599992D797Z  Emami 7 Oils in One Red Onion Hair Oil – 300ml...   
9   ZD7FF539B0AFC37C48BB2Z  Emami 7 Oils in One Cactus Hair Oil – 200ml | .

In [83]:
final_df.to_csv(r"C:\Users\Shambhavi Dixit\Downloads\final_pricing_output.csv", index=False, encoding="utf-8-sig")

In [84]:
import time
from io import BytesIO
import pandas as pd

from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import ServiceResponseError

# ====================================================================
# CLEAN COLUMN NAMES
# ====================================================================

def clean_columns(df):
    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'[\s\-]+', '_', regex=True)
    )

    return df


# ====================================================================
# AZURE CONFIG
# ====================================================================

AZURE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=thriveclientdatabase;AccountKey=t3YQjkSXWXafdSIzp48QxNgmuW8BSh+6FzHaEy6VhGbgINYgo7Sica0BZw9sAKEPbwBcjqqhK5p6+AStuxDzSA==;EndpointSuffix=core.windows.net"

AZURE_CONTAINER = "thrive-client-emami"


# ====================================================================
# CLEAN COLUMN NAMES
# ====================================================================

def clean_columns(df):
    df = df.copy()

    # Only remove leading/trailing spaces.
    # Keep original column names such as:
    # NINs, Product Name, Pricing, Rating, Reviews, Date
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )

    return df


# ====================================================================
# UPLOAD WITH RETRY
# ====================================================================

def upload_blob_with_retry(
    blob_client,
    data,
    blob_path,
    max_retries=3,
    max_concurrency=4,
    timeout=600,
):

    for attempt in range(max_retries):

        try:

            data.seek(0)

            blob_client.upload_blob(
                data,
                overwrite=True,
                max_concurrency=max_concurrency,
                timeout=timeout,
            )

            print(f"✅ Uploaded: {blob_path}")
            return

        except ServiceResponseError as e:

            wait = (2 ** attempt) * 5

            print(
                f"⚠️ Upload failed "
                f"({attempt + 1}/{max_retries}) : {blob_path}"
            )

            print(e)
            print(f"Retrying in {wait} seconds...\n")

            time.sleep(wait)

        except Exception as e:

            print(f"❌ Unexpected Error : {blob_path}")
            print(e)
            raise

    raise RuntimeError(
        f"Upload failed for {blob_path}"
    )


# ====================================================================
# UPLOAD DATAFRAME AS CSV
# ====================================================================

def upload_to_azure_csv(
    df,
    blob_path,
    connection_string,
    container_name,
):

    # ---------------------------------------------------------------
    # COPY DATAFRAME
    # ---------------------------------------------------------------

    df = df.copy()

    # ---------------------------------------------------------------
    # CLEAN COLUMN NAMES
    # ---------------------------------------------------------------

    df = clean_columns(df)

    # ---------------------------------------------------------------
    # ADD COUNTRY
    # ---------------------------------------------------------------

    df["country"] = "AE"

    # ---------------------------------------------------------------
    # CLEAN STRING COLUMNS
    # ---------------------------------------------------------------

    object_cols = df.select_dtypes(
        include=["object"]
    ).columns

    for col in object_cols:
        df[col] = df[col].fillna("").astype(str).str.strip()

    # ---------------------------------------------------------------
    # CLEAN NUMERIC COLUMNS
    # ---------------------------------------------------------------

    numeric_cols = [
        "Pricing",
        "Rating",
        "Reviews"
    ]

    for col in numeric_cols:

        if col in df.columns:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    # ---------------------------------------------------------------
    # HANDLE DATE
    # ---------------------------------------------------------------

    if "Date" in df.columns:

        df["Date"] = pd.to_datetime(
            df["Date"],
            errors="coerce"
        ).dt.strftime("%d-%b-%Y")

    # ---------------------------------------------------------------
    # HANDLE NULL VALUES
    # ---------------------------------------------------------------

    df = df.where(
        pd.notnull(df),
        None
    )

    # ---------------------------------------------------------------
    # FINAL COLUMN ORDER
    # ---------------------------------------------------------------

    required_columns = [
        "NINs",
        "Product Name",
        "Pricing",
        "Rating",
        "Reviews",
        "Date",
        "country"
    ]

    # Keep only columns that exist
    final_columns = [
        col
        for col in required_columns
        if col in df.columns
    ]

    df = df[final_columns]

    # ---------------------------------------------------------------
    # CREATE CSV BUFFER
    # ---------------------------------------------------------------

    buffer = BytesIO()

    df.to_csv(
        buffer,
        index=False,
        encoding="utf-8"
    )

    buffer.seek(0)

    # ---------------------------------------------------------------
    # CREATE AZURE CLIENT
    # ---------------------------------------------------------------

    blob_service_client = (
        BlobServiceClient.from_connection_string(
            connection_string,
            connection_timeout=600,
            read_timeout=600,
        )
    )

    blob_client = (
        blob_service_client.get_blob_client(
            container=container_name,
            blob=blob_path,
        )
    )

    # ---------------------------------------------------------------
    # UPLOAD
    # ---------------------------------------------------------------

    upload_blob_with_retry(
        blob_client=blob_client,
        data=buffer,
        blob_path=blob_path,
    )


# ====================================================================
# DATAFRAME TO UPLOAD
# ====================================================================

# Your first code creates `final_df`.
# Use that directly here.

output = final_df.copy()


# ====================================================================
# FILES TO UPLOAD
# ====================================================================

FILES_TO_UPLOAD = {

    "sanity_data/product_pricing/noon_core_pricing.csv": output

}


# ====================================================================
# UPLOAD ALL FILES
# ====================================================================

print("=" * 80)
print("STARTING AZURE UPLOAD")
print("=" * 80)


for blob_path, df in FILES_TO_UPLOAD.items():

    print("\n" + "-" * 80)
    print(f"Uploading : {blob_path}")

    # ---------------------------------------------------------------
    # CHECK NONE
    # ---------------------------------------------------------------

    if df is None:

        print("⚠️ DataFrame is None. Skipping.")
        continue

    # ---------------------------------------------------------------
    # CHECK EMPTY
    # ---------------------------------------------------------------

    if df.empty:

        print("⚠️ DataFrame is Empty. Skipping.")
        continue

    # ---------------------------------------------------------------
    # SHOW DATA
    # ---------------------------------------------------------------

    print(f"Rows    : {len(df):,}")
    print(f"Columns : {len(df.columns)}")

    print("\nColumns:")
    print(df.columns.tolist())

    # ---------------------------------------------------------------
    # UPLOAD
    # ---------------------------------------------------------------

    try:

        upload_to_azure_csv(
            df=df,
            blob_path=blob_path,
            connection_string=AZURE_CONNECTION_STRING,
            container_name=AZURE_CONTAINER,
        )

    except Exception as e:

        print(f"❌ Failed : {blob_path}")
        print(e)


print("\n" + "=" * 80)
print("🎉 ALL UPLOADS COMPLETED")
print("=" * 80)

STARTING AZURE UPLOAD

--------------------------------------------------------------------------------
Uploading : sanity_data/product_pricing/noon_core_pricing.csv
Rows    : 75
Columns : 6

Columns:
['NINs', 'Product Name', 'Pricing', 'Rating', 'Reviews', 'Date']
✅ Uploaded: sanity_data/product_pricing/noon_core_pricing.csv

🎉 ALL UPLOADS COMPLETED


# NOON_MINUTES

In [85]:
import pandas as pd
import re
import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# ============================================================================
# INPUT
# ============================================================================

INPUT_FILE = r"C:\Users\Shambhavi Dixit\Downloads\Emami - Pricing (2).xlsx"

minutes_df = pd.read_excel(
    INPUT_FILE,
    sheet_name="Noon Minutes"
)

# ============================================================================
# DRIVER
# ============================================================================

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# ============================================================================
# SCRAPE
# ============================================================================

results = []

for nin in minutes_df["NINs"].dropna():

    nin = str(nin).strip()

    url = f"https://minutes.noon.com/uae-en/now-product/{nin}/"

    print("Processing:", nin)

    try:

        driver.get(url)

        time.sleep(6)

        product_name = ""
        pricing = ""

        # PRODUCT NAME
        try:

            product_name = driver.find_element(
                By.TAG_NAME,
                "h1"
            ).text.strip()

        except:
            pass

        # PRICE
        try:

            body_text = driver.find_element(
                By.TAG_NAME,
                "body"
            ).text

            m = re.search(r'(\d+\.\d{2})', body_text)

            if m:
                pricing = m.group(1)

        except:
            pass

        results.append({
            "NINs": nin,
            "Product Name": product_name,
            "Pricing": pricing
        })

        print(product_name, pricing)

    except Exception as e:

        print(e)

driver.quit()


# SAVE# ============================================================================

# ============================================================================

out = pd.DataFrame(results)

out.to_excel(
    "Noon_Minutes_Pricing.xlsx",
    index=False
)

print("Completed")

Processing: ZA4EA14149E7DB0926923Z-1
 19.15
Processing: ZB7943C6E1E1C3E507267Z-1
 16.70
Processing: Z3472F77ACFFA103E8ED6Z-1
 17.85
Processing: Z38685AD13D6A297BDF7DZ-1
 31.55
Processing: ZE48CCD14111D3C0B6E27Z-1
 22.75
Processing: Z5540F7088905D2C64FFBZ-1
 26.45
Processing: ZBA36704959668354C95AZ-1
 6.95
Processing: ZB925B53F738D1172A5DFZ-1
 20.55
Processing: Z1730364D9AC286FC839CZ-1
 20.50
Processing: ZF4F3189FA13EFE7A2689Z-1
 23.65
Processing: ZEF0EE5D0D3DC52BAB27AZ-1
 19.40
Processing: Z0F92A2D30AC8A4AB0954Z-1
 15.25
Processing: Z7C6630A1A206F17CC55EZ-1
 18.45
Processing: Z045B32EE54200CC7E00FZ-1
 16.70
Processing: Z3C4584D211C7E7C40EC7Z-1
 13.15
Processing: ZDD36D3416ED824786598Z-1
 13.15
Completed


## 21 - Noon Pricing - Separate Exports

In [86]:
# ══════════════════════════════════════════════════════════════════════════
# NOON PRICING EXPORTS
#
# ⚠️  Noon Core pricing: your old "Noon Core" pricing cell was 100% commented
# out (a disabled Selenium scraper reading "Emami - Pricing.xlsx" sheet
# "Noon Core" and searching noon.com) — it wasn't producing anything, so
# there's currently no live Noon Core pricing dataframe to export. I've
# deleted that dead cell. If you still need this file, let me know and I'll
# re-enable/fix the scraper the same way Noon Minutes' already works below.
#
# Noon Minutes pricing: this IS live — `out` (NINs, Product Name, Pricing)
# is produced by the Selenium scrape just above. This just renames it to
# your snake_case convention and adds country/client_id, same as every
# other file.
# ══════════════════════════════════════════════════════════════════════════

NMS_PRICING_RENAME = {
    "NINs": "asin",
    "Product Name": "product_name",
    "Pricing": "pricing",
}

NMS_PRICING_COLS = ["asin", "product_name", "pricing", "country", "client_id"]

nms_pricing_export = finalize_export(out, NMS_PRICING_RENAME, NMS_PRICING_COLS)

nc_pricing_export = None  # no live source yet — see note above


In [87]:
nms_pricing_export

,asin,product_name,pricing,country,client_id
0,ZA4EA14149E7DB0926923Z-1,,19.15,AE,TH-1783660475608
1,ZB7943C6E1E1C3E507267Z-1,,16.70,AE,TH-1783660475608
2,Z3472F77ACFFA103E8ED6Z-1,,17.85,AE,TH-1783660475608
3,Z38685AD13D6A297BDF7DZ-1,,31.55,AE,TH-1783660475608
4,ZE48CCD14111D3C0B6E27Z-1,,22.75,AE,TH-1783660475608
5,Z5540F7088905D2C64FFBZ-1,,26.45,AE,TH-1783660475608
6,ZBA36704959668354C95AZ-1,,6.95,AE,TH-1783660475608
7,ZB925B53F738D1172A5DFZ-1,,20.55,AE,TH-1783660475608
8,Z1730364D9AC286FC839CZ-1,,20.50,AE,TH-1783660475608
9,ZF4F3189FA13EFE7A2689Z-1,,23.65,AE,TH-1783660475608


In [88]:
# Defensive save for marketing output
import traceback

print('\n== Marketing Output Safe Save ==')

missing = []
if 'marketing_output' not in globals():
    missing.append('marketing_output')
if 'OUT_MARK' not in globals():
    missing.append('OUT_MARK')

if missing:
    print('Cannot save marketing output — missing variables:', missing)
    # Try to show available related vars
    available = [v for v in globals().keys() if 'mark' in v.lower() or 'marketing' in v.lower()]
    print('Available variables related to marketing:', available)
else:
    try:
        df = marketing_output.copy()
        # Ensure Date exists and is datetime
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        else:
            print('Warning: marketing_output has no Date column')

        # Basic cleaning of numeric columns
        numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
        # Also coerce known numeric-like columns
        for c in ['cost','spend','impressions','clicks','sales','units']:
            if c in df.columns and c not in numeric_cols:
                df[c] = pd.to_numeric(df[c], errors='coerce')
                numeric_cols.append(c)

        # Drop columns with all NaN
        df = df.dropna(axis=1, how='all')

        # Attempt save
        print(f"Saving marketing output to: {OUT_MARK}")
        df.to_excel(OUT_MARK, index=False)
        print('Saved successfully. Shape:', df.shape)
    except Exception as e:
        print('Error while saving marketing output:')
        traceback.print_exc()



== Marketing Output Safe Save ==
Saving marketing output to: C:\Users\Shambhavi Dixit\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\client_mamangement_team\EMAMI\marketing_output_2026.xlsx
Saved successfully. Shape: (24824, 80)


In [89]:
# This cell previously contained a third, conflicting redefinition of Noon
# Minutes processing ("standardise_noon_inv_final"). That patch is no longer
# needed: load_noon_inv_files + standardise_noon_inv (defined once, in the
# "A-0 SHARED HELPERS" section of the Section A cell) now resolve dates
# deterministically and correctly include all of June. This cell is kept as
# a lightweight sanity check instead of a duplicate pipeline.

print("\n== Sanity check: Noon Minutes June coverage ==")
if "nms_inv_df" in dir() and isinstance(nms_inv_df, pd.DataFrame) and not nms_inv_df.empty:
    _jun_mask = pd.to_datetime(nms_inv_df["Date"]).dt.month == 6
    print(f"  Total NMs rows       : {len(nms_inv_df):,}")
    print(f"  June 2026 NMs rows   : {int(_jun_mask.sum()):,}")
    print(f"  Date range           : {nms_inv_df['Date'].min().date()} → {nms_inv_df['Date'].max().date()}")
    print(f"  Unique ASINs (NINs)  : {nms_inv_df['ASIN'].nunique()}")
else:
    print("  ⚠️  nms_inv_df not available — re-run the Section A cell above.")

    



== Sanity check: Noon Minutes June coverage ==
  Total NMs rows       : 1,071
  June 2026 NMs rows   : 274
  Date range           : 2026-05-18 → 2026-08-20
  Unique ASINs (NINs)  : 16


In [90]:
amz_ads_export.to_csv(r"C:\Users\Shambhavi Dixit\Downloads\Emami - Marketing.csv", index=False, encoding="utf-8-sig")

## 22 - Final Azure Push - All Channels, All File Types

In [91]:
# # ══════════════════════════════════════════════════════════════════════════
# # FINAL CONSOLIDATED AZURE PUSH
# # ══════════════════════════════════════════════════════════════════════════

# FILES_TO_UPLOAD = {
#     "sanity_data/inventory/amazon_inventory.csv": amz_inventory_export,
#     "sanity_data/ads/amazon_ads.csv":       amz_ads_export,
#     "sanity_data/orders/amazon_orders.csv":    amz_orders_export,
#     "sanity_data/buy_box/amazon_buybox.csv":    amz_buybox_export,

#     "sanity_data/ads/noon_ads.csv":         noon_ads_export,
#     "sanity_data/inventory/noon_inventory.csv":   nc_inventory_export,
#     "sanity_data/orders/noon_revenue.csv":     nc_revenue_export,

#     "sanity_data/ads/noon_minutes_ads.csv":       nms_ads_export,
#     "sanity_data/inventory/noon_minutes_inventory.csv": nms_inventory_export,
#     "sanity_data/orders/noon_minutes_revenue.csv":   nms_revenue_export,
#     "sanity_data/product_pricing/noon_minutes_pricing.csv":   nms_pricing_export,
# }

# print("=" * 80)
# print("STARTING AZURE UPLOAD")
# print("=" * 80)

# for blob_path, df in FILES_TO_UPLOAD.items():
#     print("\n" + "-" * 80)
#     print(f"Uploading : {blob_path}")

#     if df is None:
#         print("Skipping (None):", blob_path)
#         continue
#     if df.empty:
#         print("Skipping (empty):", blob_path)
#         continue

#     try:
#         upload_to_azure_csv(
#             df=df,
#             blob_path=blob_path,
#             connection_string=AZURE_CONNECTION_STRING,
#             container_name=AZURE_CONTAINER,
#         )
#     except Exception as e:
#         print(f"Failed : {blob_path}")
#         print(type(e).__name__, e)

# print("\n" + "=" * 80)
# print("ALL UPLOADS COMPLETED")
# print("=" * 80)


In [93]:
# # ============================================================
# # FINAL CONSOLIDATED AZURE PUSH
# # GENERIC UPLOADER FOR ALL DATAFRAMES
# # ============================================================

# import time
# from io import BytesIO

# import pandas as pd

# from azure.storage.blob import BlobServiceClient
# from azure.core.exceptions import (
#     ServiceResponseError,
#     AzureError
# )


# # ============================================================
# # AZURE CONFIG
# # ============================================================

# # Keep your existing values here.
# # Do NOT create another uploader elsewhere in the notebook.

# AZURE_CONTAINER = "thrive-client-emami"


# # ============================================================
# # GENERIC AZURE UPLOAD WITH RETRY
# # ============================================================

# def upload_blob_with_retry(
#     blob_client,
#     data,
#     blob_path,
#     max_retries=3,
#     max_concurrency=4,
#     timeout=600
# ):

#     for attempt in range(max_retries):

#         try:

#             data.seek(0)

#             blob_client.upload_blob(
#                 data,
#                 overwrite=True,
#                 max_concurrency=max_concurrency,
#                 timeout=timeout
#             )

#             print(
#                 f"✅ Azure upload successful: {blob_path}"
#             )

#             return True

#         except ServiceResponseError as e:

#             wait = (2 ** attempt) * 5

#             print(
#                 f"⚠️ Azure upload failed "
#                 f"(Attempt {attempt + 1}/{max_retries})"
#             )

#             print(f"Path  : {blob_path}")
#             print(f"Error : {e}")
#             print(f"Retry : {wait} seconds")

#             time.sleep(wait)

#         except AzureError as e:

#             wait = (2 ** attempt) * 5

#             print(
#                 f"⚠️ Azure error "
#                 f"(Attempt {attempt + 1}/{max_retries})"
#             )

#             print(f"Path  : {blob_path}")
#             print(f"Error : {e}")
#             print(f"Retry : {wait} seconds")

#             time.sleep(wait)

#         except Exception as e:

#             print(
#                 f"❌ Unexpected Azure error: {blob_path}"
#             )

#             print(
#                 f"{type(e).__name__}: {e}"
#             )

#             raise

#     raise RuntimeError(
#         f"Could not upload {blob_path} "
#         f"after {max_retries} attempts."
#     )


# # ============================================================
# # GENERIC DATAFRAME → AZURE CSV
# # ============================================================

# def upload_to_azure_csv(
#     df,
#     blob_path,
#     connection_string,
#     container_name
# ):

#     # --------------------------------------------------------
#     # VALIDATE DATAFRAME
#     # --------------------------------------------------------

#     if df is None:
#         raise ValueError(
#             f"DataFrame is None: {blob_path}"
#         )

#     if not isinstance(df, pd.DataFrame):
#         raise TypeError(
#             f"Expected pandas DataFrame, "
#             f"got {type(df).__name__}: {blob_path}"
#         )

#     if df.empty:
#         raise ValueError(
#             f"DataFrame is empty: {blob_path}"
#         )

#     # --------------------------------------------------------
#     # COPY
#     # --------------------------------------------------------

#     df = df.copy()

#     # --------------------------------------------------------
#     # CLEAN COLUMN NAMES
#     # --------------------------------------------------------

#     df.columns = (
#         df.columns
#         .astype(str)
#         .str.strip()
#     )

#     # --------------------------------------------------------
#     # CONVERT OBJECT COLUMNS
#     # --------------------------------------------------------

#     object_cols = df.select_dtypes(
#         include=["object"]
#     ).columns

#     for col in object_cols:

#         df[col] = (
#             df[col]
#             .fillna("")
#             .astype(str)
#         )

#     # --------------------------------------------------------
#     # REPLACE NaN
#     # --------------------------------------------------------

#     df = df.where(
#         pd.notnull(df),
#         None
#     )

#     # --------------------------------------------------------
#     # CREATE CSV IN MEMORY
#     # --------------------------------------------------------

#     buffer = BytesIO()

#     df.to_csv(
#         buffer,
#         index=False,
#         encoding="utf-8-sig"
#     )

#     buffer.seek(0)

#     # --------------------------------------------------------
#     # CREATE AZURE CLIENT
#     # --------------------------------------------------------

#     blob_service_client = (
#         BlobServiceClient.from_connection_string(
#             connection_string,
#             connection_timeout=600,
#             read_timeout=600
#         )
#     )

#     # --------------------------------------------------------
#     # GET BLOB CLIENT
#     # --------------------------------------------------------

#     blob_client = (
#         blob_service_client.get_blob_client(
#             container=container_name,
#             blob=blob_path
#         )
#     )

#     # --------------------------------------------------------
#     # UPLOAD
#     # --------------------------------------------------------

#     upload_blob_with_retry(
#         blob_client=blob_client,
#         data=buffer,
#         blob_path=blob_path
#     )

#     # --------------------------------------------------------
#     # SUCCESS INFO
#     # --------------------------------------------------------

#     print(
#         f"   Rows    : {len(df):,}"
#     )

#     print(
#         f"   Columns : {len(df.columns)}"
#     )

#     print(
#         f"   Size    : {buffer.getbuffer().nbytes / 1024:.2f} KB"
#     )

#     return True

# # ============================================================
# # ADD SALES CHANNEL — AMAZON
# # ============================================================

# amz_inventory_export = amz_inventory_export.copy()
# amz_ads_export = amz_ads_export.copy()
# amz_orders_export = amz_orders_export.copy()
# amz_buybox_export = amz_buybox_export.copy()

# amz_inventory_export["sales_channel"] = "Vendor Central"
# amz_ads_export["sales_channel"] = "Vendor Central"
# amz_orders_export["sales_channel"] = "Vendor Central"
# amz_buybox_export["sales_channel"] = "Vendor Central"

# # ============================================================
# # FILES TO UPLOAD
# # ============================================================

# FILES_TO_UPLOAD = {

#     # AMAZON
#     "sanity_data/inventory/amazon_inventory.csv":
#         amz_inventory_export,

#     "sanity_data/ads/amazon_ads.csv":
#         amz_ads_export,

#     "sanity_data/orders/amazon_orders.csv":
#         amz_orders_export,

#     "sanity_data/buy_box/amazon_buybox.csv":
#         amz_buybox_export,


#     # NOON CORE
#     "sanity_data/ads/noon_ads.csv":
#         noon_ads_export,

#     "sanity_data/inventory/noon_inventory.csv":
#         nc_inventory_export,

#     "sanity_data/orders/noon_revenue.csv":
#         nc_revenue_export,


#     # NOON MINUTES
#     "sanity_data/ads/noon_minutes_ads.csv":
#         nms_ads_export,

#     "sanity_data/inventory/noon_minutes_inventory.csv":
#         nms_inventory_export,

#     "sanity_data/orders/noon_minutes_revenue.csv":
#         nms_revenue_export,

#     "sanity_data/product_pricing/noon_minutes_pricing.csv":
#         nms_pricing_export,
# }


# # ============================================================
# # START UPLOAD
# # ============================================================

# print("=" * 80)
# print("STARTING FINAL AZURE UPLOAD")
# print("=" * 80)

# success_count = 0
# skip_count = 0
# failed_count = 0


# # ============================================================
# # LOOP THROUGH ALL FILES
# # ============================================================

# for blob_path, df in FILES_TO_UPLOAD.items():

#     print("\n" + "-" * 80)

#     print(
#         f"Uploading : {blob_path}"
#     )

#     # --------------------------------------------------------
#     # CHECK NONE
#     # --------------------------------------------------------

#     if df is None:

#         print(
#             "⚠️ Skipping because DataFrame is None"
#         )

#         skip_count += 1

#         continue


#     # --------------------------------------------------------
#     # CHECK TYPE
#     # --------------------------------------------------------

#     if not isinstance(df, pd.DataFrame):

#         print(
#             f"❌ Invalid object type: "
#             f"{type(df).__name__}"
#         )

#         failed_count += 1

#         continue


#     # --------------------------------------------------------
#     # CHECK EMPTY
#     # --------------------------------------------------------

#     if df.empty:

#         print(
#             "⚠️ Skipping because DataFrame is empty"
#         )

#         skip_count += 1

#         continue


#     # --------------------------------------------------------
#     # PRINT DATAFRAME INFO
#     # --------------------------------------------------------

#     print(
#         f"Rows    : {len(df):,}"
#     )

#     print(
#         f"Columns : {len(df.columns)}"
#     )

#     print(
#         "Columns :"
#     )

#     print(
#         df.columns.tolist()
#     )


#     # --------------------------------------------------------
#     # UPLOAD
#     # --------------------------------------------------------

#     try:

#         upload_to_azure_csv(
#             df=df,
#             blob_path=blob_path,
#             connection_string=AZURE_CONNECTION_STRING,
#             container_name=AZURE_CONTAINER
#         )

#         success_count += 1

#         print(
#             f"✅ COMPLETED: {blob_path}"
#         )

#     except Exception as e:

#         failed_count += 1

#         print(
#             f"❌ FAILED: {blob_path}"
#         )

#         print(
#             f"Error Type : {type(e).__name__}"
#         )

#         print(
#             f"Error      : {e}"
#         )


# # ============================================================
# # FINAL SUMMARY
# # ============================================================

# print("\n" + "=" * 80)
# print("AZURE UPLOAD SUMMARY")
# print("=" * 80)

# print(
#     f"Total files : {len(FILES_TO_UPLOAD)}"
# )

# print(
#     f"Successful  : {success_count}"
# )

# print(
#     f"Skipped     : {skip_count}"
# )

# print(
#     f"Failed      : {failed_count}"
# )

# print("=" * 80)

In [100]:
# # ============================================================
# # FINAL CONSOLIDATED AZURE PUSH
# # GENERIC UPLOADER FOR ALL DATAFRAMES
# # ============================================================

# import time
# from io import BytesIO

# import pandas as pd

# from azure.storage.blob import BlobServiceClient
# from azure.core.exceptions import (
#     ServiceResponseError,
#     AzureError
# )


# # ============================================================
# # AZURE CONFIG
# # ============================================================

# # Keep your existing values here.
# # Do NOT create another uploader elsewhere in the notebook.

# AZURE_CONTAINER = "thrive-client-emami"


# # ============================================================
# # GENERIC AZURE UPLOAD WITH RETRY
# # ============================================================

# def upload_blob_with_retry(
#     blob_client,
#     data,
#     blob_path,
#     max_retries=3,
#     max_concurrency=4,
#     timeout=600
# ):

#     for attempt in range(max_retries):

#         try:

#             data.seek(0)

#             blob_client.upload_blob(
#                 data,
#                 overwrite=True,
#                 max_concurrency=max_concurrency,
#                 timeout=timeout
#             )

#             print(
#                 f"✅ Azure upload successful: {blob_path}"
#             )

#             return True

#         except ServiceResponseError as e:

#             wait = (2 ** attempt) * 5

#             print(
#                 f"⚠️ Azure upload failed "
#                 f"(Attempt {attempt + 1}/{max_retries})"
#             )

#             print(f"Path  : {blob_path}")
#             print(f"Error : {e}")
#             print(f"Retry : {wait} seconds")

#             time.sleep(wait)

#         except AzureError as e:

#             wait = (2 ** attempt) * 5

#             print(
#                 f"⚠️ Azure error "
#                 f"(Attempt {attempt + 1}/{max_retries})"
#             )

#             print(f"Path  : {blob_path}")
#             print(f"Error : {e}")
#             print(f"Retry : {wait} seconds")

#             time.sleep(wait)

#         except Exception as e:

#             print(
#                 f"❌ Unexpected Azure error: {blob_path}"
#             )

#             print(
#                 f"{type(e).__name__}: {e}"
#             )

#             raise

#     raise RuntimeError(
#         f"Could not upload {blob_path} "
#         f"after {max_retries} attempts."
#     )


# # ============================================================
# # GENERIC DATAFRAME → AZURE CSV
# # ============================================================

# def upload_to_azure_csv(
#     df,
#     blob_path,
#     connection_string,
#     container_name
# ):

#     # --------------------------------------------------------
#     # VALIDATE DATAFRAME
#     # --------------------------------------------------------

#     if df is None:

#         raise ValueError(
#             f"DataFrame is None: {blob_path}"
#         )

#     if not isinstance(df, pd.DataFrame):

#         raise TypeError(
#             f"Expected pandas DataFrame, "
#             f"got {type(df).__name__}: {blob_path}"
#         )

#     if df.empty:

#         raise ValueError(
#             f"DataFrame is empty: {blob_path}"
#         )


#     # --------------------------------------------------------
#     # COPY
#     # --------------------------------------------------------

#     df = df.copy()


#     # --------------------------------------------------------
#     # CLEAN COLUMN NAMES
#     # --------------------------------------------------------

#     df.columns = (
#         df.columns
#         .astype(str)
#         .str.strip()
#     )


#     # --------------------------------------------------------
#     # CONVERT OBJECT COLUMNS
#     # --------------------------------------------------------

#     object_cols = df.select_dtypes(
#         include=["object"]
#     ).columns

#     for col in object_cols:

#         df[col] = (
#             df[col]
#             .fillna("")
#             .astype(str)
#         )


#     # --------------------------------------------------------
#     # REPLACE NaN
#     # --------------------------------------------------------

#     df = df.where(
#         pd.notnull(df),
#         None
#     )


#     # --------------------------------------------------------
#     # CREATE CSV IN MEMORY
#     # --------------------------------------------------------

#     buffer = BytesIO()

#     df.to_csv(
#         buffer,
#         index=False,
#         encoding="utf-8-sig"
#     )

#     buffer.seek(0)


#     # --------------------------------------------------------
#     # CREATE AZURE CLIENT
#     # --------------------------------------------------------

#     blob_service_client = (
#         BlobServiceClient.from_connection_string(
#             connection_string,
#             connection_timeout=600,
#             read_timeout=600
#         )
#     )


#     # --------------------------------------------------------
#     # GET BLOB CLIENT
#     # --------------------------------------------------------

#     blob_client = (
#         blob_service_client.get_blob_client(
#             container=container_name,
#             blob=blob_path
#         )
#     )


#     # --------------------------------------------------------
#     # UPLOAD
#     # --------------------------------------------------------

#     upload_blob_with_retry(
#         blob_client=blob_client,
#         data=buffer,
#         blob_path=blob_path
#     )


#     # --------------------------------------------------------
#     # SUCCESS INFO
#     # --------------------------------------------------------

#     print(
#         f"   Rows    : {len(df):,}"
#     )

#     print(
#         f"   Columns : {len(df.columns)}"
#     )

#     print(
#         f"   Size    : {buffer.getbuffer().nbytes / 1024:.2f} KB"
#     )

#     return True


# # ============================================================
# # ADD SALES CHANNEL — AMAZON
# # ============================================================

# amz_inventory_export = amz_inventory_export.copy()
# amz_ads_export = amz_ads_export.copy()
# amz_orders_export = amz_orders_export.copy()
# amz_buybox_export = amz_buybox_export.copy()

# amz_inventory_export["sales_channel"] = "Vendor Central"
# amz_ads_export["sales_channel"] = "Vendor Central"
# amz_orders_export["sales_channel"] = "Vendor Central"
# amz_buybox_export["sales_channel"] = "Vendor Central"


# # ============================================================
# # FILES TO UPLOAD
# # ============================================================

# FILES_TO_UPLOAD = {

#     # AMAZON
#     "sanity_data/inventory/amazon_inventory.csv":
#         amz_inventory_export,

#     "sanity_data/ads/amazon_ads.csv":
#         amz_ads_export,

#     "sanity_data/orders/amazon_orders.csv":
#         amz_orders_export,

#     "sanity_data/buy_box/amazon_buybox.csv":
#         amz_buybox_export,


#     # NOON CORE
#     "sanity_data/ads/noon_ads.csv":
#         noon_ads_export,

#     "sanity_data/inventory/noon_inventory.csv":
#         nc_inventory_export,

#     "sanity_data/orders/noon_revenue.csv":
#         nc_revenue_export,


#     # NOON MINUTES
#     "sanity_data/ads/noon_minutes_ads.csv":
#         nms_ads_export,

#     "sanity_data/inventory/noon_minutes_inventory.csv":
#         nms_inventory_export,

#     "sanity_data/orders/noon_minutes_revenue.csv":
#         nms_revenue_export,

#     "sanity_data/product_pricing/noon_minutes_pricing.csv":
#         nms_pricing_export,
# }


# # ============================================================
# # REMOVE DUPLICATES
# # ============================================================
# # UNIQUE KEY:
# # sales_channel + date + asin
# #
# # If the same combination occurs more than once,
# # the LAST occurrence is kept.
# # ============================================================

# print("\n" + "=" * 80)
# print("REMOVING DUPLICATES")
# print("=" * 80)

# DUPLICATE_COLUMNS = [
#     "sales_channel",
#     "date",
#     "asin"
# ]

# total_duplicates_removed = 0


# for blob_path, df in FILES_TO_UPLOAD.items():

#     if df is None:
#         continue

#     if not isinstance(df, pd.DataFrame):
#         continue

#     if df.empty:
#         continue

#     # --------------------------------------------------------
#     # CLEAN COLUMN NAMES
#     # --------------------------------------------------------

#     df.columns = (
#         df.columns
#         .astype(str)
#         .str.strip()
#     )

#     # --------------------------------------------------------
#     # CHECK REQUIRED COLUMNS
#     # --------------------------------------------------------

#     missing_columns = [
#         col
#         for col in DUPLICATE_COLUMNS
#         if col not in df.columns
#     ]

#     if missing_columns:

#         print(
#             f"\n⚠️ {blob_path}"
#         )

#         print(
#             f"   Skipped duplicate removal."
#         )

#         print(
#             f"   Missing columns: {missing_columns}"
#         )

#         continue

#     # --------------------------------------------------------
#     # BEFORE COUNT
#     # --------------------------------------------------------

#     rows_before = len(df)

#     # --------------------------------------------------------
#     # REMOVE DUPLICATES
#     # --------------------------------------------------------

#     df = df.drop_duplicates(
#         subset=DUPLICATE_COLUMNS,
#         keep="last"
#     ).reset_index(drop=True)

#     # --------------------------------------------------------
#     # AFTER COUNT
#     # --------------------------------------------------------

#     rows_after = len(df)

#     duplicates_removed = (
#         rows_before - rows_after
#     )

#     total_duplicates_removed += (
#         duplicates_removed
#     )

#     # --------------------------------------------------------
#     # UPDATE DATAFRAME IN FILES_TO_UPLOAD
#     # --------------------------------------------------------

#     FILES_TO_UPLOAD[blob_path] = df

#     # --------------------------------------------------------
#     # PRINT RESULT
#     # --------------------------------------------------------

#     print(
#         f"\n{blob_path}"
#     )

#     print(
#         f"   Rows before       : {rows_before:,}"
#     )

#     print(
#         f"   Duplicates removed: {duplicates_removed:,}"
#     )

#     print(
#         f"   Rows after        : {rows_after:,}"
#     )


# print("\n" + "-" * 80)

# print(
#     f"TOTAL DUPLICATES REMOVED: "
#     f"{total_duplicates_removed:,}"
# )

# print("=" * 80)


# # ============================================================
# # START UPLOAD
# # ============================================================

# print("\n" + "=" * 80)
# print("STARTING FINAL AZURE UPLOAD")
# print("=" * 80)

# success_count = 0
# skip_count = 0
# failed_count = 0


# # ============================================================
# # LOOP THROUGH ALL FILES
# # ============================================================

# for blob_path, df in FILES_TO_UPLOAD.items():

#     print("\n" + "-" * 80)

#     print(
#         f"Uploading : {blob_path}"
#     )


#     # --------------------------------------------------------
#     # CHECK NONE
#     # --------------------------------------------------------

#     if df is None:

#         print(
#             "⚠️ Skipping because DataFrame is None"
#         )

#         skip_count += 1

#         continue


#     # --------------------------------------------------------
#     # CHECK TYPE
#     # --------------------------------------------------------

#     if not isinstance(df, pd.DataFrame):

#         print(
#             f"❌ Invalid object type: "
#             f"{type(df).__name__}"
#         )

#         failed_count += 1

#         continue


#     # --------------------------------------------------------
#     # CHECK EMPTY
#     # --------------------------------------------------------

#     if df.empty:

#         print(
#             "⚠️ Skipping because DataFrame is empty"
#         )

#         skip_count += 1

#         continue


#     # --------------------------------------------------------
#     # PRINT DATAFRAME INFO
#     # --------------------------------------------------------

#     print(
#         f"Rows    : {len(df):,}"
#     )

#     print(
#         f"Columns : {len(df.columns)}"
#     )

#     print(
#         "Columns :"
#     )

#     print(
#         df.columns.tolist()
#     )


#     # --------------------------------------------------------
#     # UPLOAD
#     # --------------------------------------------------------

#     try:

#         upload_to_azure_csv(
#             df=df,
#             blob_path=blob_path,
#             connection_string=AZURE_CONNECTION_STRING,
#             container_name=AZURE_CONTAINER
#         )

#         success_count += 1

#         print(
#             f"✅ COMPLETED: {blob_path}"
#         )

#     except Exception as e:

#         failed_count += 1

#         print(
#             f"❌ FAILED: {blob_path}"
#         )

#         print(
#             f"Error Type : {type(e).__name__}"
#         )

#         print(
#             f"Error      : {e}"
#         )


# # ============================================================
# # FINAL SUMMARY
# # ============================================================

# print("\n" + "=" * 80)
# print("AZURE UPLOAD SUMMARY")
# print("=" * 80)

# print(
#     f"Total files          : {len(FILES_TO_UPLOAD)}"
# )

# print(
#     f"Total duplicates     : {total_duplicates_removed:,}"
# )

# print(
#     f"Successful           : {success_count}"
# )

# print(
#     f"Skipped              : {skip_count}"
# )

# print(
#     f"Failed               : {failed_count}"
# )

# print("=" * 80)

# PUSH ALL DATA TO AZURE 

In [101]:
# ============================================================
# FINAL CONSOLIDATED AZURE PUSH
# GENERIC UPLOADER FOR ALL DATAFRAMES
# ============================================================

import time
from io import BytesIO

import pandas as pd

from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import (
    ServiceResponseError,
    AzureError
)


# ============================================================
# AZURE CONFIG
# ============================================================

# Keep your existing values here.
# Do NOT create another uploader elsewhere in the notebook.

AZURE_CONTAINER = "thrive-client-emami"


# ============================================================
# GENERIC AZURE UPLOAD WITH RETRY
# ============================================================

def upload_blob_with_retry(
    blob_client,
    data,
    blob_path,
    max_retries=3,
    max_concurrency=4,
    timeout=600
):

    for attempt in range(max_retries):

        try:

            data.seek(0)

            blob_client.upload_blob(
                data,
                overwrite=True,
                max_concurrency=max_concurrency,
                timeout=timeout
            )

            print(
                f"✅ Azure upload successful: {blob_path}"
            )

            return True

        except ServiceResponseError as e:

            wait = (2 ** attempt) * 5

            print(
                f"⚠️ Azure upload failed "
                f"(Attempt {attempt + 1}/{max_retries})"
            )

            print(f"Path  : {blob_path}")
            print(f"Error : {e}")
            print(f"Retry : {wait} seconds")

            time.sleep(wait)

        except AzureError as e:

            wait = (2 ** attempt) * 5

            print(
                f"⚠️ Azure error "
                f"(Attempt {attempt + 1}/{max_retries})"
            )

            print(f"Path  : {blob_path}")
            print(f"Error : {e}")
            print(f"Retry : {wait} seconds")

            time.sleep(wait)

        except Exception as e:

            print(
                f"❌ Unexpected Azure error: {blob_path}"
            )

            print(
                f"{type(e).__name__}: {e}"
            )

            raise

    raise RuntimeError(
        f"Could not upload {blob_path} "
        f"after {max_retries} attempts."
    )


# ============================================================
# GENERIC DATAFRAME → AZURE CSV
# ============================================================

def upload_to_azure_csv(
    df,
    blob_path,
    connection_string,
    container_name
):

    # --------------------------------------------------------
    # VALIDATE DATAFRAME
    # --------------------------------------------------------

    if df is None:

        raise ValueError(
            f"DataFrame is None: {blob_path}"
        )

    if not isinstance(df, pd.DataFrame):

        raise TypeError(
            f"Expected pandas DataFrame, "
            f"got {type(df).__name__}: {blob_path}"
        )

    if df.empty:

        raise ValueError(
            f"DataFrame is empty: {blob_path}"
        )


    # --------------------------------------------------------
    # COPY
    # --------------------------------------------------------

    df = df.copy()


    # --------------------------------------------------------
    # CLEAN COLUMN NAMES
    # --------------------------------------------------------

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )


    # --------------------------------------------------------
    # CONVERT OBJECT COLUMNS
    # --------------------------------------------------------

    object_cols = df.select_dtypes(
        include=["object"]
    ).columns

    for col in object_cols:

        df[col] = (
            df[col]
            .fillna("")
            .astype(str)
        )


    # --------------------------------------------------------
    # REPLACE NaN
    # --------------------------------------------------------

    df = df.where(
        pd.notnull(df),
        None
    )


    # --------------------------------------------------------
    # CREATE CSV IN MEMORY
    # --------------------------------------------------------

    buffer = BytesIO()

    df.to_csv(
        buffer,
        index=False,
        encoding="utf-8-sig"
    )

    buffer.seek(0)


    # --------------------------------------------------------
    # CREATE AZURE CLIENT
    # --------------------------------------------------------

    blob_service_client = (
        BlobServiceClient.from_connection_string(
            connection_string,
            connection_timeout=600,
            read_timeout=600
        )
    )


    # --------------------------------------------------------
    # GET BLOB CLIENT
    # --------------------------------------------------------

    blob_client = (
        blob_service_client.get_blob_client(
            container=container_name,
            blob=blob_path
        )
    )


    # --------------------------------------------------------
    # UPLOAD
    # --------------------------------------------------------

    upload_blob_with_retry(
        blob_client=blob_client,
        data=buffer,
        blob_path=blob_path
    )


    # --------------------------------------------------------
    # SUCCESS INFO
    # --------------------------------------------------------

    print(
        f"   Rows    : {len(df):,}"
    )

    print(
        f"   Columns : {len(df.columns)}"
    )

    print(
        f"   Size    : "
        f"{buffer.getbuffer().nbytes / 1024:.2f} KB"
    )

    return True


# ============================================================
# ADD SALES CHANNEL — AMAZON
# ============================================================

amz_inventory_export = amz_inventory_export.copy()
amz_ads_export = amz_ads_export.copy()
amz_orders_export = amz_orders_export.copy()
amz_buybox_export = amz_buybox_export.copy()

amz_inventory_export["sales_channel"] = "Vendor Central"
amz_ads_export["sales_channel"] = "Vendor Central"
amz_orders_export["sales_channel"] = "Vendor Central"
amz_buybox_export["sales_channel"] = "Vendor Central"


# ============================================================
# FILES TO UPLOAD
# ============================================================

FILES_TO_UPLOAD = {

    # AMAZON
    "sanity_data/inventory/amazon_inventory.csv":
        amz_inventory_export,

    "sanity_data/ads/amazon_ads.csv":
        amz_ads_export,

    "sanity_data/orders/amazon_orders.csv":
        amz_orders_export,

    "sanity_data/buy_box/amazon_buybox.csv":
        amz_buybox_export,


    # NOON CORE
    "sanity_data/ads/noon_ads.csv":
        noon_ads_export,

    "sanity_data/inventory/noon_inventory.csv":
        nc_inventory_export,

    "sanity_data/orders/noon_revenue.csv":
        nc_revenue_export,


    # NOON MINUTES
    "sanity_data/ads/noon_minutes_ads.csv":
        nms_ads_export,

    "sanity_data/inventory/noon_minutes_inventory.csv":
        nms_inventory_export,

    "sanity_data/orders/noon_minutes_revenue.csv":
        nms_revenue_export,

    "sanity_data/product_pricing/noon_minutes_pricing.csv":
        nms_pricing_export,
}


# ============================================================
# STANDARDIZE ALL DATE COLUMNS
# FINAL FORMAT: YYYY-MM-DD
# EXAMPLE: 2026-08-24
# ============================================================

print("\n" + "=" * 100)
print("STANDARDIZING DATE COLUMNS")
print("=" * 100)


def convert_to_standard_date(series):

    # --------------------------------------------------------
    # CLEAN VALUES
    # --------------------------------------------------------

    values = (
        series
        .astype("string")
        .str.strip()
    )

    values = values.replace(
        {
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "NaT": pd.NA,
            "None": pd.NA
        }
    )


    # --------------------------------------------------------
    # CREATE EMPTY DATETIME RESULT
    # --------------------------------------------------------

    result = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns]"
    )


    # ========================================================
    # 1. YYYY-MM-DD
    # ========================================================

    mask = (
        values.notna()
        & result.isna()
        & values.str.match(
            r"^\d{4}-\d{2}-\d{2}$",
            na=False
        )
    )

    result.loc[mask] = pd.to_datetime(
        values.loc[mask],
        format="%Y-%m-%d",
        errors="coerce"
    )


    # ========================================================
    # 2. DD-MMM-YYYY
    # Example: 01-May-2026
    # ========================================================

    mask = (
        values.notna()
        & result.isna()
        & values.str.match(
            r"^\d{2}-[A-Za-z]{3}-\d{4}$",
            na=False
        )
    )

    result.loc[mask] = pd.to_datetime(
        values.loc[mask],
        format="%d-%b-%Y",
        errors="coerce"
    )


    # ========================================================
    # 3. DD-MMM-YY
    # Example: 01-May-26
    # ========================================================

    mask = (
        values.notna()
        & result.isna()
        & values.str.match(
            r"^\d{2}-[A-Za-z]{3}-\d{2}$",
            na=False
        )
    )

    result.loc[mask] = pd.to_datetime(
        values.loc[mask],
        format="%d-%b-%y",
        errors="coerce"
    )


    # ========================================================
    # 4. YYYY/MM/DD
    # Example: 2026/08/24
    # ========================================================

    mask = (
        values.notna()
        & result.isna()
        & values.str.match(
            r"^\d{4}/\d{2}/\d{2}$",
            na=False
        )
    )

    result.loc[mask] = pd.to_datetime(
        values.loc[mask],
        format="%Y/%m/%d",
        errors="coerce"
    )


    # ========================================================
    # 5. DD/MM/YYYY
    # Example: 24/08/2026
    # ========================================================

    mask = (
        values.notna()
        & result.isna()
        & values.str.match(
            r"^\d{2}/\d{2}/\d{4}$",
            na=False
        )
    )

    result.loc[mask] = pd.to_datetime(
        values.loc[mask],
        format="%d/%m/%Y",
        errors="coerce"
    )


    # ========================================================
    # 6. DD-MM-YYYY
    # Example: 24-08-2026
    # ========================================================

    mask = (
        values.notna()
        & result.isna()
        & values.str.match(
            r"^\d{2}-\d{2}-\d{4}$",
            na=False
        )
    )

    result.loc[mask] = pd.to_datetime(
        values.loc[mask],
        format="%d-%m-%Y",
        errors="coerce"
    )


    # ========================================================
    # 7. YYYY-MM-DD HH:MM:SS
    # ========================================================

    mask = (
        values.notna()
        & result.isna()
        & values.str.match(
            r"^\d{4}-\d{2}-\d{2} "
            r"\d{2}:\d{2}:\d{2}$",
            na=False
        )
    )

    result.loc[mask] = pd.to_datetime(
        values.loc[mask],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )


    # ========================================================
    # 8. ISO DATETIME
    # Example: 2026-08-24T12:30:00
    # ========================================================

    mask = (
        values.notna()
        & result.isna()
        & values.str.contains(
            "T",
            na=False
        )
    )

    if mask.any():

        result.loc[mask] = pd.to_datetime(
            values.loc[mask],
            errors="coerce"
        )


    # ========================================================
    # RETURN YYYY-MM-DD
    # ========================================================

    return result.dt.strftime("%Y-%m-%d")


# ============================================================
# APPLY DATE CONVERSION TO ALL FILES
# ============================================================

date_conversion_failed = 0

for blob_path, df in FILES_TO_UPLOAD.items():

    if df is None:
        continue

    if not isinstance(df, pd.DataFrame):
        continue

    if df.empty:
        continue


    # --------------------------------------------------------
    # FIND DATE COLUMNS
    # --------------------------------------------------------

    date_columns = [
        col
        for col in df.columns
        if "date" in str(col).lower()
    ]


    if not date_columns:

        print(
            f"\n⚠️ {blob_path}"
        )

        print(
            "   No date column found"
        )

        continue


    # --------------------------------------------------------
    # CONVERT EACH DATE COLUMN
    # --------------------------------------------------------

    for col in date_columns:

        print("\n" + "-" * 80)

        print(
            f"File   : {blob_path}"
        )

        print(
            f"Column : {col}"
        )


        original_values = df[col].copy()


        # Convert
        df[col] = convert_to_standard_date(
            df[col]
        )


        # ----------------------------------------------------
        # CHECK FAILED CONVERSIONS
        # ----------------------------------------------------

        original_text = (
            original_values
            .astype("string")
            .str.strip()
        )

        original_non_empty = (
            original_values.notna()
            & original_text.notna()
            & original_text.ne("")
            & ~original_text.str.lower().isin(
                ["nan", "nat", "none"]
            )
        )


        failed_mask = (
            original_non_empty
            & df[col].isna()
        )


        failed_count = failed_mask.sum()


        if failed_count > 0:

            date_conversion_failed += (
                failed_count
            )

            print(
                f"⚠️ Conversion failed: "
                f"{failed_count:,}"
            )

            print(
                "   Failed examples:"
            )

            failed_examples = (
                original_values[
                    failed_mask
                ]
                .drop_duplicates()
                .head(10)
            )

            for value in failed_examples:

                print(
                    f"      {value}"
                )

        else:

            print(
                "✅ All non-empty dates converted"
            )


        # ----------------------------------------------------
        # SHOW FINAL DATE EXAMPLES
        # ----------------------------------------------------

        final_examples = (
            df[col]
            .dropna()
            .drop_duplicates()
            .head(5)
            .tolist()
        )

        print(
            "Final format examples:"
        )

        for value in final_examples:

            print(
                f"   {value}"
            )


# ============================================================
# DATE CONVERSION SUMMARY
# ============================================================

print("\n" + "=" * 100)

print(
    "DATE CONVERSION COMPLETE"
)

print(
    "Final format: YYYY-MM-DD"
)

print(
    f"Total conversion failures: "
    f"{date_conversion_failed:,}"
)

print("=" * 100)


# ============================================================
# REMOVE DUPLICATES
# ============================================================
# UNIQUE KEY:
# sales_channel + date + asin
#
# If the same combination occurs more than once,
# the LAST occurrence is kept.
# ============================================================

print("\n" + "=" * 80)
print("REMOVING DUPLICATES")
print("=" * 80)


DUPLICATE_COLUMNS = [
    "sales_channel",
    "date",
    "asin"
]


total_duplicates_removed = 0


for blob_path, df in FILES_TO_UPLOAD.items():

    if df is None:
        continue

    if not isinstance(df, pd.DataFrame):
        continue

    if df.empty:
        continue


    # --------------------------------------------------------
    # CLEAN COLUMN NAMES
    # --------------------------------------------------------

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )


    # --------------------------------------------------------
    # CHECK REQUIRED COLUMNS
    # --------------------------------------------------------

    missing_columns = [
        col
        for col in DUPLICATE_COLUMNS
        if col not in df.columns
    ]


    if missing_columns:

        print(
            f"\n⚠️ {blob_path}"
        )

        print(
            "   Skipped duplicate removal."
        )

        print(
            f"   Missing columns: "
            f"{missing_columns}"
        )

        continue


    # --------------------------------------------------------
    # BEFORE COUNT
    # --------------------------------------------------------

    rows_before = len(df)


    # --------------------------------------------------------
    # REMOVE DUPLICATES
    # --------------------------------------------------------

    df = (
        df
        .drop_duplicates(
            subset=DUPLICATE_COLUMNS,
            keep="last"
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # AFTER COUNT
    # --------------------------------------------------------

    rows_after = len(df)


    duplicates_removed = (
        rows_before - rows_after
    )


    total_duplicates_removed += (
        duplicates_removed
    )


    # --------------------------------------------------------
    # UPDATE DATAFRAME
    # --------------------------------------------------------

    FILES_TO_UPLOAD[blob_path] = df


    # --------------------------------------------------------
    # PRINT RESULT
    # --------------------------------------------------------

    print(
        f"\n{blob_path}"
    )

    print(
        f"   Rows before       : "
        f"{rows_before:,}"
    )

    print(
        f"   Duplicates removed: "
        f"{duplicates_removed:,}"
    )

    print(
        f"   Rows after        : "
        f"{rows_after:,}"
    )


print("\n" + "-" * 80)

print(
    f"TOTAL DUPLICATES REMOVED: "
    f"{total_duplicates_removed:,}"
)

print("=" * 80)


# ============================================================
# FINAL DATE FORMAT CHECK BEFORE UPLOAD
# ============================================================

print("\n" + "=" * 80)
print("FINAL DATE FORMAT CHECK")
print("=" * 80)


for blob_path, df in FILES_TO_UPLOAD.items():

    if df is None or not isinstance(df, pd.DataFrame):
        continue

    date_columns = [
        col
        for col in df.columns
        if "date" in str(col).lower()
    ]

    for col in date_columns:

        sample_dates = (
            df[col]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(3)
            .tolist()
        )

        print(
            f"\n{blob_path}"
        )

        print(
            f"   Column: {col}"
        )

        print(
            f"   Sample: {sample_dates}"
        )


print("\n" + "=" * 80)
print("FINAL DATE CHECK COMPLETE")
print("=" * 80)


# ============================================================
# START UPLOAD
# ============================================================

print("\n" + "=" * 80)
print("STARTING FINAL AZURE UPLOAD")
print("=" * 80)


success_count = 0
skip_count = 0
failed_count = 0


# ============================================================
# LOOP THROUGH ALL FILES
# ============================================================

for blob_path, df in FILES_TO_UPLOAD.items():

    print("\n" + "-" * 80)

    print(
        f"Uploading : {blob_path}"
    )


    # --------------------------------------------------------
    # CHECK NONE
    # --------------------------------------------------------

    if df is None:

        print(
            "⚠️ Skipping because DataFrame is None"
        )

        skip_count += 1

        continue


    # --------------------------------------------------------
    # CHECK TYPE
    # --------------------------------------------------------

    if not isinstance(df, pd.DataFrame):

        print(
            f"❌ Invalid object type: "
            f"{type(df).__name__}"
        )

        failed_count += 1

        continue


    # --------------------------------------------------------
    # CHECK EMPTY
    # --------------------------------------------------------

    if df.empty:

        print(
            "⚠️ Skipping because DataFrame is empty"
        )

        skip_count += 1

        continue


    # --------------------------------------------------------
    # PRINT DATAFRAME INFO
    # --------------------------------------------------------

    print(
        f"Rows    : {len(df):,}"
    )

    print(
        f"Columns : {len(df.columns)}"
    )

    print(
        "Columns :"
    )

    print(
        df.columns.tolist()
    )


    # --------------------------------------------------------
    # UPLOAD
    # --------------------------------------------------------

    try:

        upload_to_azure_csv(
            df=df,
            blob_path=blob_path,
            connection_string=AZURE_CONNECTION_STRING,
            container_name=AZURE_CONTAINER
        )

        success_count += 1

        print(
            f"✅ COMPLETED: {blob_path}"
        )


    except Exception as e:

        failed_count += 1

        print(
            f"❌ FAILED: {blob_path}"
        )

        print(
            f"Error Type : {type(e).__name__}"
        )

        print(
            f"Error      : {e}"
        )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("AZURE UPLOAD SUMMARY")
print("=" * 80)


print(
    f"Total files          : "
    f"{len(FILES_TO_UPLOAD)}"
)

print(
    f"Total duplicates     : "
    f"{total_duplicates_removed:,}"
)

print(
    f"Date conversion fail : "
    f"{date_conversion_failed:,}"
)

print(
    f"Successful           : "
    f"{success_count}"
)

print(
    f"Skipped              : "
    f"{skip_count}"
)

print(
    f"Failed               : "
    f"{failed_count}"
)

print("=" * 80)


STANDARDIZING DATE COLUMNS

--------------------------------------------------------------------------------
File   : sanity_data/inventory/amazon_inventory.csv
Column : date
✅ All non-empty dates converted
Final format examples:
   2026-05-01
   2026-05-02
   2026-05-03
   2026-05-04
   2026-05-05

--------------------------------------------------------------------------------
File   : sanity_data/ads/amazon_ads.csv
Column : date
✅ All non-empty dates converted
Final format examples:
   2026-05-01
   2026-05-02
   2026-05-03
   2026-05-04
   2026-05-05

--------------------------------------------------------------------------------
File   : sanity_data/orders/amazon_orders.csv
Column : date
✅ All non-empty dates converted
Final format examples:
   2026-05-01
   2026-05-02
   2026-05-03
   2026-05-04
   2026-05-05

--------------------------------------------------------------------------------
File   : sanity_data/buy_box/amazon_buybox.csv
Column : date
✅ All non-empty dates conver